# Dataset Familiarization

## Amsterdam Airbnb Data Engineering Challenge

This notebook explores and documents the seven raw Amsterdam Airbnb datasets before any cleaning, transformation, enrichment, or statistical analysis is performed.

### Objectives

For each dataset, we will inspect:

- File name
- Number of rows
- Number of columns
- Column names
- Data types
- Missing-value counts
- Missing-value percentages
- Unique-value counts
- Minimum and maximum values
- Sample values
- Duplicate counts
- Possible primary keys
- Possible foreign keys
- Relationships between datasets
- Dataset limitations
- Business-domain meaning


This project is designed for an 8 GB RAM laptop.

- Small and medium datasets will be processed with Pandas.
- `calendar.csv.gz` and `reviews.csv.gz` will be processed using DuckDB or chunked processing.
- All seven datasets will not be loaded into memory at the same time.
- Large datasets will be processed one at a time.
- Temporary large objects will be released from memory after use.
- Compact aggregated results will be used for downstream analysis.

In [4]:
from pathlib import Path
import gc

import pandas as pd
import numpy as np
import duckdb

# Improve Pandas display settings for dataset inspection.
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)

print("Libraries imported successfully.")

Libraries imported successfully.


In [5]:
# Detect the project root dynamically.
# This allows the notebook to work whether Jupyter starts
# from the project root or from the notebooks directory.

CURRENT_PATH = Path.cwd()

if CURRENT_PATH.name == "notebooks":
    PROJECT_ROOT = CURRENT_PATH.parent
else:
    PROJECT_ROOT = CURRENT_PATH

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "amsterdam"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "data_quality"

# Create the output directory if it does not already exist.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Current working directory: {CURRENT_PATH}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data directory: {RAW_DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

Current working directory: c:\Users\saths\Desktop\MyProjects\Expernetic Data Challenge\airbnb-data-challenge\notebooks
Project root: c:\Users\saths\Desktop\MyProjects\Expernetic Data Challenge\airbnb-data-challenge
Raw data directory: c:\Users\saths\Desktop\MyProjects\Expernetic Data Challenge\airbnb-data-challenge\data\raw\amsterdam
Output directory: c:\Users\saths\Desktop\MyProjects\Expernetic Data Challenge\airbnb-data-challenge\outputs\data_quality


In [6]:
DATA_FILES = {
    "detailed_listings": RAW_DATA_DIR / "listings.csv.gz",
    "summary_listings": RAW_DATA_DIR / "listings.csv",
    "detailed_calendar": RAW_DATA_DIR / "calendar.csv.gz",
    "detailed_reviews": RAW_DATA_DIR / "reviews.csv.gz",
    "summary_reviews": RAW_DATA_DIR / "reviews.csv",
    "neighbourhoods": RAW_DATA_DIR / "neighbourhoods.csv",
    "neighbourhoods_geojson": RAW_DATA_DIR / "neighbourhoods.geojson",
}

print(f"Configured {len(DATA_FILES)} Amsterdam datasets.\n")

for dataset_name, file_path in DATA_FILES.items():
    print(f"{dataset_name}: {file_path.name}")

Configured 7 Amsterdam datasets.

detailed_listings: listings.csv.gz
summary_listings: listings.csv
detailed_calendar: calendar.csv.gz
detailed_reviews: reviews.csv.gz
summary_reviews: reviews.csv
neighbourhoods: neighbourhoods.csv
neighbourhoods_geojson: neighbourhoods.geojson


In [7]:
file_check = []

for dataset_name, file_path in DATA_FILES.items():
    file_exists = file_path.exists()

    file_size_mb = (
        round(file_path.stat().st_size / (1024 ** 2), 2)
        if file_exists
        else None
    )

    file_check.append({
        "dataset_name": dataset_name,
        "file_name": file_path.name,
        "exists": file_exists,
        "file_size_mb": file_size_mb,
    })

file_check_df = pd.DataFrame(file_check)

display(file_check_df)

,dataset_name,file_name,exists,file_size_mb
0,detailed_listings,listings.csv.gz,True,5.49
1,summary_listings,listings.csv,True,1.97
2,detailed_calendar,calendar.csv.gz,True,8.57
3,detailed_reviews,reviews.csv.gz,True,63.48
4,summary_reviews,reviews.csv,True,11.30
5,neighbourhoods,neighbourhoods.csv,True,0.00
6,neighbourhoods_geojson,neighbourhoods.geojson,True,0.18


In [8]:
missing_files = file_check_df.loc[
    ~file_check_df["exists"],
    "file_name"
].tolist()

if missing_files:
    raise FileNotFoundError(
        f"The following required Amsterdam dataset files are missing: {missing_files}"
    )

print("All seven Amsterdam dataset files were found successfully.")

All seven Amsterdam dataset files were found successfully.


## 1. Neighbourhoods Dataset Familiarization

The `neighbourhoods.csv` file contains neighbourhood reference data for Amsterdam.

This dataset will be inspected for:

- Shape
- Column names
- Data types
- Missing values
- Unique values
- Minimum and maximum values
- Sample values
- Duplicate rows
- Possible primary keys
- Relationships with other datasets
- Business-domain meaning

Load the neighbourhood dataset

In [9]:
neighbourhoods_df = pd.read_csv(
    DATA_FILES["neighbourhoods"]
)

print("neighbourhoods.csv loaded successfully.")
print(f"Rows: {neighbourhoods_df.shape[0]:,}")
print(f"Columns: {neighbourhoods_df.shape[1]}")

neighbourhoods.csv loaded successfully.
Rows: 22
Columns: 2


Preview the dataset

In [10]:
display(neighbourhoods_df.head())

,neighbourhood_group,neighbourhood
0,NaN,Bijlmer-Centrum
1,NaN,Bijlmer-Oost
2,NaN,Bos en Lommer
3,NaN,Buitenveldert - Zuidas
4,NaN,Centrum-Oost


Show dataset shape

In [11]:
print("Dataset Shape")
print("-" * 40)
print(f"Number of rows: {neighbourhoods_df.shape[0]:,}")
print(f"Number of columns: {neighbourhoods_df.shape[1]}")

Dataset Shape
----------------------------------------
Number of rows: 22
Number of columns: 2


Show column names

In [12]:
print("Column Names")
print("-" * 40)

for index, column in enumerate(neighbourhoods_df.columns, start=1):
    print(f"{index}. {column}")

Column Names
----------------------------------------
1. neighbourhood_group
2. neighbourhood


Show data types

In [13]:
data_types_df = pd.DataFrame({
    "column_name": neighbourhoods_df.columns,
    "data_type": neighbourhoods_df.dtypes.astype(str).values
})

display(data_types_df)

,column_name,data_type
0,neighbourhood_group,float64
1,neighbourhood,str


Missing-value analysis

In [14]:
missing_summary = pd.DataFrame({
    "column_name": neighbourhoods_df.columns,
    "missing_count": neighbourhoods_df.isnull().sum().values,
    "missing_percentage": (
        neighbourhoods_df.isnull().mean() * 100
    ).round(2).values
})

display(missing_summary)

,column_name,missing_count,missing_percentage
0,neighbourhood_group,22,100.0
1,neighbourhood,0,0.0


Unique-value counts

In [15]:
unique_summary = pd.DataFrame({
    "column_name": neighbourhoods_df.columns,
    "unique_count": [
        neighbourhoods_df[column].nunique(dropna=True)
        for column in neighbourhoods_df.columns
    ]
})

display(unique_summary)

,column_name,unique_count
0,neighbourhood_group,0
1,neighbourhood,22


Minimum, maximum, and sample values

In [16]:
column_profile = []

for column in neighbourhoods_df.columns:
    series = neighbourhoods_df[column]

    non_null_values = series.dropna()

    try:
        minimum_value = non_null_values.min()
    except (TypeError, ValueError):
        minimum_value = None

    try:
        maximum_value = non_null_values.max()
    except (TypeError, ValueError):
        maximum_value = None

    sample_values = (
        non_null_values
        .astype(str)
        .drop_duplicates()
        .head(3)
        .tolist()
    )

    column_profile.append({
        "column_name": column,
        "data_type": str(series.dtype),
        "missing_count": int(series.isnull().sum()),
        "missing_percentage": round(series.isnull().mean() * 100, 2),
        "unique_count": int(series.nunique(dropna=True)),
        "minimum": minimum_value,
        "maximum": maximum_value,
        "sample_values": sample_values,
    })

neighbourhoods_profile_df = pd.DataFrame(column_profile)

display(neighbourhoods_profile_df)

,column_name,data_type,missing_count,missing_percentage,unique_count,minimum,maximum,sample_values
0,neighbourhood_group,float64,22,100.0,0,NaN,NaN,[]
1,neighbourhood,str,0,0.0,22,Bijlmer-Centrum,Zuid,"[Bijlmer-Centrum, Bijlmer-Oost, Bos en Lommer]"


Duplicate row analysis

In [17]:
duplicate_count = neighbourhoods_df.duplicated().sum()

print("Duplicate Analysis")
print("-" * 40)
print(f"Total rows: {len(neighbourhoods_df):,}")
print(f"Duplicate rows: {duplicate_count:,}")
print(
    f"Duplicate percentage: "
    f"{(duplicate_count / len(neighbourhoods_df) * 100):.2f}%"
)

Duplicate Analysis
----------------------------------------
Total rows: 22
Duplicate rows: 0
Duplicate percentage: 0.00%


Candidate primary key analysis

In [18]:
candidate_keys = []

total_rows = len(neighbourhoods_df)

for column in neighbourhoods_df.columns:
    unique_count = neighbourhoods_df[column].nunique(dropna=False)
    missing_count = neighbourhoods_df[column].isnull().sum()

    is_unique = unique_count == total_rows
    has_no_missing_values = missing_count == 0

    if is_unique and has_no_missing_values:
        candidate_keys.append(column)

print("Candidate Primary Key Analysis")
print("-" * 40)

if candidate_keys:
    print("Possible single-column primary key candidates:")
    
    for column in candidate_keys:
        print(f"- {column}")
else:
    print("No single-column candidate primary key was identified.")

Candidate Primary Key Analysis
----------------------------------------
Possible single-column primary key candidates:
- neighbourhood


Dataset summary

In [19]:
neighbourhoods_dataset_summary = {
    "dataset_name": "neighbourhoods",
    "file_name": DATA_FILES["neighbourhoods"].name,
    "row_count": len(neighbourhoods_df),
    "column_count": len(neighbourhoods_df.columns),
    "duplicate_count": int(neighbourhoods_df.duplicated().sum()),
    "candidate_primary_keys": candidate_keys,
}

neighbourhoods_dataset_summary

{'dataset_name': 'neighbourhoods',
 'file_name': 'neighbourhoods.csv',
 'row_count': 22,
 'column_count': 2,
 'duplicate_count': 0,
 'candidate_primary_keys': ['neighbourhood']}

Save profile results

In [20]:
neighbourhoods_profile_path = (
    OUTPUT_DIR / "neighbourhoods_column_profile.csv"
)

neighbourhoods_profile_df.to_csv(
    neighbourhoods_profile_path,
    index=False
)

print(
    f"Neighbourhood profile saved successfully to:\n"
    f"{neighbourhoods_profile_path}"
)

Neighbourhood profile saved successfully to:
c:\Users\saths\Desktop\MyProjects\Expernetic Data Challenge\airbnb-data-challenge\outputs\data_quality\neighbourhoods_column_profile.csv


### Findings and Interpretation

The `neighbourhoods.csv` dataset contains **22 rows and 2 columns**, representing 22 unique Amsterdam neighbourhoods.

#### Key Findings

- The dataset contains **22 unique neighbourhood names**.
- No duplicate rows were identified.
- The `neighbourhood` column contains no missing values and all 22 values are unique.
- Therefore, `neighbourhood` is identified as a **candidate single-column primary key** for this dataset.
- The `neighbourhood_group` column is completely empty, with **22 missing values out of 22 rows (100%)**.
- Since `neighbourhood_group` contains no usable information, it should not be used for analysis unless valid values are available from another trusted source.

#### Important Technical Note

The minimum and maximum values for the `neighbourhood` column are based on alphabetical or lexicographic ordering:

- Minimum: `Bijlmer-Centrum`
- Maximum: `Zuid`

These values do not represent numerical minimum or maximum measurements and have limited business meaning.

#### Data Quality Assessment

Overall, the dataset is structurally clean because:

- There are no duplicate rows.
- The main neighbourhood field is complete.
- All neighbourhood names are unique.

The primary data-quality issue is the completely missing `neighbourhood_group` field.

### Candidate Keys and Relationships

#### Candidate Primary Key

The `neighbourhood` column is a candidate primary key because:

- It contains 22 values across 22 rows.
- All 22 values are unique.
- It contains no missing values.

Therefore:

**Candidate Primary Key:** `neighbourhood`

This conclusion is based on observed uniqueness and completeness. It should be treated as a candidate key rather than assuming a formally defined database primary key.

#### Potential Foreign-Key Relationships

The `neighbourhood` value may potentially be referenced by neighbourhood-related columns in:

- `listings.csv`
- `listings.csv.gz`

A likely relationship is:

```text
neighbourhoods.neighbourhood
            │
            │ potential geographic lookup relationship
            ▼
listings.neighbourhood

### Business-Domain Meaning

The `neighbourhoods.csv` dataset acts as geographic reference data for Amsterdam.

Each row represents one neighbourhood that may be associated with Airbnb listings in the city.

This dataset can support analyses such as:

- Number of listings by neighbourhood
- Listing density across Amsterdam neighbourhoods
- Average and median listing price by neighbourhood
- Room-type distribution by neighbourhood
- Availability patterns by neighbourhood
- Review activity by neighbourhood
- Comparison of market characteristics across geographic areas

The dataset is primarily a reference or lookup dataset rather than a transactional dataset.

### Dataset Limitations

The main limitations identified in `neighbourhoods.csv` are:

1. **Completely missing neighbourhood group information**

   The `neighbourhood_group` column contains 100% missing values and currently provides no analytical value.

2. **No geographic coordinates or boundaries**

   The CSV contains neighbourhood names only. Geographic boundary information is stored separately in `neighbourhoods.geojson`.

3. **Potential join inconsistencies**

   Before joining this dataset with listings data, neighbourhood names should be checked for:

   - spelling differences,
   - capitalization differences,
   - leading or trailing whitespace,
   - alternative naming conventions.

4. **No hierarchical grouping information**

   Because `neighbourhood_group` is completely empty, the CSV does not provide a higher-level geographic grouping.

5. **String minimum and maximum values are not analytically meaningful**

   The observed minimum and maximum neighbourhood values are based only on alphabetical ordering and should not be interpreted as numerical or business measurements.

## 2 Summary Listings Dataset Familiarization

The `listings.csv` file contains summary-level information about Airbnb listings in Amsterdam.

This dataset will be inspected for:

- Dataset shape
- Column names
- Data types
- Missing values
- Unique-value counts
- Minimum and maximum values
- Sample values
- Duplicate rows
- Candidate primary keys
- Potential foreign-key relationships
- Relationship with `neighbourhoods.csv`
- Business-domain meaning
- Dataset limitations

Load listings.csv

In [21]:
summary_listings_df = pd.read_csv(
    DATA_FILES["summary_listings"]
)

print("listings.csv loaded successfully.")
print(f"Rows: {summary_listings_df.shape[0]:,}")
print(f"Columns: {summary_listings_df.shape[1]}")

listings.csv loaded successfully.
Rows: 10,465
Columns: 19


Preview the first five rows

In [22]:
display(summary_listings_df.head())

,id,name,host_id,host_profile_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365,number_of_reviews_ltm,license
0,28871,Comfortable double room,124245.0,1.462510e+18,Edwin,NaN,Centrum-West,52.367750,4.89092,Private room,94.0,1.0,799,2026-06-01,4.15,2.0,11,88,0363 607B EA74 0BD8 2F6F
1,29051,Comfortable single / double room,124245.0,1.462510e+18,Edwin,NaN,Centrum-Oost,52.365840,4.89111,Private room,NaN,1.0,906,2026-06-01,4.88,2.0,2,82,0363 607B EA74 0BD8 2F6F
2,44129,Luxury design with canal view,187728.0,1.462512e+18,Tanya,NaN,Centrum-West,52.382110,4.88630,Entire home/apt,314.0,3.0,186,2026-04-30,0.96,4.0,3,5,03635399E87602900F47
3,44391,Quiet 2-bedroom Amsterdam city centre apartment,194779.0,1.462512e+18,Jan,NaN,Centrum-Oost,52.371680,4.91471,Entire home/apt,NaN,3.0,42,2022-08-20,0.22,1.0,0,0,0363 E76E F06A C1DD 172C
4,48373,Cozy family home in Amsterdam South,220434.0,1.462513e+18,Vesna & Misha,NaN,Buitenveldert - Zuidas,52.327808,4.87680,Entire home/apt,NaN,3.0,5,2024-04-28,0.14,1.0,0,0,0363 4A2B A6AD 0196 F684


Dataset shape

In [23]:
print("Dataset Shape")
print("-" * 40)
print(f"Number of rows: {summary_listings_df.shape[0]:,}")
print(f"Number of columns: {summary_listings_df.shape[1]}")

Dataset Shape
----------------------------------------
Number of rows: 10,465
Number of columns: 19


Column names

In [24]:
print("Column Names")
print("-" * 40)

for index, column in enumerate(summary_listings_df.columns, start=1):
    print(f"{index}. {column}")

Column Names
----------------------------------------
1. id
2. name
3. host_id
4. host_profile_id
5. host_name
6. neighbourhood_group
7. neighbourhood
8. latitude
9. longitude
10. room_type
11. price
12. minimum_nights
13. number_of_reviews
14. last_review
15. reviews_per_month
16. calculated_host_listings_count
17. availability_365
18. number_of_reviews_ltm
19. license


Data types

In [25]:
summary_listings_dtypes = pd.DataFrame({
    "column_name": summary_listings_df.columns,
    "data_type": summary_listings_df.dtypes.astype(str).values
})

display(summary_listings_dtypes)

,column_name,data_type
0,id,int64
1,name,str
2,host_id,float64
3,host_profile_id,float64
4,host_name,str
5,neighbourhood_group,float64
6,neighbourhood,str
7,latitude,float64
8,longitude,float64
9,room_type,str


Missing-value analysis

In [26]:
summary_listings_missing = pd.DataFrame({
    "column_name": summary_listings_df.columns,
    "missing_count": summary_listings_df.isnull().sum().values,
    "missing_percentage": (
        summary_listings_df.isnull().mean() * 100
    ).round(2).values
})

display(
    summary_listings_missing.sort_values(
        by="missing_percentage",
        ascending=False
    ).reset_index(drop=True)
)

,column_name,missing_count,missing_percentage
0,neighbourhood_group,10465,100.00
1,price,3994,38.17
2,last_review,1033,9.87
3,reviews_per_month,1033,9.87
4,license,319,3.05
5,host_name,238,2.27
6,host_profile_id,233,2.23
7,host_id,96,0.92
8,calculated_host_listings_count,96,0.92
9,minimum_nights,3,0.03


Unique-value counts

In [27]:
summary_listings_unique = pd.DataFrame({
    "column_name": summary_listings_df.columns,
    "unique_count": [
        summary_listings_df[column].nunique(dropna=True)
        for column in summary_listings_df.columns
    ]
})

display(
    summary_listings_unique.sort_values(
        by="unique_count",
        ascending=False
    ).reset_index(drop=True)
)

,column_name,unique_count
0,id,10465
1,name,10148
2,host_id,9104
3,host_profile_id,9076
4,license,9032
5,longitude,8576
6,latitude,7569
7,host_name,3793
8,last_review,1506
9,price,842


Combined column profile

In [28]:
summary_listings_profile = []

for column in summary_listings_df.columns:
    series = summary_listings_df[column]
    non_null_values = series.dropna()

    # Calculate min/max only for numeric and datetime columns.
    if (
        pd.api.types.is_numeric_dtype(series)
        or pd.api.types.is_datetime64_any_dtype(series)
    ):
        minimum_value = (
            non_null_values.min()
            if not non_null_values.empty
            else None
        )

        maximum_value = (
            non_null_values.max()
            if not non_null_values.empty
            else None
        )
    else:
        minimum_value = None
        maximum_value = None

    sample_values = (
        non_null_values
        .astype(str)
        .drop_duplicates()
        .head(3)
        .tolist()
    )

    summary_listings_profile.append({
        "column_name": column,
        "data_type": str(series.dtype),
        "missing_count": int(series.isnull().sum()),
        "missing_percentage": round(
            series.isnull().mean() * 100,
            2
        ),
        "unique_count": int(series.nunique(dropna=True)),
        "minimum": minimum_value,
        "maximum": maximum_value,
        "sample_values": sample_values,
    })

summary_listings_profile_df = pd.DataFrame(
    summary_listings_profile
)

display(summary_listings_profile_df)

,column_name,data_type,missing_count,missing_percentage,unique_count,minimum,maximum,sample_values
0,id,int64,0,0.00,10465,2.887100e+04,1.708159e+18,"[28871, 29051, 44129]"
1,name,str,0,0.00,10148,NaN,NaN,"[Comfortable double room, Comfortable single / double room, Luxury design with canal view]"
2,host_id,float64,96,0.92,9104,3.592000e+03,1.705253e+18,"[124245.0, 187728.0, 194779.0]"
3,host_profile_id,float64,233,2.23,9076,1.462506e+18,1.707866e+18,"[1.462510208428039e+18, 1.4625121252931446e+18, 1.4625123369036503e+18]"
4,host_name,str,238,2.27,3793,NaN,NaN,"[Edwin, Tanya, Jan]"
5,neighbourhood_group,float64,10465,100.00,0,NaN,NaN,[]
6,neighbourhood,str,0,0.00,22,NaN,NaN,"[Centrum-West, Centrum-Oost, Buitenveldert - Zuidas]"
7,latitude,float64,0,0.00,7569,5.229028e+01,5.242516e+01,"[52.36775, 52.36584, 52.38211]"
8,longitude,float64,0,0.00,8576,4.755870e+00,5.028150e+00,"[4.89092, 4.89111, 4.8863]"
9,room_type,str,0,0.00,4,NaN,NaN,"[Private room, Entire home/apt, Hotel room]"


Duplicate-row analysis

In [29]:
summary_listings_duplicate_count = (
    summary_listings_df.duplicated().sum()
)

print("Duplicate Analysis")
print("-" * 40)
print(f"Total rows: {len(summary_listings_df):,}")
print(
    f"Duplicate rows: "
    f"{summary_listings_duplicate_count:,}"
)

duplicate_percentage = (
    summary_listings_duplicate_count
    / len(summary_listings_df)
    * 100
)

print(
    f"Duplicate percentage: "
    f"{duplicate_percentage:.2f}%"
)

Duplicate Analysis
----------------------------------------
Total rows: 10,465
Duplicate rows: 0
Duplicate percentage: 0.00%


Candidate primary-key analysis

In [30]:
summary_listings_candidate_keys = []

total_rows = len(summary_listings_df)

for column in summary_listings_df.columns:
    unique_count = summary_listings_df[column].nunique(
        dropna=False
    )
    missing_count = summary_listings_df[column].isnull().sum()

    is_unique = unique_count == total_rows
    has_no_missing_values = missing_count == 0

    if is_unique and has_no_missing_values:
        summary_listings_candidate_keys.append(column)

print("Candidate Primary Key Analysis")
print("-" * 40)

if summary_listings_candidate_keys:
    print("Possible single-column primary key candidates:")

    for column in summary_listings_candidate_keys:
        print(f"- {column}")
else:
    print(
        "No single-column candidate primary key "
        "was identified."
    )

Candidate Primary Key Analysis
----------------------------------------
Possible single-column primary key candidates:
- id


Explicitly validate id if present

In [31]:
if "id" in summary_listings_df.columns:
    print("Listing ID Validation")
    print("-" * 40)

    print(
        f"Total rows: "
        f"{len(summary_listings_df):,}"
    )

    print(
        f"Unique IDs: "
        f"{summary_listings_df['id'].nunique(dropna=True):,}"
    )

    print(
        f"Missing IDs: "
        f"{summary_listings_df['id'].isnull().sum():,}"
    )

    print(
        f"Duplicate IDs: "
        f"{summary_listings_df['id'].duplicated().sum():,}"
    )
else:
    print("The 'id' column was not found.")

Listing ID Validation
----------------------------------------
Total rows: 10,465
Unique IDs: 10,465
Missing IDs: 0
Duplicate IDs: 0


Inspect categorical values

In [32]:
low_cardinality_columns = []

for column in summary_listings_df.columns:
    unique_count = summary_listings_df[column].nunique(
        dropna=True
    )

    if unique_count <= 30:
        low_cardinality_columns.append({
            "column_name": column,
            "unique_count": unique_count
        })

low_cardinality_df = pd.DataFrame(
    low_cardinality_columns
)

display(low_cardinality_df)

,column_name,unique_count
0,neighbourhood_group,0
1,neighbourhood,22
2,room_type,4
3,calculated_host_listings_count,20


In [33]:
if "room_type" in summary_listings_df.columns:
    display(
        summary_listings_df["room_type"]
        .value_counts(dropna=False)
        .rename_axis("room_type")
        .reset_index(name="count")
    )

,room_type,count
0,Entire home/apt,8489
1,Private room,1929
2,Hotel room,26
3,Shared room,21


In [34]:
if "neighbourhood" in summary_listings_df.columns:
    display(
        summary_listings_df["neighbourhood"]
        .value_counts(dropna=False)
        .rename_axis("neighbourhood")
        .reset_index(name="listing_count")
    )

,neighbourhood,listing_count
0,De Baarsjes - Oud-West,1825
1,Centrum-West,1217
2,De Pijp - Rivierenbuurt,1201
3,Centrum-Oost,933
4,Westerpark,720
5,Zuid,709
6,Oud-Oost,670
7,Bos en Lommer,546
8,Oud-Noord,479
9,Oostelijk Havengebied - Indische Buurt,411


Verify the neighbourhood relationship

In [35]:
if "neighbourhood" in summary_listings_df.columns:
    listing_neighbourhoods = set(
        summary_listings_df["neighbourhood"]
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
    )

    reference_neighbourhoods = set(
        neighbourhoods_df["neighbourhood"]
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
    )

    unmatched_listing_neighbourhoods = (
        listing_neighbourhoods
        - reference_neighbourhoods
    )

    unused_reference_neighbourhoods = (
        reference_neighbourhoods
        - listing_neighbourhoods
    )

    print("Neighbourhood Relationship Validation")
    print("-" * 50)

    print(
        f"Unique neighbourhoods in listings.csv: "
        f"{len(listing_neighbourhoods)}"
    )

    print(
        f"Unique neighbourhoods in neighbourhoods.csv: "
        f"{len(reference_neighbourhoods)}"
    )

    print(
        f"Unmatched listing neighbourhoods: "
        f"{len(unmatched_listing_neighbourhoods)}"
    )

    print(
        f"Unused reference neighbourhoods: "
        f"{len(unused_reference_neighbourhoods)}"
    )
else:
    print(
        "The 'neighbourhood' column does not exist "
        "in listings.csv."
    )

Neighbourhood Relationship Validation
--------------------------------------------------
Unique neighbourhoods in listings.csv: 22
Unique neighbourhoods in neighbourhoods.csv: 22
Unmatched listing neighbourhoods: 0
Unused reference neighbourhoods: 0


In [36]:
if unmatched_listing_neighbourhoods:
    print("Listing neighbourhoods not found in reference data:")

    for neighbourhood in sorted(
        unmatched_listing_neighbourhoods
    ):
        print(f"- {neighbourhood}")
else:
    print(
        "All listing neighbourhood values match "
        "the neighbourhood reference dataset."
    )

All listing neighbourhood values match the neighbourhood reference dataset.


In [37]:
if unused_reference_neighbourhoods:
    print(
        "Reference neighbourhoods with no listings "
        "in the current summary dataset:"
    )

    for neighbourhood in sorted(
        unused_reference_neighbourhoods
    ):
        print(f"- {neighbourhood}")
else:
    print(
        "Every reference neighbourhood appears "
        "in the current listings dataset."
    )

Every reference neighbourhood appears in the current listings dataset.


Relationship coverage percentage

In [38]:
if "neighbourhood" in summary_listings_df.columns:
    valid_neighbourhood_mask = (
        summary_listings_df["neighbourhood"]
        .astype(str)
        .str.strip()
        .isin(reference_neighbourhoods)
    )

    matched_rows = int(valid_neighbourhood_mask.sum())
    total_listing_rows = len(summary_listings_df)

    match_percentage = (
        matched_rows / total_listing_rows * 100
        if total_listing_rows > 0
        else 0
    )

    print("Neighbourhood Join Coverage")
    print("-" * 40)
    print(f"Matched listing rows: {matched_rows:,}")
    print(f"Total listing rows: {total_listing_rows:,}")
    print(f"Match coverage: {match_percentage:.2f}%")

Neighbourhood Join Coverage
----------------------------------------
Matched listing rows: 10,465
Total listing rows: 10,465
Match coverage: 100.00%


Dataset-level summary

In [39]:
summary_listings_dataset_summary = {
    "dataset_name": "summary_listings",
    "file_name": DATA_FILES["summary_listings"].name,
    "row_count": len(summary_listings_df),
    "column_count": len(summary_listings_df.columns),
    "duplicate_count": int(
        summary_listings_df.duplicated().sum()
    ),
    "candidate_primary_keys": (
        summary_listings_candidate_keys
    ),
}

summary_listings_dataset_summary

{'dataset_name': 'summary_listings',
 'file_name': 'listings.csv',
 'row_count': 10465,
 'column_count': 19,
 'duplicate_count': 0,
 'candidate_primary_keys': ['id']}

Save the profile result

In [40]:
summary_listings_profile_path = (
    OUTPUT_DIR / "summary_listings_column_profile.csv"
)

summary_listings_profile_df.to_csv(
    summary_listings_profile_path,
    index=False
)

print(
    "Summary listings profile saved successfully to:\n"
    f"{summary_listings_profile_path}"
)

Summary listings profile saved successfully to:
c:\Users\saths\Desktop\MyProjects\Expernetic Data Challenge\airbnb-data-challenge\outputs\data_quality\summary_listings_column_profile.csv


### Findings and Interpretation

The `listings.csv` dataset contains **10,465 rows and 19 columns**, with each row representing one summary-level Airbnb listing in Amsterdam.

#### Key Findings

- The dataset contains **10,465 unique listing IDs**.
- No duplicate rows were identified.
- The `id` column contains no missing values and no duplicate values, making it a strong **candidate primary key**.
- The dataset contains listings from all **22 Amsterdam neighbourhoods** available in the neighbourhood reference dataset.
- Four room types are present:
  - Entire home/apt: 8,489 listings
  - Private room: 1,929 listings
  - Hotel room: 26 listings
  - Shared room: 21 listings

The Amsterdam Airbnb market in this dataset is therefore heavily dominated by `Entire home/apt` listings, followed by `Private room` listings.

#### Major Missing-Value Findings

The most important missing-value patterns are:

- `neighbourhood_group`: 10,465 missing values — **100.00%**
- `price`: 3,994 missing values — **38.17%**
- `last_review`: 1,033 missing values — **9.87%**
- `reviews_per_month`: 1,033 missing values — **9.87%**
- `license`: 319 missing values — **3.05%**
- `host_name`: 238 missing values — **2.27%**
- `host_profile_id`: 233 missing values — **2.23%**
- `host_id`: 96 missing values — **0.92%**
- `calculated_host_listings_count`: 96 missing values — **0.92%**
- `minimum_nights`: 3 missing values — **0.03%**

The identical missing counts for `last_review` and `reviews_per_month` suggest that these two fields may be structurally related. Listings without recorded review activity may not have values for either field. This should be verified later against `number_of_reviews`.

The large proportion of missing `price` values is a significant limitation because pricing is central to later exploratory analysis, statistical testing, and business interpretation.

#### Range Observations

Several variables contain values that should later be validated during the data-quality phase:

- `price`: minimum 15, maximum 11,412
- `minimum_nights`: minimum 1, maximum 800
- `number_of_reviews`: minimum 0, maximum 5,603
- `reviews_per_month`: minimum 0.01, maximum 92.56
- `number_of_reviews_ltm`: minimum 0, maximum 813
- `availability_365`: minimum 0, maximum 365

These values should not automatically be classified as errors. They should first be investigated as potential valid extremes, business-rule violations, or outliers.

The `availability_365` values remain within the expected range of 0 to 365 days.

### Candidate Keys and Relationships

#### Candidate Primary Key

The `id` column is a strong candidate primary key for the `listings.csv` dataset because it satisfies the main observed requirements for a unique record identifier:

- Total rows: 10,465
- Unique `id` values: 10,465
- Missing `id` values: 0
- Duplicate `id` values: 0

Therefore:

**Candidate Primary Key:** `id`

This conclusion is based on observed uniqueness and completeness within the current dataset. Since the source is a CSV file rather than a relational database with enforced constraints, `id` is described as a candidate primary key rather than a formally enforced primary key.

---

#### Host Relationship

The `host_id` column contains 9,104 unique non-null host identifiers across 10,465 listing records.

Because multiple listings can share the same `host_id`, this column is not suitable as a primary key for the listings dataset.

Instead, it represents a potential one-to-many relationship between hosts and listings:

```text
Host
  │
  │ host_id
  ▼
Listing
````

This means:

**One host may manage one or more Airbnb listings.**

However, the relationship is not complete for every row because:

* 96 listings have missing `host_id` values.
* `host_id` is currently stored as `float64`.
* Some identifier values are very large and may require safer data-type handling later.

The host relationship should therefore be validated further during the cleaning and enrichment phases.

---

#### Neighbourhood Relationship

The relationship between `listings.csv` and `neighbourhoods.csv` was explicitly validated using the `neighbourhood` field.

The logical relationship is:

```text
neighbourhoods.csv
    neighbourhood
          │
          │ logical reference relationship
          ▼
listings.csv
    neighbourhood
```

Validation results:

* Unique neighbourhoods in `listings.csv`: 22
* Unique neighbourhoods in `neighbourhoods.csv`: 22
* Unmatched listing neighbourhoods: 0
* Unused reference neighbourhoods: 0
* Matched listing rows: 10,465
* Total listing rows: 10,465
* Join coverage: 100.00%

Therefore, every listing neighbourhood matches a valid value in the neighbourhood reference dataset.

This supports treating:

* `neighbourhoods.neighbourhood` as a candidate lookup-table primary key.
* `listings.neighbourhood` as a logical foreign-key-like reference to that lookup table.

Because CSV files do not enforce relational constraints, this should be described as a logical foreign-key relationship rather than a formally enforced database foreign key.

---

#### Potential Relationships with Other Datasets

The `id` field in `listings.csv` is expected to act as the central listing identifier for relationships with other Amsterdam Airbnb datasets.

Potential relationships include:

```text
listings.csv.id
       │
       ├──────────────► calendar.csv.gz.listing_id
       │
       ├──────────────► reviews.csv.gz.listing_id
       │
       └──────────────► reviews.csv.listing_id
```

These relationships must be verified later using the actual datasets before being confirmed.

The expected relationship structure is:

```text
                    Host
                      │
                   host_id
                      │
                      ▼
                Listing
                id / listing_id
                 │         │
                 │         │
                 ▼         ▼
              Calendar   Reviews
```

Likely relationship types:

* One host to many listings.
* One neighbourhood to many listings.
* One listing to many calendar records.
* One listing to many detailed review records.
* One listing to one or more summary review-related records, depending on the actual structure of `reviews.csv`.

---

#### Relationship Validation Status

At this stage, the following relationships have been validated:

* `neighbourhoods.neighbourhood` → `listings.neighbourhood`

  * Status: Validated
  * Coverage: 100.00%

The following relationships are still pending validation:

* `listings.id` → `calendar.listing_id`
* `listings.id` → `reviews.csv.gz.listing_id`
* `listings.id` → `reviews.csv.listing_id`
* Host-level relationships involving `host_id`

These will be tested when the remaining datasets are profiled.

---

#### Key Engineering Interpretation

The `listings.csv` dataset appears to be the central listing-level entity in the Amsterdam Airbnb data model.

Its `id` column provides a stable candidate identifier, while fields such as `host_id` and `neighbourhood` connect listings to host-level and geographic information.

This makes `listings.csv` a strong foundation for the later:

* enriched listing master dataset,
* analytical star schema,
* DuckDB data model,
* EDA,
* statistical testing,
* business analysis.


### Identifier Data-Type Observation

An important technical issue was identified in the host-related identifier columns.

The following columns were inferred by Pandas as `float64`:

- `host_id`
- `host_profile_id`

This is potentially risky because some identifier values are extremely large, reaching approximately `10^18`.

Identifiers are labels rather than numerical measurements, so they should not normally be stored as floating-point values. Large integers represented as `float64` may lose exact precision because floating-point numbers cannot represent every very large integer exactly.

This could create problems during:

- joins between datasets,
- duplicate checks,
- host-level aggregation,
- relationship validation,
- comparisons between identifiers.

For example, two originally different large integer IDs could potentially be represented inaccurately after floating-point conversion.

The likely reason these columns were inferred as `float64` is that they contain missing values:

- `host_id`: 96 missing values
- `host_profile_id`: 233 missing values

During the cleaning and standardization phase, these identifier columns should be converted to a safer data type.

Possible options include:

- Pandas nullable `Int64`, when all non-null values are valid integers.
- String type, when exact identifier preservation is the priority.

No changes will be made to the raw dataset during Dataset Familiarization. The original source data will remain unchanged, and any data-type corrections will be applied only to processed datasets.

#### Engineering Decision

For this project, identifier fields should be treated as categorical identifiers rather than numerical measurements.

Therefore, before using `host_id` or `host_profile_id` in joins or aggregations, their values should be validated and converted to a precision-safe data type.

This issue will be handled during the later **Cleaning and Standardization** phase.

### Business-Domain Meaning

The `listings.csv` dataset represents the summary-level Airbnb accommodation supply available in Amsterdam.

Each row represents one Airbnb listing.

The dataset contains information about:

- Listing identity
- Listing name
- Host identity
- Host profile identity
- Host name
- Neighbourhood
- Geographic coordinates
- Room type
- Price
- Minimum-stay requirement
- Review activity
- Host portfolio size
- Annual availability
- Recent review activity
- License information

The grain of the dataset is:

**One row represents one Airbnb listing.**

This makes `listings.csv` one of the central datasets in the project and a strong foundation for the later enriched listing master dataset.

#### Business Questions Supported by This Dataset

The dataset can support business analysis such as:

- Which neighbourhoods contain the highest number of listings?
- Which room types dominate the Amsterdam Airbnb market?
- How do prices differ across room types?
- How do prices vary between neighbourhoods?
- Which neighbourhoods have the highest listing concentration?
- How concentrated is market supply among multi-listing hosts?
- Which listings receive the highest number of reviews?
- How does listing availability differ across room types and neighbourhoods?
- Which accommodation segments may command pricing premiums?
- Which areas may show stronger competition or higher market density?

#### Main Business Entities Represented

The dataset includes several important business entities and concepts.

**Listing**

A listing represents an individual Airbnb accommodation offered to guests.

The main listing identifier is:

`id`

---

**Host**

A host represents the person or entity associated with one or more Airbnb listings.

Relevant columns include:

- `host_id`
- `host_profile_id`
- `host_name`
- `calculated_host_listings_count`

Multiple listings may belong to the same host.

---

**Neighbourhood**

The `neighbourhood` column identifies the geographic area in which a listing is located.

This allows geographic comparisons such as:

- listing density,
- pricing differences,
- room-type distribution,
- availability patterns,
- review activity.

The relationship with `neighbourhoods.csv` has already been validated with 100% join coverage.

---

**Room Type**

The `room_type` column describes the accommodation category.

The current dataset contains four room types:

- Entire home/apt
- Private room
- Hotel room
- Shared room

The Amsterdam market represented in this dataset is heavily dominated by `Entire home/apt` listings.

---

**Price**

The `price` column represents the listed nightly price where available.

Price is one of the most important variables for later:

- EDA,
- room-type comparison,
- neighbourhood analysis,
- statistical testing,
- business recommendations.

However, because 38.17% of price values are missing, price-based analysis must clearly report the valid sample size used.

---

**Review Activity**

The dataset includes several fields related to review activity:

- `number_of_reviews`
- `last_review`
- `reviews_per_month`
- `number_of_reviews_ltm`

These variables can be used as indicators of listing activity and guest engagement.

However, review count should not be interpreted as an exact booking count because not every guest necessarily leaves a review.

---

**Availability**

The `availability_365` column describes how many days a listing is shown as available within a 365-day period.

This field may support analyses of:

- listing availability patterns,
- differences across room types,
- neighbourhood-level availability,
- possible occupancy proxies.

However, availability should not be directly interpreted as actual occupancy or vacancy because unavailable dates may result from bookings, host blocks, maintenance, or other reasons.

---

#### Analytical Importance

This dataset is likely to become the primary listing-level source for:

- data cleaning,
- data enrichment,
- host-level analysis,
- neighbourhood analysis,
- price analysis,
- availability analysis,
- review-performance analysis,
- statistical hypothesis testing,
- business recommendations.

It will also serve as the foundation for creating the later:

`enriched_listing_master.parquet`

where listing information can be combined with aggregated review, calendar, host, and neighbourhood features.

### Dataset Limitations

The `listings.csv` dataset is structurally useful and suitable for downstream analysis, but several important limitations must be considered before cleaning, enrichment, exploratory data analysis, statistical testing, or business interpretation.

#### 1. Substantial Missing Price Data

The `price` column contains:

- 3,994 missing values
- 38.17% missing data

This is the most significant limitation in the dataset because price is central to later analyses such as:

- price distribution,
- price by room type,
- neighbourhood pricing,
- statistical comparison of room types,
- business recommendations.

Any price-based analysis will therefore use a reduced subset of records unless a justified alternative treatment is applied.

Missing prices should not be automatically replaced with zero because zero would incorrectly represent a real listing price.

The valid sample size should always be reported for price-based analysis.

---

#### 2. Completely Missing Neighbourhood Group

The `neighbourhood_group` column contains:

- 10,465 missing values
- 100.00% missing data

Therefore, this field currently provides no analytical value.

It should not be used in downstream analysis unless valid values are obtained from another trusted source.

The raw column should still remain unchanged in the original source dataset.

---

#### 3. Missing Review Information

The following review-related columns contain missing values:

- `last_review`: 1,033 missing values
- `reviews_per_month`: 1,033 missing values

Both fields have exactly the same missing count of 1,033 records, representing 9.87% of the dataset.

This suggests that the missingness may be structurally related to listings with no recorded review activity.

However, this relationship should be explicitly validated against `number_of_reviews` before deciding how to handle the missing values.

For example, a missing `reviews_per_month` value should not automatically be replaced with zero unless the listing is confirmed to have no reviews.

---

#### 4. Missing Host Information

Some listings contain incomplete host-related information:

- `host_id`: 96 missing values
- `host_profile_id`: 233 missing values
- `host_name`: 238 missing values
- `calculated_host_listings_count`: 96 missing values

These missing values may affect:

- host-level aggregation,
- host portfolio analysis,
- multi-listing host classification,
- host concentration analysis,
- joins involving host identifiers.

Host-based analyses should clearly account for these incomplete records.

---

#### 5. Potential Precision Risk in Identifier Columns

The following identifier columns were inferred as `float64`:

- `host_id`
- `host_profile_id`

Some identifier values are extremely large, reaching approximately `10^18`.

Large integer identifiers stored as floating-point values may lose exact precision.

This could affect:

- joins,
- comparisons,
- duplicate checks,
- grouping,
- relationship validation.

These columns should therefore be converted to a safer precision-preserving type during the cleaning phase.

---

#### 6. Potential Outliers and Extreme Values

Several columns contain extreme values that require further validation:

- `price`: maximum 11,412
- `minimum_nights`: maximum 800
- `number_of_reviews`: maximum 5,603
- `reviews_per_month`: maximum 92.56
- `number_of_reviews_ltm`: maximum 813

These values should not automatically be classified as incorrect.

They may represent:

- legitimate extreme observations,
- special listing types,
- unusual host policies,
- data collection issues,
- possible business-rule violations.

Further validation is required before deciding whether any values should be capped, excluded, flagged, or preserved.

---

#### 7. Review Counts Are Not Equivalent to Booking Counts

The following columns measure review activity:

- `number_of_reviews`
- `reviews_per_month`
- `number_of_reviews_ltm`

However, review volume should not be interpreted as an exact measure of booking volume because not every guest necessarily leaves a review.

Therefore, review counts may only serve as an imperfect proxy for demand or listing activity.

---

#### 8. Availability Is Not Equivalent to Occupancy

The `availability_365` field contains values between 0 and 365.

Although this field is useful for understanding listing availability, it should not be interpreted directly as actual vacancy or occupancy.

An unavailable date may result from:

- a confirmed booking,
- a host manually blocking the date,
- maintenance,
- personal use,
- listing suspension,
- other operational reasons.

Therefore, calculations such as:

`1 - availability_rate`

should only be described as an **occupancy proxy**, not true occupancy.

---

#### 9. Snapshot Nature of the Dataset

The dataset represents a snapshot of the Amsterdam Airbnb market at a particular collection time.

Therefore, it does not necessarily capture:

- complete historical trends,
- all previous listing activity,
- long-term host behaviour,
- complete seasonal patterns,
- actual future market conditions.

Historical and time-series conclusions should therefore be made cautiously.

---

#### 10. CSV Relationships Are Not Formally Enforced

Although the relationship between `listings.csv` and `neighbourhoods.csv` achieved 100% join coverage, the source files are CSV datasets rather than relational database tables.

Therefore:

- primary-key constraints are not formally enforced,
- foreign-key constraints are not formally enforced,
- uniqueness must be tested manually,
- referential integrity must be validated explicitly.

The current project will therefore treat identified keys and relationships as logical data-model relationships based on observed evidence.

---

#### 11. Geographic Coordinates Require Domain Validation

The `latitude` and `longitude` fields contain no missing values, which is a positive quality characteristic.

However, completeness alone does not guarantee geographic validity.

These coordinates should later be checked for:

- valid latitude range,
- valid longitude range,
- plausible location within or near Amsterdam,
- possible coordinate anomalies.

---

#### 12. License Information Is Incomplete

The `license` column contains:

- 319 missing values
- 3.05% missing data

This may affect any future analysis related to listing registration or licensing status.

Missing license information should not automatically be interpreted as proof that a listing is unlicensed, because missing data may have multiple causes.

---

#### Overall Limitation Summary

The most important limitations are:

- substantial missing price data,
- a completely empty `neighbourhood_group` column,
- missing review-related information,
- incomplete host identifiers,
- potential precision risk in large identifier fields,
- extreme values requiring validation,
- the snapshot nature of the dataset,
- availability not representing true occupancy,
- review counts not representing exact bookings.

Despite these limitations, the dataset remains highly valuable for downstream analysis after targeted cleaning, validation, and documented handling of missing values.

### Data Quality Assessment

The `listings.csv` dataset is structurally strong at the listing level and provides a useful foundation for downstream data engineering, exploratory analysis, statistical testing, and business interpretation.

#### Positive Data Quality Characteristics

Several important quality checks produced positive results:

- Total rows: **10,465**
- Total columns: **19**
- Duplicate rows: **0**
- Duplicate percentage: **0.00%**
- Unique listing IDs: **10,465**
- Missing listing IDs: **0**
- Duplicate listing IDs: **0**

The `id` column therefore provides a strong candidate primary key for the dataset.

The following important fields are also complete:

- `id`
- `name`
- `neighbourhood`
- `latitude`
- `longitude`
- `room_type`
- `number_of_reviews`
- `availability_365`
- `number_of_reviews_ltm`

This provides a strong basis for listing-level, geographic, room-type, review, and availability analysis.

---

#### Neighbourhood Referential Integrity

The relationship between `listings.csv` and `neighbourhoods.csv` was explicitly validated.

Results:

- Unique neighbourhoods in `listings.csv`: **22**
- Unique neighbourhoods in `neighbourhoods.csv`: **22**
- Unmatched listing neighbourhoods: **0**
- Unused reference neighbourhoods: **0**
- Matched listing rows: **10,465**
- Total listing rows: **10,465**
- Join coverage: **100.00%**

This indicates complete observed referential consistency between:

`listings.neighbourhood`

and:

`neighbourhoods.neighbourhood`

The result supports treating `listings.neighbourhood` as a logical foreign-key-like reference to the neighbourhood lookup dataset.

---

#### Room-Type Consistency

The `room_type` column contains four distinct categories:

- Entire home/apt: **8,489 listings**
- Private room: **1,929 listings**
- Hotel room: **26 listings**
- Shared room: **21 listings**

No missing room-type values were identified.

The category structure appears internally consistent and suitable for later segmentation analysis.

However, room-type categories should still be reviewed during the validation phase for unexpected whitespace, spelling differences, or inconsistent formatting.

---

#### Geographic Completeness

Both geographic coordinate fields are complete:

- `latitude`: 0 missing values
- `longitude`: 0 missing values

Observed ranges:

- Latitude: approximately **52.29028 to 52.42516**
- Longitude: approximately **4.75587 to 5.02815**

These coordinates should later undergo domain and geographic validation to confirm that they fall within plausible Amsterdam boundaries.

Completeness alone does not guarantee geographic correctness.

---

#### Availability Validation

The `availability_365` column has:

- Minimum value: **0**
- Maximum value: **365**
- Missing values: **0**

The observed values therefore remain within the expected annual range of 0 to 365 days.

This is a positive data-quality result.

However, availability should not be interpreted as true vacancy or occupancy because unavailable dates may result from bookings, host blocks, maintenance, or other operational reasons.

---

#### Main Data Quality Concerns

The most important data-quality issues are:

1. `neighbourhood_group` is 100% missing.
2. `price` is missing for 38.17% of listings.
3. `last_review` and `reviews_per_month` are each missing for 9.87% of listings.
4. Some host-related fields are incomplete.
5. Large identifier values are currently stored as floating-point numbers.
6. Several extreme values require later outlier and domain validation.
7. `last_review` is currently stored as a string rather than a datetime type.

---

#### Missing-Value Severity Assessment

| Column | Missing Count | Missing Percentage | Initial Severity |
|---|---:|---:|---|
| `neighbourhood_group` | 10,465 | 100.00% | Critical |
| `price` | 3,994 | 38.17% | High |
| `last_review` | 1,033 | 9.87% | Moderate |
| `reviews_per_month` | 1,033 | 9.87% | Moderate |
| `license` | 319 | 3.05% | Low |
| `host_name` | 238 | 2.27% | Low |
| `host_profile_id` | 233 | 2.23% | Low |
| `host_id` | 96 | 0.92% | Low |
| `calculated_host_listings_count` | 96 | 0.92% | Low |
| `minimum_nights` | 3 | 0.03% | Very Low |

These severity labels are initial prioritization categories rather than automatic cleaning decisions.

Actual treatment should depend on:

- business meaning,
- downstream analytical use,
- relationship with other columns,
- availability of supporting data,
- risk of introducing false information.

---

#### Outlier and Domain-Validation Priorities

The following fields should receive priority during the later validation phase:

- `price`
- `minimum_nights`
- `number_of_reviews`
- `reviews_per_month`
- `number_of_reviews_ltm`
- `latitude`
- `longitude`
- `availability_365`
- `host_id`
- `host_profile_id`

Observed extreme values include:

- Maximum price: **11,412**
- Maximum minimum stay: **800 nights**
- Maximum number of reviews: **5,603**
- Maximum reviews per month: **92.56**
- Maximum reviews in the last twelve months: **813**

These values should be investigated before any removal or transformation decision is made.

---

#### Overall Data Quality Assessment

Overall, `listings.csv` is suitable for downstream use after targeted cleaning and validation.

Its strongest characteristics are:

- unique and complete listing identifiers,
- no duplicate rows,
- complete neighbourhood information,
- 100% neighbourhood reference coverage,
- complete geographic coordinates,
- complete room-type information,
- valid observed availability range.

Its most important weaknesses are:

- substantial missing price data,
- a completely empty `neighbourhood_group` field,
- missing review-related information,
- incomplete host-related fields,
- precision risk in large identifiers stored as `float64`,
- extreme values requiring validation.

**Overall assessment: Suitable for downstream analysis after targeted cleaning, type correction, missing-value handling, outlier validation, and documented engineering decisions.**

## 3. Summary Reviews Dataset Familiarization

The `reviews.csv` file contains summary-level review records associated with Airbnb listings in Amsterdam.

This dataset will be inspected for:

- Dataset shape
- Column names
- Data types
- Missing values
- Unique-value counts
- Minimum and maximum values
- Sample values
- Duplicate rows
- Candidate primary keys
- Potential foreign-key relationships
- Relationship with `listings.csv`
- Business-domain meaning
- Dataset limitations

The main relationship to investigate is expected to involve:

`reviews.csv.listing_id`

and:

`listings.csv.id`

However, this relationship will be verified using the actual dataset values rather than assumed.

In [41]:
summary_reviews_df = pd.read_csv(
    DATA_FILES["summary_reviews"]
)

print("reviews.csv loaded successfully.")
print(f"Rows: {summary_reviews_df.shape[0]:,}")
print(f"Columns: {summary_reviews_df.shape[1]}")

reviews.csv loaded successfully.
Rows: 545,162
Columns: 2


In [42]:
display(summary_reviews_df.head())

,listing_id,date
0,28871,2010-08-22
1,28871,2014-10-19
2,28871,2015-02-09
3,28871,2015-05-18
4,28871,2015-05-20


In [43]:
print("Dataset Shape")
print("-" * 40)
print(f"Number of rows: {summary_reviews_df.shape[0]:,}")
print(f"Number of columns: {summary_reviews_df.shape[1]}")

Dataset Shape
----------------------------------------
Number of rows: 545,162
Number of columns: 2


In [44]:
print("Column Names")
print("-" * 40)

for index, column in enumerate(summary_reviews_df.columns, start=1):
    print(f"{index}. {column}")

Column Names
----------------------------------------
1. listing_id
2. date


In [45]:
summary_reviews_dtypes = pd.DataFrame({
    "column_name": summary_reviews_df.columns,
    "data_type": summary_reviews_df.dtypes.astype(str).values
})

display(summary_reviews_dtypes)

,column_name,data_type
0,listing_id,int64
1,date,str


Missing-value analysis

In [46]:
summary_reviews_missing = pd.DataFrame({
    "column_name": summary_reviews_df.columns,
    "missing_count": summary_reviews_df.isnull().sum().values,
    "missing_percentage": (
        summary_reviews_df.isnull().mean() * 100
    ).round(2).values
})

display(summary_reviews_missing)

,column_name,missing_count,missing_percentage
0,listing_id,0,0.0
1,date,0,0.0


Unique-value counts

In [47]:
summary_reviews_unique = pd.DataFrame({
    "column_name": summary_reviews_df.columns,
    "unique_count": [
        summary_reviews_df[column].nunique(dropna=True)
        for column in summary_reviews_df.columns
    ]
})

display(summary_reviews_unique)

,column_name,unique_count
0,listing_id,9432
1,date,5327


Combined column profile

In [48]:
summary_reviews_profile = []

for column in summary_reviews_df.columns:
    series = summary_reviews_df[column]
    non_null_values = series.dropna()

    if (
        pd.api.types.is_numeric_dtype(series)
        or pd.api.types.is_datetime64_any_dtype(series)
    ):
        minimum_value = (
            non_null_values.min()
            if not non_null_values.empty
            else None
        )

        maximum_value = (
            non_null_values.max()
            if not non_null_values.empty
            else None
        )
    else:
        minimum_value = None
        maximum_value = None

    sample_values = (
        non_null_values
        .astype(str)
        .drop_duplicates()
        .head(3)
        .tolist()
    )

    summary_reviews_profile.append({
        "column_name": column,
        "data_type": str(series.dtype),
        "missing_count": int(series.isnull().sum()),
        "missing_percentage": round(
            series.isnull().mean() * 100,
            2
        ),
        "unique_count": int(
            series.nunique(dropna=True)
        ),
        "minimum": minimum_value,
        "maximum": maximum_value,
        "sample_values": sample_values,
    })

summary_reviews_profile_df = pd.DataFrame(
    summary_reviews_profile
)

display(summary_reviews_profile_df)

,column_name,data_type,missing_count,missing_percentage,unique_count,minimum,maximum,sample_values
0,listing_id,int64,0,0.0,9432,28871.0,1.704341e+18,"[28871, 29051, 44129]"
1,date,str,0,0.0,5327,NaN,NaN,"[2010-08-22, 2014-10-19, 2015-02-09]"


Duplicate-row analysis

In [49]:
summary_reviews_duplicate_count = (
    summary_reviews_df.duplicated().sum()
)

print("Duplicate Analysis")
print("-" * 40)
print(f"Total rows: {len(summary_reviews_df):,}")
print(
    f"Duplicate rows: "
    f"{summary_reviews_duplicate_count:,}"
)

duplicate_percentage = (
    summary_reviews_duplicate_count
    / len(summary_reviews_df)
    * 100
)

print(
    f"Duplicate percentage: "
    f"{duplicate_percentage:.2f}%"
)

Duplicate Analysis
----------------------------------------
Total rows: 545,162
Duplicate rows: 22,541
Duplicate percentage: 4.13%


In [50]:
if summary_reviews_duplicate_count > 0:
    print("Sample duplicate rows:")
    
    display(
        summary_reviews_df[
            summary_reviews_df.duplicated(
                keep=False
            )
        ].head(20)
    )
else:
    print("No duplicate rows found.")

Sample duplicate rows:


,listing_id,date
958,29051,2016-03-10
959,29051,2016-03-10
4214,327285,2013-02-26
4215,327285,2013-02-26
4518,327285,2016-01-05
4519,327285,2016-01-05
4553,327285,2016-05-04
4554,327285,2016-05-04
4604,327285,2016-11-08
4605,327285,2016-11-08


Candidate primary-key analysis

In [51]:
summary_reviews_candidate_keys = []

total_rows = len(summary_reviews_df)

for column in summary_reviews_df.columns:
    unique_count = summary_reviews_df[column].nunique(
        dropna=False
    )
    missing_count = (
        summary_reviews_df[column].isnull().sum()
    )

    is_unique = unique_count == total_rows
    has_no_missing_values = missing_count == 0

    if is_unique and has_no_missing_values:
        summary_reviews_candidate_keys.append(column)

print("Candidate Primary Key Analysis")
print("-" * 40)

if summary_reviews_candidate_keys:
    print(
        "Possible single-column primary key candidates:"
    )

    for column in summary_reviews_candidate_keys:
        print(f"- {column}")
else:
    print(
        "No single-column candidate primary key "
        "was identified."
    )

Candidate Primary Key Analysis
----------------------------------------
No single-column candidate primary key was identified.


Composite-key validation

In [52]:
composite_duplicate_count = (
    summary_reviews_df.duplicated(
        subset=["listing_id", "date"]
    ).sum()
)

print("Composite Key Validation")
print("-" * 40)
print(
    "Candidate composite key: "
    "(listing_id, date)"
)
print(
    f"Duplicate combinations: "
    f"{composite_duplicate_count:,}"
)

if composite_duplicate_count == 0:
    print(
        "The combination (listing_id, date) "
        "is unique across all rows."
    )
else:
    print(
        "The combination (listing_id, date) "
        "is not unique."
    )

Composite Key Validation
----------------------------------------
Candidate composite key: (listing_id, date)
Duplicate combinations: 22,541
The combination (listing_id, date) is not unique.


Validate listing_id

In [53]:
print("Listing ID Validation")
print("-" * 40)

print(
    f"Total review rows: "
    f"{len(summary_reviews_df):,}"
)

print(
    f"Unique listing IDs: "
    f"{summary_reviews_df['listing_id'].nunique(dropna=True):,}"
)

print(
    f"Missing listing IDs: "
    f"{summary_reviews_df['listing_id'].isnull().sum():,}"
)

Listing ID Validation
----------------------------------------
Total review rows: 545,162
Unique listing IDs: 9,432
Missing listing IDs: 0


Review counts per listing

In [54]:
reviews_per_listing = (
    summary_reviews_df
    .groupby("listing_id")
    .size()
    .reset_index(name="review_count")
)

display(
    reviews_per_listing
    .sort_values(
        by="review_count",
        ascending=False
    )
    .head(10)
)

,listing_id,review_count
3755,50383849,5603
2768,32485135,4082
3531,45045046,3257
3762,50452179,1993
2979,35927687,1773
2668,30762441,1706
5601,905677609043113133,1611
140,802052,1551
4927,746060021045144380,1398
3627,46591985,1304


In [55]:
print("Review Count per Listing")
print("-" * 40)
print(
    f"Listings with at least one review: "
    f"{len(reviews_per_listing):,}"
)
print(
    f"Minimum reviews per reviewed listing: "
    f"{reviews_per_listing['review_count'].min():,}"
)
print(
    f"Maximum reviews per reviewed listing: "
    f"{reviews_per_listing['review_count'].max():,}"
)
print(
    f"Median reviews per reviewed listing: "
    f"{reviews_per_listing['review_count'].median():,.0f}"
)

Review Count per Listing
----------------------------------------
Listings with at least one review: 9,432
Minimum reviews per reviewed listing: 1
Maximum reviews per reviewed listing: 5,603
Median reviews per reviewed listing: 13


Validate relationship with listings.csv

In [56]:
review_listing_ids = set(
    summary_reviews_df["listing_id"]
    .dropna()
    .unique()
)

listing_ids = set(
    summary_listings_df["id"]
    .dropna()
    .unique()
)

unmatched_review_listing_ids = (
    review_listing_ids - listing_ids
)

listings_without_summary_reviews = (
    listing_ids - review_listing_ids
)

print("Review-to-Listing Relationship Validation")
print("-" * 55)

print(
    f"Unique listing IDs in reviews.csv: "
    f"{len(review_listing_ids):,}"
)

print(
    f"Unique listing IDs in listings.csv: "
    f"{len(listing_ids):,}"
)

print(
    f"Review listing IDs not found in listings.csv: "
    f"{len(unmatched_review_listing_ids):,}"
)

print(
    f"Listings with no record in reviews.csv: "
    f"{len(listings_without_summary_reviews):,}"
)

Review-to-Listing Relationship Validation
-------------------------------------------------------
Unique listing IDs in reviews.csv: 9,432
Unique listing IDs in listings.csv: 10,465
Review listing IDs not found in listings.csv: 0
Listings with no record in reviews.csv: 1,033


Relationship coverage

In [57]:
matched_review_listing_ids = (
    review_listing_ids & listing_ids
)

relationship_coverage = (
    len(matched_review_listing_ids)
    / len(review_listing_ids)
    * 100
    if review_listing_ids
    else 0
)

print("Review Relationship Coverage")
print("-" * 40)

print(
    f"Matched unique review listing IDs: "
    f"{len(matched_review_listing_ids):,}"
)

print(
    f"Total unique review listing IDs: "
    f"{len(review_listing_ids):,}"
)

print(
    f"Relationship coverage: "
    f"{relationship_coverage:.2f}%"
)

Review Relationship Coverage
----------------------------------------
Matched unique review listing IDs: 9,432
Total unique review listing IDs: 9,432
Relationship coverage: 100.00%


Inspect unmatched IDs only if needed

In [58]:
if unmatched_review_listing_ids:
    print(
        "Sample review listing IDs not found "
        "in listings.csv:"
    )

    for listing_id in list(
        sorted(unmatched_review_listing_ids)
    )[:20]:
        print(f"- {listing_id}")
else:
    print(
        "All review listing IDs match "
        "listings.csv."
    )

All review listing IDs match listings.csv.


Raw date range inspection

In [59]:
parsed_review_dates = pd.to_datetime(
    summary_reviews_df["date"],
    errors="coerce"
)

print("Review Date Validation")
print("-" * 40)

print(
    f"Earliest review date: "
    f"{parsed_review_dates.min()}"
)

print(
    f"Latest review date: "
    f"{parsed_review_dates.max()}"
)

print(
    f"Invalid/unparseable dates: "
    f"{parsed_review_dates.isna().sum():,}"
)

Review Date Validation
----------------------------------------
Earliest review date: 2010-08-16 00:00:00
Latest review date: 2026-06-28 00:00:00
Invalid/unparseable dates: 0


Dataset-level summary

In [60]:
summary_reviews_dataset_summary = {
    "dataset_name": "summary_reviews",
    "file_name": DATA_FILES["summary_reviews"].name,
    "row_count": len(summary_reviews_df),
    "column_count": len(summary_reviews_df.columns),
    "duplicate_count": int(
        summary_reviews_df.duplicated().sum()
    ),
    "candidate_primary_keys": (
        summary_reviews_candidate_keys
    ),
    "composite_key_candidate": (
        ["listing_id", "date"]
        if composite_duplicate_count == 0
        else None
    ),
}

summary_reviews_dataset_summary

{'dataset_name': 'summary_reviews',
 'file_name': 'reviews.csv',
 'row_count': 545162,
 'column_count': 2,
 'duplicate_count': 22541,
 'candidate_primary_keys': [],
 'composite_key_candidate': None}

Save the profile

In [61]:
summary_reviews_profile_path = (
    OUTPUT_DIR / "summary_reviews_column_profile.csv"
)

summary_reviews_profile_df.to_csv(
    summary_reviews_profile_path,
    index=False
)

print(
    "Summary reviews profile saved successfully to:\n"
    f"{summary_reviews_profile_path}"
)

Summary reviews profile saved successfully to:
c:\Users\saths\Desktop\MyProjects\Expernetic Data Challenge\airbnb-data-challenge\outputs\data_quality\summary_reviews_column_profile.csv


### Findings and Interpretation

The `reviews.csv` dataset contains **545,162 rows and 2 columns**, representing summary-level review activity associated with Airbnb listings in Amsterdam.

The grain of the dataset is:

**One row represents one recorded review event for a listing on a particular date.**

However, because the dataset does not contain an individual review identifier, multiple reviews for the same listing on the same date cannot be uniquely distinguished from this file alone.

---

#### Key Findings

- Total rows: **545,162**
- Total columns: **2**
- Unique listing IDs: **9,432**
- Unique review dates: **5,327**
- Missing `listing_id` values: **0**
- Missing `date` values: **0**
- Exact repeated `(listing_id, date)` rows: **22,541**
- Duplicate percentage: **4.13%**
- Earliest review date: **2010-08-16**
- Latest review date: **2026-06-28**
- Invalid or unparseable dates: **0**

No single-column candidate primary key was identified.

The combination:

`(listing_id, date)`

is also not unique because 22,541 repeated combinations were found.

#### Review Activity by Listing

The dataset contains review records for **9,432 unique listings**.

Among listings with at least one recorded review:

- Minimum reviews per listing: **1**
- Maximum reviews per listing: **5,603**
- Median reviews per listing: **13**

The listing with the highest recorded review count has **5,603 review records**.

This matches the maximum `number_of_reviews` value previously observed in `listings.csv`, providing useful cross-dataset consistency evidence for at least that extreme value.

However, complete row-level consistency between the two files should be explicitly validated later before concluding that review counts match for every listing.


#### Important Duplicate Interpretation

The dataset contains **22,541 rows that are duplicates when considering only the available columns `listing_id` and `date`**.

However, these rows should not automatically be classified as erroneous duplicates.

A repeated combination such as:

`listing_id = 29051, date = 2016-03-10`

may legitimately represent multiple different guests submitting reviews for the same listing on the same day.

Because `reviews.csv` contains only:

- `listing_id`
- `date`

and does not contain an individual `review_id`, reviewer identifier, or review text, it is impossible to determine from this file alone whether repeated `(listing_id, date)` rows are:

- legitimate separate reviews submitted on the same date,
- true accidental duplicate records,
- or repeated records introduced during data collection.

Therefore, these 22,541 rows should be described as:

**Repeated listing-date combinations requiring cautious interpretation, rather than confirmed erroneous duplicates.**

They should not be automatically removed during cleaning without further evidence.

The more detailed `reviews.csv.gz` dataset may later provide additional identifiers that help determine whether these repeated combinations represent distinct review records.


#### Relationship with `listings.csv`

The relationship between `reviews.csv` and `listings.csv` was explicitly validated using:

`reviews.csv.listing_id`

and:

`listings.csv.id`

Validation results:

- Unique listing IDs in `reviews.csv`: **9,432**
- Unique listing IDs in `listings.csv`: **10,465**
- Review listing IDs not found in `listings.csv`: **0**
- Listings with no record in `reviews.csv`: **1,033**
- Matched unique review listing IDs: **9,432**
- Relationship coverage for review listing IDs: **100.00%**

Therefore, every listing identifier referenced by `reviews.csv` exists in `listings.csv`.

This provides strong evidence of referential consistency between the two datasets.

#### Important Cross-Dataset Missingness Finding

A particularly important relationship was identified between the review data and the missing values previously observed in `listings.csv`.

In `listings.csv`:

- `last_review` has **1,033 missing values**.
- `reviews_per_month` has **1,033 missing values**.

In `reviews.csv`:

- Exactly **1,033 listings have no review records**.

This exact numerical agreement strongly suggests that the missing values in `last_review` and `reviews_per_month` are structurally associated with listings that have no recorded review history.

This is an important data-engineering finding because it indicates that these missing values may not represent random data-quality failures.

Instead, they may have a valid business meaning:

**The listing has no recorded reviews from which a last review date or monthly review rate could be calculated.**

This relationship should still be validated explicitly at the listing-ID level before making a final cleaning or imputation decision.

For example, before replacing missing `reviews_per_month` values with zero, the project should verify that every affected listing genuinely has no review records.


#### Date Quality

The raw `date` column was initially loaded as a string.

A temporary datetime conversion was performed for validation without modifying the original raw column.

Results:

- Earliest review date: **2010-08-16**
- Latest review date: **2026-06-28**
- Invalid or unparseable dates: **0**

This indicates that all 545,162 review date values were successfully parsed as valid dates.

During the later cleaning and standardization phase, the `date` field should be converted from string to a proper datetime type in the processed dataset.


#### Review History Coverage

The review records span approximately from:

**August 2010 to June 2026**

This provides substantial historical coverage for review activity.

However, the dataset should not automatically be treated as a complete representation of every historical booking or guest stay because:

- not every guest necessarily leaves a review,
- historical data coverage may be affected by scraping limitations,
- listing activity may begin or end at different times,
- removed listings may not be represented in the current listings snapshot.

Therefore, review activity should be interpreted as an indicator of observed engagement rather than an exact record of bookings.

#### Data Quality Interpretation

The dataset has several strong quality characteristics:

- No missing listing identifiers.
- No missing review dates.
- All date values are parseable.
- Every review listing ID matches a listing in `listings.csv`.
- Review-to-listing relationship coverage is 100.00%.

The main issue requiring careful interpretation is the presence of **22,541 repeated `(listing_id, date)` combinations**.

Because multiple genuine reviews can occur for the same listing on the same day and no individual review identifier exists in this summary dataset, these records should not be automatically removed as duplicate errors.

Overall, the dataset is structurally strong for:

- review-frequency analysis,
- listing-level review counts,
- temporal review trends,
- first and last review date calculations,
- cross-dataset validation,
- listing-level enrichment.

Further validation with the detailed `reviews.csv.gz` dataset should help clarify review-level uniqueness and provide a richer understanding of the review entity.

### Candidate Keys and Relationships

#### Single-Column Candidate Primary Key

No single-column candidate primary key was identified in `reviews.csv`.

The available columns are:

- `listing_id`
- `date`

Neither column is unique across all 545,162 rows:

- `listing_id` contains 9,432 unique values.
- `date` contains 5,327 unique values.

Therefore, neither field can uniquely identify an individual review record by itself.

---

#### Composite-Key Validation

The combination:

`(listing_id, date)`

was tested as a possible composite key.

Validation result:

- Duplicate `(listing_id, date)` combinations: **22,541**

Therefore, the pair:

`(listing_id, date)`

is not unique and cannot be treated as a candidate composite primary key.

This means that the current summary review dataset does not contain enough information to uniquely identify every individual review row.

---

#### Important Interpretation of Repeated Listing-Date Combinations

The presence of repeated `(listing_id, date)` combinations does not automatically mean that these rows are erroneous duplicates.

Multiple different guests may submit reviews for the same listing on the same date.

Because `reviews.csv` contains only:

- `listing_id`
- `date`

and does not contain:

- an individual review ID,
- a reviewer ID,
- reviewer name,
- review text,

it is impossible to distinguish separate reviews occurring on the same listing and date using this summary dataset alone.

Therefore, the repeated rows should be described as:

**Repeated listing-date combinations requiring cautious interpretation, rather than confirmed duplicate errors.**

They should not be automatically deleted without additional evidence.

The more detailed `reviews.csv.gz` dataset may later provide individual review identifiers that allow proper review-level uniqueness validation.

---

#### Foreign-Key Relationship with `listings.csv`

The `listing_id` column in `reviews.csv` was explicitly validated against the `id` column in `listings.csv`.

The logical relationship is:

```text
listings.csv
    id
    │
    │ one-to-many relationship
    ▼
reviews.csv
    listing_id
````

Validation results:

* Unique listing IDs in `reviews.csv`: **9,432**
* Unique listing IDs in `listings.csv`: **10,465**
* Review listing IDs not found in `listings.csv`: **0**
* Listings with no review records: **1,033**
* Matched unique review listing IDs: **9,432**
* Relationship coverage: **100.00%**

Therefore, every listing identifier referenced in `reviews.csv` exists in `listings.csv`.

This supports treating:

* `listings.id` as the parent listing identifier.
* `reviews.listing_id` as a logical foreign key referencing the listings dataset.

Because these are CSV files rather than relational database tables with enforced constraints, this is described as a logical foreign-key relationship rather than a formally enforced database foreign key.

---

#### Relationship Cardinality

The observed relationship between listings and reviews is:

**One listing can have zero, one, or many review records.**

The relationship can be represented as:

```text
Listing
   │
   │ 1
   │
   │
   │ 0..many
   ▼
Reviews
```

This is supported by the data because:

* `listings.csv` contains 10,465 listings.
* `reviews.csv` contains review records for 9,432 listings.
* 1,033 listings have no review records.
* Some listings contain thousands of review records.

Therefore, the relationship is best described as:

**One-to-many from Listing to Reviews, with some listings having zero reviews.**

---

#### Cross-Dataset Missingness Relationship

An important relationship was identified between `reviews.csv` and review-related missing values in `listings.csv`.

In `listings.csv`:

* `last_review` has 1,033 missing values.
* `reviews_per_month` has 1,033 missing values.

In `reviews.csv`:

* 1,033 listings have no review records.

This exact numerical agreement strongly suggests that the missing review-related fields in `listings.csv` may correspond to listings without review history.

The likely relationship is:

```text
Listing has no rows in reviews.csv
                │
                ▼
last_review = missing
reviews_per_month = missing
```

This is strong evidence of structural missingness rather than random missingness.

However, the relationship should still be validated explicitly at the listing-ID level before using it as the basis for cleaning decisions.

---

#### Potential Relationship with `reviews.csv.gz`

The detailed reviews dataset is expected to provide richer review-level information.

A likely relationship is:

```text
reviews.csv
    listing_id + date
          │
          ▼
reviews.csv.gz
    listing_id + date + review-level fields
```

The detailed file may include additional fields such as:

* review ID,
* reviewer ID,
* reviewer name,
* review comments.

These fields may help explain why multiple records share the same `listing_id` and `date` in the summary review dataset.

This relationship will be validated later when `reviews.csv.gz` is profiled using DuckDB or another memory-efficient method.

---

#### Relationship Validation Status

At this stage, the following relationship has been validated:

* `listings.id` → `reviews.listing_id`

  * Status: **Validated**
  * Coverage: **100.00%**
  * Relationship type: **One-to-many**

The following relationships are still pending validation:

* `listings.id` → `reviews.csv.gz.listing_id`
* `reviews.csv` → `reviews.csv.gz`
* Review-level uniqueness using any detailed review identifier available in `reviews.csv.gz`

---

#### Key Engineering Interpretation

The `reviews.csv` dataset is a child dataset of `listings.csv`.

Its main role is to record historical review activity for listings.

The `listing_id` field links each review record back to the corresponding listing, while the `date` field provides the temporal dimension of review activity.

The dataset is suitable for:

* review counts per listing,
* first review date,
* last review date,
* monthly review activity,
* temporal review trends,
* cross-dataset consistency checks,
* listing-level enrichment.

However, the absence of an individual review identifier means that this summary file alone cannot uniquely identify every review event.


### Business-Domain Meaning

The `reviews.csv` dataset represents historical review activity associated with Airbnb listings in Amsterdam.

Each row contains:

- `listing_id`
- `date`

The dataset grain is:

**One row represents one observed review event for a listing on a specific date.**

However, because the dataset does not include a unique review identifier, multiple rows with the same `listing_id` and `date` cannot be uniquely distinguished from this file alone.

---

#### Main Business Entity Represented

The primary business entity represented in this dataset is the **Review**.

A review is a record of guest feedback associated with an Airbnb listing.

In this summary dataset, only two attributes are available:

- the listing being reviewed,
- the date on which the review was recorded.

The dataset does not contain:

- review text,
- reviewer identity,
- rating score,
- review ID,
- sentiment information.

Therefore, its main value is in measuring review activity over time rather than analyzing review content.

---

#### Relationship to Listings

Each review record is associated with a listing through:

`listing_id`

This creates a one-to-many relationship:

```text
Listing
   │
   │ one listing
   ▼
Reviews
   many review records
````

The relationship was explicitly validated:

* Unique listing IDs in `reviews.csv`: **9,432**
* Unique listing IDs in `listings.csv`: **10,465**
* Unmatched review listing IDs: **0**
* Relationship coverage: **100.00%**

This means every listing referenced in `reviews.csv` exists in `listings.csv`.

---

#### Business Questions Supported by This Dataset

The dataset can support questions such as:

* Which listings receive the most reviews?
* Which listings receive very few reviews?
* How many listings have no review history?
* What is the earliest observed review date?
* What is the latest observed review date?
* How does review activity change over time?
* Which listings show the strongest review activity?
* How long has a listing been receiving reviews?
* Which listings have recently received reviews?
* How can review history enrich the listing-level analytical dataset?

---

#### Review Count as an Activity Indicator

The number of review rows associated with each listing can be used as an indicator of listing activity and guest engagement.

For the current dataset:

* Listings with at least one review: **9,432**
* Minimum reviews per reviewed listing: **1**
* Maximum reviews per reviewed listing: **5,603**
* Median reviews per reviewed listing: **13**

A listing with many reviews may indicate stronger historical guest activity.

However, review count should not be treated as an exact booking count because:

* not every guest leaves a review,
* review behavior may vary over time,
* platform policies may influence review activity.

Therefore, review count is best treated as an **activity proxy** rather than a direct measure of bookings.

---

#### Time-Based Business Value

The review dates span from:

* Earliest review date: **2010-08-16**
* Latest review date: **2026-06-28**

This allows the project to study temporal review activity over a long historical period.

Potential derived features include:

* first review date,
* last review date,
* review count,
* reviews per year,
* review frequency,
* days since last review,
* listing review lifespan.

These features can later be aggregated to one row per listing and joined into the enriched listing master dataset.

---

#### Importance for Listing-Level Enrichment

The raw `reviews.csv` dataset contains 545,162 rows.

For downstream analysis, it should not be joined directly to the listings table at full row level because this would expand the listing dataset into a much larger many-row table.

Instead, review data should first be aggregated to one row per listing.

For example:

```text
listing_id
review_count
first_review_date
last_review_date
review_lifespan_days
recent_review_count
review_frequency
```

Then the compact aggregated result can be joined to the listing master dataset.

This supports the project's 8 GB RAM optimization strategy and avoids unnecessary memory expansion.

---

#### Cross-Dataset Business Insight

An important consistency pattern was identified between `reviews.csv` and `listings.csv`.

Exactly **1,033 listings** have no review rows in `reviews.csv`.

The same number of listings in `listings.csv` have missing values for:

* `last_review`
* `reviews_per_month`

This suggests that these missing review-related fields may have a clear business meaning:

**The listing has no recorded review history.**

This makes the review dataset useful not only for analysis but also for validating and interpreting missing values in other datasets.

---

#### Analytical Importance

The `reviews.csv` dataset is valuable for:

* measuring listing review activity,
* identifying reviewed and unreviewed listings,
* calculating first and last review dates,
* studying temporal trends,
* creating listing-level review features,
* validating review-related fields in `listings.csv`,
* enriching the final analytical dataset.

Its strongest role in this project is as a historical activity source that can be transformed into compact listing-level features for later EDA, statistical analysis, and business interpretation.

### Dataset Limitations

The `reviews.csv` dataset is valuable for understanding historical review activity, but several important limitations must be considered before using it for cleaning, enrichment, temporal analysis, or business interpretation.

---

#### 1. No Unique Review Identifier

The dataset contains only two columns:

- `listing_id`
- `date`

There is no individual `review_id`.

As a result, the dataset cannot uniquely identify every review record.

This is confirmed by the fact that:

- No single-column primary key was found.
- The composite combination `(listing_id, date)` is also not unique.

Therefore, individual review-level uniqueness cannot be established from this summary dataset alone.

---

#### 2. Repeated Listing-Date Combinations

The dataset contains:

- **22,541 repeated rows**
- **4.13% of total records**

These repetitions occur when the same `listing_id` and `date` combination appears more than once.

However, these rows should not automatically be treated as erroneous duplicates.

Possible explanations include:

- Multiple guests reviewing the same listing on the same day.
- Legitimate separate review events sharing the same date.
- Duplicate records introduced during scraping or source preparation.

Because no unique review identifier exists in this file, these possibilities cannot be distinguished with certainty.

Therefore:

**The 22,541 repeated listing-date combinations should be preserved during familiarization and investigated further using `reviews.csv.gz`.**

---

#### 3. Summary-Level Structure

The dataset provides only:

- listing identifier,
- review date.

It does not include richer review-level attributes such as:

- review ID,
- reviewer ID,
- reviewer name,
- review text,
- review rating,
- sentiment,
- language.

This limits the types of analysis that can be performed directly from this file.

For example, it cannot support:

- sentiment analysis,
- reviewer-level behaviour analysis,
- review-content analysis,
- unique review identification.

---

#### 4. Review Count Is Not Equivalent to Booking Count

The number of review records associated with a listing should not be interpreted as the exact number of bookings.

Not every guest necessarily leaves a review.

Therefore:

`review_count`

should be treated as an **activity proxy** rather than an exact booking measure.

A listing with more reviews may have greater historical guest activity, but the dataset alone cannot prove the exact number of stays or bookings.

---

#### 5. Listings Without Reviews Are Absent From This File

The dataset contains review records for:

**9,432 unique listings**

However, `listings.csv` contains:

**10,465 listings**

Therefore:

**1,033 listings have no review record in `reviews.csv`.**

These listings are completely absent from the review dataset rather than appearing with a zero review count.

This is important when aggregating review data because an inner join would remove these 1,033 listings.

To preserve all listings, downstream enrichment should normally use the listings dataset as the parent table and apply a left join with aggregated review features.

---

#### 6. Date Column Stored as String

The `date` column was initially loaded as:

`str`

rather than as a datetime type.

Although all 545,162 date values were successfully parsed during validation, the raw field is not yet in its ideal analytical format.

During the cleaning phase, it should be converted to a proper datetime type for:

- date filtering,
- monthly aggregation,
- yearly aggregation,
- calculating first and last review dates,
- measuring review lifespan,
- calculating recency.

The raw dataset itself should remain unchanged.

---

#### 7. Historical Coverage Does Not Guarantee Completeness

The observed review period spans:

- Earliest review date: **2010-08-16**
- Latest review date: **2026-06-28**

Although this provides long-term historical coverage, the dataset should not automatically be assumed to represent every review ever created.

Potential reasons include:

- removed listings,
- deleted reviews,
- changes in scraping coverage,
- differences in source availability,
- platform changes over time.

Therefore, the dataset should be interpreted as the observed review history available in the current source rather than a guaranteed complete historical record.

---

#### 8. Current Listings Snapshot vs Historical Reviews

The review dataset contains historical activity, while `listings.csv` represents a current listing snapshot.

This creates an important analytical limitation.

A listing may have historical reviews even though its current characteristics may have changed over time, such as:

- price,
- host,
- room configuration,
- availability,
- listing description.

Therefore, current listing attributes should not automatically be assumed to represent the exact conditions that existed when older reviews were created.

---

#### 9. Extreme Review Counts Require Careful Interpretation

The review-count distribution is highly uneven.

Observed values include:

- Minimum reviews per reviewed listing: **1**
- Median reviews per reviewed listing: **13**
- Maximum reviews per listing: **5,603**

Listings with very high review counts may represent:

- highly popular listings,
- long-running listings,
- professionally managed properties,
- listings with unusually high turnover,
- possible data anomalies.

These extreme values should be investigated rather than automatically removed.

---

#### 10. No Direct Evidence of Review Quality or Sentiment

Because `reviews.csv` contains no review text or rating information, it cannot determine whether a review was:

- positive,
- negative,
- neutral,
- detailed,
- short,
- recent enough to reflect current listing quality.

Therefore, a high review count should not automatically be interpreted as high customer satisfaction.

Review volume and review quality are different concepts.

---

#### 11. Repeated Dates May Affect Simple Temporal Counts

Because multiple reviews can occur on the same date for the same listing, counting unique dates is not equivalent to counting reviews.

For example:

`1 unique review date`

may still represent multiple review records.

Therefore, later analysis must clearly distinguish between:

- total review count,
- unique review dates,
- reviews per month,
- reviews per year.

---

#### 12. Relationship Is Logical, Not Formally Enforced

The relationship:

`listings.id → reviews.listing_id`

achieved 100.00% observed coverage.

However, these are CSV files rather than relational database tables.

Therefore:

- foreign-key constraints are not formally enforced,
- referential integrity must be validated manually,
- future dataset versions may contain different results.

The relationship should therefore be described as a validated logical foreign-key relationship.

---

#### 13. Review-Based Demand Measures Are Proxies

The dataset can support calculated measures such as:

- review frequency,
- recent review count,
- review recency,
- first review date,
- last review date.

However, these are still review-based indicators.

They should not be interpreted as direct measures of:

- occupancy,
- bookings,
- revenue,
- profitability.

Any business interpretation must clearly use careful wording such as:

**review activity proxy**

rather than claiming actual booking demand.

---

#### Overall Limitation Summary

The most important limitations of `reviews.csv` are:

- absence of a unique review identifier,
- 22,541 repeated listing-date combinations that cannot be conclusively classified as errors,
- lack of review text and reviewer information,
- inability to equate reviews directly with bookings,
- absence of listings with zero reviews,
- string-based raw date storage,
- possible incompleteness of historical coverage,
- mismatch between historical review events and current listing attributes.

Despite these limitations, the dataset remains highly useful for:

- review-frequency analysis,
- listing-level aggregation,
- temporal trend analysis,
- review recency calculations,
- cross-dataset consistency validation,
- enrichment of the final listing master dataset.

### Data Quality Assessment

The `reviews.csv` dataset is structurally simple but highly valuable for historical review activity analysis and cross-dataset validation.

It contains:

- **545,162 rows**
- **2 columns**
- `listing_id`
- `date`

The dataset is complete with respect to its available fields, but several important limitations and interpretation issues remain.

---

#### Positive Data Quality Characteristics

The dataset has several strong quality characteristics:

- Missing `listing_id` values: **0**
- Missing `date` values: **0**
- Invalid or unparseable dates: **0**
- Unique listing IDs: **9,432**
- All review listing IDs exist in `listings.csv`
- Review-to-listing relationship coverage: **100.00%**

These results indicate that the available review records are structurally complete and maintain strong referential consistency with the listings dataset.

---

#### Listing Identifier Quality

The `listing_id` field contains:

- Total review rows: **545,162**
- Unique listing IDs: **9,432**
- Missing listing IDs: **0**

Every `listing_id` found in `reviews.csv` was matched successfully to an `id` in `listings.csv`.

Validation results:

- Unique listing IDs in `reviews.csv`: **9,432**
- Unique listing IDs in `listings.csv`: **10,465**
- Unmatched review listing IDs: **0**
- Listings with no review record: **1,033**
- Relationship coverage: **100.00%**

This is a strong positive data-quality result and supports treating:

`reviews.listing_id`

as a logical foreign-key-like reference to:

`listings.id`

---

#### Date Quality

The raw `date` column was initially loaded as a string.

A temporary datetime conversion was performed for validation without changing the original raw dataset.

Results:

- Earliest review date: **2010-08-16**
- Latest review date: **2026-06-28**
- Invalid or unparseable dates: **0**

All 545,162 date values were successfully parsed.

This indicates strong date-format consistency.

During the later cleaning phase, the field should be converted to a proper datetime type in the processed dataset.

---

#### Duplicate and Repeated-Row Assessment

The dataset contains:

- Exact repeated rows: **22,541**
- Duplicate percentage: **4.13%**

Because the dataset contains only:

- `listing_id`
- `date`

the repeated rows are identical when considering all available columns.

However, these rows cannot automatically be classified as erroneous duplicates.

Possible explanations include:

- multiple guests reviewing the same listing on the same day,
- distinct review events sharing the same date,
- genuine duplicate records introduced during scraping or processing.

Since no unique review identifier exists in this summary dataset, these possibilities cannot be distinguished with certainty.

Therefore, the repeated rows should be classified as:

**Potential duplicate or repeated listing-date records requiring further investigation.**

They should not be automatically removed without additional evidence from the detailed `reviews.csv.gz` dataset.

---

#### Primary-Key Assessment

No valid single-column primary key was identified.

The fields have the following uniqueness:

- `listing_id`: 9,432 unique values
- `date`: 5,327 unique values

Neither field uniquely identifies all 545,162 rows.

The composite combination:

`(listing_id, date)`

was also tested.

Results:

- Duplicate combinations: **22,541**

Therefore, `(listing_id, date)` is not a valid candidate composite primary key.

The dataset does not currently contain enough information to uniquely identify each individual review record.

---

#### Review Count Distribution

Review activity varies substantially across listings.

Observed results:

- Listings with at least one review: **9,432**
- Minimum reviews per reviewed listing: **1**
- Median reviews per reviewed listing: **13**
- Maximum reviews per reviewed listing: **5,603**

This indicates a highly uneven distribution of review activity.

Some listings have only a single review, while others have thousands.

These extreme values should not automatically be considered errors because they may represent long-running, highly active, or professionally managed listings.

However, extreme review counts should be investigated during later EDA and validation.

---

#### Cross-Dataset Consistency

A strong consistency pattern was identified between `reviews.csv` and `listings.csv`.

In `reviews.csv`:

- **1,033 listings have no review records**

In `listings.csv`:

- `last_review` has **1,033 missing values**
- `reviews_per_month` has **1,033 missing values**

This exact numerical match strongly suggests that the missing review-related fields in `listings.csv` correspond to listings without recorded review history.

This is an important data-quality finding because it indicates likely structural missingness rather than random missingness.

However, the relationship should still be validated at the individual listing-ID level before making a final imputation decision.

---

#### Main Data Quality Concerns

The primary concerns identified are:

1. **No unique review identifier**

   Individual review records cannot be uniquely identified from this summary dataset alone.

2. **22,541 repeated listing-date rows**

   These records may represent legitimate same-day reviews or actual duplicates.

3. **Raw date stored as string**

   The `date` field should be converted to datetime during cleaning.

4. **Listings without reviews are absent**

   The 1,033 listings with no reviews do not appear in this dataset, so downstream enrichment should use a left join from the listings table.

5. **Review count is only an activity proxy**

   Review volume should not be interpreted as exact booking demand.

6. **Historical reviews may not reflect current listing conditions**

   Current listing attributes such as price, host, or availability may differ from the conditions that existed when older reviews were created.

---

#### Data Quality Strengths

The strongest characteristics of `reviews.csv` are:

- complete listing identifiers,
- complete dates,
- fully parseable date values,
- 100% referential coverage to `listings.csv`,
- long historical coverage,
- useful listing-level review activity information.

These characteristics make the dataset suitable for:

- review count aggregation,
- first and last review date calculations,
- review recency features,
- temporal trend analysis,
- listing-level enrichment,
- validation of review-related fields in `listings.csv`.

---

#### Overall Data Quality Assessment

Overall, `reviews.csv` is a structurally strong dataset for review-activity analysis and listing enrichment.

Its most important strengths are:

- no missing values,
- fully valid date values,
- complete relationship coverage with `listings.csv`,
- long historical coverage.

Its main limitations are:

- no unique review identifier,
- 22,541 repeated listing-date rows that require cautious interpretation,
- absence of zero-review listings,
- inability to distinguish review count from actual booking activity.

**Overall assessment: Suitable for downstream aggregation and temporal analysis after careful handling of repeated listing-date records, conversion of the date field to datetime, and documented use of review metrics as activity proxies rather than exact booking measures.**

## 4. Detailed Listings Dataset Familiarization

The `listings.csv.gz` file contains detailed listing-level information for Airbnb properties in Amsterdam.

Compared with the summary `listings.csv` dataset, this file is expected to contain a much richer set of attributes related to:

- Listing identity
- Host information
- Listing descriptions
- Property characteristics
- Room and accommodation details
- Pricing
- Availability
- Reviews
- Geographic information
- Amenities
- Licensing and booking rules

This dataset will be inspected for:

- Dataset shape
- Column names
- Data types
- Missing values
- Unique-value counts
- Minimum and maximum values
- Sample values
- Duplicate rows
- Candidate primary keys
- Potential foreign-key relationships
- Relationship with `listings.csv`
- Relationship with `neighbourhoods.csv`
- Business-domain meaning
- Dataset limitations

Because this project is designed for an 8 GB RAM laptop, the detailed listings file will be loaded and processed independently rather than keeping all seven raw datasets in memory at the same time.

In [62]:
detailed_listings_df = pd.read_csv(
    DATA_FILES["detailed_listings"],
    compression="gzip",
    low_memory=False
)

print("listings.csv.gz loaded successfully.")
print(f"Rows: {detailed_listings_df.shape[0]:,}")
print(f"Columns: {detailed_listings_df.shape[1]}")

listings.csv.gz loaded successfully.
Rows: 10,369
Columns: 90


In [63]:
display(detailed_listings_df.head())

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,host_url,host_profile_id,host_profile_url,host_name,host_since,hosts_time_as_user_years,hosts_time_as_user_months,hosts_time_as_host_years,hosts_time_as_host_months,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_thumbnail_url,host_picture_url,host_neighbourhood,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms,beds,amenities,price,price_quote_checkin_date,price_quote_checkout_date,price_quote_total_price,price_quote_price_per_night,price_quote_raw,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,calendar_updated,has_availability,availability_30,availability_60,availability_90,availability_365,calendar_last_scraped,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,28871,https://www.airbnb.com/rooms/28871,20260615212022,2026-06-16,previous scrape,Comfortable double room,Basic bedroom in the center of Amsterdam.,NaN,https://a0.muscache.com/pictures/160889/362340f7_original.jpg,124245,https://www.airbnb.com/users/show/124245,1.462510e+18,https://www.airbnb.com/users/profile/1462510208428038971,Edwin,NaN,16.0,1.0,15.0,5.0,"Amsterdam, The Netherlands",Hi,NaN,NaN,NaN,t,NaN,https://a0.muscache.com/im/pictures/user/9986bbdb-632f-42b5-a866-8e3307184977.jpg?aki_policy=pro...,NaN,2.0,NaN,NaN,t,t,NaN,Centrum-West,NaN,52.367750,4.89092,Private room in rental unit,Private room,2,NaN,1 shared bath,NaN,NaN,"[""Heating"", ""Fire extinguisher"", ""Hot water"", ""Refrigerator"", ""Bed linens"", ""Private entrance"", ...",$94.00,2027-05-29,2027-05-31,188.0,94.00,"{""quote"": {""taxes"": null, ""currency"": ""EUR"", ""date_match"": null, ""service_fee"": null, ""total_pri...",1.0,730.0,1.0,2.0,730.0,730.0,2.0,730.0,NaN,t,1,1,1,11,2026-06-16,799,88,5,2,95,255,23970.0,2010-08-22,2026-06-01,4.86,4.89,4.84,4.94,4.94,4.93,4.83,0363 607B EA74 0BD8 2F6F,NaN,2,0,2,0,4.15
1,29051,https://www.airbnb.com/rooms/29051,20260615212022,2026-06-16,previous scrape,Comfortable single / double room,This room can also be rented as a single or a small double<br /><br />Also a cat lives here,NaN,https://a0.muscache.com/pictures/162009/bd6be2f8_original.jpg,124245,https://www.airbnb.com/users/show/124245,1.462510e+18,https://www.airbnb.com/users/profile/1462510208428038971,Edwin,NaN,16.0,1.0,15.0,5.0,"Amsterdam, The Netherlands",Hi,NaN,NaN,NaN,t,NaN,https://a0.muscache.com/im/pictures/user/9986bbdb-632f-42b5-a866-8e3307184977.jpg?aki_policy=pro...,NaN,2.0,NaN,NaN,t,t,NaN,Centrum-Oost,NaN,52.365840,4.89111,Private room in condo,Private room,2,NaN,1 shared bath,NaN,NaN,"[""Heating"", ""Fire extinguisher"", ""Hot water"", ""Refrigerator"", ""Shower gel"", ""Bed linens"", ""Priva...",NaN,NaN,NaN,NaN,NaN,NaN,1.0,730.0,1.0,2.0,730.0,730.0,2.0,730.0,NaN,t,0,2,2,2,2026-06-16,906,82,6,2,85,255,NaN,2011-03-16,2026-06-01,4.82,4.88,4.83,4.93,4.93,4.88,4.79,0363 607B EA74 0BD8 2F6F,NaN,2,0,2,0,4.88
2,44129,https://www.airbnb.com/rooms/44129,20260615212022,2026-06-24,previous scrape,Luxury design with canal view,"Welcome to my little gem<br /><br />Cozy, bright and romantic one bedroom apartment on the first

In [64]:
print("Dataset Shape")
print("-" * 40)
print(f"Number of rows: {detailed_listings_df.shape[0]:,}")
print(f"Number of columns: {detailed_listings_df.shape[1]}")

Dataset Shape
----------------------------------------
Number of rows: 10,369
Number of columns: 90


In [65]:
print("Column Names")
print("-" * 40)

for index, column in enumerate(
    detailed_listings_df.columns,
    start=1
):
    print(f"{index}. {column}")

Column Names
----------------------------------------
1. id
2. listing_url
3. scrape_id
4. last_scraped
5. source
6. name
7. description
8. neighborhood_overview
9. picture_url
10. host_id
11. host_url
12. host_profile_id
13. host_profile_url
14. host_name
15. host_since
16. hosts_time_as_user_years
17. hosts_time_as_user_months
18. hosts_time_as_host_years
19. hosts_time_as_host_months
20. host_location
21. host_about
22. host_response_time
23. host_response_rate
24. host_acceptance_rate
25. host_is_superhost
26. host_thumbnail_url
27. host_picture_url
28. host_neighbourhood
29. host_listings_count
30. host_total_listings_count
31. host_verifications
32. host_has_profile_pic
33. host_identity_verified
34. neighbourhood
35. neighbourhood_cleansed
36. neighbourhood_group_cleansed
37. latitude
38. longitude
39. property_type
40. room_type
41. accommodates
42. bathrooms
43. bathrooms_text
44. bedrooms
45. beds
46. amenities
47. price
48. price_quote_checkin_date
49. price_quote_checkout_d

Data types

In [66]:
detailed_listings_dtypes = pd.DataFrame({
    "column_name": detailed_listings_df.columns,
    "data_type": detailed_listings_df.dtypes.astype(str).values
})

display(detailed_listings_dtypes)

,column_name,data_type
0,id,int64
1,listing_url,str
2,scrape_id,int64
3,last_scraped,str
4,source,str
5,name,str
6,description,str
7,neighborhood_overview,float64
8,picture_url,str
9,host_id,int64


Missing-value analysis

In [67]:
detailed_listings_missing = pd.DataFrame({
    "column_name": detailed_listings_df.columns,
    "missing_count": detailed_listings_df.isnull().sum().values,
    "missing_percentage": (
        detailed_listings_df.isnull().mean() * 100
    ).round(2).values
})

display(
    detailed_listings_missing
    .sort_values(
        by="missing_percentage",
        ascending=False
    )
    .reset_index(drop=True)
)

,column_name,missing_count,missing_percentage
0,neighborhood_overview,10369,100.00
1,host_since,10369,100.00
2,host_response_time,10369,100.00
3,host_thumbnail_url,10369,100.00
4,host_acceptance_rate,10369,100.00
5,host_response_rate,10369,100.00
6,host_verifications,10369,100.00
7,neighbourhood,10369,100.00
8,host_total_listings_count,10369,100.00
9,host_neighbourhood,10369,100.00


In [68]:
columns_with_missing = (
    detailed_listings_missing[
        detailed_listings_missing["missing_count"] > 0
    ]
    .sort_values(
        by="missing_percentage",
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    f"Columns with missing values: "
    f"{len(columns_with_missing)} out of "
    f"{detailed_listings_df.shape[1]}"
)

display(columns_with_missing)

Columns with missing values: 57 out of 90


,column_name,missing_count,missing_percentage
0,neighborhood_overview,10369,100.00
1,host_since,10369,100.00
2,host_total_listings_count,10369,100.00
3,neighbourhood,10369,100.00
4,host_verifications,10369,100.00
5,host_response_rate,10369,100.00
6,host_acceptance_rate,10369,100.00
7,host_thumbnail_url,10369,100.00
8,host_response_time,10369,100.00
9,host_neighbourhood,10369,100.00


Unique-value counts

In [69]:
detailed_listings_unique = pd.DataFrame({
    "column_name": detailed_listings_df.columns,
    "unique_count": [
        detailed_listings_df[column].nunique(dropna=True)
        for column in detailed_listings_df.columns
    ]
})

display(
    detailed_listings_unique
    .sort_values(
        by="unique_count",
        ascending=False
    )
    .reset_index(drop=True)
)

,column_name,unique_count
0,id,10369
1,listing_url,10369
2,picture_url,10315
3,name,10076
4,amenities,10009
5,description,9640
6,host_url,9104
7,host_id,9104
8,host_profile_url,9076
9,host_profile_id,9067


Combined column profile

In [70]:
detailed_listings_profile = []

for column in detailed_listings_df.columns:
    series = detailed_listings_df[column]
    non_null_values = series.dropna()

    # Calculate min/max only for numeric or datetime columns.
    if (
        pd.api.types.is_numeric_dtype(series)
        or pd.api.types.is_datetime64_any_dtype(series)
    ):
        minimum_value = (
            non_null_values.min()
            if not non_null_values.empty
            else None
        )

        maximum_value = (
            non_null_values.max()
            if not non_null_values.empty
            else None
        )
    else:
        minimum_value = None
        maximum_value = None

    # Keep only short samples so large descriptions,
    # URLs, amenities, and JSON text do not flood output.
    sample_values = (
        non_null_values
        .astype(str)
        .drop_duplicates()
        .head(3)
        .apply(
            lambda value: (
                value[:100] + "..."
                if len(value) > 100
                else value
            )
        )
        .tolist()
    )

    detailed_listings_profile.append({
        "column_name": column,
        "data_type": str(series.dtype),
        "missing_count": int(series.isnull().sum()),
        "missing_percentage": round(
            series.isnull().mean() * 100,
            2
        ),
        "unique_count": int(
            series.nunique(dropna=True)
        ),
        "minimum": minimum_value,
        "maximum": maximum_value,
        "sample_values": sample_values,
    })

detailed_listings_profile_df = pd.DataFrame(
    detailed_listings_profile
)

display(detailed_listings_profile_df)

,column_name,data_type,missing_count,missing_percentage,unique_count,minimum,maximum,sample_values
0,id,int64,0,0.00,10369,2.887100e+04,1.708159e+18,"[28871, 29051, 44129]"
1,listing_url,str,0,0.00,10369,NaN,NaN,"[https://www.airbnb.com/rooms/28871, https://www.airbnb.com/rooms/29051, https://www.airbnb.com/..."
2,scrape_id,int64,0,0.00,1,2.026062e+13,2.026062e+13,[20260615212022]
3,last_scraped,str,0,0.00,3,NaN,NaN,"[2026-06-16, 2026-06-24, 2026-06-15]"
4,source,str,0,0.00,2,NaN,NaN,"[previous scrape, city scrape]"
5,name,str,0,0.00,10076,NaN,NaN,"[Comfortable double room, Comfortable single / double room, Luxury design with canal view]"
6,description,str,450,4.34,9640,NaN,NaN,"[Basic bedroom in the center of Amsterdam., This room can also be rented as a single or a small ..."
7,neighborhood_overview,float64,10369,100.00,0,NaN,NaN,[]
8,picture_url,str,0,0.00,10315,NaN,NaN,"[https://a0.muscache.com/pictures/160889/362340f7_original.jpg, https://a0.muscache.com/pictures..."
9,host_id,int64,0,0.00,9104,3.592000e+03,1.705253e+18,"[124245, 187728, 194779]"


Duplicate-row analysis

In [71]:
detailed_listings_duplicate_count = (
    detailed_listings_df.duplicated().sum()
)

print("Duplicate Analysis")
print("-" * 40)
print(
    f"Total rows: "
    f"{len(detailed_listings_df):,}"
)
print(
    f"Duplicate rows: "
    f"{detailed_listings_duplicate_count:,}"
)

duplicate_percentage = (
    detailed_listings_duplicate_count
    / len(detailed_listings_df)
    * 100
)

print(
    f"Duplicate percentage: "
    f"{duplicate_percentage:.2f}%"
)

Duplicate Analysis
----------------------------------------
Total rows: 10,369
Duplicate rows: 0
Duplicate percentage: 0.00%


Candidate primary-key analysis

In [72]:
detailed_listings_candidate_keys = []

total_rows = len(detailed_listings_df)

for column in detailed_listings_df.columns:
    unique_count = detailed_listings_df[column].nunique(
        dropna=False
    )

    missing_count = (
        detailed_listings_df[column]
        .isnull()
        .sum()
    )

    is_unique = unique_count == total_rows
    has_no_missing_values = missing_count == 0

    if is_unique and has_no_missing_values:
        detailed_listings_candidate_keys.append(column)

print("Candidate Primary Key Analysis")
print("-" * 40)

if detailed_listings_candidate_keys:
    print(
        "Possible single-column primary key candidates:"
    )

    for column in detailed_listings_candidate_keys:
        print(f"- {column}")
else:
    print(
        "No single-column candidate primary key "
        "was identified."
    )

Candidate Primary Key Analysis
----------------------------------------
Possible single-column primary key candidates:
- id
- listing_url


Explicit listing ID validation

In [73]:
print("Detailed Listing ID Validation")
print("-" * 40)

print(
    f"Total rows: "
    f"{len(detailed_listings_df):,}"
)

print(
    f"Unique IDs: "
    f"{detailed_listings_df['id'].nunique(dropna=True):,}"
)

print(
    f"Missing IDs: "
    f"{detailed_listings_df['id'].isnull().sum():,}"
)

print(
    f"Duplicate IDs: "
    f"{detailed_listings_df['id'].duplicated().sum():,}"
)

Detailed Listing ID Validation
----------------------------------------
Total rows: 10,369
Unique IDs: 10,369
Missing IDs: 0
Duplicate IDs: 0


Compare detailed and summary listing IDs

In [74]:
detailed_listing_ids = set(
    detailed_listings_df["id"]
    .dropna()
    .unique()
)

summary_listing_ids = set(
    summary_listings_df["id"]
    .dropna()
    .unique()
)

common_listing_ids = (
    detailed_listing_ids
    & summary_listing_ids
)

detailed_only_listing_ids = (
    detailed_listing_ids
    - summary_listing_ids
)

summary_only_listing_ids = (
    summary_listing_ids
    - detailed_listing_ids
)

print("Detailed vs Summary Listings Relationship")
print("-" * 55)

print(
    f"Unique IDs in listings.csv.gz: "
    f"{len(detailed_listing_ids):,}"
)

print(
    f"Unique IDs in listings.csv: "
    f"{len(summary_listing_ids):,}"
)

print(
    f"IDs present in both datasets: "
    f"{len(common_listing_ids):,}"
)

print(
    f"IDs only in listings.csv.gz: "
    f"{len(detailed_only_listing_ids):,}"
)

print(
    f"IDs only in listings.csv: "
    f"{len(summary_only_listing_ids):,}"
)

Detailed vs Summary Listings Relationship
-------------------------------------------------------
Unique IDs in listings.csv.gz: 10,369
Unique IDs in listings.csv: 10,465
IDs present in both datasets: 10,369
IDs only in listings.csv.gz: 0
IDs only in listings.csv: 96


In [75]:
detailed_to_summary_coverage = (
    len(common_listing_ids)
    / len(detailed_listing_ids)
    * 100
    if detailed_listing_ids
    else 0
)

summary_to_detailed_coverage = (
    len(common_listing_ids)
    / len(summary_listing_ids)
    * 100
    if summary_listing_ids
    else 0
)

print("Listing ID Coverage")
print("-" * 40)

print(
    f"Detailed IDs found in summary dataset: "
    f"{detailed_to_summary_coverage:.2f}%"
)

print(
    f"Summary IDs found in detailed dataset: "
    f"{summary_to_detailed_coverage:.2f}%"
)

Listing ID Coverage
----------------------------------------
Detailed IDs found in summary dataset: 100.00%
Summary IDs found in detailed dataset: 99.08%


In [76]:
if detailed_only_listing_ids:
    print(
        "\nSample IDs present only in listings.csv.gz:"
    )

    print(
        sorted(detailed_only_listing_ids)[:10]
    )

if summary_only_listing_ids:
    print(
        "\nSample IDs present only in listings.csv:"
    )

    print(
        sorted(summary_only_listing_ids)[:10]
    )


Sample IDs present only in listings.csv:
[np.int64(1051276165819384554), np.int64(1080400371674885906), np.int64(1080400391115118344), np.int64(1080404233704746629), np.int64(1080404315599095169), np.int64(1095467564153718913), np.int64(1095467732227004425), np.int64(1095467839679422773), np.int64(1131589609552213167), np.int64(1132334737447631275)]


Neighbourhood field assessment

In [77]:
neighbourhood_columns = [
    "neighbourhood",
    "neighbourhood_cleansed",
    "neighbourhood_group_cleansed"
]

neighbourhood_field_summary = []

for column in neighbourhood_columns:
    if column in detailed_listings_df.columns:
        neighbourhood_field_summary.append({
            "column_name": column,
            "missing_count": int(
                detailed_listings_df[column]
                .isnull()
                .sum()
            ),
            "missing_percentage": round(
                detailed_listings_df[column]
                .isnull()
                .mean()
                * 100,
                2
            ),
            "unique_count": int(
                detailed_listings_df[column]
                .nunique(dropna=True)
            ),
        })

neighbourhood_field_summary_df = pd.DataFrame(
    neighbourhood_field_summary
)

display(neighbourhood_field_summary_df)

,column_name,missing_count,missing_percentage,unique_count
0,neighbourhood,10369,100.0,0
1,neighbourhood_cleansed,0,0.0,22
2,neighbourhood_group_cleansed,10369,100.0,0


Validate neighbourhood relationship

In [78]:
detailed_neighbourhoods = set(
    detailed_listings_df["neighbourhood_cleansed"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

reference_neighbourhoods = set(
    neighbourhoods_df["neighbourhood"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

unmatched_detailed_neighbourhoods = (
    detailed_neighbourhoods
    - reference_neighbourhoods
)

unused_reference_neighbourhoods_detailed = (
    reference_neighbourhoods
    - detailed_neighbourhoods
)

print("Detailed Listings Neighbourhood Validation")
print("-" * 55)

print(
    f"Unique neighbourhoods in listings.csv.gz: "
    f"{len(detailed_neighbourhoods)}"
)

print(
    f"Unique neighbourhoods in neighbourhoods.csv: "
    f"{len(reference_neighbourhoods)}"
)

print(
    f"Unmatched detailed neighbourhoods: "
    f"{len(unmatched_detailed_neighbourhoods)}"
)

print(
    f"Unused reference neighbourhoods: "
    f"{len(unused_reference_neighbourhoods_detailed)}"
)

Detailed Listings Neighbourhood Validation
-------------------------------------------------------
Unique neighbourhoods in listings.csv.gz: 22
Unique neighbourhoods in neighbourhoods.csv: 22
Unmatched detailed neighbourhoods: 0
Unused reference neighbourhoods: 0


In [79]:
if unmatched_detailed_neighbourhoods:
    print(
        "Detailed listing neighbourhoods not found "
        "in reference data:"
    )

    for neighbourhood in sorted(
        unmatched_detailed_neighbourhoods
    ):
        print(f"- {neighbourhood}")
else:
    print(
        "All detailed listing neighbourhood values "
        "match the neighbourhood reference dataset."
    )

All detailed listing neighbourhood values match the neighbourhood reference dataset.


Neighbourhood join coverage

In [80]:
valid_detailed_neighbourhood_mask = (
    detailed_listings_df["neighbourhood_cleansed"]
    .astype(str)
    .str.strip()
    .isin(reference_neighbourhoods)
)

matched_detailed_rows = int(
    valid_detailed_neighbourhood_mask.sum()
)

total_detailed_rows = len(
    detailed_listings_df
)

detailed_neighbourhood_match_percentage = (
    matched_detailed_rows
    / total_detailed_rows
    * 100
    if total_detailed_rows > 0
    else 0
)

print("Detailed Neighbourhood Join Coverage")
print("-" * 45)

print(
    f"Matched detailed listing rows: "
    f"{matched_detailed_rows:,}"
)

print(
    f"Total detailed listing rows: "
    f"{total_detailed_rows:,}"
)

print(
    f"Match coverage: "
    f"{detailed_neighbourhood_match_percentage:.2f}%"
)

Detailed Neighbourhood Join Coverage
---------------------------------------------
Matched detailed listing rows: 10,369
Total detailed listing rows: 10,369
Match coverage: 100.00%


Important categorical fields

In [81]:
categorical_columns_to_check = [
    "source",
    "room_type",
    "property_type",
    "host_is_superhost",
    "host_response_time",
    "host_has_profile_pic",
    "host_identity_verified",
    "has_availability",
    "instant_bookable"
]

for column in categorical_columns_to_check:
    if column in detailed_listings_df.columns:
        print(f"\n{column}")
        print("-" * 50)

        display(
            detailed_listings_df[column]
            .value_counts(dropna=False)
            .rename_axis(column)
            .reset_index(name="count")
            .head(30)
        )


source
--------------------------------------------------


,source,count
0,city scrape,5864
1,previous scrape,4505



room_type
--------------------------------------------------


,room_type,count
0,Entire home/apt,8489
1,Private room,1833
2,Hotel room,26
3,Shared room,21



property_type
--------------------------------------------------


...

,host_identity_verified,count
0,t,10187
1,NaN,142
2,f,40



has_availability
--------------------------------------------------


,has_availability,count
0,t,10081
1,f,152
2,NaN,136



instant_bookable
--------------------------------------------------


,instant_bookable,count
0,NaN,10369


Room type distribution

In [82]:
display(
    detailed_listings_df["room_type"]
    .value_counts(dropna=False)
    .rename_axis("room_type")
    .reset_index(name="listing_count")
)

,room_type,listing_count
0,Entire home/apt,8489
1,Private room,1833
2,Hotel room,26
3,Shared room,21


Date-column validation

In [83]:
date_columns_to_check = [
    "last_scraped",
    "host_since",
    "price_quote_checkin_date",
    "price_quote_checkout_date",
    "calendar_last_scraped",
    "first_review",
    "last_review"
]

date_validation_results = []

for column in date_columns_to_check:
    if column in detailed_listings_df.columns:
        parsed_dates = pd.to_datetime(
            detailed_listings_df[column],
            errors="coerce"
        )

        raw_non_null_count = (
            detailed_listings_df[column]
            .notna()
            .sum()
        )

        invalid_count = int(
            raw_non_null_count
            - parsed_dates.notna().sum()
        )

        date_validation_results.append({
            "column_name": column,
            "raw_non_null_count": int(
                raw_non_null_count
            ),
            "invalid_date_count": invalid_count,
            "earliest_date": (
                parsed_dates.min()
                if parsed_dates.notna().any()
                else None
            ),
            "latest_date": (
                parsed_dates.max()
                if parsed_dates.notna().any()
                else None
            ),
        })

detailed_date_validation_df = pd.DataFrame(
    date_validation_results
)

display(detailed_date_validation_df)

,column_name,raw_non_null_count,invalid_date_count,earliest_date,latest_date
0,last_scraped,10369,0,2026-06-15,2026-06-24
1,host_since,0,0,NaT,NaT
2,price_quote_checkin_date,6508,0,2026-06-16,2027-06-11
3,price_quote_checkout_date,6508,0,2026-06-17,2027-06-16
4,calendar_last_scraped,10369,0,2026-06-15,2026-06-24
5,first_review,9346,0,2010-08-16,2026-06-18
6,last_review,9346,0,2014-04-22,2026-06-24


Key numeric range validation summary

In [84]:
numeric_columns_to_check = [
    "latitude",
    "longitude",
    "accommodates",
    "bathrooms",
    "bedrooms",
    "beds",
    "minimum_nights",
    "maximum_nights",
    "availability_30",
    "availability_60",
    "availability_90",
    "availability_365",
    "number_of_reviews",
    "number_of_reviews_ltm",
    "number_of_reviews_l30d",
    "review_scores_rating",
    "reviews_per_month",
    "estimated_occupancy_l365d",
    "estimated_revenue_l365d"
]

numeric_range_results = []

for column in numeric_columns_to_check:
    if column in detailed_listings_df.columns:
        series = detailed_listings_df[column]

        numeric_range_results.append({
            "column_name": column,
            "missing_count": int(
                series.isnull().sum()
            ),
            "minimum": (
                series.min()
                if series.notna().any()
                else None
            ),
            "maximum": (
                series.max()
                if series.notna().any()
                else None
            ),
            "median": (
                series.median()
                if series.notna().any()
                else None
            ),
        })

detailed_numeric_range_df = pd.DataFrame(
    numeric_range_results
)

display(detailed_numeric_range_df)

,column_name,missing_count,minimum,maximum,median
0,latitude,0,52.290276,5.242516e+01,52.365765
1,longitude,0,4.755870,5.028150e+00,4.887434
2,accommodates,0,1.000000,1.600000e+01,2.000000
3,bathrooms,4956,0.500000,1.700000e+01,1.000000
4,bedrooms,1150,0.000000,1.700000e+01,1.000000
5,beds,4652,1.000000,3.300000e+01,1.000000
6,minimum_nights,3,1.000000,8.000000e+02,2.000000
7,maximum_nights,3,1.000000,2.147484e+09,40.000000
8,availability_30,0,0.000000,3.000000e+01,0.000000
9,availability_60,0,0.000000,6.000000e+01,4.000000


Dataset-level summary

In [85]:
detailed_listings_dataset_summary = {
    "dataset_name": "detailed_listings",
    "file_name": DATA_FILES["detailed_listings"].name,
    "row_count": len(detailed_listings_df),
    "column_count": len(detailed_listings_df.columns),
    "duplicate_count": int(
        detailed_listings_df.duplicated().sum()
    ),
    "candidate_primary_keys": (
        detailed_listings_candidate_keys
    ),
    "unique_listing_ids": int(
        detailed_listings_df["id"]
        .nunique(dropna=True)
    ),
}

detailed_listings_dataset_summary

{'dataset_name': 'detailed_listings',
 'file_name': 'listings.csv.gz',
 'row_count': 10369,
 'column_count': 90,
 'duplicate_count': 0,
 'candidate_primary_keys': ['id', 'listing_url'],
 'unique_listing_ids': 10369}

Save profile results

In [86]:
detailed_listings_profile_path = (
    OUTPUT_DIR / "detailed_listings_column_profile.csv"
)

detailed_listings_profile_df.to_csv(
    detailed_listings_profile_path,
    index=False
)

print(
    "Detailed listings profile saved successfully to:\n"
    f"{detailed_listings_profile_path}"
)

Detailed listings profile saved successfully to:
c:\Users\saths\Desktop\MyProjects\Expernetic Data Challenge\airbnb-data-challenge\outputs\data_quality\detailed_listings_column_profile.csv


### Findings and Interpretation

The `listings.csv.gz` dataset contains **10,369 rows and 90 columns**, with each row representing one detailed Airbnb listing in Amsterdam.

Compared with the summary `listings.csv` dataset, this file provides substantially richer information about:

- listing identity,
- host characteristics,
- property and room details,
- geographic location,
- amenities,
- pricing,
- booking rules,
- availability,
- review activity,
- review scores,
- estimated occupancy,
- estimated revenue,
- licensing.

The grain of this dataset is:

**One row represents one Airbnb listing.**

---

#### Key Structural Findings

The dataset contains:

- Total rows: **10,369**
- Total columns: **90**
- Duplicate rows: **0**
- Duplicate percentage: **0.00%**
- Unique listing IDs: **10,369**
- Missing listing IDs: **0**
- Duplicate listing IDs: **0**

Two columns were identified as technically unique and complete:

- `id`
- `listing_url`

However, `id` is the most appropriate semantic candidate primary key because it is the actual listing identifier, while `listing_url` is a derived URL representation of that listing identity.

Therefore:

**Preferred candidate primary key: `id`**

---

#### Detailed vs Summary Listings Difference

The detailed `listings.csv.gz` and summary `listings.csv` datasets do not contain exactly the same number of listings.

Observed results:

- Unique IDs in `listings.csv.gz`: **10,369**
- Unique IDs in `listings.csv`: **10,465**
- IDs present in both datasets: **10,369**
- IDs only in `listings.csv.gz`: **0**
- IDs only in `listings.csv`: **96**

Coverage results:

- Detailed listing IDs found in the summary dataset: **100.00%**
- Summary listing IDs found in the detailed dataset: **99.08%**

This means that every listing in the detailed dataset also appears in the summary dataset, while 96 summary listings are absent from the detailed dataset.

The cause of this difference should not be assumed without additional evidence.

Possible explanations may include:

- differences in file-generation logic,
- scrape timing,
- source coverage,
- incomplete detailed records,
- filtering rules applied to the detailed export.

Further validation is required before assigning a final explanation.

---

#### Important Cross-Dataset Pattern Requiring Validation

A notable numerical pattern was observed across the summary and detailed listings files.

In `listings.csv`:

- Total listings: **10,465**
- Private room listings: **1,929**
- Missing `host_id` values: **96**

In `listings.csv.gz`:

- Total listings: **10,369**
- Private room listings: **1,833**

The difference in total listing count is exactly:

**10,465 - 10,369 = 96**

The difference in private-room count is also exactly:

**1,929 - 1,833 = 96**

The summary dataset also contains exactly **96 missing `host_id` values**.

This numerical alignment is potentially meaningful, but it does not by itself prove that the same 96 records are responsible for all three observations.

Therefore, a later row-level validation should test whether the 96 summary-only listings:

- are all private-room listings,
- all have missing `host_id`,
- account exactly for the observed differences.

Until that validation is performed, this should be documented as a **strong observed pattern requiring confirmation**, not as a confirmed causal relationship.

---

#### Missing-Value Overview

The dataset contains missing values in:

**57 out of 90 columns**

Thirteen columns are completely missing:

- `neighborhood_overview`
- `host_since`
- `host_response_time`
- `host_response_rate`
- `host_acceptance_rate`
- `host_thumbnail_url`
- `host_neighbourhood`
- `host_total_listings_count`
- `host_verifications`
- `neighbourhood`
- `neighbourhood_group_cleansed`
- `calendar_updated`
- `instant_bookable`

These columns contain **10,369 missing values each, representing 100.00% missingness**.

They currently provide no analytical information in this dataset and should not be used downstream unless values can be obtained from another trusted source.

The original raw data should still remain unchanged.

---

#### Major Partially Missing Fields

Several analytically important fields have substantial missingness.

##### Host description

`host_about`

- Missing values: **4,959**
- Missing percentage: **47.83%**

Almost half of the listings have no available host biography.

---

##### Bathrooms

`bathrooms`

- Missing values: **4,956**
- Missing percentage: **47.80%**

However, `bathrooms_text` is almost complete, with only 9 missing values.

This suggests that bathroom information may still be recoverable from the text field for many records where the numeric `bathrooms` field is missing.

Any such derivation should be documented explicitly during cleaning.

---

##### Beds

`beds`

- Missing values: **4,652**
- Missing percentage: **44.86%**

This limits direct analysis involving bed count unless missing values are handled carefully.

---

##### Price and related fields

The following fields each contain **3,992 missing values (38.50%)**:

- `price`
- `price_quote_total_price`
- `price_quote_price_per_night`
- `estimated_revenue_l365d`

This is a significant limitation because price is central to later EDA, statistical testing, and business analysis.

The raw `price` column is also stored as a string because values contain currency formatting such as:

`$94.00`

During cleaning, this field should be converted into a numeric representation while preserving the original raw data.

---

##### Price quote fields

The following each contain **3,861 missing values (37.24%)**:

- `price_quote_checkin_date`
- `price_quote_checkout_date`
- `price_quote_raw`

These fields are therefore available only for a subset of listings.

---

##### Bedrooms

`bedrooms`

- Missing values: **1,150**
- Missing percentage: **11.09%**

---

##### Review-related fields

Most review-related fields contain approximately **1,023 missing values (9.87%)**:

- `first_review`
- `last_review`
- `review_scores_rating`
- `review_scores_accuracy`
- `review_scores_checkin`
- `review_scores_communication`
- `review_scores_location`
- `review_scores_value`
- `reviews_per_month`

`review_scores_cleanliness` contains:

- Missing values: **1,024**
- Missing percentage: **9.88%**

These missing patterns are likely connected to listings without sufficient review history, but this should be validated before any imputation decision is made.

---

#### Neighbourhood Field Assessment

Three neighbourhood-related fields were examined:

| Column | Missing Percentage | Unique Values |
|---|---:|---:|
| `neighbourhood` | 100.00% | 0 |
| `neighbourhood_cleansed` | 0.00% | 22 |
| `neighbourhood_group_cleansed` | 100.00% | 0 |

The only usable neighbourhood field is:

`neighbourhood_cleansed`

It contains:

- 0 missing values,
- 22 unique neighbourhoods.

Its values were explicitly validated against `neighbourhoods.csv`.

Results:

- Unique neighbourhoods in `listings.csv.gz`: **22**
- Unique neighbourhoods in `neighbourhoods.csv`: **22**
- Unmatched detailed neighbourhoods: **0**
- Unused reference neighbourhoods: **0**
- Matched detailed listing rows: **10,369**
- Total detailed listing rows: **10,369**
- Join coverage: **100.00%**

Therefore, every detailed listing contains a valid neighbourhood value that matches the neighbourhood reference dataset.

This is a strong positive referential-integrity result.

---

#### Room-Type Distribution

The detailed dataset contains four room types:

| Room Type | Listing Count |
|---|---:|
| Entire home/apt | 8,489 |
| Private room | 1,833 |
| Hotel room | 26 |
| Shared room | 21 |

The market is therefore heavily dominated by `Entire home/apt` listings.

Notably, the detailed and summary datasets contain identical counts for:

- Entire home/apt: **8,489**
- Hotel room: **26**
- Shared room: **21**

Only the `Private room` category differs:

- Summary listings: **1,929**
- Detailed listings: **1,833**
- Difference: **96**

This exactly matches the overall 96-listing difference between the two files and should be validated at the individual listing-ID level.

---

#### Property-Type Diversity

The dataset contains **59 distinct property types**.

The largest observed categories include:

- Entire rental unit: **4,451**
- Entire condo: **1,803**
- Entire home: **1,183**
- Private room in rental unit: **382**
- Room in hotel: **359**
- Private room in bed and breakfast: **301**
- Entire loft: **251**
- Entire townhouse: **225**
- Houseboat: **190**

This demonstrates substantial diversity in Amsterdam accommodation supply.

The distinction between `property_type` and `room_type` is important:

- `room_type` provides a broad accommodation category.
- `property_type` provides a more detailed description of the physical property.

Both fields may therefore support different levels of segmentation.

---

#### Host Superhost Status

The `host_is_superhost` field contains:

- Non-superhost (`f`): **8,513**
- Superhost (`t`): **1,714**
- Missing: **142**

This field is particularly important because it may be used later for the planned statistical hypothesis comparing review scores between superhost and non-superhost listings.

The 142 missing values should be excluded or handled explicitly during statistical testing rather than automatically assigned to either category.

---

#### Host Identity and Profile Information

The following results were observed:

##### Host profile picture

- `t`: **10,072**
- `f`: **155**
- Missing: **142**

##### Host identity verification

- `t`: **10,187**
- `f`: **40**
- Missing: **142**

The consistent count of 142 missing values across several host fields suggests a related missing-data pattern affecting a subset of host records.

This should be investigated later.

---

#### Availability Fields

The dataset contains complete availability information for:

- `availability_30`
- `availability_60`
- `availability_90`
- `availability_365`

Observed ranges are internally consistent:

- `availability_30`: 0 to 30
- `availability_60`: 0 to 60
- `availability_90`: 0 to 90
- `availability_365`: 0 to 365

These results indicate that the observed availability values fall within their expected upper limits.

However, availability must not be interpreted directly as true occupancy or vacancy.

Unavailable dates may result from:

- confirmed bookings,
- host blocking,
- maintenance,
- personal use,
- listing suspension,
- other operational reasons.

---

#### Date Quality

Seven date-related columns were checked using temporary datetime parsing without modifying the raw dataset.

Results:

| Column | Invalid Dates | Earliest Date | Latest Date |
|---|---:|---|---|
| `last_scraped` | 0 | 2026-06-15 | 2026-06-24 |
| `host_since` | 0 | No available values | No available values |
| `price_quote_checkin_date` | 0 | 2026-06-16 | 2027-06-11 |
| `price_quote_checkout_date` | 0 | 2026-06-17 | 2027-06-16 |
| `calendar_last_scraped` | 0 | 2026-06-15 | 2026-06-24 |
| `first_review` | 0 | 2010-08-16 | 2026-06-18 |
| `last_review` | 0 | 2014-04-22 | 2026-06-24 |

All available date values were successfully parsed.

This indicates strong date-format consistency.

The `host_since` column contains no values and therefore provides no usable host-registration date information.

---

#### Numeric Range Observations

Several important numeric fields have plausible bounded ranges:

- `latitude`: approximately 52.290276 to 52.42516
- `longitude`: approximately 4.75587 to 5.02815
- `accommodates`: 1 to 16
- `availability_30`: 0 to 30
- `availability_60`: 0 to 60
- `availability_90`: 0 to 90
- `availability_365`: 0 to 365
- `review_scores_rating`: 1 to 5

These fields appear internally plausible based on their expected meaning, although geographic validation and formal domain checks should still be performed later.

---

#### Extreme Values Requiring Validation

Several fields contain unusually large values that should be investigated during the validation and EDA phases.

Examples include:

- Maximum `minimum_nights`: **800**
- Maximum `maximum_nights`: **2,147,483,647**
- Maximum `number_of_reviews`: **5,603**
- Maximum `number_of_reviews_ltm`: **813**
- Maximum `number_of_reviews_l30d`: **41**
- Maximum `reviews_per_month`: **92.56**
- Maximum `estimated_revenue_l365d`: **344,955**
- Maximum `beds`: **33**
- Maximum `bedrooms`: **17**
- Maximum `bathrooms`: **17**

These values should not automatically be classified as errors.

They may represent:

- legitimate extreme cases,
- special property configurations,
- host-defined business rules,
- source-system sentinel values,
- data collection artifacts,
- possible anomalies.

Further validation is required before deciding whether to preserve, cap, transform, flag, or exclude them.

The value:

`2,147,483,647`

is particularly notable because it corresponds to the maximum signed 32-bit integer value and may potentially function as a source-system sentinel or unrestricted-maximum placeholder.

This interpretation should be treated as a hypothesis requiring validation rather than a confirmed fact.

---

#### Review Score Quality

The review score fields generally range from:

**1.0 to 5.0**

This includes:

- `review_scores_rating`
- `review_scores_accuracy`
- `review_scores_cleanliness`
- `review_scores_checkin`
- `review_scores_communication`
- `review_scores_location`
- `review_scores_value`

The observed ranges therefore fall within the expected 1-to-5 rating scale.

However, approximately 9.87% of records are missing most review-score values, likely because some listings have insufficient or no review history.

---

#### Estimated Occupancy and Revenue

The dataset includes source-derived fields:

- `estimated_occupancy_l365d`
- `estimated_revenue_l365d`

Observed ranges:

- Estimated occupancy: **0 to 255**
- Estimated revenue: **0 to 344,955**

These fields must be interpreted cautiously.

They are estimated measures and should not be presented as exact:

- actual occupancy,
- confirmed bookings,
- verified host earnings,
- platform revenue.

Any analysis using these fields should clearly preserve the word **estimated**.

---

#### Overall Interpretation

The detailed `listings.csv.gz` dataset is the richest listing-level source available in the Amsterdam dataset collection.

Its main strengths are:

- unique and complete listing IDs,
- zero duplicate rows,
- 90 detailed attributes,
- complete geographic coordinates,
- complete room-type information,
- 100% neighbourhood reference coverage,
- complete availability ranges,
- valid review-score ranges,
- fully parseable available dates.

Its main data-quality challenges are:

- 57 columns with some level of missingness,
- 13 completely empty columns,
- 38.50% missing price data,
- substantial missingness in bathrooms, beds, and host descriptions,
- approximately 9.87% missing review information,
- large identifier values requiring precision-safe handling,
- extreme values requiring domain validation,
- a 96-listing difference from the summary listings file.

Overall assessment:

**The dataset is highly valuable for downstream cleaning, enrichment, EDA, statistical testing, and business analysis, but requires targeted missing-value handling, careful data-type correction, explicit domain validation, and documented treatment of extreme values before analytical use.**

### Candidate Keys and Relationships

The `listings.csv.gz` dataset acts as the most detailed listing-level source in the Amsterdam Airbnb dataset collection.

Several important keys and cross-dataset relationships were identified and validated.

---

#### Candidate Primary Key

Two columns are technically unique and complete:

- `id`
- `listing_url`

Observed results:

- Total rows: **10,369**
- Unique `id` values: **10,369**
- Missing `id` values: **0**
- Duplicate `id` values: **0**
- Unique `listing_url` values: **10,369**
- Missing `listing_url` values: **0**

Therefore, both fields are technically unique.

However, the preferred semantic candidate primary key is:

**`id`**

The reason is that `id` is the actual Airbnb listing identifier, while `listing_url` is a URL representation derived from the same listing identity.

The preferred key relationship is therefore:

```text
Detailed Listing
Primary Key Candidate: id
````

The `listing_url` field should be treated as an alternate unique identifier rather than the main primary key.

Because the source is a CSV file rather than a relational database with enforced constraints, these are described as candidate keys based on observed uniqueness and completeness.

---

#### Relationship with Summary Listings

The relationship between `listings.csv.gz` and `listings.csv` was explicitly validated using the listing identifier.

The relationship is:

```text
listings.csv
    id
     │
     │ listing identity
     ▼
listings.csv.gz
    id
```

Validation results:

* Unique IDs in `listings.csv.gz`: **10,369**
* Unique IDs in `listings.csv`: **10,465**
* IDs present in both datasets: **10,369**
* IDs only in `listings.csv.gz`: **0**
* IDs only in `listings.csv`: **96**

Coverage:

* Detailed IDs found in summary listings: **100.00%**
* Summary IDs found in detailed listings: **99.08%**

Therefore, every listing in `listings.csv.gz` exists in `listings.csv`.

However, 96 listings appearing in the summary dataset do not appear in the detailed dataset.

The cause of this difference has not yet been confirmed.

Possible explanations may include:

* differences in scrape timing,
* differences in file-generation rules,
* incomplete detailed records,
* filtering applied to the detailed export,
* differences between source snapshots.

No specific cause should be stated as fact without further evidence.

The observed relationship can therefore be described as:

**A near one-to-one logical relationship based on listing ID, with complete coverage from detailed listings to summary listings and 96 summary-only listing records.**

---

#### Important 96-Listing Pattern

A notable numerical pattern was identified.

In `listings.csv`:

* Total listings: **10,465**
* Private room listings: **1,929**
* Missing `host_id` values: **96**

In `listings.csv.gz`:

* Total listings: **10,369**
* Private room listings: **1,833**

The differences are:

```text
Total listing difference:
10,465 - 10,369 = 96

Private room difference:
1,929 - 1,833 = 96
```

The summary dataset also contains exactly:

```text
96 missing host_id values
```

This creates a strong numerical pattern.

However, these matching counts do not by themselves prove that the same 96 records account for all three observations.

A later row-level validation should test whether the 96 summary-only listings:

* are all `Private room` listings,
* all have missing `host_id`,
* exactly explain the summary-to-detailed difference.

Until explicitly validated, this should be described as:

**A strong cross-dataset pattern requiring confirmation, rather than a confirmed relationship.**

---

#### Neighbourhood Relationship

Three neighbourhood-related columns exist in the detailed listings dataset:

* `neighbourhood`
* `neighbourhood_cleansed`
* `neighbourhood_group_cleansed`

Observed results:

| Column                         | Missing Percentage | Unique Values |
| ------------------------------ | -----------------: | ------------: |
| `neighbourhood`                |            100.00% |             0 |
| `neighbourhood_cleansed`       |              0.00% |            22 |
| `neighbourhood_group_cleansed` |            100.00% |             0 |

Therefore, the usable geographic field is:

**`neighbourhood_cleansed`**

This field was explicitly validated against:

`neighbourhoods.csv.neighbourhood`

The relationship is:

```text
neighbourhoods.csv
    neighbourhood
          │
          │ logical reference relationship
          ▼
listings.csv.gz
    neighbourhood_cleansed
```

Validation results:

* Unique neighbourhoods in `listings.csv.gz`: **22**
* Unique neighbourhoods in `neighbourhoods.csv`: **22**
* Unmatched detailed neighbourhoods: **0**
* Unused reference neighbourhoods: **0**
* Matched detailed listing rows: **10,369**
* Total detailed listing rows: **10,369**
* Match coverage: **100.00%**

Therefore, every detailed listing contains a valid `neighbourhood_cleansed` value that matches the neighbourhood reference table.

This supports treating:

* `neighbourhoods.neighbourhood` as a candidate lookup-table key.
* `listings.csv.gz.neighbourhood_cleansed` as a logical foreign-key-like reference.

Because the source files are CSVs, the relationship is logical rather than formally enforced.

---

#### Host Relationship

The `host_id` column links listings to hosts.

Observed results:

* Total listings: **10,369**
* Unique `host_id` values: **9,104**
* Missing `host_id` values: **0**

Since multiple listings can share the same `host_id`, this column is not a primary key for the listing dataset.

Instead, it represents a one-to-many relationship:

```text
Host
  │
  │ host_id
  │
  ▼
Listing
```

This can be interpreted as:

**One host may manage one or more listings.**

The observed cardinality is:

```text
Host 1 ───────< Many Listings
```

This relationship will be useful later for:

* host portfolio analysis,
* host concentration analysis,
* identifying single-listing and multi-listing hosts,
* building a host dimension table,
* creating derived host-level features.

---

#### Host Profile Relationship

The dataset also contains:

* `host_profile_id`
* `host_profile_url`

These fields appear to represent host profile information.

However:

* `host_profile_id` has **142 missing values**.
* `host_profile_url` has **137 missing values**.
* `host_profile_id` is stored as `float64`.
* some values are extremely large.

Therefore, the host profile relationship should be treated cautiously until:

* precision-safe data types are applied,
* missing values are investigated,
* cross-field consistency is validated.

The primary host relationship for downstream work should therefore use `host_id`, not `host_profile_id`, unless further validation supports otherwise.

---

#### Relationship with Summary Reviews

The expected review relationship is:

```text
listings.csv.gz
    id
     │
     │ one listing
     ▼
reviews.csv
    listing_id
```

The `reviews.csv` dataset previously showed:

* 9,432 unique listing IDs,
* 100.00% relationship coverage against `listings.csv`.

A later validation should also explicitly compare:

`listings.csv.gz.id`

with:

`reviews.csv.listing_id`

because the detailed listings file contains 96 fewer listing IDs than the summary listings dataset.

This may affect the number of reviewed and unreviewed listings represented in each source.

---

#### Expected Relationship with Detailed Reviews

The expected relationship with `reviews.csv.gz` is:

```text
listings.csv.gz
    id
     │
     │ one-to-many
     ▼
reviews.csv.gz
    listing_id
```

The expected cardinality is:

```text
Listing 1 ───────< Many Detailed Reviews
```

This relationship must be explicitly validated later using DuckDB or chunked processing.

The detailed reviews file may also contain fields such as:

* review ID,
* reviewer ID,
* reviewer name,
* review comments.

These fields may help resolve the repeated `(listing_id, date)` combinations found in `reviews.csv`.

---

#### Expected Relationship with Calendar Data

The expected calendar relationship is:

```text
listings.csv.gz
    id
     │
     │ one-to-many
     ▼
calendar.csv.gz
    listing_id
```

The expected cardinality is:

```text
Listing 1 ───────< Many Daily Calendar Records
```

Each listing may have multiple daily calendar rows representing different dates.

This relationship should later support calculations such as:

* availability rate,
* average calendar price,
* weekday price,
* weekend price,
* monthly pricing patterns,
* availability patterns.

However, the relationship must be explicitly validated when `calendar.csv.gz` is profiled.

---

#### Relationship with `neighbourhoods.geojson`

The geographic relationship is expected to involve the neighbourhood identifier.

A likely logical relationship is:

```text
listings.csv.gz
    neighbourhood_cleansed
              │
              ▼
neighbourhoods.geojson
    neighbourhood name / property
```

This relationship will allow listings to be connected to geographic boundary polygons for optional mapping or spatial analysis.

The actual GeoJSON property names must be inspected before confirming the exact join field.

---

#### Overall Relationship Structure

The current logical data model can be represented as:

```text
                         Host
                          │
                       host_id
                          │
                          ▼
                    Detailed Listing
                   listings.csv.gz.id
                    /      |       \
                   /       |        \
                  ▼        ▼         ▼
             Reviews    Calendar   Neighbourhood
            listing_id  listing_id  reference
```

A more complete relationship view is:

```text
neighbourhoods.csv
    neighbourhood
          │
          ▼
listings.csv.gz
    neighbourhood_cleansed
          │
          │
          ├──────────────► reviews.csv.listing_id
          │
          ├──────────────► reviews.csv.gz.listing_id
          │
          ├──────────────► calendar.csv.gz.listing_id
          │
          └──────────────► listings.csv.id
```

---

#### Relationship Validation Status

Validated relationships:

* `listings.csv.gz.id` → `listings.csv.id`

  * Status: **Validated**
  * Detailed-to-summary coverage: **100.00%**
  * Summary-to-detailed coverage: **99.08%**

* `listings.csv.gz.neighbourhood_cleansed` → `neighbourhoods.csv.neighbourhood`

  * Status: **Validated**
  * Coverage: **100.00%**

Observed host relationship:

* `host_id` → multiple listing rows

  * Status: **Observed**
  * Relationship type: **One-to-many**

Pending validation:

* `listings.csv.gz.id` → `reviews.csv.listing_id`
* `listings.csv.gz.id` → `reviews.csv.gz.listing_id`
* `listings.csv.gz.id` → `calendar.csv.gz.listing_id`
* `listings.csv.gz.neighbourhood_cleansed` → GeoJSON neighbourhood property
* row-level explanation of the 96 summary-only listings

---

#### Key Engineering Interpretation

The detailed `listings.csv.gz` dataset is the central listing-level entity in the Amsterdam Airbnb data model.

Its `id` field is the preferred listing identifier, while:

* `host_id` connects listings to hosts,
* `neighbourhood_cleansed` connects listings to neighbourhood reference data,
* `listing_id` in review and calendar datasets is expected to connect child records back to listings.

This makes `listings.csv.gz` a strong foundation for:

* the enriched listing master dataset,
* the DuckDB analytical model,
* a `dim_listing` table,
* host analysis,
* neighbourhood analysis,
* review enrichment,
* calendar enrichment,
* EDA,
* statistical testing,
* business interpretation.


### Identifier Data-Type Observation

An important technical issue was identified in several identifier-related fields in the detailed `listings.csv.gz` dataset.

Identifiers should be treated as labels rather than numerical measurements. Therefore, preserving their exact values is more important than performing arithmetic operations on them.

---

#### Listing Identifier

The main listing identifier is:

`id`

Observed characteristics:

- Data type: `int64`
- Total values: 10,369
- Unique values: 10,369
- Missing values: 0
- Duplicate values: 0

The `id` field is currently stored safely as an integer and is the preferred candidate primary key for the dataset.

No immediate type correction is required for this field.

---

#### Host Identifier

The `host_id` field has the following characteristics:

- Data type: `int64`
- Unique values: 9,104
- Missing values: 0

This field is currently stored as an integer and can be used for:

- host grouping,
- host portfolio analysis,
- host concentration analysis,
- relationship validation.

However, some `host_id` values are extremely large, reaching approximately `10^18`.

Although the current `int64` representation can safely store values within the signed 64-bit integer range, the field should still be treated as an identifier rather than a numerical quantity.

No arithmetic operations should be performed on `host_id`.

---

#### Host Profile Identifier

The `host_profile_id` field requires more careful handling.

Observed characteristics:

- Data type: `float64`
- Missing values: 142
- Missing percentage: 1.37%
- Unique non-null values: 9,067
- Some values are approximately `10^18`

The likely reason Pandas inferred this field as `float64` is that the column contains missing values.

This creates a potential precision problem.

A `float64` cannot represent every very large integer exactly. Therefore, extremely large identifiers may lose precision when stored as floating-point values.

This could affect:

- equality comparisons,
- joins,
- duplicate detection,
- grouping,
- host-profile relationship validation.

---

#### Example Precision Risk

A value such as:

`1462510208428038971`

may be displayed internally in scientific notation as approximately:

`1.462510208428039e+18`

The scientific notation itself is not the problem.

The real risk is that converting a very large integer identifier to `float64` may alter its exact value because floating-point representation has limited integer precision.

Since identifiers must match exactly during joins, even a small precision loss could produce incorrect results.

---

#### Cross-Dataset Type Difference

The same identifier fields may be represented differently across files.

For example:

- `host_id` in `listings.csv.gz` is stored as `int64`.
- `host_id` in `listings.csv` was stored as `float64` because 96 values were missing.

This creates a potential join risk because the same conceptual identifier may have different data types across datasets.

Before performing cross-dataset joins, identifier columns should therefore be standardized to a common precision-safe type.

---

#### Recommended Data-Type Strategy

During the cleaning and standardization phase, identifier fields should be handled using one of the following approaches.

##### Option 1 — Pandas Nullable Integer

Use:

`Int64`

This allows:

- exact integer representation,
- support for missing values,
- safer joins between datasets.

This is appropriate when all non-null identifier values are valid integers.

---

##### Option 2 — String Type

Use:

`string`

This is appropriate when:

- exact textual preservation is the highest priority,
- identifiers are never used mathematically,
- source formatting may vary,
- future datasets may contain non-numeric identifier values.

---

#### Recommended Project Decision

For this project, the preferred approach is:

- keep `id` as `int64` where complete and safe,
- convert identifier columns containing missing values to nullable `Int64` when all non-null values are valid integers,
- use string type only when exact textual preservation is more appropriate.

The following fields should receive explicit type validation:

- `id`
- `host_id`
- `host_profile_id`
- `scrape_id`

In particular, `host_profile_id` should not remain as `float64` in processed outputs if exact identifier preservation is required.

---

#### Important Raw-Data Principle

No changes should be made to the original raw `listings.csv.gz` file.

The raw dataset should remain unchanged.

Any data-type corrections should be applied only to:

- cleaned datasets,
- processed Parquet files,
- enriched listing master datasets,
- DuckDB analytical tables.

This preserves source integrity and supports reproducibility.

---

#### Engineering Interpretation

Identifier handling is not a cosmetic issue.

Incorrect identifier types can directly affect:

- referential integrity,
- join accuracy,
- duplicate detection,
- host-level aggregation,
- cross-dataset validation.

Therefore, large identifier fields should be treated as precision-sensitive categorical keys rather than ordinary numerical variables.

This issue will be addressed during the later **Cleaning and Standardization** phase.

### Business-Domain Meaning

The `listings.csv.gz` dataset is the most detailed listing-level source in the Amsterdam Airbnb dataset collection.

Each row represents one Airbnb listing and contains a broad set of attributes describing:

- the listing itself,
- the host,
- the property,
- location,
- room and accommodation characteristics,
- amenities,
- pricing,
- stay restrictions,
- availability,
- review activity,
- review scores,
- estimated occupancy,
- estimated revenue,
- licensing.

The grain of the dataset is:

**One row represents one Airbnb listing.**

Because of its 90 columns and rich feature coverage, this dataset is expected to be one of the primary sources for the later cleaned, enriched, and analytical datasets.

---

#### Listing

A listing represents an individual Airbnb accommodation offered to guests.

The primary listing identifier is:

`id`

Other listing-related fields include:

- `listing_url`
- `name`
- `description`
- `picture_url`
- `property_type`
- `room_type`
- `accommodates`
- `bathrooms`
- `bedrooms`
- `beds`
- `amenities`

These attributes describe what the accommodation is, what it offers, and how many guests it can potentially accommodate.

The listing entity is central to the project because other datasets such as reviews and calendar records are expected to connect back to listings through `listing_id`.

---

#### Host

A host represents the person or entity responsible for one or more Airbnb listings.

Relevant host-related fields include:

- `host_id`
- `host_url`
- `host_profile_id`
- `host_profile_url`
- `host_name`
- `host_location`
- `host_about`
- `host_is_superhost`
- `host_listings_count`
- `host_has_profile_pic`
- `host_identity_verified`
- `calculated_host_listings_count`

The observed relationship is:

**One host may manage one or more listings.**

This supports later analyses such as:

- single-listing vs multi-listing hosts,
- host portfolio concentration,
- professional hosting activity,
- superhost performance,
- supply concentration.

---

#### Neighbourhood

The usable geographic field is:

`neighbourhood_cleansed`

This field contains 22 unique Amsterdam neighbourhoods and achieved 100.00% match coverage with the `neighbourhoods.csv` reference dataset.

Neighbourhood information supports analyses such as:

- listing density by neighbourhood,
- price comparison by neighbourhood,
- availability patterns,
- room-type composition,
- review performance,
- geographic market segmentation.

The other fields:

- `neighbourhood`
- `neighbourhood_group_cleansed`

are completely missing in the current detailed dataset and provide no present analytical value.

---

#### Geographic Coordinates

The dataset contains:

- `latitude`
- `longitude`

These fields identify the approximate geographic position of each listing.

They can support:

- geographic validation,
- map visualizations,
- neighbourhood-level clustering,
- location-based analysis.

However, the coordinates should later be checked for plausible Amsterdam bounds before being used in geographic analysis.

---

#### Property Type

The `property_type` field provides detailed information about the physical accommodation category.

The dataset contains 59 distinct property types, including examples such as:

- Entire rental unit
- Entire condo
- Entire home
- Private room in rental unit
- Room in hotel
- Houseboat
- Boat
- Entire loft

This field provides finer segmentation than `room_type`.

It can support more detailed business questions such as:

- Which property categories command higher prices?
- Which property types are most common?
- Which property categories receive stronger review activity?
- How do accommodation formats differ across neighbourhoods?

---

#### Room Type

The `room_type` field provides a broader accommodation classification.

The dataset contains four room types:

- Entire home/apt: 8,489 listings
- Private room: 1,833 listings
- Hotel room: 26 listings
- Shared room: 21 listings

The market represented in this detailed dataset is therefore heavily dominated by:

`Entire home/apt`

Room type is expected to be one of the most important segmentation variables in later:

- price analysis,
- EDA,
- statistical testing,
- business recommendations.

---

#### Accommodation Capacity

Relevant fields include:

- `accommodates`
- `bathrooms`
- `bathrooms_text`
- `bedrooms`
- `beds`

These fields describe the physical capacity and accommodation structure of each property.

They may support analysis such as:

- price per guest,
- price per bedroom,
- relationship between capacity and price,
- accommodation size by neighbourhood,
- property segmentation.

However, substantial missingness exists in `bathrooms` and `beds`, so these variables require careful treatment.

---

#### Amenities

The `amenities` field describes the features and facilities available at each listing.

Examples may include:

- heating,
- refrigerator,
- fire extinguisher,
- private entrance,
- bed linens,
- stove,
- air conditioning.

This field is high-cardinality and text-like.

It may later support derived features such as:

- amenity count,
- presence of selected important amenities,
- premium amenity indicators.

However, full text processing of amenities is not a priority unless time allows.

---

#### Price

The raw `price` field contains nightly listing prices in text format, such as:

`$94.00`

The field is missing for 3,992 listings, representing 38.50% of the dataset.

Price is central to later analyses such as:

- price distribution,
- price by room type,
- neighbourhood pricing,
- property-type pricing,
- statistical comparison between market segments.

During cleaning, the raw price field should be converted to a numeric value while preserving the original raw data.

---

#### Price Quote Information

The dataset also contains:

- `price_quote_checkin_date`
- `price_quote_checkout_date`
- `price_quote_total_price`
- `price_quote_price_per_night`
- `price_quote_raw`

These fields provide additional pricing information for specific check-in and checkout dates.

They may support:

- quote-level price validation,
- price consistency checks,
- nightly price derivation.

However, these fields contain substantial missingness and should be used only where valid data exists.

---

#### Minimum and Maximum Stay Rules

Relevant fields include:

- `minimum_nights`
- `maximum_nights`
- `minimum_minimum_nights`
- `maximum_minimum_nights`
- `minimum_maximum_nights`
- `maximum_maximum_nights`
- `minimum_nights_avg_ntm`
- `maximum_nights_avg_ntm`

These fields describe booking-length restrictions.

They can support analysis such as:

- restrictive vs flexible listings,
- long-stay requirements,
- differences in stay rules across room types,
- potential outlier detection.

Some extreme values, particularly maximum-night limits of `2,147,483,647`, require later validation.

---

#### Availability

The dataset includes availability measures for different time horizons:

- `availability_30`
- `availability_60`
- `availability_90`
- `availability_365`
- `availability_eoy`

These variables describe the number of days a listing is marked as available within a given period.

Observed ranges are internally consistent for:

- `availability_30`: 0 to 30
- `availability_60`: 0 to 60
- `availability_90`: 0 to 90
- `availability_365`: 0 to 365

These fields can support:

- availability distribution analysis,
- room-type comparison,
- neighbourhood comparison,
- listing-level availability features.

However, availability should not be interpreted directly as occupancy.

Unavailable dates may result from:

- bookings,
- host blocks,
- maintenance,
- personal use,
- listing suspension,
- other operational reasons.

---

#### Review Activity

The dataset contains several review-volume fields:

- `number_of_reviews`
- `number_of_reviews_ltm`
- `number_of_reviews_l30d`
- `number_of_reviews_ly`
- `reviews_per_month`

These fields can help measure listing activity and guest engagement.

They may support:

- identifying highly reviewed listings,
- recent review activity,
- review-frequency analysis,
- listing activity segmentation.

However, review count should not be interpreted as an exact booking count.

It is better treated as an activity proxy.

---

#### Review Dates

Relevant fields include:

- `first_review`
- `last_review`

These fields describe the observed review-history period for each listing.

They can support derived features such as:

- review lifespan,
- days since last review,
- years of review activity,
- recent review status.

All available date values were successfully parsed during validation.

---

#### Review Scores

The dataset contains detailed review-score dimensions:

- `review_scores_rating`
- `review_scores_accuracy`
- `review_scores_cleanliness`
- `review_scores_checkin`
- `review_scores_communication`
- `review_scores_location`
- `review_scores_value`

These fields generally range from 1 to 5.

They support analysis such as:

- overall guest satisfaction,
- quality comparison by room type,
- superhost vs non-superhost performance,
- neighbourhood-level review quality,
- identification of weak performance dimensions.

The `review_scores_rating` field is particularly important for the planned hypothesis comparing superhost and non-superhost listings.

---

#### Superhost Status

The `host_is_superhost` field indicates whether a host is marked as:

- `t` — superhost
- `f` — non-superhost

Observed counts:

- Non-superhost: 8,513
- Superhost: 1,714
- Missing: 142

This field is important for later statistical testing and business interpretation.

Missing values should not be automatically assigned to either category.

---

#### Estimated Occupancy

The field:

`estimated_occupancy_l365d`

contains an estimated occupancy-related measure for the last 365 days.

Observed range:

- Minimum: 0
- Maximum: 255

This field may support market activity analysis.

However, it must be described carefully as an estimated measure rather than actual verified occupancy.

---

#### Estimated Revenue

The field:

`estimated_revenue_l365d`

contains an estimated revenue measure for the last 365 days.

Observed range:

- Minimum: 0
- Maximum: 344,955

This field can support revenue-oriented analysis, but it must never be described as confirmed host earnings.

The correct interpretation should preserve the term:

**Estimated revenue**

because the value may be based on assumptions or proxy calculations.

---

#### Licensing

The `license` field contains licensing information for many listings.

It may support analysis of:

- presence or absence of license data,
- listing registration patterns.

However, 223 values are missing.

Missing license information should not automatically be interpreted as proof of non-compliance or unlicensed operation.

---

#### Source Information

The `source` field contains two categories:

- `city scrape`: 5,864 listings
- `previous scrape`: 4,505 listings

This indicates that records may originate from different scrape sources or collection states.

This source distinction may help explain differences in field completeness or snapshot timing.

It should be considered when investigating data inconsistencies.

---

#### Analytical Importance

The detailed `listings.csv.gz` dataset is expected to be the main source for building:

`enriched_listing_master.parquet`

Potential derived features include:

- host portfolio size,
- host tenure,
- price per bedroom,
- price per guest,
- review frequency,
- listing age,
- review recency,
- accommodation capacity metrics,
- superhost indicators,
- neighbourhood segmentation,
- estimated activity metrics.

This dataset is therefore central to:

- data cleaning,
- validation,
- enrichment,
- DuckDB modeling,
- EDA,
- statistical testing,
- business interpretation.

Overall, it provides the most complete listing-level representation of the Amsterdam Airbnb market in the available source files.

### Dataset Limitations

The `listings.csv.gz` dataset is the richest listing-level source in the Amsterdam Airbnb dataset collection, but it also contains several important limitations that must be considered before cleaning, enrichment, EDA, statistical testing, or business interpretation.

---

#### 1. Large Number of Columns with Missing Values

The dataset contains:

- 90 total columns
- 57 columns with at least one missing value

This means that a large proportion of the available schema requires careful treatment.

Missing values should not be handled using a single blanket rule such as:

`fillna(0)`

Instead, each field should be interpreted according to its business meaning.

For example:

- missing `price` does not mean zero price,
- missing `review_scores_rating` does not mean zero rating,
- missing `host_about` means no available host description,
- missing `last_review` may indicate no recorded review history.

Therefore, missing-value handling must be context-dependent.

---

#### 2. Thirteen Completely Empty Columns

The following columns contain 100% missing values:

- `neighborhood_overview`
- `host_since`
- `host_response_time`
- `host_response_rate`
- `host_acceptance_rate`
- `host_thumbnail_url`
- `host_neighbourhood`
- `host_total_listings_count`
- `host_verifications`
- `neighbourhood`
- `neighbourhood_group_cleansed`
- `calendar_updated`
- `instant_bookable`

These columns currently provide no analytical value.

They should not be used in downstream analysis unless values can be obtained from another trusted source.

However, the original raw dataset should remain unchanged.

Any exclusion decision should apply only to cleaned or processed outputs.

---

#### 3. Substantial Missing Price Information

The `price` column contains:

- 3,992 missing values
- 38.50% missing data

This is one of the most important limitations because price is central to later analyses such as:

- price distribution,
- room-type comparison,
- neighbourhood pricing,
- property-type pricing,
- statistical hypothesis testing,
- business recommendations.

Price-based analysis will therefore use a reduced sample unless another justified source is used to recover missing values.

Missing prices must not be automatically replaced with zero.

The valid sample size should always be reported for pricing analysis.

---

#### 4. Raw Price Stored as Text

The `price` field is stored as a string and contains values such as:

`$94.00`

This means it cannot be used directly for numerical analysis.

During cleaning, the field must be converted into a numeric representation.

For example:

`$313.67`

should become:

`313.67`

The raw value should still be preserved or remain available in the unchanged source file.

---

#### 5. Missing Price Quote Information

The following fields contain substantial missingness:

- `price_quote_checkin_date`: 37.24%
- `price_quote_checkout_date`: 37.24%
- `price_quote_raw`: 37.24%
- `price_quote_total_price`: 38.50%
- `price_quote_price_per_night`: 38.50%

Therefore, quote-based analysis can only be performed on a subset of listings.

The missingness pattern suggests that these fields may be generated only where quote information was available.

This should be treated as a source-availability limitation rather than automatically as a data error.

---

#### 6. Missing Estimated Revenue Values

The `estimated_revenue_l365d` field contains:

- 3,992 missing values
- 38.50% missing data

This is the same missing count as:

- `price`
- `price_quote_total_price`
- `price_quote_price_per_night`

This numerical alignment may indicate that estimated revenue depends on available price-related information.

However, that relationship should be explicitly tested before being stated as a confirmed dependency.

Additionally, estimated revenue must not be interpreted as actual host earnings.

It should always be labeled as:

**Estimated revenue**

---

#### 7. Missing Bathroom Information

The numeric `bathrooms` field contains:

- 4,956 missing values
- 47.80% missing data

However, the `bathrooms_text` field contains only:

- 9 missing values
- 0.09% missing data

This suggests that bathroom information may still be available in textual form even when the numeric field is missing.

For example:

- `1 bath`
- `1.5 baths`
- `1 shared bath`

A later cleaning step may derive a numeric bathroom count from `bathrooms_text`.

However, this transformation must be documented carefully because textual values may contain qualifiers such as:

- shared,
- private,
- half-bath.

These concepts should not be oversimplified without preserving relevant meaning.

---

#### 8. Missing Bed and Bedroom Information

The dataset contains:

- `beds`: 4,652 missing values, or 44.86%
- `bedrooms`: 1,150 missing values, or 11.09%

These missing values limit analysis involving:

- price per bed,
- price per bedroom,
- accommodation capacity,
- property size comparison.

They should not automatically be replaced with zero.

A missing value and a genuine zero value may have different business meanings.

---

#### 9. Missing Host Information

Several host-related fields contain missing values.

Examples include:

- `host_about`: 47.83% missing
- `host_location`: 13.23% missing
- `host_profile_id`: 1.37% missing
- `host_name`: 1.37% missing
- `host_is_superhost`: 1.37% missing
- `host_has_profile_pic`: 1.37% missing
- `host_identity_verified`: 1.37% missing
- `host_listings_count`: 1.37% missing

This may reduce completeness for:

- host-level analysis,
- superhost comparison,
- host concentration analysis,
- host profile enrichment.

Missing host attributes should be handled explicitly rather than assigned default values without evidence.

---

#### 10. Missing Review Information

Most review-related fields contain approximately 9.87% missing values.

These include:

- `first_review`
- `last_review`
- `review_scores_rating`
- `review_scores_accuracy`
- `review_scores_checkin`
- `review_scores_communication`
- `review_scores_location`
- `review_scores_value`
- `reviews_per_month`

The `review_scores_cleanliness` field contains:

- 1,024 missing values
- 9.88% missing data

These missing values may correspond to listings without sufficient review history.

However, this relationship should be validated against:

- `number_of_reviews`
- `reviews.csv`
- `reviews.csv.gz`

before making final cleaning decisions.

---

#### 11. Current Snapshot and Historical Review Data Are Mixed

The dataset contains current listing attributes such as:

- price,
- room type,
- availability,
- host status,

alongside historical review information such as:

- first review date,
- last review date,
- review counts.

This creates a temporal limitation.

A listing's current attributes may not be identical to the conditions that existed when older reviews were created.

For example:

- the host may have changed,
- the price may have changed,
- the room configuration may have changed,
- amenities may have changed.

Therefore, historical review activity should not automatically be interpreted using current listing attributes as if they were constant over time.

---

#### 12. Detailed and Summary Listings Are Not Fully Identical

The summary and detailed listing datasets contain different row counts:

- `listings.csv`: 10,465 listings
- `listings.csv.gz`: 10,369 listings

The relationship validation found:

- 10,369 IDs present in both datasets
- 0 IDs only in the detailed dataset
- 96 IDs only in the summary dataset

Therefore, the two files are not perfectly identical representations of the listing population.

The cause of the 96-listing difference is not yet confirmed.

Possible reasons may include:

- different scrape timing,
- filtering,
- incomplete detailed records,
- different export logic,
- source coverage differences.

No cause should be stated as fact without evidence.

---

#### 13. The 96-Listing Difference Requires Row-Level Validation

A notable pattern was observed:

- Summary listings contain 96 more rows.
- Summary listings contain 96 more private-room listings.
- Summary listings contain 96 missing `host_id` values.

These matching counts are potentially meaningful.

However, numerical equality alone does not prove that the same 96 records explain every difference.

A later row-level validation should explicitly test whether the 96 summary-only listings:

- are all private rooms,
- all have missing host IDs,
- exactly account for the detailed-to-summary mismatch.

Until then, this should be treated as an observed pattern, not a confirmed explanation.

---

#### 14. Large Identifiers Require Precision-Safe Handling

The `host_profile_id` field is stored as `float64` and contains very large values around `10^18`.

Floating-point representation may lose exact integer precision for very large identifiers.

This can affect:

- joins,
- duplicate detection,
- equality comparisons,
- grouping,
- relationship validation.

Before using this field in relationships, it should be converted to a precision-safe type such as:

- nullable `Int64`, if valid,
- string.

---

#### 15. Extreme Maximum-Night Values

The following fields contain values up to:

`2,147,483,647`

including:

- `maximum_nights`
- `minimum_maximum_nights`
- `maximum_maximum_nights`
- `maximum_nights_avg_ntm`

This value is unusually large and corresponds to the maximum signed 32-bit integer.

It may potentially represent:

- an unrestricted maximum stay,
- a system default,
- a sentinel value,
- a source artifact.

However, its exact meaning has not yet been confirmed.

It should therefore be flagged for investigation and not automatically treated as a genuine stay duration.

---

#### 16. Extreme Minimum Stay Values

The `minimum_nights` field has:

- minimum: 1
- maximum: 800

A minimum stay of 800 nights is highly unusual.

This could represent:

- a valid long-term listing policy,
- a temporary host configuration,
- an extreme outlier,
- a data-quality issue.

It should be investigated before removal or modification.

---

#### 17. Extreme Review Activity

The dataset contains very large review-related values:

- Maximum `number_of_reviews`: 5,603
- Maximum `number_of_reviews_ltm`: 813
- Maximum `number_of_reviews_l30d`: 41
- Maximum `reviews_per_month`: 92.56

These values should not automatically be treated as errors.

They may represent:

- highly active listings,
- long-running listings,
- professionally managed properties,
- data anomalies.

They require validation and contextual interpretation.

---

#### 18. Availability Is Not Occupancy

The dataset contains:

- `availability_30`
- `availability_60`
- `availability_90`
- `availability_365`

These values fall within their expected ranges.

However, availability must not be interpreted directly as occupancy.

A date marked unavailable may result from:

- a booking,
- host blocking,
- maintenance,
- personal use,
- temporary listing suspension,
- other reasons.

Therefore, any measure based on:

`1 - availability_rate`

must be described as an:

**occupancy proxy**

and not as true occupancy.

---

#### 19. Estimated Occupancy Is Not Verified Occupancy

The field:

`estimated_occupancy_l365d`

contains values from 0 to 255.

This is an estimated measure.

It should not be described as:

- actual booked nights,
- verified occupancy,
- confirmed stays.

All analysis using this field should preserve the word:

**estimated**

---

#### 20. Estimated Revenue Is Not Actual Host Earnings

The field:

`estimated_revenue_l365d`

contains values up to:

**344,955**

This should not be interpreted as exact or verified host earnings.

Estimated revenue may depend on assumptions, source-specific formulas, or proxy measures.

Therefore, it must be described as:

**Estimated revenue**

and not actual revenue.

---

#### 21. Review Count Is Not Booking Count

Review activity fields such as:

- `number_of_reviews`
- `number_of_reviews_ltm`
- `number_of_reviews_l30d`

should not be interpreted as exact booking counts.

Not every guest necessarily leaves a review.

Therefore, review volume is better treated as:

**An activity or demand proxy**

rather than a direct booking measure.

---

#### 22. High-Cardinality Text Fields Are Expensive to Analyze

Several fields contain high-cardinality text:

- `description`
- `host_about`
- `amenities`
- `price_quote_raw`

These fields may require:

- text parsing,
- JSON parsing,
- NLP,
- additional memory,
- additional processing time.

Given the two-day execution constraint, extensive NLP or deep text analysis should not be prioritized unless all core work is already complete.

---

#### 23. Amenities Are Stored as Text-Like Lists

The `amenities` field contains values resembling lists.

Before using this field analytically, it may require:

- parsing,
- normalization,
- deduplication,
- standardization of amenity names.

Potential derived features could include:

- amenity count,
- selected premium amenity indicators,
- safety feature indicators.

However, this is optional and should not delay higher-priority engineering work.

---

#### 24. Source Values May Reflect Different Collection States

The `source` field contains:

- `city scrape`: 5,864 listings
- `previous scrape`: 4,505 listings

This means records may come from different collection states.

Differences in:

- completeness,
- scrape dates,
- field availability,

may potentially be related to the source type.

This should be considered when investigating inconsistencies.

---

#### 25. Multiple Scrape Dates Exist

The dataset contains three unique values for:

- `last_scraped`
- `calendar_last_scraped`

This means not all rows necessarily correspond to exactly the same collection date.

Therefore, the dataset should not automatically be treated as a perfectly synchronized single-day snapshot.

This may matter when comparing:

- price,
- availability,
- review counts,
- current listing state.

---

#### 26. Some Fields Are Derived or Source-Generated

Several variables appear to be calculated or source-derived, such as:

- `calculated_host_listings_count`
- `estimated_occupancy_l365d`
- `estimated_revenue_l365d`
- `minimum_nights_avg_ntm`
- `maximum_nights_avg_ntm`

These should not be treated as raw direct observations without understanding their meaning.

Derived metrics may depend on source-specific formulas or assumptions.

Their interpretation should remain cautious.

---

#### 27. CSV Relationships Are Not Formally Enforced

Although several relationships were validated successfully, the source files are CSV datasets rather than relational database tables.

Therefore:

- primary keys are not formally enforced,
- foreign keys are not formally enforced,
- referential integrity must be tested manually,
- future data versions may produce different results.

All relationships should therefore be described as logical or validated relationships rather than database-enforced constraints.

---

#### Overall Limitation Summary

The most important limitations of `listings.csv.gz` are:

- 57 of 90 columns contain missing data.
- 13 columns are completely empty.
- 38.50% of prices are missing.
- Bathroom and bed information have substantial missingness.
- Approximately 9.87% of review-related fields are missing.
- Large identifier values require precision-safe handling.
- Summary and detailed listing files differ by 96 records.
- Several extreme values require validation.
- Availability is not equivalent to occupancy.
- Review count is not equivalent to booking count.
- Estimated occupancy and revenue are not verified operational measures.
- Multiple scrape dates reduce perfect snapshot consistency.

Despite these limitations, the dataset remains highly valuable because it provides the richest listing-level information available and can support:

- cleaning,
- enrichment,
- host analysis,
- pricing analysis,
- neighbourhood analysis,
- review analysis,
- statistical testing,
- DuckDB modeling,
- business interpretation.

**Overall assessment: Highly valuable for downstream analysis after targeted cleaning, type correction, missing-value handling, domain validation, and documented treatment of derived and extreme values.**

### Data Quality Assessment

The `listings.csv.gz` dataset is the richest listing-level dataset in the Amsterdam Airbnb collection and provides a strong foundation for downstream data engineering, enrichment, EDA, statistical analysis, and business interpretation.

The dataset contains:

- **10,369 rows**
- **90 columns**
- **0 duplicate rows**
- **10,369 unique listing IDs**
- **0 missing listing IDs**
- **0 duplicate listing IDs**

Overall, the dataset is structurally strong at the listing level, but it contains substantial missingness, several extreme values, multiple fully empty columns, and some data-type issues that require targeted cleaning and validation.

---

#### Positive Data Quality Characteristics

Several important quality checks produced strong results.

##### Listing identity

- Total rows: **10,369**
- Unique `id` values: **10,369**
- Missing `id` values: **0**
- Duplicate `id` values: **0**
- Duplicate full rows: **0**

This makes `id` a strong candidate primary key.

The `listing_url` field is also unique and complete, but `id` remains the preferred semantic identifier.

---

#### Neighbourhood Referential Integrity

The usable neighbourhood field is:

`neighbourhood_cleansed`

Observed results:

- Missing values: **0**
- Unique values: **22**

This field was validated against:

`neighbourhoods.csv.neighbourhood`

Validation results:

- Unique neighbourhoods in detailed listings: **22**
- Unique neighbourhoods in neighbourhood reference: **22**
- Unmatched detailed neighbourhoods: **0**
- Unused reference neighbourhoods: **0**
- Matched detailed listing rows: **10,369**
- Total detailed listing rows: **10,369**
- Match coverage: **100.00%**

This is a strong positive data-quality result.

It supports treating:

`listings.csv.gz.neighbourhood_cleansed`

as a logical foreign-key-like reference to:

`neighbourhoods.csv.neighbourhood`

---

#### Detailed and Summary Listings Consistency

The detailed and summary listings datasets were compared using `id`.

Results:

- Unique IDs in `listings.csv.gz`: **10,369**
- Unique IDs in `listings.csv`: **10,465**
- IDs present in both datasets: **10,369**
- IDs only in detailed listings: **0**
- IDs only in summary listings: **96**

Coverage:

- Detailed IDs found in summary listings: **100.00%**
- Summary IDs found in detailed listings: **99.08%**

This shows strong but not perfect cross-dataset consistency.

Every detailed listing exists in the summary dataset, but 96 summary listings are absent from the detailed file.

The cause has not yet been confirmed and requires further row-level investigation.

---

#### Missing-Value Quality Assessment

The dataset contains:

- **90 total columns**
- **57 columns with at least one missing value**
- **13 columns with 100% missingness**

The 13 completely empty columns are:

- `neighborhood_overview`
- `host_since`
- `host_response_time`
- `host_response_rate`
- `host_acceptance_rate`
- `host_thumbnail_url`
- `host_neighbourhood`
- `host_total_listings_count`
- `host_verifications`
- `neighbourhood`
- `neighbourhood_group_cleansed`
- `calendar_updated`
- `instant_bookable`

These fields currently provide no analytical value.

They should remain unchanged in the raw source but may be excluded from cleaned or analytical outputs if not recoverable from another trusted source.

---

#### Missing-Value Severity Assessment

| Column | Missing Count | Missing Percentage | Initial Severity |
|---|---:|---:|---|
| `neighborhood_overview` | 10,369 | 100.00% | Critical |
| `host_since` | 10,369 | 100.00% | Critical |
| `host_response_time` | 10,369 | 100.00% | Critical |
| `host_response_rate` | 10,369 | 100.00% | Critical |
| `host_acceptance_rate` | 10,369 | 100.00% | Critical |
| `host_thumbnail_url` | 10,369 | 100.00% | Critical |
| `host_neighbourhood` | 10,369 | 100.00% | Critical |
| `host_total_listings_count` | 10,369 | 100.00% | Critical |
| `host_verifications` | 10,369 | 100.00% | Critical |
| `neighbourhood` | 10,369 | 100.00% | Critical |
| `neighbourhood_group_cleansed` | 10,369 | 100.00% | Critical |
| `calendar_updated` | 10,369 | 100.00% | Critical |
| `instant_bookable` | 10,369 | 100.00% | Critical |
| `host_about` | 4,959 | 47.83% | High |
| `bathrooms` | 4,956 | 47.80% | High |
| `beds` | 4,652 | 44.86% | High |
| `price` | 3,992 | 38.50% | High |
| `price_quote_total_price` | 3,992 | 38.50% | High |
| `price_quote_price_per_night` | 3,992 | 38.50% | High |
| `estimated_revenue_l365d` | 3,992 | 38.50% | High |
| `price_quote_raw` | 3,861 | 37.24% | High |
| `price_quote_checkin_date` | 3,861 | 37.24% | High |
| `price_quote_checkout_date` | 3,861 | 37.24% | High |
| `host_location` | 1,372 | 13.23% | Moderate |
| `bedrooms` | 1,150 | 11.09% | Moderate |
| Most review-related fields | ~1,023 | ~9.87% | Moderate |
| `description` | 450 | 4.34% | Low |
| `license` | 223 | 2.15% | Low |
| Several host fields | 142 | 1.37% | Low |
| `has_availability` | 136 | 1.31% | Low |
| `bathrooms_text` | 9 | 0.09% | Very Low |
| Several stay-rule fields | 3 | 0.03% | Very Low |

These severity labels are only prioritization aids.

They do not automatically determine whether a field should be dropped, filled, preserved, or transformed.

---

#### Price Data Quality

The `price` field has:

- Missing values: **3,992**
- Missing percentage: **38.50%**
- Unique non-null values: **2,709**

The field is stored as text, with values such as:

`$94.00`

This means it requires cleaning before numerical analysis.

The related numeric field:

`price_quote_price_per_night`

ranges from approximately:

- Minimum: **14.70**
- Maximum: **11,412.00**

The high missing rate and extreme maximum value require careful handling.

Missing prices should not be replaced with zero.

Price-based analyses should clearly report the number of valid records used.

---

#### Bathroom and Bed Information

The numeric bathroom field has:

- Missing values: **4,956**
- Missing percentage: **47.80%**

However:

`bathrooms_text`

has only:

- 9 missing values
- 0.09% missingness

This suggests that bathroom information may still be recoverable from text.

Similarly:

- `beds`: 44.86% missing
- `bedrooms`: 11.09% missing

These fields require context-aware handling.

Zero and missing values must not be treated as equivalent.

---

#### Review Data Quality

Most review-related columns contain approximately 9.87% missingness.

Examples include:

- `first_review`
- `last_review`
- `review_scores_rating`
- `review_scores_accuracy`
- `review_scores_checkin`
- `review_scores_communication`
- `review_scores_location`
- `review_scores_value`
- `reviews_per_month`

Observed rating ranges remain valid:

- Minimum rating: **1.0**
- Maximum rating: **5.0**

This is a positive domain-validation result.

However, missing review values likely have structural meaning and may correspond to listings without review history.

This should be validated before any imputation.

---

#### Date Quality

Available date fields were temporarily parsed for validation.

No invalid or unparseable date values were found.

Observed results include:

- `last_scraped`: 2026-06-15 to 2026-06-24
- `price_quote_checkin_date`: 2026-06-16 to 2027-06-11
- `price_quote_checkout_date`: 2026-06-17 to 2027-06-16
- `calendar_last_scraped`: 2026-06-15 to 2026-06-24
- `first_review`: 2010-08-16 to 2026-06-18
- `last_review`: 2014-04-22 to 2026-06-24

All available values were successfully parsed.

This indicates strong date-format consistency.

The main date-quality limitation is that:

`host_since`

is entirely missing.

---

#### Availability Validation

Observed availability ranges are internally valid:

- `availability_30`: 0 to 30
- `availability_60`: 0 to 60
- `availability_90`: 0 to 90
- `availability_365`: 0 to 365

There are no missing values in these four fields.

This is a strong positive data-quality characteristic.

However, availability must not be interpreted as true occupancy.

---

#### Geographic Validation Readiness

The geographic coordinate fields are complete:

- `latitude`: 0 missing values
- `longitude`: 0 missing values

Observed ranges:

- Latitude: approximately **52.290276 to 52.42516**
- Longitude: approximately **4.75587 to 5.02815**

These values appear plausible for Amsterdam, but a formal geographic validation step should still confirm that coordinates fall within appropriate city bounds.

---

#### Room-Type Consistency

The `room_type` field contains four categories:

- Entire home/apt: **8,489**
- Private room: **1,833**
- Hotel room: **26**
- Shared room: **21**

There are:

- 0 missing room-type values
- 4 unique categories

This is a strong positive quality result.

No unexpected room-type category was observed.

---

#### Property-Type Diversity

The `property_type` field contains:

- **59 unique categories**

The largest categories include:

- Entire rental unit
- Entire condo
- Entire home
- Private room in rental unit
- Room in hotel
- Private room in bed and breakfast
- Entire loft
- Entire townhouse
- Houseboat

The field is complete with no missing values.

Its high category count is not inherently a quality problem but may require grouping for some analyses.

---

#### Host Data Quality

The `host_id` field is complete:

- Missing values: **0**
- Unique host IDs: **9,104**

This supports strong host-level grouping.

However, several host attributes contain missing values.

A recurring pattern of **142 missing values** appears across fields such as:

- `host_profile_id`
- `host_name`
- `host_is_superhost`
- `host_has_profile_pic`
- `host_identity_verified`
- `host_listings_count`

This repeated count suggests a shared missingness pattern affecting a subset of host records.

Further investigation is warranted.

---

#### Identifier Precision Risk

The `host_profile_id` field is stored as:

`float64`

It contains very large values approximately around `10^18`.

This creates a possible precision-loss risk.

Before using this field for:

- joins,
- comparisons,
- grouping,
- deduplication,

it should be converted to a precision-safe type.

This issue is important because incorrect identifier representation can break referential integrity.

---

#### Extreme-Value Validation Priorities

Several fields contain unusually large values.

Examples include:

- Maximum `minimum_nights`: **800**
- Maximum `maximum_nights`: **2,147,483,647**
- Maximum `beds`: **33**
- Maximum `bedrooms`: **17**
- Maximum `bathrooms`: **17**
- Maximum `number_of_reviews`: **5,603**
- Maximum `number_of_reviews_ltm`: **813**
- Maximum `number_of_reviews_l30d`: **41**
- Maximum `reviews_per_month`: **92.56**
- Maximum `estimated_revenue_l365d`: **344,955**

These values should be investigated, not automatically removed.

Some may represent:

- valid extreme records,
- special properties,
- host-defined restrictions,
- source-system defaults,
- sentinel values,
- data anomalies.

---

#### Maximum-Stay Sentinel Risk

The value:

`2,147,483,647`

appears in several maximum-stay-related fields.

This is noteworthy because it equals the maximum signed 32-bit integer.

It may indicate:

- a system default,
- an effectively unlimited stay value,
- a sentinel,
- a source artifact.

This is not yet confirmed and should remain a validation hypothesis.

---

#### Source Consistency

The `source` field contains:

- `city scrape`: **5,864**
- `previous scrape`: **4,505**

This means the dataset includes records from different source states.

The dataset also contains multiple scrape dates.

Therefore, the file should not automatically be interpreted as a perfectly synchronized single-day snapshot.

This may explain some completeness differences and should be considered during later validation.

---

#### Cross-Dataset 96-Listing Pattern

A notable pattern exists across the detailed and summary listing files.

Observed differences:

- Summary listings contain 96 more rows.
- Summary listings contain 96 more private-room records.
- Summary listings contain 96 missing `host_id` values.

This is a strong numerical pattern.

However, row-level validation has not yet confirmed that the same 96 listings explain every difference.

Therefore, the correct quality interpretation is:

**A strong cross-dataset consistency pattern requiring explicit confirmation.**

It should not yet be treated as a proven causal explanation.

---

#### Main Data Quality Strengths

The strongest characteristics of `listings.csv.gz` are:

- complete and unique listing IDs,
- zero duplicate rows,
- 100% neighbourhood reference coverage,
- complete geographic coordinates,
- complete room-type information,
- internally valid availability ranges,
- valid 1-to-5 review-score ranges,
- fully parseable available date values,
- complete host IDs,
- rich 90-column schema.

---

#### Main Data Quality Concerns

The most important concerns are:

1. 57 of 90 columns contain missing values.
2. 13 columns are entirely empty.
3. Price is missing for 38.50% of listings.
4. Bathrooms and beds have substantial missingness.
5. Review-related fields are missing for approximately 9.87% of listings.
6. `host_profile_id` may suffer from floating-point precision risk.
7. Extreme values require domain validation.
8. Detailed and summary listing files differ by 96 records.
9. Multiple scrape dates reduce perfect snapshot consistency.
10. Some source-derived variables require cautious interpretation.

---

#### Overall Data Quality Assessment

Overall, `listings.csv.gz` is highly valuable and structurally strong at the listing level.

Its best qualities are:

- unique listing identity,
- no duplicate rows,
- strong neighbourhood referential integrity,
- complete location and room-type information,
- valid availability ranges,
- parseable date values.

Its main weaknesses are:

- widespread missingness,
- 13 fully empty columns,
- substantial missing price data,
- incomplete property-capacity fields,
- possible identifier precision risks,
- extreme values requiring investigation,
- imperfect consistency with the summary listing file.

**Overall assessment: Highly suitable for downstream cleaning, enrichment, EDA, statistical testing, and DuckDB modeling after targeted missing-value handling, precision-safe identifier treatment, price cleaning, domain validation, and documented treatment of extreme and derived values.**

In [87]:
# Release the detailed listings DataFrame now that profiling is complete.
# Compact profiling summaries and ID sets remain available.

del detailed_listings_df
gc.collect()

print("Detailed listings DataFrame released from memory.")

Detailed listings DataFrame released from memory.


## 5. Neighbourhoods GeoJSON Dataset Familiarization

The `neighbourhoods.geojson` file contains geographic boundary information for Amsterdam neighbourhoods.

Unlike the CSV datasets, this file uses the GeoJSON format, which represents geographic features using:

- Geometry
- Coordinates
- Feature properties
- Geographic boundary polygons

This dataset will be inspected for:

- File structure
- Top-level GeoJSON type
- Number of geographic features
- Feature properties
- Geometry types
- Missing properties
- Duplicate neighbourhood identifiers
- Candidate geographic keys
- Relationship with `neighbourhoods.csv`
- Relationship with `listings.csv.gz`
- Dataset limitations
- Business-domain meaning

The main relationship to investigate is expected to involve a neighbourhood name or identifier stored in the GeoJSON feature properties.

The exact join field will be verified from the actual file structure rather than assumed.

Import JSON library and load the GeoJSON

In [88]:
import json

with open(
    DATA_FILES["neighbourhoods_geojson"],
    "r",
    encoding="utf-8"
) as file:
    neighbourhoods_geojson = json.load(file)

print("neighbourhoods.geojson loaded successfully.")

neighbourhoods.geojson loaded successfully.


Inspect top-level GeoJSON structure

In [89]:
print("Top-Level GeoJSON Structure")
print("-" * 50)

print(f"Top-level type: {type(neighbourhoods_geojson).__name__}")
print(f"Top-level keys: {list(neighbourhoods_geojson.keys())}")
print(f"GeoJSON type: {neighbourhoods_geojson.get('type')}")

Top-Level GeoJSON Structure
--------------------------------------------------
Top-level type: dict
Top-level keys: ['type', 'features']
GeoJSON type: FeatureCollection


Count geographic features

In [90]:
features = neighbourhoods_geojson.get("features", [])

print("GeoJSON Feature Count")
print("-" * 40)
print(f"Number of geographic features: {len(features):,}")

GeoJSON Feature Count
----------------------------------------
Number of geographic features: 22


Inspect the first feature structure

In [91]:
if features:
    first_feature = features[0]

    print("First Feature Structure")
    print("-" * 50)

    print(f"Feature keys: {list(first_feature.keys())}")
    print(f"Feature type: {first_feature.get('type')}")
    print(
        f"Property keys: "
        f"{list(first_feature.get('properties', {}).keys())}"
    )
    print(
        f"Geometry type: "
        f"{first_feature.get('geometry', {}).get('type')}"
    )
else:
    print("No geographic features found.")

First Feature Structure
--------------------------------------------------
Feature keys: ['type', 'geometry', 'properties']
Feature type: Feature
Property keys: ['neighbourhood', 'neighbourhood_group']
Geometry type: MultiPolygon


Display first feature properties

In [92]:
if features:
    first_properties = first_feature.get(
        "properties",
        {}
    )

    print("First Feature Properties")
    print("-" * 50)

    for key, value in first_properties.items():
        print(f"{key}: {value}")

First Feature Properties
--------------------------------------------------
neighbourhood: Bijlmer-Oost
neighbourhood_group: None


Inspect geometry type

In [93]:
if features:
    first_geometry = first_feature.get(
        "geometry",
        {}
    )

    print("First Feature Geometry")
    print("-" * 50)

    print(
        f"Geometry type: "
        f"{first_geometry.get('type')}"
    )

    coordinates = first_geometry.get(
        "coordinates"
    )

    print(
        f"Coordinates object type: "
        f"{type(coordinates).__name__}"
    )

First Feature Geometry
--------------------------------------------------
Geometry type: MultiPolygon
Coordinates object type: list


Geometry-type distribution

In [94]:
geometry_types = []

for feature in features:
    geometry = feature.get("geometry") or {}

    geometry_types.append(
        geometry.get("type")
    )

geometry_type_counts = (
    pd.Series(
        geometry_types,
        name="geometry_type"
    )
    .value_counts(dropna=False)
    .rename_axis("geometry_type")
    .reset_index(name="feature_count")
)

display(geometry_type_counts)

,geometry_type,feature_count
0,MultiPolygon,22


Convert feature properties to a compact DataFrame

In [95]:
geojson_properties = []

for feature_index, feature in enumerate(features):
    properties = feature.get("properties") or {}

    geojson_properties.append({
        "feature_index": feature_index,
        **properties
    })

geojson_properties_df = pd.DataFrame(
    geojson_properties
)

print("GeoJSON properties extracted successfully.")
print(f"Rows: {geojson_properties_df.shape[0]:,}")
print(f"Columns: {geojson_properties_df.shape[1]}")

display(geojson_properties_df.head())

GeoJSON properties extracted successfully.
Rows: 22
Columns: 3


,feature_index,neighbourhood,neighbourhood_group
0,0,Bijlmer-Oost,None
1,1,Oud-Noord,None
2,2,Noord-Oost,None
3,3,Noord-West,None
4,4,IJburg - Zeeburgereiland,None


Show property column names

In [96]:
print("GeoJSON Property Column Names")
print("-" * 50)

for index, column in enumerate(
    geojson_properties_df.columns,
    start=1
):
    print(f"{index}. {column}")

GeoJSON Property Column Names
--------------------------------------------------
1. feature_index
2. neighbourhood
3. neighbourhood_group


Property data types

In [97]:
geojson_property_dtypes = pd.DataFrame({
    "column_name": geojson_properties_df.columns,
    "data_type": (
        geojson_properties_df
        .dtypes
        .astype(str)
        .values
    )
})

display(geojson_property_dtypes)

,column_name,data_type
0,feature_index,int64
1,neighbourhood,str
2,neighbourhood_group,object


Missing-value analysis

In [98]:
geojson_missing_summary = pd.DataFrame({
    "column_name": geojson_properties_df.columns,
    "missing_count": (
        geojson_properties_df
        .isnull()
        .sum()
        .values
    ),
    "missing_percentage": (
        geojson_properties_df
        .isnull()
        .mean()
        .mul(100)
        .round(2)
        .values
    )
})

display(
    geojson_missing_summary
    .sort_values(
        by="missing_percentage",
        ascending=False
    )
    .reset_index(drop=True)
)

,column_name,missing_count,missing_percentage
0,neighbourhood_group,22,100.0
1,feature_index,0,0.0
2,neighbourhood,0,0.0


Unique-value counts

In [99]:
geojson_unique_summary = pd.DataFrame({
    "column_name": geojson_properties_df.columns,
    "unique_count": [
        geojson_properties_df[column]
        .nunique(dropna=True)
        for column in geojson_properties_df.columns
    ]
})

display(geojson_unique_summary)

,column_name,unique_count
0,feature_index,22
1,neighbourhood,22
2,neighbourhood_group,0


Duplicate feature analysis

In [100]:
geojson_duplicate_count = (
    geojson_properties_df
    .drop(columns=["feature_index"])
    .duplicated()
    .sum()
)

print("GeoJSON Duplicate Feature Property Analysis")
print("-" * 50)
print(f"Total features: {len(features):,}")
print(f"Duplicate property rows: {geojson_duplicate_count:,}")

duplicate_percentage = (
    geojson_duplicate_count
    / len(features)
    * 100
    if features
    else 0
)

print(f"Duplicate percentage: {duplicate_percentage:.2f}%")

GeoJSON Duplicate Feature Property Analysis
--------------------------------------------------
Total features: 22
Duplicate property rows: 0
Duplicate percentage: 0.00%


Candidate key validation

In [101]:
geojson_candidate_keys = []

source_property_columns = [
    column
    for column in geojson_properties_df.columns
    if column != "feature_index"
]

total_features = len(geojson_properties_df)

for column in source_property_columns:
    unique_count = (
        geojson_properties_df[column]
        .nunique(dropna=False)
    )

    missing_count = (
        geojson_properties_df[column]
        .isnull()
        .sum()
    )

    is_unique = unique_count == total_features
    has_no_missing_values = missing_count == 0

    if is_unique and has_no_missing_values:
        geojson_candidate_keys.append(column)

print("Candidate Key Analysis")
print("-" * 40)

if geojson_candidate_keys:
    print("Possible source-property candidate keys:")

    for column in geojson_candidate_keys:
        print(f"- {column}")
else:
    print("No source-property candidate key identified.")

Candidate Key Analysis
----------------------------------------
Possible source-property candidate keys:
- neighbourhood


Explicit neighbourhood-key validation

In [102]:
print("GeoJSON Neighbourhood Key Validation")
print("-" * 50)

print(
    f"Total features: "
    f"{len(geojson_properties_df):,}"
)

print(
    f"Unique neighbourhoods: "
    f"{geojson_properties_df['neighbourhood'].nunique(dropna=True):,}"
)

print(
    f"Missing neighbourhoods: "
    f"{geojson_properties_df['neighbourhood'].isnull().sum():,}"
)

print(
    f"Duplicate neighbourhoods: "
    f"{geojson_properties_df['neighbourhood'].duplicated().sum():,}"
)

GeoJSON Neighbourhood Key Validation
--------------------------------------------------
Total features: 22
Unique neighbourhoods: 22
Missing neighbourhoods: 0
Duplicate neighbourhoods: 0


Validate relationship with neighbourhoods.csv

In [103]:
geojson_neighbourhoods = set(
    geojson_properties_df["neighbourhood"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

csv_neighbourhoods = set(
    neighbourhoods_df["neighbourhood"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

geojson_not_in_csv = (
    geojson_neighbourhoods
    - csv_neighbourhoods
)

csv_not_in_geojson = (
    csv_neighbourhoods
    - geojson_neighbourhoods
)

print("GeoJSON-to-CSV Neighbourhood Relationship Validation")
print("-" * 60)

print(
    f"Unique neighbourhoods in GeoJSON: "
    f"{len(geojson_neighbourhoods)}"
)

print(
    f"Unique neighbourhoods in neighbourhoods.csv: "
    f"{len(csv_neighbourhoods)}"
)

print(
    f"GeoJSON neighbourhoods not found in CSV: "
    f"{len(geojson_not_in_csv)}"
)

print(
    f"CSV neighbourhoods not found in GeoJSON: "
    f"{len(csv_not_in_geojson)}"
)

GeoJSON-to-CSV Neighbourhood Relationship Validation
------------------------------------------------------------
Unique neighbourhoods in GeoJSON: 22
Unique neighbourhoods in neighbourhoods.csv: 22
GeoJSON neighbourhoods not found in CSV: 0
CSV neighbourhoods not found in GeoJSON: 0


In [104]:
if geojson_not_in_csv:
    print("\nGeoJSON neighbourhoods not found in CSV:")

    for neighbourhood in sorted(geojson_not_in_csv):
        print(f"- {neighbourhood}")
else:
    print(
        "\nAll GeoJSON neighbourhoods match "
        "neighbourhoods.csv."
    )

if csv_not_in_geojson:
    print("\nCSV neighbourhoods not found in GeoJSON:")

    for neighbourhood in sorted(csv_not_in_geojson):
        print(f"- {neighbourhood}")
else:
    print(
        "Every CSV neighbourhood has a corresponding "
        "GeoJSON feature."
    )


All GeoJSON neighbourhoods match neighbourhoods.csv.
Every CSV neighbourhood has a corresponding GeoJSON feature.


Calculate relationship coverage

In [105]:
matched_geojson_neighbourhoods = (
    geojson_neighbourhoods
    & csv_neighbourhoods
)

geojson_relationship_coverage = (
    len(matched_geojson_neighbourhoods)
    / len(geojson_neighbourhoods)
    * 100
    if geojson_neighbourhoods
    else 0
)

print("GeoJSON Neighbourhood Relationship Coverage")
print("-" * 50)

print(
    f"Matched GeoJSON neighbourhoods: "
    f"{len(matched_geojson_neighbourhoods)}"
)

print(
    f"Total GeoJSON neighbourhoods: "
    f"{len(geojson_neighbourhoods)}"
)

print(
    f"Relationship coverage: "
    f"{geojson_relationship_coverage:.2f}%"
)

GeoJSON Neighbourhood Relationship Coverage
--------------------------------------------------
Matched GeoJSON neighbourhoods: 22
Total GeoJSON neighbourhoods: 22
Relationship coverage: 100.00%


Geometry completeness validation

In [106]:
geometry_validation_results = []

for feature_index, feature in enumerate(features):
    geometry = feature.get("geometry")

    if geometry is None:
        geometry_type = None
        coordinates_present = False
    else:
        geometry_type = geometry.get("type")
        coordinates = geometry.get("coordinates")

        coordinates_present = (
            coordinates is not None
            and len(coordinates) > 0
        )

    properties = feature.get("properties") or {}

    geometry_validation_results.append({
        "feature_index": feature_index,
        "neighbourhood": properties.get("neighbourhood"),
        "geometry_type": geometry_type,
        "geometry_missing": geometry is None,
        "coordinates_present": coordinates_present,
    })

geojson_geometry_validation_df = pd.DataFrame(
    geometry_validation_results
)

display(geojson_geometry_validation_df)

,feature_index,neighbourhood,geometry_type,geometry_missing,coordinates_present
0,0,Bijlmer-Oost,MultiPolygon,False,True
1,1,Oud-Noord,MultiPolygon,False,True
2,2,Noord-Oost,MultiPolygon,False,True
3,3,Noord-West,MultiPolygon,False,True
4,4,IJburg - Zeeburgereiland,MultiPolygon,False,True
5,5,Centrum-West,MultiPolygon,False,True
6,6,Oostelijk Havengebied - Indische Buurt,MultiPolygon,False,True
7,7,Centrum-Oost,MultiPolygon,False,True
8,8,Oud-Oost,MultiPolygon,False,True
9,9,Westerpark,MultiPolygon,False,True


In [107]:
print("Geometry Completeness Summary")
print("-" * 45)

print(
    f"Total features: "
    f"{len(geojson_geometry_validation_df):,}"
)

print(
    f"Missing geometries: "
    f"{geojson_geometry_validation_df['geometry_missing'].sum():,}"
)

print(
    f"Features without coordinates: "
    f"{(~geojson_geometry_validation_df['coordinates_present']).sum():,}"
)

Geometry Completeness Summary
---------------------------------------------
Total features: 22
Missing geometries: 0
Features without coordinates: 0


Validate geometry types

In [108]:
unexpected_geometry_types = (
    geojson_geometry_validation_df.loc[
        geojson_geometry_validation_df["geometry_type"]
        != "MultiPolygon",
        "geometry_type"
    ]
    .dropna()
    .unique()
    .tolist()
)

print("Geometry Type Validation")
print("-" * 40)

if unexpected_geometry_types:
    print(
        "Unexpected geometry types found: "
        f"{unexpected_geometry_types}"
    )
else:
    print(
        "All 22 geographic features use "
        "MultiPolygon geometry."
    )

Geometry Type Validation
----------------------------------------
All 22 geographic features use MultiPolygon geometry.


Validate relationship with detailed listings

In [109]:
geojson_not_in_detailed_listings = (
    geojson_neighbourhoods
    - detailed_neighbourhoods
)

detailed_listings_not_in_geojson = (
    detailed_neighbourhoods
    - geojson_neighbourhoods
)

print("GeoJSON-to-Detailed Listings Relationship")
print("-" * 55)

print(
    f"GeoJSON neighbourhoods not used by detailed listings: "
    f"{len(geojson_not_in_detailed_listings)}"
)

print(
    f"Detailed listing neighbourhoods absent from GeoJSON: "
    f"{len(detailed_listings_not_in_geojson)}"
)

GeoJSON-to-Detailed Listings Relationship
-------------------------------------------------------
GeoJSON neighbourhoods not used by detailed listings: 0
Detailed listing neighbourhoods absent from GeoJSON: 0


In [110]:
if (
    not geojson_not_in_detailed_listings
    and not detailed_listings_not_in_geojson
):
    print(
        "All 22 GeoJSON neighbourhoods match "
        "the detailed listings neighbourhood set."
    )

All 22 GeoJSON neighbourhoods match the detailed listings neighbourhood set.


Dataset-level summary

In [111]:
geojson_dataset_summary = {
    "dataset_name": "neighbourhoods_geojson",
    "file_name": DATA_FILES["neighbourhoods_geojson"].name,
    "geojson_type": neighbourhoods_geojson.get("type"),
    "feature_count": len(features),
    "property_column_count": len(
        geojson_properties_df.columns
    ) - 1,  # exclude generated feature_index
    "duplicate_property_rows": int(
        geojson_duplicate_count
    ),
    "candidate_keys": geojson_candidate_keys,
    "geometry_types": (
        geometry_type_counts["geometry_type"]
        .dropna()
        .tolist()
    ),
    "missing_geometries": int(
        geojson_geometry_validation_df[
            "geometry_missing"
        ].sum()
    ),
    "relationship_coverage_with_neighbourhoods_csv": round(
        geojson_relationship_coverage,
        2
    ),
}

geojson_dataset_summary

{'dataset_name': 'neighbourhoods_geojson',
 'file_name': 'neighbourhoods.geojson',
 'geojson_type': 'FeatureCollection',
 'feature_count': 22,
 'property_column_count': 2,
 'duplicate_property_rows': 0,
 'candidate_keys': ['neighbourhood'],
 'geometry_types': ['MultiPolygon'],
 'missing_geometries': 0,
 'relationship_coverage_with_neighbourhoods_csv': 100.0}

Save profile results

In [112]:
geojson_properties_profile_path = (
    OUTPUT_DIR / "neighbourhoods_geojson_property_profile.csv"
)

geojson_properties_df.to_csv(
    geojson_properties_profile_path,
    index=False
)

geojson_geometry_profile_path = (
    OUTPUT_DIR / "neighbourhoods_geojson_geometry_validation.csv"
)

geojson_geometry_validation_df.to_csv(
    geojson_geometry_profile_path,
    index=False
)

print("GeoJSON profiling outputs saved successfully.")

print(
    f"\nProperty profile:\n"
    f"{geojson_properties_profile_path}"
)

print(
    f"\nGeometry validation:\n"
    f"{geojson_geometry_profile_path}"
)

GeoJSON profiling outputs saved successfully.

Property profile:
c:\Users\saths\Desktop\MyProjects\Expernetic Data Challenge\airbnb-data-challenge\outputs\data_quality\neighbourhoods_geojson_property_profile.csv

Geometry validation:
c:\Users\saths\Desktop\MyProjects\Expernetic Data Challenge\airbnb-data-challenge\outputs\data_quality\neighbourhoods_geojson_geometry_validation.csv


### Findings and Interpretation

The `neighbourhoods.geojson` dataset contains geographic boundary information for Amsterdam neighbourhoods.

The file is structured as a GeoJSON:

`FeatureCollection`

with:

- **22 geographic features**
- **2 source property fields**
- `neighbourhood`
- `neighbourhood_group`

Each geographic feature represents one Amsterdam neighbourhood and contains a corresponding `MultiPolygon` geometry.

---

#### Key Structural Findings

Observed results:

- GeoJSON type: **FeatureCollection**
- Total geographic features: **22**
- Source property columns: **2**
- Duplicate property rows: **0**
- Duplicate percentage: **0.00%**
- Unique neighbourhood values: **22**
- Missing neighbourhood values: **0**
- Duplicate neighbourhood values: **0**
- Missing geometries: **0**
- Features without coordinates: **0**
- Geometry type: **MultiPolygon for all 22 features**

These results indicate a clean geographic reference structure.

---

#### Candidate Geographic Key

The `neighbourhood` field was identified as a strong candidate key because:

- Total features: **22**
- Unique neighbourhoods: **22**
- Missing neighbourhoods: **0**
- Duplicate neighbourhoods: **0**

Therefore:

**Candidate geographic key: `neighbourhood`**

This field can be used to connect GeoJSON boundary features to neighbourhood-level reference and listing datasets.

---

#### Neighbourhood Group Missingness

The `neighbourhood_group` property is completely missing.

Observed results:

- Missing values: **22**
- Missing percentage: **100.00%**
- Unique non-null values: **0**

Therefore, `neighbourhood_group` currently provides no usable analytical information.

This is consistent with the earlier `neighbourhoods.csv` dataset, where the corresponding `neighbourhood_group` field was also 100% missing.

This cross-dataset consistency suggests that higher-level neighbourhood-group information is not available in the current Amsterdam source files.

The original raw data should remain unchanged, while the empty field may be excluded from processed analytical outputs if it is not required.

---

#### Relationship with `neighbourhoods.csv`

The GeoJSON neighbourhood names were explicitly validated against the neighbourhood reference CSV.

Validation results:

- Unique neighbourhoods in GeoJSON: **22**
- Unique neighbourhoods in `neighbourhoods.csv`: **22**
- GeoJSON neighbourhoods not found in CSV: **0**
- CSV neighbourhoods not found in GeoJSON: **0**
- Matched neighbourhoods: **22**
- Relationship coverage: **100.00%**

Therefore, every neighbourhood in the GeoJSON has a corresponding record in `neighbourhoods.csv`, and every neighbourhood in the CSV has a corresponding geographic boundary feature.

This supports the logical relationship:

```text
neighbourhoods.csv
    neighbourhood
          │
          │ 1-to-1 logical relationship
          ▼
neighbourhoods.geojson
    neighbourhood
````

The relationship is complete for all 22 Amsterdam neighbourhoods.

---

#### Relationship with Detailed Listings

The GeoJSON neighbourhood set was also compared with:

`listings.csv.gz.neighbourhood_cleansed`

Results:

* GeoJSON neighbourhoods not used by detailed listings: **0**
* Detailed listing neighbourhoods absent from GeoJSON: **0**

Therefore, all 22 neighbourhoods represented in the detailed listings dataset have corresponding geographic boundary features.

The relationship can be represented as:

```text
neighbourhoods.geojson
    neighbourhood
          │
          │ geographic lookup relationship
          ▼
listings.csv.gz
    neighbourhood_cleansed
```

This supports future geographic analysis such as:

* listing density by neighbourhood,
* average price by neighbourhood,
* availability patterns,
* review activity,
* mapping listing markets using neighbourhood polygons.

---

#### Geometry Completeness

All 22 geographic features contain valid geometry objects.

Observed results:

* Missing geometries: **0**
* Features without coordinates: **0**

Therefore, every neighbourhood has usable geographic boundary information.

This is a strong positive data-quality result.

---

#### Geometry Type Consistency

All 22 geographic features use:

`MultiPolygon`

geometry.

No unexpected geometry types were found.

This consistency simplifies later spatial processing because the project does not need to handle a mixture of:

* `Polygon`
* `MultiPolygon`
* `Point`
* or other geometry types.

Every neighbourhood boundary can be interpreted using the same general geographic structure.

---

#### Business Interpretation

The GeoJSON file functions as the geographic boundary layer for the Amsterdam Airbnb dataset collection.

Its main purpose is not transactional analysis but geographic enrichment.

It can be used to connect neighbourhood-level business metrics with actual geographic shapes.

For example, later analysis may produce metrics such as:

* listing count by neighbourhood,
* median price by neighbourhood,
* room-type distribution,
* availability rate,
* review activity.

These values can then be joined to the GeoJSON and displayed on a choropleth or neighbourhood map.

The logical flow is:

```text
Detailed Listings
       │
       │ aggregate by neighbourhood
       ▼
Neighbourhood Metrics
       │
       │ join using neighbourhood name
       ▼
GeoJSON Boundaries
       │
       ▼
Geographic Visualization
```

---

#### Important Geographic Data Interpretation

The GeoJSON polygons define neighbourhood boundaries rather than individual listing locations.

Individual listing coordinates are available separately through:

* `latitude`
* `longitude`

in the detailed listings dataset.

Therefore:

* GeoJSON should be used for neighbourhood boundary visualization and aggregation.
* Listing coordinates should be used for point-level geographic analysis.

These two geographic sources provide complementary levels of detail.

---

#### Overall Interpretation

The `neighbourhoods.geojson` dataset is structurally clean and highly consistent with the other Amsterdam geographic reference data.

Its strongest characteristics are:

* 22 complete geographic features,
* 22 unique neighbourhood names,
* zero duplicate property rows,
* zero missing geometries,
* zero features without coordinates,
* consistent `MultiPolygon` geometry,
* 100% relationship coverage with `neighbourhoods.csv`,
* complete neighbourhood alignment with the detailed listings dataset.

The only notable limitation identified so far is that:

`neighbourhood_group`

is entirely missing.

Overall assessment:

**The dataset is suitable for neighbourhood-level geographic enrichment and mapping without major structural cleaning, although spatial analysis should still be treated as optional unless the core engineering, EDA, statistics, and report requirements are already complete.**



### Candidate Keys and Relationships

The `neighbourhoods.geojson` dataset acts as the geographic boundary reference layer for the Amsterdam Airbnb datasets.

Each GeoJSON feature represents one neighbourhood and contains:

- a `neighbourhood` property,
- a `neighbourhood_group` property,
- a geographic `MultiPolygon` geometry.

The main purpose of this dataset is to connect neighbourhood-level analytical results with geographic boundary shapes.

---

#### Candidate Geographic Key

The `neighbourhood` property was identified as a strong candidate key.

Validation results:

- Total geographic features: **22**
- Unique `neighbourhood` values: **22**
- Missing `neighbourhood` values: **0**
- Duplicate `neighbourhood` values: **0**

Therefore:

**Candidate geographic key: `neighbourhood`**

This field uniquely identifies each neighbourhood feature in the current GeoJSON dataset.

Because the source is a GeoJSON file rather than a relational database with formally enforced key constraints, `neighbourhood` should be described as a candidate key based on observed uniqueness and completeness.

---

#### Non-Key Property: `neighbourhood_group`

The `neighbourhood_group` property cannot be used as a key because:

- Missing values: **22**
- Missing percentage: **100.00%**
- Unique non-null values: **0**

Therefore, it contains no usable identifying or grouping information in the current dataset.

This is consistent with the corresponding field in `neighbourhoods.csv`, which is also completely missing.

---

#### Relationship with `neighbourhoods.csv`

The `neighbourhood` property in the GeoJSON was explicitly validated against:

`neighbourhoods.csv.neighbourhood`

The logical relationship is:

```text
neighbourhoods.csv
    neighbourhood
          │
          │ one-to-one logical relationship
          ▼
neighbourhoods.geojson
    neighbourhood
````

Validation results:

* Unique neighbourhoods in GeoJSON: **22**
* Unique neighbourhoods in `neighbourhoods.csv`: **22**
* GeoJSON neighbourhoods not found in CSV: **0**
* CSV neighbourhoods not found in GeoJSON: **0**
* Matched neighbourhoods: **22**
* Relationship coverage: **100.00%**

Therefore, every GeoJSON feature has a corresponding neighbourhood record in the CSV reference dataset, and every CSV neighbourhood has a corresponding GeoJSON feature.

The observed relationship is effectively:

**One neighbourhood reference record to one geographic boundary feature.**

Because the source files are not relational database tables, this relationship is logical rather than formally enforced.

---

#### Relationship with Detailed Listings

The GeoJSON neighbourhood values were also compared with:

`listings.csv.gz.neighbourhood_cleansed`

The relationship is:

```text
neighbourhoods.geojson
    neighbourhood
          │
          │ one neighbourhood to many listings
          ▼
listings.csv.gz
    neighbourhood_cleansed
```

Validation results:

* GeoJSON neighbourhoods not used by detailed listings: **0**
* Detailed listing neighbourhoods absent from GeoJSON: **0**

Therefore, all 22 neighbourhoods represented in the detailed listings dataset have corresponding geographic boundary features.

The relationship type is:

**One-to-many**

because one neighbourhood boundary may be associated with many listing records.

This can be represented as:

```text
Neighbourhood Boundary
          │
          │ 1
          │
          │
          │ many
          ▼
       Listings
```

---

#### Relationship with Summary Listings

The summary listings dataset also contains a `neighbourhood` field.

The logical relationship is:

```text
neighbourhoods.geojson
    neighbourhood
          │
          ▼
listings.csv
    neighbourhood
```

Earlier validation showed that:

* `listings.csv` contains all 22 neighbourhoods,
* all listing neighbourhood values match `neighbourhoods.csv`,
* neighbourhood join coverage is 100.00%.

Because the GeoJSON and CSV neighbourhood reference datasets also match perfectly, the GeoJSON can logically be connected to the summary listings dataset through the same neighbourhood values.

The relationship type is:

**One neighbourhood boundary to many summary listing records.**

---

#### Relationship Across Geographic Reference Files

The three main geographic representations can be viewed together as:

```text
neighbourhoods.csv
    neighbourhood
          │
          │ 1-to-1
          ▼
neighbourhoods.geojson
    neighbourhood
          │
          │ 1-to-many
          ▼
listings.csv.gz
    neighbourhood_cleansed
```

The summary listings dataset can also be included:

```text
                     neighbourhoods.csv
                         neighbourhood
                              │
                              │ 1-to-1
                              ▼
                   neighbourhoods.geojson
                         neighbourhood
                         /            \
                        /              \
                       ▼                ▼
              listings.csv       listings.csv.gz
              neighbourhood      neighbourhood_cleansed
```

This provides a consistent geographic relationship structure across the Amsterdam datasets.

---

#### Geographic Relationship Cardinality

The observed relationships are:

##### `neighbourhoods.csv` to `neighbourhoods.geojson`

**One-to-one**

Each of the 22 neighbourhood reference rows has exactly one matching geographic boundary feature.

---

##### `neighbourhoods.geojson` to `listings.csv.gz`

**One-to-many**

One neighbourhood boundary may contain many Airbnb listings.

---

##### `neighbourhoods.geojson` to `listings.csv`

**One-to-many**

One neighbourhood boundary may correspond to many summary listing records.

---

#### Geometry Relationship

Each neighbourhood feature also contains one `MultiPolygon` geometry.

Therefore, the feature structure can be represented as:

```text
Neighbourhood Name
       │
       │ identifies
       ▼
GeoJSON Feature
       │
       │ contains
       ▼
MultiPolygon Geometry
```

Observed results:

* Total features: **22**
* MultiPolygon geometries: **22**
* Missing geometries: **0**
* Features without coordinates: **0**

This means every candidate neighbourhood key is associated with a complete geographic boundary.

---

#### Geographic Enrichment Relationship

A future analytical workflow may use the relationship:

```text
Listings
    │
    │ group by neighbourhood
    ▼
Neighbourhood-Level Metrics
    │
    │ join on neighbourhood
    ▼
GeoJSON Boundary Features
    │
    ▼
Map Visualization
```

For example, listing data may first be aggregated into measures such as:

* listing count,
* median price,
* average availability,
* room-type distribution,
* average review score.

These aggregated results can then be joined to the GeoJSON using:

`neighbourhood`

This enables choropleth maps and geographic business analysis without joining individual listing rows directly to complex geometry objects.

---

#### Relationship Validation Status

Validated relationships:

* `neighbourhoods.geojson.neighbourhood`
  → `neighbourhoods.csv.neighbourhood`

  * Status: **Validated**
  * Coverage: **100.00%**
  * Relationship type: **One-to-one**

* `neighbourhoods.geojson.neighbourhood`
  → `listings.csv.gz.neighbourhood_cleansed`

  * Status: **Validated**
  * Coverage across neighbourhood sets: **100.00%**
  * Relationship type: **One-to-many**

Logical relationship supported by previous validation:

* `neighbourhoods.geojson.neighbourhood`
  → `listings.csv.neighbourhood`

  * Status: **Supported through matching reference values**
  * Relationship type: **One-to-many**

---

#### Key Engineering Interpretation

The `neighbourhoods.geojson` dataset is the geographic representation of the Amsterdam neighbourhood entity.

Its `neighbourhood` property acts as the main logical join key connecting:

* neighbourhood reference metadata,
* detailed listings,
* summary listings,
* future aggregated business metrics.

The key relationship structure is:

```text
Neighbourhood Reference
          │
          ▼
Geographic Boundary
          │
          ▼
Listing-Level Data
          │
          ▼
Neighbourhood Metrics and Maps
```

This makes the GeoJSON valuable for geographic enrichment and visualization while keeping the analytical grain separate from the individual listing records.

### Business-Domain Meaning

The `neighbourhoods.geojson` dataset represents the geographic boundary layer for Amsterdam neighbourhoods.

Unlike the CSV datasets, which primarily contain tabular attributes, this file stores geographic shapes that define the spatial boundaries of each neighbourhood.

The grain of the dataset is:

**One GeoJSON feature represents one Amsterdam neighbourhood boundary.**

Each feature contains:

- `neighbourhood`
- `neighbourhood_group`
- `geometry`

The geometry is represented using:

`MultiPolygon`

for all 22 neighbourhood features.

---

#### Main Business Entity Represented

The primary business entity represented in this dataset is the:

**Neighbourhood**

A neighbourhood represents a geographic market area within Amsterdam.

Each neighbourhood can contain multiple Airbnb listings and can be used as a natural geographic unit for comparing:

- listing supply,
- price levels,
- room-type composition,
- availability,
- review activity,
- host concentration,
- estimated revenue,
- other market indicators.

The `neighbourhood` property acts as the main business identifier for each geographic feature.

---

#### Geographic Boundary Meaning

Each feature contains a `MultiPolygon` geometry.

This geometry defines the geographic boundary of the corresponding Amsterdam neighbourhood.

The relationship can be understood as:

```text
Neighbourhood Name
        │
        ▼
Geographic Boundary
        │
        ▼
Area on the Map
````

The boundary does not represent an individual listing.

Instead, it represents the geographic area within which multiple listings may be located.

---

#### Relationship to Listings

The GeoJSON neighbourhood boundaries can be connected to listing-level data using neighbourhood names.

For detailed listings:

```text
neighbourhoods.geojson.neighbourhood
                │
                ▼
listings.csv.gz.neighbourhood_cleansed
```

For summary listings:

```text
neighbourhoods.geojson.neighbourhood
                │
                ▼
listings.csv.neighbourhood
```

This allows listing-level measures to be aggregated by neighbourhood and then joined to geographic boundaries.

For example:

```text
Listings
    │
    │ group by neighbourhood
    ▼
Neighbourhood Metrics
    │
    │ join on neighbourhood
    ▼
GeoJSON Boundaries
    │
    ▼
Map Visualization
```

---

#### Business Questions Supported by This Dataset

The GeoJSON file can support geographic business questions such as:

* Which neighbourhoods contain the highest number of listings?
* Which neighbourhoods have the highest median prices?
* Where are entire-home listings concentrated?
* Which areas have the highest private-room share?
* Which neighbourhoods show greater availability?
* Which areas receive more review activity?
* Where are higher-rated listings concentrated?
* Which neighbourhoods may represent premium market segments?
* Which areas show stronger host concentration?
* Which neighbourhoods may support future market expansion?

The GeoJSON itself does not provide these business metrics.

Instead, it provides the geographic structure needed to visualize metrics calculated from other datasets.

---

#### Example Geographic Analysis

Suppose the listings dataset is aggregated by neighbourhood to calculate:

* listing count,
* median price,
* average review score,
* average availability.

The result may look conceptually like:

```text
neighbourhood                  listing_count    median_price
Centrum-West                         ...              ...
Centrum-Oost                         ...              ...
Zuid                                 ...              ...
```

This aggregated table can then be joined to the GeoJSON through:

`neighbourhood`

The final result can support a choropleth map showing how a selected metric varies across Amsterdam.

---

#### Neighbourhood-Level Market Segmentation

Neighbourhood boundaries are useful because Airbnb markets are not geographically uniform.

Different areas may vary in:

* accommodation supply,
* property type,
* tourism demand,
* price levels,
* review activity,
* host concentration,
* regulatory environment,
* listing availability.

Therefore, city-wide averages may hide important geographic differences.

Neighbourhood-level analysis can provide more realistic market segmentation.

For example:

**A city-wide median price may be less informative than separate median prices for Centrum-West, Zuid, De Pijp - Rivierenbuurt, and other neighbourhoods.**

---

#### Geographic Visualization Value

The main analytical value of `neighbourhoods.geojson` is enabling map-based communication.

Potential visualizations include:

* listing density map,
* median price choropleth,
* availability map,
* average review score map,
* room-type concentration map,
* estimated revenue map.

These maps can make geographic differences easier to understand than tables alone.

However, geographic visualization should remain optional if it risks delaying the core engineering, EDA, statistics, documentation, or report deliverables.

---

#### Difference Between GeoJSON Boundaries and Listing Coordinates

The project contains two different forms of geographic information.

##### Neighbourhood boundaries

Provided by:

`neighbourhoods.geojson`

These represent geographic areas.

##### Listing coordinates

Provided by:

* `latitude`
* `longitude`

in the detailed listings dataset.

These represent individual listing locations.

The distinction is:

```text
GeoJSON MultiPolygon
        │
        ▼
Neighbourhood-level area

Latitude + Longitude
        │
        ▼
Individual listing point
```

These two forms of geographic data serve different analytical purposes.

---

#### Aggregation Strategy

For efficient analysis, listing-level data should first be aggregated by neighbourhood.

For example:

```text
10,369 detailed listing rows
          │
          │ group by neighbourhood
          ▼
22 neighbourhood-level rows
          │
          │ join with GeoJSON
          ▼
22 geographic features with business metrics
```

This is computationally efficient and aligns well with the project's 8 GB RAM strategy.

It avoids unnecessarily attaching complex geometry objects to every individual listing row.

---

#### Relationship with `neighbourhoods.csv`

The `neighbourhoods.csv` file provides a simple tabular neighbourhood reference.

The GeoJSON provides the spatial version of the same business entity.

The relationship is:

```text
neighbourhoods.csv
    neighbourhood
          │
          │ 1-to-1
          ▼
neighbourhoods.geojson
    neighbourhood
```

All 22 neighbourhood values matched successfully.

Therefore:

* the CSV acts as a lightweight reference table,
* the GeoJSON acts as the geographic boundary layer.

---

#### Importance for Business Storytelling

Geographic maps can make business findings easier to communicate.

For example, instead of only stating:

> Certain neighbourhoods contain more expensive listings.

A map can visually show where those higher-priced areas are located.

This may help:

* market analysts,
* product managers,
* hosts,
* revenue strategists,
* researchers,

understand geographic market patterns more intuitively.

---

#### Analytical Importance

The `neighbourhoods.geojson` dataset is valuable for:

* geographic enrichment,
* neighbourhood-level aggregation,
* map visualizations,
* spatial business storytelling,
* validating neighbourhood coverage,
* connecting business metrics to physical geographic areas.

It is not intended to function as a transactional dataset.

Its primary role is to add geographic meaning to analytical results derived from listings, reviews, calendar data, and future enriched datasets.

---

#### Overall Business Interpretation

The `neighbourhoods.geojson` file provides the spatial representation of Amsterdam's 22 neighbourhoods.

Its strongest business value comes from enabling neighbourhood-level market metrics to be displayed geographically.

The main analytical flow is:

```text
Listing-Level Data
        │
        ▼
Neighbourhood Aggregation
        │
        ▼
Join with GeoJSON
        │
        ▼
Geographic Visualization
        │
        ▼
Business Insight
```

Overall, this dataset supports geographic market segmentation and map-based storytelling, while the actual business metrics must be calculated from other datasets.

### Dataset Limitations

The `neighbourhoods.geojson` dataset is structurally clean and provides complete geographic coverage for all 22 Amsterdam neighbourhoods represented in the project.

However, several limitations should still be considered before using it for spatial analysis, mapping, or business interpretation.

---

#### 1. Completely Missing `neighbourhood_group`

The `neighbourhood_group` property contains:

- 22 missing values
- 100.00% missing data
- 0 unique non-null values

Therefore, this field provides no usable grouping information.

This is consistent with `neighbourhoods.csv`, where the same field is also completely missing.

As a result, the project cannot use a higher-level neighbourhood grouping from the available source data.

The raw source should remain unchanged, but the field may be excluded from processed outputs if it is not needed.

---

#### 2. Geographic Boundaries Do Not Contain Business Metrics

The GeoJSON file contains:

- neighbourhood names,
- geographic geometries,
- boundary coordinates.

It does not contain business metrics such as:

- listing count,
- price,
- availability,
- review activity,
- room type,
- host information,
- estimated revenue.

Therefore, meaningful business analysis requires joining the GeoJSON with aggregated data from other datasets.

The GeoJSON acts as a geographic reference layer rather than a standalone analytical dataset.

---

#### 3. Neighbourhood Names Are the Main Join Key

The primary logical join field is:

`neighbourhood`

This field achieved 100.00% coverage against `neighbourhoods.csv` and detailed listings.

However, text-based keys can be sensitive to:

- spelling differences,
- capitalization differences,
- leading or trailing whitespace,
- special characters,
- naming convention changes.

The current dataset shows perfect consistency, but future dataset versions may not.

Therefore, neighbourhood names should still be normalized and validated before automated joins in a reusable pipeline.

---

#### 4. Boundaries Represent Areas, Not Exact Listing Locations

The GeoJSON polygons represent neighbourhood boundaries.

They do not represent:

- exact listing coordinates,
- exact building locations,
- street addresses,
- property boundaries.

Individual listing locations are represented separately through:

- `latitude`
- `longitude`

in `listings.csv.gz`.

Therefore, neighbourhood-level maps should not be interpreted as showing exact property locations.

---

#### 5. Spatial Accuracy Depends on Source Boundary Definitions

The GeoJSON geometry reflects the boundary definitions available in the source dataset.

Possible limitations include:

- administrative boundary changes,
- boundary generalization,
- outdated geographic definitions,
- differences between official and platform-specific neighbourhood definitions.

The project should therefore treat the geometry as the source-provided neighbourhood boundary representation rather than assuming it is an authoritative legal boundary.

---

#### 6. All Features Use `MultiPolygon`

All 22 features use:

`MultiPolygon`

geometry.

This is structurally consistent and simplifies processing.

However, `MultiPolygon` geometries can be more complex than simple polygons because one neighbourhood may contain:

- disconnected geographic areas,
- islands,
- multiple separate polygon parts.

Any future spatial processing should therefore preserve the full geometry rather than simplifying it carelessly.

---

#### 7. Coordinate Reference System Is Not Explicitly Documented in the Current Profiling Results

The current inspection confirmed:

- valid GeoJSON structure,
- complete geometries,
- complete coordinates.

However, the coordinate reference system was not explicitly extracted or validated during familiarization.

GeoJSON commonly uses longitude and latitude coordinates, but the project should avoid making unsupported assumptions about spatial reference details without explicit verification.

If geographic calculations such as:

- distance,
- area,
- spatial containment,

are later performed, coordinate-system handling should be validated carefully.

---

#### 8. No Direct Area Measurements

The GeoJSON file does not currently provide derived measures such as:

- neighbourhood area,
- perimeter,
- population density,
- listing density per square kilometre.

These measures would require additional spatial processing.

For example:

```text
Listing density
=
Number of listings
/
Neighbourhood area
````

Such analysis may be useful but is optional and should not delay the higher-priority engineering and analytical tasks.

---

#### 9. Mapping Can Add Technical Overhead

Using the GeoJSON for visual maps may require additional libraries such as:

* GeoPandas,
* Folium,
* Plotly,
* GeoJSON-aware visualization tools.

This can introduce:

* installation complexity,
* dependency issues,
* spatial library compatibility problems,
* extra debugging time.

Given the short project timeline, geographic visualization should remain optional unless the core pipeline, EDA, statistics, documentation, and report are already secure. This aligns with the depth-first strategy in the project plan. 

---

#### 10. Geographic Analysis Should Use Aggregated Data

The recommended workflow is:

```text
Listing-Level Data
        │
        ▼
Aggregate by Neighbourhood
        │
        ▼
22 Neighbourhood-Level Rows
        │
        ▼
Join with GeoJSON
        │
        ▼
Map Visualization
```

Attaching full geometry objects directly to every listing record is unnecessary for most analyses and could increase memory use.

This is particularly important because the project is designed for an 8 GB RAM laptop.

---

#### 11. No Temporal Dimension

The GeoJSON contains static geographic boundaries only.

It does not contain:

* scrape date,
* effective date,
* historical boundary changes,
* version date.

Therefore, the dataset cannot independently show whether neighbourhood boundaries changed over time.

Any temporal geographic interpretation should be made cautiously.

---

#### 12. No Listing-Level Spatial Validation Was Performed Yet

Although all neighbourhood names match perfectly between the GeoJSON and the detailed listings dataset, the current familiarization phase has not yet checked whether each listing's:

* latitude,
* longitude

actually falls inside its assigned neighbourhood polygon.

That would require spatial point-in-polygon validation.

This may be useful as an advanced data-quality check, but it is optional and should be attempted only if time permits.

---

#### 13. Geometry Complexity Was Not Fully Measured

The current profiling confirmed:

* 22 complete geometries,
* all are `MultiPolygon`,
* no missing coordinates.

However, it did not calculate:

* number of polygons per feature,
* number of coordinate points,
* geometry complexity,
* invalid self-intersections,
* topology errors.

These advanced spatial validation tasks are not necessary for the core assignment unless geographic analysis becomes a major focus.

---

#### 14. Geographic Boundaries Should Not Be Overinterpreted

Neighbourhood boundaries are useful for market segmentation, but business conditions may vary significantly within a single neighbourhood.

For example, two listings within the same neighbourhood may differ in:

* distance to city centre,
* access to public transport,
* proximity to attractions,
* property quality,
* local demand.

Therefore, neighbourhood-level averages should not be treated as perfectly representative of every individual listing in that area.

---

#### Overall Limitation Summary

The main limitations of `neighbourhoods.geojson` are:

* `neighbourhood_group` is 100% missing.
* The file contains geographic boundaries but no business metrics.
* Text-based neighbourhood names are used as join keys.
* Boundaries represent areas rather than exact listing locations.
* Coordinate-system details were not explicitly validated during familiarization.
* No spatial point-in-polygon validation has yet been performed.
* Advanced geometry validity and topology checks remain optional.
* Geographic visualization may add technical overhead.

Despite these limitations, the dataset is structurally clean and highly suitable for:

* neighbourhood-level geographic enrichment,
* choropleth maps,
* geographic storytelling,
* spatial validation,
* visualizing aggregated business metrics.

**Overall assessment: Suitable for geographic enrichment and optional mapping, with no major structural issues identified beyond the completely missing `neighbourhood_group` field and the usual limitations of spatial boundary data.**

### Data Quality Assessment

The `neighbourhoods.geojson` dataset is a small, well-structured geographic reference file containing boundary information for Amsterdam neighbourhoods.

The dataset contains:

- **22 geographic features**
- **2 source property fields**
- **0 duplicate property rows**
- **22 unique neighbourhood names**
- **0 missing neighbourhood names**
- **0 duplicate neighbourhood names**
- **0 missing geometries**
- **0 features without coordinates**
- **22 `MultiPolygon` geometries**
- **100.00% relationship coverage with `neighbourhoods.csv`**

Overall, the dataset demonstrates very strong structural and relational quality.

---

#### GeoJSON Structure Quality

The file was successfully parsed as:

`FeatureCollection`

Observed top-level structure:

- `type`
- `features`

The dataset contains:

- **22 geographic features**

Each feature contains:

- `type`
- `geometry`
- `properties`

This indicates that the file follows a consistent GeoJSON feature structure.

No structural loading errors were encountered during profiling.

---

#### Feature Count Validation

The GeoJSON contains:

- **22 geographic features**

This matches the number of unique neighbourhoods found in:

`neighbourhoods.csv`

which also contains:

- **22 neighbourhood records**

The matching feature counts provide an initial positive consistency signal.

More importantly, explicit relationship validation confirmed that all 22 neighbourhood names match exactly between both datasets.

---

#### Candidate Key Quality

The `neighbourhood` property was validated as a strong candidate key.

Observed results:

- Total features: **22**
- Unique neighbourhood values: **22**
- Missing neighbourhood values: **0**
- Duplicate neighbourhood values: **0**

Therefore:

**Candidate key: `neighbourhood`**

The field is both complete and unique within the current GeoJSON dataset.

Because the source is a GeoJSON file rather than a relational database, this key is logically identified rather than formally enforced.

---

#### Duplicate Analysis

Duplicate property-row validation found:

- Total features: **22**
- Duplicate property rows: **0**
- Duplicate percentage: **0.00%**

This means no two geographic features contain identical source-property combinations.

The absence of duplicate neighbourhood names further confirms that each neighbourhood is represented once in the property data.

---

#### Missing-Value Assessment

The GeoJSON contains two source properties:

| Property | Missing Count | Missing Percentage | Unique Non-Null Values |
|---|---:|---:|---:|
| `neighbourhood` | 0 | 0.00% | 22 |
| `neighbourhood_group` | 22 | 100.00% | 0 |

The `neighbourhood` property is fully complete.

The `neighbourhood_group` property is completely missing and therefore provides no usable analytical value.

This is consistent with the corresponding `neighbourhood_group` field in `neighbourhoods.csv`, which is also 100% missing.

This suggests that higher-level neighbourhood grouping information is simply unavailable in the current Amsterdam source data.

---

#### Geometry Completeness

All 22 geographic features contain geometry.

Observed results:

- Total features: **22**
- Missing geometries: **0**
- Features without coordinates: **0**

This is a strong positive data-quality result.

Every neighbourhood feature contains usable geographic boundary information.

---

#### Geometry-Type Consistency

All 22 features use the same geometry type:

`MultiPolygon`

Observed geometry distribution:

| Geometry Type | Feature Count |
|---|---:|
| MultiPolygon | 22 |

No unexpected geometry types were found.

This consistency simplifies future geographic processing because the project does not need to handle mixed geometry types.

---

#### Coordinate Presence Validation

All 22 features contain coordinate data.

Observed results:

- Features with coordinates: **22**
- Features without coordinates: **0**

This confirms that no neighbourhood boundary is structurally empty.

However, coordinate presence alone does not guarantee advanced spatial validity.

The current familiarization phase did not yet test:

- self-intersections,
- topology errors,
- invalid ring structures,
- coordinate reference system details,
- polygon orientation.

These checks are optional and are not required for the current core workflow unless spatial analysis becomes a major project focus.

---

#### Relationship Quality with `neighbourhoods.csv`

The GeoJSON neighbourhood values were explicitly validated against:

`neighbourhoods.csv.neighbourhood`

Results:

- Unique neighbourhoods in GeoJSON: **22**
- Unique neighbourhoods in CSV: **22**
- GeoJSON neighbourhoods absent from CSV: **0**
- CSV neighbourhoods absent from GeoJSON: **0**
- Matched neighbourhoods: **22**
- Relationship coverage: **100.00%**

This represents complete cross-dataset referential consistency.

The logical relationship is:

```text
neighbourhoods.csv
    neighbourhood
          │
          │ one-to-one
          ▼
neighbourhoods.geojson
    neighbourhood
````

This is one of the strongest data-quality results identified so far.

---

#### Relationship Quality with Detailed Listings

The GeoJSON neighbourhood set was also compared with:

`listings.csv.gz.neighbourhood_cleansed`

Results:

* GeoJSON neighbourhoods not used by detailed listings: **0**
* Detailed-listing neighbourhoods absent from GeoJSON: **0**

Therefore, all 22 neighbourhoods represented in the detailed listings dataset are also represented in the GeoJSON.

The logical relationship is:

```text
neighbourhoods.geojson
    neighbourhood
          │
          │ one-to-many
          ▼
listings.csv.gz
    neighbourhood_cleansed
```

This provides complete neighbourhood-level geographic coverage for the detailed listings dataset.

---

#### Relationship Quality with Summary Listings

The summary listings dataset also contains all 22 neighbourhoods.

Because:

* `listings.csv.neighbourhood`
* `neighbourhoods.csv.neighbourhood`
* `neighbourhoods.geojson.neighbourhood`

all use the same 22 neighbourhood values, the geographic reference relationship is highly consistent across the project.

This supports stable neighbourhood-based joins between:

* summary listings,
* detailed listings,
* neighbourhood CSV reference data,
* GeoJSON boundaries.

---

#### Text-Key Quality

The main relationship key is a text field:

`neighbourhood`

Text-based keys may normally be vulnerable to:

* whitespace differences,
* capitalization differences,
* spelling changes,
* punctuation differences,
* naming convention changes.

However, the current data achieved:

* exact value agreement,
* zero unmatched neighbourhoods,
* 100.00% relationship coverage.

Therefore, no naming inconsistency is currently present.

For a reusable pipeline, text normalization and relationship validation should still be retained to protect against future source changes.

---

#### Geographic Coverage Quality

All 22 neighbourhoods represented in:

* `neighbourhoods.csv`
* `listings.csv.gz`

have corresponding GeoJSON features.

This means there is no missing geographic boundary for any neighbourhood currently used by the detailed listing population.

Therefore, future neighbourhood-level maps can theoretically cover the entire represented neighbourhood set.

---

#### Main Data Quality Strengths

The strongest characteristics of `neighbourhoods.geojson` are:

1. Valid `FeatureCollection` structure.
2. Exactly 22 geographic features.
3. 22 unique neighbourhood names.
4. Zero missing neighbourhood names.
5. Zero duplicate neighbourhood names.
6. Zero duplicate property rows.
7. Zero missing geometries.
8. Zero features without coordinates.
9. Consistent `MultiPolygon` geometry across all 22 features.
10. 100.00% relationship coverage with `neighbourhoods.csv`.
11. Complete neighbourhood alignment with detailed listings.
12. Clean suitability for neighbourhood-level geographic enrichment.

---

#### Main Data Quality Concerns

The main concerns are limited:

1. `neighbourhood_group` is 100% missing.
2. The coordinate reference system was not explicitly validated during familiarization.
3. Advanced geometry validity checks were not performed.
4. Neighbourhood names are text-based join keys.
5. No point-in-polygon validation has yet confirmed whether every listing coordinate lies inside its assigned neighbourhood boundary.
6. The file contains geographic shapes but no business metrics.

These concerns do not prevent normal neighbourhood-level enrichment or mapping.

---

#### Data Quality Risk Classification

A practical quality classification is:

| Quality Area                            | Assessment                                       |
| --------------------------------------- | ------------------------------------------------ |
| File structure                          | Excellent                                        |
| Candidate key quality                   | Excellent                                        |
| Duplicate quality                       | Excellent                                        |
| Neighbourhood completeness              | Excellent                                        |
| Geometry completeness                   | Excellent                                        |
| Geometry-type consistency               | Excellent                                        |
| CSV relationship integrity              | Excellent                                        |
| Detailed-listing relationship integrity | Excellent                                        |
| Property completeness                   | Good, except fully missing `neighbourhood_group` |
| Advanced spatial validity               | Not yet tested                                   |

---

#### Cleaning Requirements

Very little cleaning is required.

Recommended actions include:

* preserve the raw GeoJSON unchanged,
* retain `neighbourhood` as the main geographic join key,
* optionally exclude `neighbourhood_group` from processed outputs because it is 100% missing,
* normalize neighbourhood strings in the reusable pipeline,
* validate joins whenever new source files are introduced,
* perform advanced geometry checks only if spatial analysis becomes a priority.

No major structural repair is required.

---

#### Overall Data Quality Assessment

The `neighbourhoods.geojson` dataset has excellent structural and relational quality.

It provides:

* complete geographic coverage for all 22 neighbourhoods,
* unique and complete neighbourhood identifiers,
* zero duplicate property rows,
* complete geometry and coordinate availability,
* consistent `MultiPolygon` geometry,
* perfect alignment with the neighbourhood CSV reference,
* complete neighbourhood-set alignment with detailed listings.

Its only major field-level weakness is that:

`neighbourhood_group`

is completely missing.

**Overall assessment: Excellent-quality geographic reference data that is ready for neighbourhood-level enrichment and optional mapping with minimal cleaning required.**

## 6. Calendar Dataset Familiarization

The `calendar.csv.gz` dataset contains listing-level calendar records for Amsterdam Airbnb listings.

This is expected to be a large dataset because one listing may have multiple records across different calendar dates.

To protect memory usage on an 8 GB RAM laptop, the full compressed file will not be loaded into Pandas. Instead, DuckDB will query the compressed CSV directly and return only compact profiling results.

The dataset will be inspected for:

- File structure
- Row count
- Column count
- Column names
- Data types
- Missing values
- Unique-value counts
- Minimum and maximum values
- Sample values
- Duplicate rows
- Candidate keys
- Listing relationships
- Date coverage
- Availability values
- Price-related fields
- Business-domain meaning
- Dataset limitations
- Data quality issues

The expected relationship is:

`calendar.csv.gz.listing_id` → `listings.csv.gz.id`

However, the actual schema and relationship will be explicitly validated before any conclusion is made.

Create a DuckDB connection

In [113]:
# Create an in-memory DuckDB connection for profiling large files.

calendar_connection = duckdb.connect(database=":memory:")

print("DuckDB connection created successfully.")

DuckDB connection created successfully.


Prepare the calendar file path

In [114]:
calendar_file_path = str(
    DATA_FILES["detailed_calendar"]
).replace("\\", "/")

print("Calendar file path:")
print(calendar_file_path)

Calendar file path:
c:/Users/saths/Desktop/MyProjects/Expernetic Data Challenge/airbnb-data-challenge/data/raw/amsterdam/calendar.csv.gz


Create a DuckDB view over the compressed CSV

In [115]:
calendar_connection.execute(f"""
    CREATE OR REPLACE VIEW calendar_raw AS
    SELECT *
    FROM read_csv_auto(
        '{calendar_file_path}',
        compression='gzip',
        header=true,
        sample_size=-1,
        all_varchar=false
    )
""")

print("DuckDB calendar view created successfully.")

DuckDB calendar view created successfully.


Inspect the schema

In [116]:
calendar_schema_df = calendar_connection.execute("""
    DESCRIBE calendar_raw
""").fetchdf()

print("Calendar Schema")
print("-" * 50)

display(calendar_schema_df)

Calendar Schema
--------------------------------------------------


,column_name,column_type,null,key,default,extra
0,listing_id,BIGINT,YES,None,None,None
1,date,DATE,YES,None,None,None
2,available,BOOLEAN,YES,None,None,None
3,minimum_nights,BIGINT,YES,None,None,None
4,maximum_nights,BIGINT,YES,None,None,None


Show column names

In [117]:
calendar_columns = calendar_schema_df["column_name"].tolist()

print("Calendar Column Names")
print("-" * 50)

for index, column in enumerate(calendar_columns, start=1):
    print(f"{index}. {column}")

Calendar Column Names
--------------------------------------------------
1. listing_id
2. date
3. available
4. minimum_nights
5. maximum_nights


Row and column count

In [118]:
calendar_row_count = calendar_connection.execute("""
    SELECT COUNT(*)
    FROM calendar_raw
""").fetchone()[0]

calendar_column_count = len(calendar_columns)

print("Calendar Dataset Size")
print("-" * 40)

print(f"Rows: {calendar_row_count:,}")
print(f"Columns: {calendar_column_count}")

Calendar Dataset Size
----------------------------------------
Rows: 3,819,725
Columns: 5


Preview five rows

In [119]:
calendar_sample_df = calendar_connection.execute("""
    SELECT *
    FROM calendar_raw
    LIMIT 5
""").fetchdf()

display(calendar_sample_df)

,listing_id,date,available,minimum_nights,maximum_nights
0,28871,2026-06-16,False,2,730
1,28871,2026-06-17,False,2,730
2,28871,2026-06-18,False,2,730
3,28871,2026-06-19,False,2,730
4,28871,2026-06-20,False,2,730


Build missing-value summary dynamically

In [120]:
calendar_missing_expressions = []

for column in calendar_columns:
    expression = (
        f'SUM(CASE WHEN "{column}" IS NULL THEN 1 ELSE 0 END) '
        f'AS "{column}"'
    )

    calendar_missing_expressions.append(expression)

calendar_missing_query = f"""
    SELECT
        {", ".join(calendar_missing_expressions)}
    FROM calendar_raw
"""

calendar_missing_counts = (
    calendar_connection
    .execute(calendar_missing_query)
    .fetchdf()
    .iloc[0]
)

calendar_missing_summary_df = pd.DataFrame({
    "column_name": calendar_columns,
    "missing_count": [
        int(calendar_missing_counts[column])
        for column in calendar_columns
    ]
})

calendar_missing_summary_df["missing_percentage"] = (
    calendar_missing_summary_df["missing_count"]
    / calendar_row_count
    * 100
).round(2)

calendar_missing_summary_df = (
    calendar_missing_summary_df
    .sort_values(
        by="missing_percentage",
        ascending=False
    )
    .reset_index(drop=True)
)

display(calendar_missing_summary_df)

,column_name,missing_count,missing_percentage
0,listing_id,0,0.0
1,date,0,0.0
2,available,0,0.0
3,minimum_nights,0,0.0
4,maximum_nights,0,0.0


Unique-value counts

In [121]:
calendar_unique_results = []

for column in calendar_columns:
    unique_count = calendar_connection.execute(f"""
        SELECT COUNT(DISTINCT "{column}")
        FROM calendar_raw
    """).fetchone()[0]

    calendar_unique_results.append({
        "column_name": column,
        "unique_count": int(unique_count)
    })

calendar_unique_summary_df = pd.DataFrame(
    calendar_unique_results
)

display(calendar_unique_summary_df)

,column_name,unique_count
0,listing_id,10465
1,date,378
2,available,2
3,minimum_nights,56
4,maximum_nights,169


Compact profiling summary

In [122]:
calendar_profile_df = (
    calendar_schema_df[
        ["column_name", "column_type"]
    ]
    .merge(
        calendar_missing_summary_df,
        on="column_name",
        how="left"
    )
    .merge(
        calendar_unique_summary_df,
        on="column_name",
        how="left"
    )
)

display(calendar_profile_df)

,column_name,column_type,missing_count,missing_percentage,unique_count
0,listing_id,BIGINT,0,0.0,10465
1,date,DATE,0,0.0,378
2,available,BOOLEAN,0,0.0,2
3,minimum_nights,BIGINT,0,0.0,56
4,maximum_nights,BIGINT,0,0.0,169


Check expected important columns

In [123]:
expected_calendar_columns = [
    "listing_id",
    "date",
    "available",
    "price",
    "adjusted_price",
    "minimum_nights",
    "maximum_nights"
]

print("Expected Calendar Column Check")
print("-" * 50)

for column in expected_calendar_columns:
    status = (
        "FOUND"
        if column in calendar_columns
        else "NOT FOUND"
    )

    print(f"{column}: {status}")

Expected Calendar Column Check
--------------------------------------------------
listing_id: FOUND
date: FOUND
available: FOUND
price: NOT FOUND
adjusted_price: NOT FOUND
minimum_nights: FOUND
maximum_nights: FOUND


Display sample values for each column

In [124]:
calendar_sample_values = []

for column in calendar_columns:
    sample_values = calendar_connection.execute(f"""
        SELECT DISTINCT "{column}"
        FROM calendar_raw
        WHERE "{column}" IS NOT NULL
        LIMIT 3
    """).fetchall()

    sample_values = [
        row[0]
        for row in sample_values
    ]

    calendar_sample_values.append({
        "column_name": column,
        "sample_values": sample_values
    })

calendar_sample_values_df = pd.DataFrame(
    calendar_sample_values
)

display(calendar_sample_values_df)

,column_name,sample_values
0,listing_id,"[18017858, 18172462, 18248976]"
1,date,"[2026-06-18, 2026-07-18, 2026-08-11]"
2,available,"[False, True]"
3,minimum_nights,"[26, 11, 16]"
4,maximum_nights,"[16, 11, 26]"


Full duplicate-row analysis

In [125]:
calendar_duplicate_summary_df = calendar_connection.execute("""
    SELECT
        COUNT(*) AS duplicate_groups,
        COALESCE(SUM(row_count - 1), 0) AS duplicate_extra_rows
    FROM (
        SELECT
            listing_id,
            date,
            available,
            minimum_nights,
            maximum_nights,
            COUNT(*) AS row_count
        FROM calendar_raw
        GROUP BY
            listing_id,
            date,
            available,
            minimum_nights,
            maximum_nights
        HAVING COUNT(*) > 1
    )
""").fetchdf()

calendar_duplicate_group_count = int(
    calendar_duplicate_summary_df.loc[
        0,
        "duplicate_groups"
    ]
)

calendar_duplicate_extra_rows = int(
    calendar_duplicate_summary_df.loc[
        0,
        "duplicate_extra_rows"
    ]
)

calendar_duplicate_percentage = (
    calendar_duplicate_extra_rows
    / calendar_row_count
    * 100
)

print("Calendar Full Duplicate-Row Analysis")
print("-" * 55)

print(
    f"Total rows: "
    f"{calendar_row_count:,}"
)

print(
    f"Duplicate groups: "
    f"{calendar_duplicate_group_count:,}"
)

print(
    f"Extra duplicate rows: "
    f"{calendar_duplicate_extra_rows:,}"
)

print(
    f"Duplicate percentage: "
    f"{calendar_duplicate_percentage:.4f}%"
)

Calendar Full Duplicate-Row Analysis
-------------------------------------------------------
Total rows: 3,819,725
Duplicate groups: 0
Extra duplicate rows: 0
Duplicate percentage: 0.0000%


In [126]:
if calendar_duplicate_group_count > 0:
    calendar_duplicate_examples_df = (
        calendar_connection.execute("""
            SELECT
                listing_id,
                date,
                available,
                minimum_nights,
                maximum_nights,
                COUNT(*) AS occurrence_count
            FROM calendar_raw
            GROUP BY
                listing_id,
                date,
                available,
                minimum_nights,
                maximum_nights
            HAVING COUNT(*) > 1
            ORDER BY occurrence_count DESC
            LIMIT 10
        """).fetchdf()
    )

    display(calendar_duplicate_examples_df)

else:
    print("No complete duplicate rows found.")

No complete duplicate rows found.


Composite key validation: (listing_id, date)

In [127]:
calendar_composite_key_summary_df = (
    calendar_connection.execute("""
        SELECT
            COUNT(*) AS duplicate_key_groups,
            COALESCE(SUM(row_count - 1), 0)
                AS extra_rows_from_duplicate_keys
        FROM (
            SELECT
                listing_id,
                date,
                COUNT(*) AS row_count
            FROM calendar_raw
            GROUP BY
                listing_id,
                date
            HAVING COUNT(*) > 1
        )
    """).fetchdf()
)

calendar_duplicate_key_groups = int(
    calendar_composite_key_summary_df.loc[
        0,
        "duplicate_key_groups"
    ]
)

calendar_duplicate_key_extra_rows = int(
    calendar_composite_key_summary_df.loc[
        0,
        "extra_rows_from_duplicate_keys"
    ]
)

print("Calendar Composite Key Validation")
print("-" * 55)

print("Candidate composite key: (listing_id, date)")

print(
    f"Duplicate key combinations: "
    f"{calendar_duplicate_key_groups:,}"
)

print(
    f"Extra rows caused by duplicate keys: "
    f"{calendar_duplicate_key_extra_rows:,}"
)

Calendar Composite Key Validation
-------------------------------------------------------
Candidate composite key: (listing_id, date)
Duplicate key combinations: 0
Extra rows caused by duplicate keys: 0


In [128]:
if calendar_duplicate_key_groups > 0:
    calendar_duplicate_key_examples_df = (
        calendar_connection.execute("""
            SELECT
                listing_id,
                date,
                COUNT(*) AS occurrence_count
            FROM calendar_raw
            GROUP BY
                listing_id,
                date
            HAVING COUNT(*) > 1
            ORDER BY occurrence_count DESC
            LIMIT 10
        """).fetchdf()
    )

    display(calendar_duplicate_key_examples_df)

else:
    print(
        "The combination (listing_id, date) "
        "is unique across the dataset."
    )

The combination (listing_id, date) is unique across the dataset.


Explicit listing identifier validation

In [129]:
calendar_listing_id_validation_df = (
    calendar_connection.execute("""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT listing_id)
                AS unique_listing_ids,
            SUM(
                CASE
                    WHEN listing_id IS NULL THEN 1
                    ELSE 0
                END
            ) AS missing_listing_ids
        FROM calendar_raw
    """).fetchdf()
)

display(calendar_listing_id_validation_df)

,total_rows,unique_listing_ids,missing_listing_ids
0,3819725,10465,0.0


In [130]:
print("Calendar Listing ID Validation")
print("-" * 45)

print(
    f"Total calendar rows: "
    f"{calendar_row_count:,}"
)

print(
    f"Unique listing IDs: "
    f"{calendar_listing_id_validation_df.loc[0, 'unique_listing_ids']:,}"
)

print(
    f"Missing listing IDs: "
    f"{calendar_listing_id_validation_df.loc[0, 'missing_listing_ids']:,}"
)

Calendar Listing ID Validation
---------------------------------------------
Total calendar rows: 3,819,725
Unique listing IDs: 10,465
Missing listing IDs: 0.0


Extract compact calendar listing ID set

In [131]:
calendar_listing_ids = set(
    calendar_connection.execute("""
        SELECT DISTINCT listing_id
        FROM calendar_raw
        WHERE listing_id IS NOT NULL
    """).fetchnumpy()["listing_id"].tolist()
)

print("Calendar listing ID set created.")
print(
    f"Unique calendar listing IDs: "
    f"{len(calendar_listing_ids):,}"
)

Calendar listing ID set created.
Unique calendar listing IDs: 10,465


Reconstruct the summary listing ID set

In [132]:
summary_listing_ids = (
    detailed_listing_ids
    | summary_only_listing_ids
)

print("Summary listing ID set reconstructed.")
print(
    f"Unique summary listing IDs: "
    f"{len(summary_listing_ids):,}"
)

Summary listing ID set reconstructed.
Unique summary listing IDs: 10,465


Validate calendar relationship with summary listings

In [133]:
calendar_not_in_summary = (
    calendar_listing_ids
    - summary_listing_ids
)

summary_not_in_calendar = (
    summary_listing_ids
    - calendar_listing_ids
)

calendar_summary_common_ids = (
    calendar_listing_ids
    & summary_listing_ids
)

calendar_to_summary_coverage = (
    len(calendar_summary_common_ids)
    / len(calendar_listing_ids)
    * 100
    if calendar_listing_ids
    else 0
)

summary_to_calendar_coverage = (
    len(calendar_summary_common_ids)
    / len(summary_listing_ids)
    * 100
    if summary_listing_ids
    else 0
)

print("Calendar-to-Summary Listings Relationship")
print("-" * 60)

print(
    f"Unique calendar listing IDs: "
    f"{len(calendar_listing_ids):,}"
)

print(
    f"Unique summary listing IDs: "
    f"{len(summary_listing_ids):,}"
)

print(
    f"Common listing IDs: "
    f"{len(calendar_summary_common_ids):,}"
)

print(
    f"Calendar IDs not found in summary listings: "
    f"{len(calendar_not_in_summary):,}"
)

print(
    f"Summary listing IDs absent from calendar: "
    f"{len(summary_not_in_calendar):,}"
)

print(
    f"Calendar-to-summary coverage: "
    f"{calendar_to_summary_coverage:.2f}%"
)

print(
    f"Summary-to-calendar coverage: "
    f"{summary_to_calendar_coverage:.2f}%"
)

Calendar-to-Summary Listings Relationship
------------------------------------------------------------
Unique calendar listing IDs: 10,465
Unique summary listing IDs: 10,465
Common listing IDs: 10,465
Calendar IDs not found in summary listings: 0
Summary listing IDs absent from calendar: 0
Calendar-to-summary coverage: 100.00%
Summary-to-calendar coverage: 100.00%


Validate calendar relationship with detailed listings

In [134]:
calendar_not_in_detailed = (
    calendar_listing_ids
    - detailed_listing_ids
)

detailed_not_in_calendar = (
    detailed_listing_ids
    - calendar_listing_ids
)

calendar_detailed_common_ids = (
    calendar_listing_ids
    & detailed_listing_ids
)

calendar_to_detailed_coverage = (
    len(calendar_detailed_common_ids)
    / len(calendar_listing_ids)
    * 100
    if calendar_listing_ids
    else 0
)

detailed_to_calendar_coverage = (
    len(calendar_detailed_common_ids)
    / len(detailed_listing_ids)
    * 100
    if detailed_listing_ids
    else 0
)

print("Calendar-to-Detailed Listings Relationship")
print("-" * 60)

print(
    f"Unique calendar listing IDs: "
    f"{len(calendar_listing_ids):,}"
)

print(
    f"Unique detailed listing IDs: "
    f"{len(detailed_listing_ids):,}"
)

print(
    f"Common listing IDs: "
    f"{len(calendar_detailed_common_ids):,}"
)

print(
    f"Calendar IDs not found in detailed listings: "
    f"{len(calendar_not_in_detailed):,}"
)

print(
    f"Detailed listing IDs absent from calendar: "
    f"{len(detailed_not_in_calendar):,}"
)

print(
    f"Calendar-to-detailed coverage: "
    f"{calendar_to_detailed_coverage:.2f}%"
)

print(
    f"Detailed-to-calendar coverage: "
    f"{detailed_to_calendar_coverage:.2f}%"
)

Calendar-to-Detailed Listings Relationship
------------------------------------------------------------
Unique calendar listing IDs: 10,465
Unique detailed listing IDs: 10,369
Common listing IDs: 10,369
Calendar IDs not found in detailed listings: 96
Detailed listing IDs absent from calendar: 0
Calendar-to-detailed coverage: 99.08%
Detailed-to-calendar coverage: 100.00%


Validate the 96-listing cross-dataset pattern

In [135]:
calendar_extra_vs_detailed_matches_summary_only = (
    calendar_not_in_detailed
    == summary_only_listing_ids
)

print("96-Listing Cross-Dataset Pattern Validation")
print("-" * 60)

print(
    f"Calendar IDs absent from detailed listings: "
    f"{len(calendar_not_in_detailed):,}"
)

print(
    f"Summary-only listing IDs: "
    f"{len(summary_only_listing_ids):,}"
)

print(
    "Exact set match between calendar-extra IDs "
    "and summary-only IDs: "
    f"{calendar_extra_vs_detailed_matches_summary_only}"
)

96-Listing Cross-Dataset Pattern Validation
------------------------------------------------------------
Calendar IDs absent from detailed listings: 96
Summary-only listing IDs: 96
Exact set match between calendar-extra IDs and summary-only IDs: True


Date-range validation

In [136]:
calendar_date_summary_df = (
    calendar_connection.execute("""
        SELECT
            MIN(date) AS earliest_date,
            MAX(date) AS latest_date,
            COUNT(DISTINCT date)
                AS unique_dates,
            COUNT(*) FILTER (
                WHERE date IS NULL
            ) AS missing_dates
        FROM calendar_raw
    """).fetchdf()
)

print("Calendar Date Range Validation")
print("-" * 50)

display(calendar_date_summary_df)

Calendar Date Range Validation
--------------------------------------------------


,earliest_date,latest_date,unique_dates,missing_dates
0,2026-06-15,2027-06-27,378,0


Calendar window length by listing

In [137]:
calendar_rows_per_listing_summary_df = (
    calendar_connection.execute("""
        WITH listing_calendar_counts AS (
            SELECT
                listing_id,
                COUNT(*) AS calendar_row_count,
                COUNT(DISTINCT date)
                    AS unique_date_count,
                MIN(date) AS first_calendar_date,
                MAX(date) AS last_calendar_date
            FROM calendar_raw
            GROUP BY listing_id
        )

        SELECT
            MIN(calendar_row_count)
                AS min_rows_per_listing,
            MAX(calendar_row_count)
                AS max_rows_per_listing,
            AVG(calendar_row_count)
                AS avg_rows_per_listing,
            MIN(unique_date_count)
                AS min_unique_dates_per_listing,
            MAX(unique_date_count)
                AS max_unique_dates_per_listing,
            AVG(unique_date_count)
                AS avg_unique_dates_per_listing
        FROM listing_calendar_counts
    """).fetchdf()
)

print("Calendar Records per Listing")
print("-" * 50)

display(calendar_rows_per_listing_summary_df)

Calendar Records per Listing
--------------------------------------------------


,min_rows_per_listing,max_rows_per_listing,avg_rows_per_listing,min_unique_dates_per_listing,max_unique_dates_per_listing,avg_unique_dates_per_listing
0,365,365,365.0,365,365,365.0


Distribution of calendar row counts per listing

In [138]:
calendar_row_count_distribution_df = (
    calendar_connection.execute("""
        WITH listing_calendar_counts AS (
            SELECT
                listing_id,
                COUNT(*) AS calendar_row_count
            FROM calendar_raw
            GROUP BY listing_id
        )

        SELECT
            calendar_row_count,
            COUNT(*) AS listing_count
        FROM listing_calendar_counts
        GROUP BY calendar_row_count
        ORDER BY calendar_row_count
    """).fetchdf()
)

display(calendar_row_count_distribution_df)

,calendar_row_count,listing_count
0,365,10465


Calendar window date ranges by listing

In [139]:
calendar_window_distribution_df = (
    calendar_connection.execute("""
        WITH listing_windows AS (
            SELECT
                listing_id,
                MIN(date) AS first_calendar_date,
                MAX(date) AS last_calendar_date,
                COUNT(DISTINCT date)
                    AS unique_date_count
            FROM calendar_raw
            GROUP BY listing_id
        )

        SELECT
            first_calendar_date,
            last_calendar_date,
            unique_date_count,
            COUNT(*) AS listing_count
        FROM listing_windows
        GROUP BY
            first_calendar_date,
            last_calendar_date,
            unique_date_count
        ORDER BY
            first_calendar_date,
            last_calendar_date
    """).fetchdf()
)

print("Calendar Window Distribution")
print("-" * 50)

display(calendar_window_distribution_df)

Calendar Window Distribution
--------------------------------------------------


,first_calendar_date,last_calendar_date,unique_date_count,listing_count
0,2026-06-15,2027-06-14,365,499
1,2026-06-16,2027-06-15,365,6450
2,2026-06-24,2027-06-23,365,3420
3,2026-06-28,2027-06-27,365,96


Availability distribution

In [140]:
calendar_availability_distribution_df = (
    calendar_connection.execute("""
        SELECT
            available,
            COUNT(*) AS row_count,
            ROUND(
                COUNT(*) * 100.0
                / SUM(COUNT(*)) OVER (),
                2
            ) AS percentage
        FROM calendar_raw
        GROUP BY available
        ORDER BY available
    """).fetchdf()
)

print("Calendar Availability Distribution")
print("-" * 50)

display(calendar_availability_distribution_df)

Calendar Availability Distribution
--------------------------------------------------


,available,row_count,percentage
0,False,2820970,73.85
1,True,998755,26.15


Availability by listing summary

In [141]:
calendar_listing_availability_summary_df = (
    calendar_connection.execute("""
        WITH listing_availability AS (
            SELECT
                listing_id,
                COUNT(*) AS total_calendar_days,
                SUM(
                    CASE
                        WHEN available = TRUE
                        THEN 1
                        ELSE 0
                    END
                ) AS available_days,
                SUM(
                    CASE
                        WHEN available = FALSE
                        THEN 1
                        ELSE 0
                    END
                ) AS unavailable_days
            FROM calendar_raw
            GROUP BY listing_id
        )

        SELECT
            MIN(available_days)
                AS min_available_days,
            MAX(available_days)
                AS max_available_days,
            AVG(available_days)
                AS avg_available_days,
            MIN(unavailable_days)
                AS min_unavailable_days,
            MAX(unavailable_days)
                AS max_unavailable_days,
            AVG(unavailable_days)
                AS avg_unavailable_days
        FROM listing_availability
    """).fetchdf()
)

display(calendar_listing_availability_summary_df)

,min_available_days,max_available_days,avg_available_days,min_unavailable_days,max_unavailable_days,avg_unavailable_days
0,0.0,365.0,95.437649,0.0,365.0,269.562351


Minimum-nights range analysis

In [142]:
calendar_minimum_nights_summary_df = (
    calendar_connection.execute("""
        SELECT
            MIN(minimum_nights)
                AS minimum_value,
            MAX(minimum_nights)
                AS maximum_value,
            MEDIAN(minimum_nights)
                AS median_value,
            AVG(minimum_nights)
                AS average_value,
            COUNT(DISTINCT minimum_nights)
                AS unique_values
        FROM calendar_raw
    """).fetchdf()
)

print("Calendar Minimum Nights Summary")
print("-" * 50)

display(calendar_minimum_nights_summary_df)

Calendar Minimum Nights Summary
--------------------------------------------------


,minimum_value,maximum_value,median_value,average_value,unique_values
0,1,800,3.0,4.099971,56


Maximum-nights range analysis

In [143]:
calendar_maximum_nights_summary_df = (
    calendar_connection.execute("""
        SELECT
            MIN(maximum_nights)
                AS minimum_value,
            MAX(maximum_nights)
                AS maximum_value,
            MEDIAN(maximum_nights)
                AS median_value,
            AVG(maximum_nights)
                AS average_value,
            COUNT(DISTINCT maximum_nights)
                AS unique_values
        FROM calendar_raw
    """).fetchdf()
)

print("Calendar Maximum Nights Summary")
print("-" * 50)

display(calendar_maximum_nights_summary_df)

Calendar Maximum Nights Summary
--------------------------------------------------


,minimum_value,maximum_value,median_value,average_value,unique_values
0,1,2147483647,30.0,410752.611216,169


Inspect extreme stay-rule values

In [144]:
calendar_extreme_stay_rules_df = (
    calendar_connection.execute("""
        SELECT
            listing_id,
            date,
            minimum_nights,
            maximum_nights
        FROM calendar_raw
        ORDER BY
            maximum_nights DESC,
            minimum_nights DESC
        LIMIT 20
    """).fetchdf()
)

display(calendar_extreme_stay_rules_df)

,listing_id,date,minimum_nights,maximum_nights
0,39529472,2026-06-27,8,2147483647
1,39529278,2026-09-10,6,2147483647
2,39529278,2026-10-02,6,2147483647
3,39529278,2026-08-01,6,2147483647
4,39529278,2026-07-31,6,2147483647
5,39529278,2026-09-11,6,2147483647
6,39529278,2026-09-19,6,2147483647
7,39529278,2026-08-22,6,2147483647
8,39529278,2026-09-09,6,2147483647
9,39529278,2026-09-05,6,2147483647


Check invalid logical stay-rule combinations

In [145]:
calendar_invalid_stay_rule_count = (
    calendar_connection.execute("""
        SELECT COUNT(*)
        FROM calendar_raw
        WHERE minimum_nights > maximum_nights
    """).fetchone()[0]
)

calendar_invalid_stay_rule_percentage = (
    calendar_invalid_stay_rule_count
    / calendar_row_count
    * 100
)

print("Calendar Stay-Rule Consistency Check")
print("-" * 50)

print(
    f"Rows where minimum_nights > maximum_nights: "
    f"{calendar_invalid_stay_rule_count:,}"
)

print(
    f"Percentage of all calendar rows: "
    f"{calendar_invalid_stay_rule_percentage:.4f}%"
)

Calendar Stay-Rule Consistency Check
--------------------------------------------------
Rows where minimum_nights > maximum_nights: 0
Percentage of all calendar rows: 0.0000%


In [146]:
if calendar_invalid_stay_rule_count > 0:
    calendar_invalid_stay_rule_examples_df = (
        calendar_connection.execute("""
            SELECT
                listing_id,
                date,
                minimum_nights,
                maximum_nights
            FROM calendar_raw
            WHERE minimum_nights > maximum_nights
            LIMIT 20
        """).fetchdf()
    )

    display(calendar_invalid_stay_rule_examples_df)

Verify price-column absence

In [147]:
calendar_price_columns = [
    column
    for column in [
        "price",
        "adjusted_price"
    ]
    if column in calendar_columns
]

print("Calendar Price Field Validation")
print("-" * 45)

if calendar_price_columns:
    print(
        "Available price-related fields: "
        f"{calendar_price_columns}"
    )
else:
    print(
        "No price or adjusted_price columns "
        "exist in this calendar dataset."
    )

Calendar Price Field Validation
---------------------------------------------
No price or adjusted_price columns exist in this calendar dataset.


Dataset summary

In [148]:
calendar_dataset_summary = {
    "dataset_name": "detailed_calendar",
    "file_name": DATA_FILES["detailed_calendar"].name,
    "row_count": calendar_row_count,
    "column_count": calendar_column_count,
    "unique_listing_ids": len(calendar_listing_ids),
    "unique_dates": int(
        calendar_date_summary_df.loc[
            0,
            "unique_dates"
        ]
    ),
    "duplicate_extra_rows": (
        calendar_duplicate_extra_rows
    ),
    "candidate_composite_key": (
        ["listing_id", "date"]
        if calendar_duplicate_key_groups == 0
        else None
    ),
    "calendar_to_summary_coverage": round(
        calendar_to_summary_coverage,
        2
    ),
    "calendar_to_detailed_coverage": round(
        calendar_to_detailed_coverage,
        2
    ),
    "missing_values_total": int(
        calendar_missing_summary_df[
            "missing_count"
        ].sum()
    ),
    "price_columns_present": (
        calendar_price_columns
    ),
}

calendar_dataset_summary

{'dataset_name': 'detailed_calendar',
 'file_name': 'calendar.csv.gz',
 'row_count': 3819725,
 'column_count': 5,
 'unique_listing_ids': 10465,
 'unique_dates': 378,
 'duplicate_extra_rows': 0,
 'candidate_composite_key': ['listing_id', 'date'],
 'calendar_to_summary_coverage': 100.0,
 'calendar_to_detailed_coverage': 99.08,
 'missing_values_total': 0,
 'price_columns_present': []}

Save compact profiling outputs

In [149]:
calendar_profile_path = (
    OUTPUT_DIR
    / "calendar_csv_gz_column_profile.csv"
)

calendar_profile_df.to_csv(
    calendar_profile_path,
    index=False
)

calendar_window_profile_path = (
    OUTPUT_DIR
    / "calendar_window_distribution.csv"
)

calendar_window_distribution_df.to_csv(
    calendar_window_profile_path,
    index=False
)

calendar_availability_profile_path = (
    OUTPUT_DIR
    / "calendar_availability_distribution.csv"
)

calendar_availability_distribution_df.to_csv(
    calendar_availability_profile_path,
    index=False
)

print("Calendar profiling outputs saved successfully.")

print(
    f"\nColumn profile:\n"
    f"{calendar_profile_path}"
)

print(
    f"\nCalendar-window profile:\n"
    f"{calendar_window_profile_path}"
)

print(
    f"\nAvailability profile:\n"
    f"{calendar_availability_profile_path}"
)

Calendar profiling outputs saved successfully.

Column profile:
c:\Users\saths\Desktop\MyProjects\Expernetic Data Challenge\airbnb-data-challenge\outputs\data_quality\calendar_csv_gz_column_profile.csv

Calendar-window profile:
c:\Users\saths\Desktop\MyProjects\Expernetic Data Challenge\airbnb-data-challenge\outputs\data_quality\calendar_window_distribution.csv

Availability profile:
c:\Users\saths\Desktop\MyProjects\Expernetic Data Challenge\airbnb-data-challenge\outputs\data_quality\calendar_availability_distribution.csv


### Findings and Interpretation

The `calendar.csv.gz` dataset contains daily calendar-level records for Amsterdam Airbnb listings.

The grain of the dataset is:

**One row represents one listing on one calendar date.**

The dataset contains:

- **3,819,725 rows**
- **5 columns**
- **10,465 unique listing IDs**
- **378 distinct calendar dates across the full dataset**
- **0 missing values**
- **0 complete duplicate rows**
- **0 duplicate `(listing_id, date)` combinations**

The dataset was profiled directly with DuckDB rather than loading all 3.8 million rows into Pandas, which is more appropriate for an 8 GB RAM environment.

---

#### Dataset Structure

The five columns are:

- `listing_id`
- `date`
- `available`
- `minimum_nights`
- `maximum_nights`

Observed DuckDB data types are:

| Column | Data Type |
|---|---|
| `listing_id` | BIGINT |
| `date` | DATE |
| `available` | BOOLEAN |
| `minimum_nights` | BIGINT |
| `maximum_nights` | BIGINT |

The schema is compact and focused on:

- listing identity,
- calendar dates,
- availability status,
- minimum-stay rules,
- maximum-stay rules.

Unlike many Airbnb calendar datasets, this file does **not** contain:

- `price`
- `adjusted_price`

Therefore, calendar-level price analysis cannot be performed from this source.

---

#### Dataset Size

Observed results:

- Total rows: **3,819,725**
- Total columns: **5**
- Unique listing IDs: **10,465**

Every listing has exactly:

**365 calendar records**

This was explicitly validated.

Observed records per listing:

- Minimum rows per listing: **365**
- Maximum rows per listing: **365**
- Average rows per listing: **365.0**

The distribution confirms:

| Calendar Rows per Listing | Number of Listings |
|---:|---:|
| 365 | 10,465 |

Therefore, every listing represented in the calendar dataset has one full 365-day calendar window.

---

#### Missing-Value Assessment

No missing values were found in any of the five columns.

| Column | Missing Count | Missing Percentage |
|---|---:|---:|
| `listing_id` | 0 | 0.00% |
| `date` | 0 | 0.00% |
| `available` | 0 | 0.00% |
| `minimum_nights` | 0 | 0.00% |
| `maximum_nights` | 0 | 0.00% |

This is a strong positive data-quality result.

The complete availability of:

- listing identifiers,
- dates,
- availability flags,
- minimum-night rules,
- maximum-night rules

makes the dataset structurally reliable for calendar-based aggregation.

---

#### Unique-Value Assessment

Observed unique-value counts are:

| Column | Unique Values |
|---|---:|
| `listing_id` | 10,465 |
| `date` | 378 |
| `available` | 2 |
| `minimum_nights` | 56 |
| `maximum_nights` | 169 |

The `available` column contains only:

- `True`
- `False`

which is consistent with its Boolean meaning.

---

#### Duplicate-Row Analysis

No complete duplicate rows were found.

Observed results:

- Total rows: **3,819,725**
- Duplicate groups: **0**
- Extra duplicate rows: **0**
- Duplicate percentage: **0.0000%**

This indicates that every complete calendar record is unique.

---

#### Candidate Composite Key

The combination:

`(listing_id, date)`

was tested as a candidate composite key.

Observed results:

- Duplicate key combinations: **0**
- Extra rows caused by duplicate key combinations: **0**

Therefore:

**Candidate composite key: (`listing_id`, `date`)**

This means each listing has at most one calendar record for a given date.

This is consistent with the dataset grain:

**One listing-date combination per row.**

Because the source is a CSV file rather than a relational database table, this key is logically identified through profiling rather than formally enforced.

---

#### Listing Identifier Quality

The `listing_id` field is complete.

Observed results:

- Total calendar rows: **3,819,725**
- Unique listing IDs: **10,465**
- Missing listing IDs: **0**

This field is therefore suitable for linking calendar records to listing-level datasets.

The expected relationship is:

```text
Listing
   │
   │ 1
   │
   │ many
   ▼
Calendar Records
````

One listing can have many calendar rows, with one row for each date in its 365-day window.

---

#### Relationship with Summary Listings

The calendar listing IDs were explicitly compared with:

`listings.csv.id`

Observed results:

* Unique calendar listing IDs: **10,465**
* Unique summary listing IDs: **10,465**
* Common listing IDs: **10,465**
* Calendar IDs not found in summary listings: **0**
* Summary listing IDs absent from calendar: **0**
* Calendar-to-summary coverage: **100.00%**
* Summary-to-calendar coverage: **100.00%**

Therefore, the calendar dataset and the summary listings dataset represent exactly the same set of 10,465 listing IDs.

This is a strong cross-dataset consistency result.

The relationship is:

```text
listings.csv
    id
     │
     │ one-to-many
     ▼
calendar.csv.gz
    listing_id
```

---

#### Relationship with Detailed Listings

The calendar IDs were also compared with:

`listings.csv.gz.id`

Observed results:

* Unique calendar listing IDs: **10,465**
* Unique detailed listing IDs: **10,369**
* Common listing IDs: **10,369**
* Calendar IDs not found in detailed listings: **96**
* Detailed listing IDs absent from calendar: **0**
* Calendar-to-detailed coverage: **99.08%**
* Detailed-to-calendar coverage: **100.00%**

Therefore, every detailed listing has calendar data.

However, the calendar dataset contains 96 additional listing IDs that do not appear in `listings.csv.gz`.

---

#### Confirmed 96-Listing Cross-Dataset Pattern

Earlier profiling identified:

* 10,465 listings in `listings.csv`
* 10,369 listings in `listings.csv.gz`
* 96 listings present only in the summary file

The calendar dataset also contains exactly 96 listing IDs that are absent from the detailed listings file.

This was explicitly validated at the set level.

Observed result:

```text
Exact set match between calendar-extra IDs
and summary-only IDs: True
```

Therefore, the following can now be confirmed:

**The 96 listing IDs present in `listings.csv` but absent from `listings.csv.gz` are exactly the same 96 listing IDs represented in the calendar dataset but absent from the detailed listings dataset.**

This strengthens the cross-dataset interpretation:

```text
Summary Listings:   10,465 IDs
Calendar:           10,465 IDs
Detailed Listings:  10,369 IDs
                         │
                         └── 96 specific IDs absent
```

The actual reason why these 96 listings are absent from the detailed listings file is still not confirmed.

Possible causes such as scrape timing, source logic, filtering, or incomplete detailed records remain hypotheses only.

---

#### Date Coverage

Observed overall date coverage is:

* Earliest date: **2026-06-15**
* Latest date: **2027-06-27**
* Distinct dates across full dataset: **378**
* Missing dates: **0**

At first glance, 378 distinct dates may appear inconsistent with the fact that every listing has exactly 365 calendar records.

However, further validation showed that individual listings use different 365-day calendar windows.

---

#### Calendar Window Distribution

Four distinct calendar windows were identified.

| First Calendar Date | Last Calendar Date | Unique Dates per Listing | Listing Count |
| ------------------- | ------------------ | -----------------------: | ------------: |
| 2026-06-15          | 2027-06-14         |                      365 |           499 |
| 2026-06-16          | 2027-06-15         |                      365 |         6,450 |
| 2026-06-24          | 2027-06-23         |                      365 |         3,420 |
| 2026-06-28          | 2027-06-27         |                      365 |            96 |

The listing counts sum to:

**10,465**

Therefore, every listing still has exactly 365 dates, but listings do not all begin their calendar window on the same date.

This explains why the full dataset contains:

**378 distinct dates**

across all listings.

---

#### Calendar Window Interpretation

The different date windows likely reflect different scrape or source timing.

However, that exact cause should not be stated as confirmed without explicit source documentation.

The evidence supports the following conclusion:

**The dataset contains multiple listing-specific 365-day calendar windows rather than one identical 365-day date range shared by every listing.**

This matters because comparisons across listings should account for slightly different calendar periods.

For example, not every listing has calendar data for exactly the same set of dates.

---

#### Important 96-Listing Calendar Window Pattern

The final calendar window is:

* First date: **2026-06-28**
* Last date: **2027-06-27**
* Listings: **96**

This count exactly matches the 96 listings that:

* appear in `listings.csv`,
* appear in `calendar.csv.gz`,
* do not appear in `listings.csv.gz`.

This is another strong numerical alignment.

Because the exact listing ID sets were already confirmed to match, it is reasonable to conclude that these same 96 summary-only listings share the 2026-06-28 to 2027-06-27 calendar window.

This may provide useful evidence when investigating why these records differ from the detailed listing population.

---

#### Availability Distribution

The `available` field contains two values:

* `False`
* `True`

Observed results:

| Availability | Row Count | Percentage |
| ------------ | --------: | ---------: |
| False        | 2,820,970 |     73.85% |
| True         |   998,755 |     26.15% |

Therefore:

* **73.85%** of calendar records are marked unavailable.
* **26.15%** are marked available.

However, an unavailable date must not automatically be interpreted as a confirmed booking.

A date may be unavailable because of:

* a guest booking,
* host blocking,
* maintenance,
* personal use,
* temporary suspension,
* other operational reasons.

Therefore, availability data should be treated as a measure of:

**Calendar availability status**

rather than direct confirmed occupancy.

---

#### Availability by Listing

At the listing level:

* Minimum available days: **0**
* Maximum available days: **365**
* Average available days: approximately **95.44**

For unavailable days:

* Minimum unavailable days: **0**
* Maximum unavailable days: **365**
* Average unavailable days: approximately **269.56**

This shows substantial variation across listings.

Some listings are:

* unavailable for all 365 days,
* available for all 365 days,
* available for only part of the year.

This can support later derived features such as:

* available-day count,
* availability rate,
* unavailable-day count.

For example:

```text
availability_rate
=
available_days / total_calendar_days
```

Since each listing has exactly 365 calendar rows:

```text
availability_rate
=
available_days / 365
```

---

#### Minimum-Night Rules

Observed `minimum_nights` statistics are:

* Minimum: **1**
* Maximum: **800**
* Median: **3**
* Average: approximately **4.10**
* Unique values: **56**

The median of 3 nights suggests that many records use relatively short minimum stays.

However, the maximum value of 800 nights is extreme and should be investigated.

It should not automatically be removed because it may represent:

* a genuine host restriction,
* a long-term accommodation policy,
* a temporary configuration,
* an outlier,
* a source-data issue.

Further validation is required before cleaning decisions are made.

---

#### Maximum-Night Rules

Observed `maximum_nights` statistics are:

* Minimum: **1**
* Maximum: **2,147,483,647**
* Median: **30**
* Average: approximately **410,752.61**
* Unique values: **169**

The extremely large maximum value strongly inflates the arithmetic average.

Therefore, the average maximum-night value is not representative of the typical listing.

The median of:

**30 nights**

provides a more robust summary of central tendency.

---

#### Extreme Maximum-Night Value

The value:

`2,147,483,647`

appears in the `maximum_nights` field.

This is noteworthy because it is the maximum value of a signed 32-bit integer.

It may represent:

* an effectively unlimited maximum stay,
* a source-system default,
* a sentinel value,
* a technical artifact.

However, its exact business meaning has not yet been confirmed.

Therefore, it should be flagged for investigation and not automatically treated as a genuine intended stay duration.

---

#### Stay-Rule Logical Consistency

The dataset was checked for rows where:

`minimum_nights > maximum_nights`

Observed results:

* Invalid rows: **0**
* Invalid percentage: **0.0000%**

Therefore, every calendar row satisfies:

```text
minimum_nights <= maximum_nights
```

This is a strong logical consistency result.

---

#### Price-Field Absence

The following expected calendar fields are not present:

* `price`
* `adjusted_price`

Therefore, this dataset cannot directly support calendar-level analyses such as:

* weekday price,
* weekend price,
* monthly price changes,
* daily price variation,
* seasonal price variation.

Any price-related analysis must instead use other available listing-level pricing fields.

This limitation should be explicitly documented so that downstream analysis does not claim to use calendar pricing that does not exist in the source.

---

#### Main Data Quality Strengths

The strongest characteristics of `calendar.csv.gz` are:

1. **3,819,725 successfully queryable rows.**
2. **0 missing values across all five columns.**
3. **0 complete duplicate rows.**
4. **Unique `(listing_id, date)` combinations.**
5. **Exactly 365 rows for every listing.**
6. **Exactly 365 unique dates for every listing.**
7. **100.00% bidirectional ID coverage with `listings.csv`.**
8. **100.00% detailed-listing-to-calendar coverage.**
9. **Complete Boolean availability values.**
10. **0 logically invalid cases where `minimum_nights > maximum_nights`.**

---

#### Main Data Quality Concerns

The most important concerns are:

1. The full dataset spans 378 distinct dates because listings use four different 365-day calendar windows.
2. The calendar dataset contains 96 listings absent from the detailed listings file.
3. `minimum_nights` reaches an extreme value of 800.
4. `maximum_nights` reaches 2,147,483,647.
5. The arithmetic average of `maximum_nights` is distorted by extreme values.
6. The dataset has no `price` or `adjusted_price` fields.
7. Unavailable dates must not be interpreted directly as confirmed bookings.

---

#### Overall Interpretation

The `calendar.csv.gz` dataset is structurally strong and highly consistent.

Its grain is clearly defined as:

**One listing-date record per row.**

The candidate composite key is:

`(listing_id, date)`

The dataset provides:

* complete listing identifiers,
* complete dates,
* complete availability indicators,
* complete stay-rule information,
* exactly 365 records per listing,
* perfect alignment with the summary listings population.

It also confirms an important cross-dataset finding:

**The 96 listings present in the summary listings file but absent from the detailed listings file are exactly the same 96 additional listing IDs represented in the calendar dataset.**

Overall assessment:

**The dataset is highly suitable for availability analysis, listing-level calendar aggregation, stay-rule validation, and enriched listing features, with the main cautions being multiple calendar windows, extreme stay-rule values, absence of price fields, and the need to avoid treating unavailable dates as confirmed bookings.**

### Candidate Keys and Relationships

The `calendar.csv.gz` dataset contains daily calendar records for Amsterdam Airbnb listings.

Its grain is:

**One row represents one listing on one calendar date.**

The dataset contains:

- **3,819,725 rows**
- **10,465 unique listing IDs**
- **378 distinct dates across the full dataset**
- **365 calendar records for every listing**
- **0 missing listing IDs**
- **0 missing dates**
- **0 complete duplicate rows**
- **0 duplicate `(listing_id, date)` combinations**

These results provide strong evidence for a composite key and clear one-to-many relationships with the listing datasets.

---

#### Candidate Composite Key

Neither `listing_id` nor `date` is individually unique.

Observed unique counts are:

- Unique `listing_id` values: **10,465**
- Unique `date` values: **378**
- Total rows: **3,819,725**

Therefore, neither field can independently identify one calendar row.

However, the combination:

`(listing_id, date)`

was explicitly tested.

Validation results:

- Duplicate `(listing_id, date)` groups: **0**
- Extra rows caused by duplicate key combinations: **0**
- Missing `listing_id` values: **0**
- Missing `date` values: **0**

Therefore:

**Candidate composite key: (`listing_id`, `date`)**

This composite key uniquely identifies every calendar record in the current dataset.

The logical structure is:

```text
listing_id + date
        │
        ▼
One unique calendar record
````

Because the data is stored in a CSV file rather than a relational database table, this key is logically identified through profiling rather than formally enforced.

---

#### Why `listing_id` Alone Is Not a Primary Key

The `listing_id` field contains:

* **10,465 unique values**
* across **3,819,725 total rows**

Each listing appears exactly:

**365 times**

because every listing has one calendar record for each date in its individual 365-day calendar window.

Therefore:

`listing_id`

is not unique at the calendar-record level.

Instead, it acts as the foreign-key-like relationship field linking calendar records back to the listing entity.

---

#### Why `date` Alone Is Not a Primary Key

The `date` column contains:

* **378 unique dates**

Each date may appear for many different listings.

Therefore:

`date`

cannot uniquely identify one calendar row.

Its role is to identify the day represented by a particular listing calendar record.

---

#### Candidate Key Structure

The key structure can be represented as:

```text
listing_id
    +
   date
    │
    ▼
Unique Calendar Record
```

For example:

```text
listing_id = 28871
date       = 2026-06-16
```

together identify one specific calendar row.

---

#### Relationship with Summary Listings

The calendar dataset was explicitly validated against:

`listings.csv.id`

The relationship is:

```text
listings.csv
    id
     │
     │ one-to-many
     ▼
calendar.csv.gz
    listing_id
```

Validation results:

* Unique calendar listing IDs: **10,465**
* Unique summary listing IDs: **10,465**
* Common listing IDs: **10,465**
* Calendar IDs absent from summary listings: **0**
* Summary listing IDs absent from calendar: **0**

Coverage:

* Calendar-to-summary coverage: **100.00%**
* Summary-to-calendar coverage: **100.00%**

Therefore, both datasets contain exactly the same set of listing IDs.

This is a strong referential-integrity result.

The logical cardinality is:

```text
Summary Listing 1 ───────< Many Calendar Records
```

Because every listing has exactly 365 calendar rows, the observed relationship is more specifically:

```text
One summary listing
        │
        ▼
365 calendar records
```

---

#### Relationship with Detailed Listings

The calendar dataset was also compared with:

`listings.csv.gz.id`

The relationship is:

```text
listings.csv.gz
    id
     │
     │ one-to-many
     ▼
calendar.csv.gz
    listing_id
```

Validation results:

* Unique calendar listing IDs: **10,465**
* Unique detailed listing IDs: **10,369**
* Common listing IDs: **10,369**
* Calendar IDs absent from detailed listings: **96**
* Detailed listing IDs absent from calendar: **0**

Coverage:

* Calendar-to-detailed coverage: **99.08%**
* Detailed-to-calendar coverage: **100.00%**

Therefore:

* every detailed listing has corresponding calendar data,
* but the calendar dataset contains 96 additional listing IDs that do not appear in the detailed listings file.

---

#### Confirmed Relationship of the 96 Additional Listing IDs

Earlier validation found:

* **96 IDs** present in `listings.csv` but absent from `listings.csv.gz`.

The calendar relationship check also identified:

* **96 IDs** present in `calendar.csv.gz` but absent from `listings.csv.gz`.

These two ID sets were explicitly compared.

Observed result:

```text
Exact set match between calendar-extra IDs
and summary-only IDs: True
```

Therefore, it is confirmed that:

**The 96 listing IDs present in the summary listings file but absent from the detailed listings file are exactly the same 96 additional listing IDs found in the calendar dataset.**

The cross-dataset relationship can be represented as:

```text
                    listings.csv
                      10,465 IDs
                          │
                          │ 100% ID match
                          ▼
                    calendar.csv.gz
                      10,465 IDs
                          │
                          │ 10,369 IDs overlap
                          ▼
                   listings.csv.gz
                      10,369 IDs

Difference:
96 specific IDs are present in summary listings
and calendar data but absent from detailed listings.
```

The reason why these 96 listings are missing from the detailed listings file remains unconfirmed.

---

#### Calendar Records per Listing Relationship

Every one of the 10,465 listings has exactly:

**365 calendar records**

Validation results:

* Minimum rows per listing: **365**
* Maximum rows per listing: **365**
* Average rows per listing: **365.0**
* Minimum unique dates per listing: **365**
* Maximum unique dates per listing: **365**
* Average unique dates per listing: **365.0**

The distribution is:

| Calendar Records per Listing | Number of Listings |
| ---------------------------: | -----------------: |
|                          365 |             10,465 |

Therefore, the observed relationship is consistently:

```text
Listing 1 ───────< 365 Calendar Records
```

No listing has fewer or more than 365 calendar rows.

---

#### Calendar Window Relationship

Although each listing has exactly 365 unique calendar dates, the full dataset contains:

**378 distinct dates**

This occurs because listings use four different calendar windows.

| First Date | Last Date  | Unique Dates per Listing | Listings |
| ---------- | ---------- | -----------------------: | -------: |
| 2026-06-15 | 2027-06-14 |                      365 |      499 |
| 2026-06-16 | 2027-06-15 |                      365 |    6,450 |
| 2026-06-24 | 2027-06-23 |                      365 |    3,420 |
| 2026-06-28 | 2027-06-27 |                      365 |       96 |

Therefore, the relationship is not:

```text
All listings
    │
    ▼
Same exact 365 dates
```

Instead, it is:

```text
Each listing
    │
    ▼
One complete 365-day calendar window
```

with different groups of listings starting on different dates.

---

#### Important 96-Listing Calendar Window Pattern

One calendar-window group contains exactly:

**96 listings**

with dates from:

* 2026-06-28
* to 2027-06-27

This count matches:

* the 96 summary-only listings,
* the 96 calendar IDs absent from detailed listings.

However, the exact listing ID set of this 96-listing calendar-window group has not yet been explicitly compared with the known `summary_only_listing_ids` set.

Therefore, the correct interpretation is:

**This is a strong numerical pattern requiring row-level set validation before concluding that the same 96 listings form the 2026-06-28 calendar-window group.**

---

#### Relationship with Availability

Each unique `(listing_id, date)` record contains one Boolean availability value:

* `True`
* `False`

The relationship is:

```text
Listing
   │
   ▼
Calendar Date
   │
   ▼
Availability Status
```

Each candidate-key combination therefore has exactly one recorded availability status.

Observed availability distribution:

* `False`: **2,820,970 rows**
* `True`: **998,755 rows**

This field does not create a separate entity or relationship table.

Instead, it is an attribute of each listing-date calendar record.

---

#### Relationship with Stay Rules

Each calendar record also contains:

* `minimum_nights`
* `maximum_nights`

Therefore, every unique `(listing_id, date)` combination is associated with one minimum-stay rule and one maximum-stay rule.

The structure is:

```text
(listing_id, date)
        │
        ├── available
        ├── minimum_nights
        └── maximum_nights
```

These fields may vary across dates for the same listing.

Therefore, they should be treated as calendar-level attributes rather than automatically assumed to be constant listing-level attributes.

---

#### Logical Stay-Rule Consistency

The relationship between:

* `minimum_nights`
* `maximum_nights`

was explicitly checked.

Observed result:

* Rows where `minimum_nights > maximum_nights`: **0**
* Percentage: **0.0000%**

Therefore, every calendar row satisfies:

```text
minimum_nights <= maximum_nights
```

This is a strong logical-integrity result.

---

#### Expected Relationship with Detailed Reviews

There is no direct key relationship between:

`calendar.csv.gz`

and:

`reviews.csv.gz`

because both are child datasets of the listing entity.

Their relationship is indirect through listing ID:

```text
                 Listing
                    │
           ┌────────┴────────┐
           ▼                 ▼
       Calendar           Reviews
       listing_id         listing_id
```

Calendar data contains one record per listing-date combination, while review data is expected to contain one record per individual review.

These datasets may later be aggregated separately to listing level and joined through:

`listing_id`

---

#### Expected Relationship with Enriched Listing Master

During the enrichment phase, calendar data should first be aggregated to listing level.

Possible derived calendar features include:

* total calendar days,
* available days,
* unavailable days,
* availability rate,
* minimum observed minimum-night requirement,
* maximum observed minimum-night requirement,
* median minimum-night requirement,
* selected maximum-night indicators.

The recommended relationship is:

```text
calendar.csv.gz
3,819,725 calendar rows
        │
        │ aggregate by listing_id
        ▼
10,465 listing-level calendar summaries
        │
        │ join on listing ID
        ▼
Enriched Listing Master
```

This avoids joining millions of raw calendar rows directly to the listing master.

---

#### Relationship Cardinality Summary

The main relationships are:

##### Summary Listings to Calendar

```text
listings.csv.id
       1
       │
       │
       ▼
      many
calendar.csv.gz.listing_id
```

Status: **Validated**

Coverage: **100.00% in both directions**

Observed calendar records per listing: **365**

---

##### Detailed Listings to Calendar

```text
listings.csv.gz.id
        1
        │
        │
        ▼
       many
calendar.csv.gz.listing_id
```

Status: **Validated**

Coverage:

* Detailed-to-calendar: **100.00%**
* Calendar-to-detailed: **99.08%**

Difference: **96 calendar listing IDs absent from detailed listings**

---

##### Calendar Composite Key

```text
(listing_id, date)
```

Status: **Validated candidate composite key**

Duplicate combinations: **0**

---

#### Relationship Validation Status

| Relationship                                           | Status                       | Coverage / Result                                         |
| ------------------------------------------------------ | ---------------------------- | --------------------------------------------------------- |
| `(listing_id, date)` as composite key                  | Validated                    | 0 duplicate combinations                                  |
| `calendar.listing_id` → `listings.csv.id`              | Validated                    | 100.00% both directions                                   |
| `calendar.listing_id` → `listings.csv.gz.id`           | Validated                    | 99.08% calendar-to-detailed; 100.00% detailed-to-calendar |
| Calendar-extra IDs → summary-only IDs                  | Validated                    | Exact set match = True                                    |
| Calendar rows per listing                              | Validated                    | Exactly 365 for every listing                             |
| `minimum_nights <= maximum_nights`                     | Validated                    | 100.00% logically consistent                              |
| 96-listing final calendar window → summary-only 96 IDs | Not yet explicitly validated | Strong numerical pattern only                             |

---

#### Overall Key and Relationship Interpretation

The `calendar.csv.gz` dataset has a clear and reliable relational structure.

Its preferred candidate composite key is:

`(listing_id, date)`

The `listing_id` field acts as the main foreign-key-like relationship field connecting calendar records to listings.

The strongest relationship findings are:

* all 10,465 calendar listing IDs exactly match the 10,465 summary listing IDs,
* every detailed listing has calendar data,
* the 96 calendar IDs absent from detailed listings exactly match the 96 summary-only listing IDs,
* every listing has exactly 365 unique calendar records,
* there are no duplicate `(listing_id, date)` combinations.

The core relationship structure is:

```text
                        Listing
                           │
                           │ listing_id
                           ▼
                      Calendar Row
                   (listing_id, date)
                     /      |       \
                    ▼       ▼        ▼
              Availability  Min Stay  Max Stay
```

Overall, the dataset demonstrates strong key integrity and clear one-to-many relationships, making it highly suitable for listing-level calendar aggregation and enrichment.

### Business-Domain Meaning

The `calendar.csv.gz` dataset represents daily availability and stay-rule information for Amsterdam Airbnb listings.

The grain of the dataset is:

**One row represents one Airbnb listing on one calendar date.**

Each calendar record contains:

- `listing_id`
- `date`
- `available`
- `minimum_nights`
- `maximum_nights`

The dataset contains:

- **3,819,725 calendar records**
- **10,465 unique listings**
- **365 calendar records per listing**
- **378 distinct dates across the full dataset**
- **4 different 365-day calendar windows**

Its primary business role is to describe how each listing is configured across future calendar dates.

---

#### Main Business Entity Represented

The central business entity is the:

**Listing Calendar Record**

A listing calendar record represents the status and stay restrictions of one Airbnb listing on one specific date.

The logical structure is:

```text
Listing
   │
   ▼
Calendar Date
   │
   ├── Availability Status
   ├── Minimum Stay Requirement
   └── Maximum Stay Requirement
````

The candidate composite key is:

`(listing_id, date)`

because each listing-date combination appears exactly once.

---

#### Listing Relationship

The `listing_id` field connects calendar records to the listing entity.

The relationship is:

```text
Listing
   │
   │ one
   ▼
Calendar Records
   │
   │ many
   ▼
365 dated records per listing
```

Each of the 10,465 listings has exactly:

**365 calendar records**

This supports listing-level calendar aggregation and enrichment.

---

#### Availability Status

The `available` field indicates whether a listing is marked as available on a particular calendar date.

Observed values are:

* `True`
* `False`

Observed distribution:

| Availability Status | Row Count | Percentage |
| ------------------- | --------: | ---------: |
| False               | 2,820,970 |     73.85% |
| True                |   998,755 |     26.15% |

Therefore:

* **26.15%** of all calendar rows are marked available.
* **73.85%** are marked unavailable.

This field can support derived measures such as:

```text
available_days
```

```text
unavailable_days
```

```text
availability_rate
=
available_days / total_calendar_days
```

Because every listing has exactly 365 calendar rows:

```text
availability_rate
=
available_days / 365
```

---

#### Important Availability Interpretation

A calendar row marked:

`False`

should not automatically be interpreted as a confirmed booking.

A date may be unavailable because of:

* a guest reservation,
* host blocking,
* maintenance,
* personal use,
* temporary suspension,
* other operational reasons.

Therefore, the correct business interpretation is:

**Unavailable calendar day**

rather than:

**Confirmed booked night**

Similarly, availability should not automatically be treated as the inverse of occupancy.

---

#### Listing-Level Availability Variation

The listing-level summary shows:

* Minimum available days: **0**
* Maximum available days: **365**
* Average available days: approximately **95.44**

For unavailable days:

* Minimum unavailable days: **0**
* Maximum unavailable days: **365**
* Average unavailable days: approximately **269.56**

This means some listings are:

* available for the full 365-day window,
* unavailable for the full 365-day window,
* available only during part of the year.

This variation can support listing segmentation.

For example:

```text
High availability
Medium availability
Low availability
No availability
```

The exact thresholds should be defined explicitly if such segments are created.

---

#### Minimum Stay Requirement

The `minimum_nights` field represents the minimum number of nights required for a booking starting under the relevant calendar conditions.

Observed statistics are:

* Minimum: **1**
* Maximum: **800**
* Median: **3**
* Average: approximately **4.10**
* Unique values: **56**

This field can support business analysis such as:

* identifying short-stay-friendly listings,
* identifying long-stay restrictions,
* comparing stay flexibility across listings,
* comparing minimum-night rules by neighbourhood or room type.

The median value of:

**3 nights**

provides a more representative typical value than the maximum.

---

#### Maximum Stay Requirement

The `maximum_nights` field represents the maximum permitted stay duration associated with a calendar record.

Observed statistics are:

* Minimum: **1**
* Maximum: **2,147,483,647**
* Median: **30**
* Average: approximately **410,752.61**
* Unique values: **169**

The very large maximum value strongly distorts the arithmetic mean.

Therefore, the median of:

**30 nights**

is more useful for describing typical maximum-stay rules.

The extreme value:

`2,147,483,647`

should be treated cautiously because it may represent:

* an effectively unlimited stay,
* a source-system default,
* a sentinel,
* a technical artifact.

Its exact meaning is not yet confirmed.

---

#### Stay-Rule Flexibility

The combination of:

* `minimum_nights`
* `maximum_nights`

describes booking flexibility.

For example:

```text
Low minimum_nights
        │
        ▼
More flexible for short stays
```

and:

```text
High minimum_nights
        │
        ▼
More restrictive for short stays
```

These rules may be useful for comparing:

* neighbourhoods,
* room types,
* hosts,
* property categories.

However, the interpretation should remain cautious because some extreme values may reflect technical defaults or unusual listing configurations.

---

#### Stay-Rule Logical Consistency

Every calendar row satisfies:

```text
minimum_nights <= maximum_nights
```

Observed results:

* Invalid rows: **0**
* Invalid percentage: **0.0000%**

This is a strong logical consistency result.

It means no calendar record contains a minimum stay that is greater than its maximum stay.

---

#### Multiple Calendar Windows

The full dataset contains:

**378 distinct dates**

However, every listing has exactly:

**365 calendar dates**

This occurs because different listing groups use different 365-day windows.

Observed calendar windows are:

| First Date | Last Date  | Listings |
| ---------- | ---------- | -------: |
| 2026-06-15 | 2027-06-14 |      499 |
| 2026-06-16 | 2027-06-15 |    6,450 |
| 2026-06-24 | 2027-06-23 |    3,420 |
| 2026-06-28 | 2027-06-27 |       96 |

Therefore, the dataset should not be interpreted as one single synchronized 365-day snapshot shared by every listing.

Instead:

**Each listing has one complete 365-day calendar window, but the starting date differs across groups of listings.**

---

#### Business Meaning of Different Calendar Windows

Different calendar windows may reflect:

* different scrape dates,
* different source states,
* different collection timing.

However, the exact cause has not been confirmed.

From a business-analysis perspective, this means:

* availability comparisons are still possible,
* but exact date-level comparisons should account for differing date ranges,
* not every listing is represented on every one of the 378 dates.

This is especially important for seasonal or month-specific analysis.

---

#### Confirmed 96-Listing Cross-Dataset Relationship

The calendar dataset contains:

* **10,465 unique listing IDs**

The detailed listings dataset contains:

* **10,369 unique listing IDs**

The difference is:

**96 listings**

These 96 calendar IDs were explicitly validated as exactly the same 96 IDs that:

* appear in `listings.csv`,
* do not appear in `listings.csv.gz`.

Therefore, the calendar dataset aligns perfectly with the summary listing population.

The relationship is:

```text
Summary Listings
    10,465 IDs
         │
         │ exact ID match
         ▼
Calendar
    10,465 IDs
```

while:

```text
Detailed Listings
    10,369 IDs
```

omit those same 96 IDs.

The reason for this difference remains unknown.

---

#### Calendar as a Source for Enriched Listing Features

The raw calendar dataset contains:

**3,819,725 rows**

This is too granular to join directly into a listing-level analytical table without aggregation.

The recommended workflow is:

```text
Raw Calendar
3,819,725 rows
        │
        │ aggregate by listing_id
        ▼
Listing-Level Calendar Summary
10,465 rows
        │
        │ join with listings
        ▼
Enriched Listing Master
```

Potential derived listing-level features include:

* `calendar_days`
* `available_days`
* `unavailable_days`
* `availability_rate`
* `min_minimum_nights`
* `max_minimum_nights`
* `median_minimum_nights`
* `min_maximum_nights`
* `max_maximum_nights`

Only useful and interpretable features should be retained.

---

#### Availability as a Market Signal

Availability may provide a useful signal about listing activity or host behavior.

For example, a listing with very low availability may be:

* highly demanded,
* heavily booked,
* manually blocked by the host,
* unavailable for operational reasons.

Therefore, low availability should not be interpreted as confirmed high demand without additional evidence.

The safest description is:

**Calendar availability pattern**

rather than direct occupancy or booking demand.

---

#### Relationship with Listing Characteristics

After aggregation, calendar features can be combined with listing attributes such as:

* neighbourhood,
* room type,
* property type,
* accommodates,
* host status,
* review score.

This can support questions such as:

* Which neighbourhoods have the highest availability rates?
* Do entire homes have different availability patterns from private rooms?
* Do superhost listings differ in calendar availability?
* Are highly reviewed listings less available?
* Do minimum-stay requirements vary by property type?

These relationships can support EDA and business interpretation.

---

#### No Calendar-Level Price Analysis

The actual calendar schema contains no:

* `price`
* `adjusted_price`

fields.

Therefore, this dataset cannot support direct calendar-level price analysis such as:

* weekday vs weekend pricing,
* monthly price changes,
* seasonal price variation,
* date-specific price analysis.

Any such analysis would require another pricing source.

This limitation should be clearly documented to avoid unsupported conclusions.

---

#### Difference Between Availability and Occupancy

The concepts must remain distinct:

```text
Availability
=
Whether a date is marked available
```

```text
Occupancy
=
Whether a listing was actually occupied
```

The current calendar dataset directly contains only:

**Availability**

It does not directly confirm:

* booking status,
* guest stay,
* actual occupancy,
* actual revenue.

Therefore, any derived metric based on unavailable days should be described cautiously.

---

#### Business Questions Supported by the Dataset

The calendar dataset can support questions such as:

* How many days is each listing available?
* Which listings are available for the full year?
* Which listings have no available days?
* Which neighbourhoods show higher average availability?
* Which room types have more restrictive stay requirements?
* Which listings have unusual minimum-night rules?
* How do calendar windows differ across listing groups?
* Are there listing-level differences in availability patterns?

These questions can provide useful operational and market insights.

---

#### Analytical Importance

The `calendar.csv.gz` dataset is important for:

* availability analysis,
* stay-rule analysis,
* listing-level feature engineering,
* cross-dataset relationship validation,
* calendar aggregation,
* enriched listing master construction,
* operational interpretation.

Its greatest value comes from converting millions of daily calendar rows into compact listing-level features.

---

#### Overall Business Interpretation

The `calendar.csv.gz` dataset describes the daily availability status and stay restrictions of Amsterdam Airbnb listings.

Its central business relationship is:

```text
Listing
   │
   ▼
365 Calendar Dates
   │
   ├── Availability
   ├── Minimum Stay
   └── Maximum Stay
```

The dataset is especially useful for deriving listing-level measures such as:

* availability rate,
* available-day count,
* unavailable-day count,
* stay-rule indicators.

However, important interpretation rules must be preserved:

* unavailable does not necessarily mean booked,
* availability does not equal occupancy,
* calendar data does not contain price fields,
* different listings use different 365-day date windows,
* extreme stay-rule values require cautious validation.

Overall, the dataset provides strong operational calendar information and is highly valuable for listing-level enrichment after efficient DuckDB aggregation.

### Data Quality Assessment

The `calendar.csv.gz` dataset is structurally strong, complete, and highly consistent with the summary listings dataset.

The dataset contains:

- **3,819,725 rows**
- **5 columns**
- **10,465 unique listing IDs**
- **378 distinct dates across the full dataset**
- **365 calendar records for every listing**
- **0 missing values**
- **0 complete duplicate rows**
- **0 duplicate `(listing_id, date)` combinations**
- **100.00% bidirectional ID coverage with `listings.csv`**
- **100.00% detailed-listing-to-calendar coverage**
- **0 rows where `minimum_nights > maximum_nights`**

Overall, the dataset demonstrates excellent structural and relational quality, although several analytical limitations and extreme stay-rule values require careful treatment.

---

#### Structural Quality

The dataset contains five fields:

- `listing_id`
- `date`
- `available`
- `minimum_nights`
- `maximum_nights`

Observed DuckDB data types are:

| Column | Data Type |
|---|---|
| `listing_id` | BIGINT |
| `date` | DATE |
| `available` | BOOLEAN |
| `minimum_nights` | BIGINT |
| `maximum_nights` | BIGINT |

The schema is compact and strongly typed.

Important positive characteristics include:

- `listing_id` is stored as `BIGINT`, which safely preserves large identifiers.
- `date` is correctly interpreted as `DATE`.
- `available` is correctly interpreted as `BOOLEAN`.
- stay-rule fields are represented as integer values.

No immediate data-type correction is required for these five fields.

---

#### Missing-Value Quality

No missing values were found in any column.

| Column | Missing Count | Missing Percentage |
|---|---:|---:|
| `listing_id` | 0 | 0.00% |
| `date` | 0 | 0.00% |
| `available` | 0 | 0.00% |
| `minimum_nights` | 0 | 0.00% |
| `maximum_nights` | 0 | 0.00% |

Total missing values across the dataset:

**0**

This is an excellent completeness result, particularly for a dataset containing more than 3.8 million records.

---

#### Duplicate-Row Quality

Complete duplicate rows were explicitly tested.

Observed results:

- Total rows: **3,819,725**
- Duplicate groups: **0**
- Extra duplicate rows: **0**
- Duplicate percentage: **0.0000%**

Therefore, no complete duplicate records were found.

This is a strong positive data-quality characteristic.

---

#### Candidate Composite-Key Quality

The combination:

`(listing_id, date)`

was tested as a candidate composite key.

Observed results:

- Duplicate key combinations: **0**
- Extra rows caused by duplicate keys: **0**
- Missing `listing_id` values: **0**
- Missing `date` values: **0**

Therefore:

**Candidate composite key: (`listing_id`, `date`)**

The candidate key is complete and unique across all 3,819,725 records.

This confirms the intended dataset grain:

**One row per listing per calendar date.**

---

#### Listing Identifier Quality

The `listing_id` field contains:

- **3,819,725 total values**
- **10,465 unique listing IDs**
- **0 missing listing IDs**

The field is suitable for linking calendar records to listing datasets.

Its `BIGINT` type also avoids the floating-point precision concern observed in some identifier fields from other datasets.

---

#### Relationship Quality with Summary Listings

The calendar dataset was explicitly compared with:

`listings.csv.id`

Observed results:

- Unique calendar listing IDs: **10,465**
- Unique summary listing IDs: **10,465**
- Common listing IDs: **10,465**
- Calendar IDs not found in summary listings: **0**
- Summary listing IDs absent from calendar: **0**

Coverage:

- Calendar-to-summary coverage: **100.00%**
- Summary-to-calendar coverage: **100.00%**

This means the two datasets contain exactly the same listing population.

The relationship is:

```text
listings.csv.id
       1
       │
       ▼
      many
calendar.csv.gz.listing_id
````

This is one of the strongest cross-dataset integrity results identified so far.

---

#### Relationship Quality with Detailed Listings

The calendar dataset was also compared with:

`listings.csv.gz.id`

Observed results:

* Unique calendar listing IDs: **10,465**
* Unique detailed listing IDs: **10,369**
* Common listing IDs: **10,369**
* Calendar IDs absent from detailed listings: **96**
* Detailed listing IDs absent from calendar: **0**

Coverage:

* Calendar-to-detailed coverage: **99.08%**
* Detailed-to-calendar coverage: **100.00%**

Therefore:

* every detailed listing has calendar data,
* 96 calendar listing IDs do not appear in the detailed listing dataset.

This is not a calendar-integrity failure because those same 96 IDs are valid records in the summary listings dataset.

---

#### Confirmed 96-Listing Cross-Dataset Consistency

The 96 calendar listing IDs absent from the detailed listings file were explicitly compared with the previously identified 96 summary-only listing IDs.

Observed result:

```text
Exact set match between calendar-extra IDs
and summary-only IDs: True
```

Therefore, it is confirmed that:

**The exact same 96 listing IDs appear in `listings.csv` and `calendar.csv.gz` but are absent from `listings.csv.gz`.**

This is a strong cross-dataset consistency result.

The remaining issue is not whether the IDs match, but why the detailed listings file omits them.

That cause remains unconfirmed.

---

#### Calendar Coverage Quality

Every listing has exactly:

**365 calendar records**

Observed results:

* Minimum rows per listing: **365**
* Maximum rows per listing: **365**
* Average rows per listing: **365.0**

Unique dates per listing:

* Minimum unique dates: **365**
* Maximum unique dates: **365**
* Average unique dates: **365.0**

Distribution:

| Calendar Rows per Listing | Number of Listings |
| ------------------------: | -----------------: |
|                       365 |             10,465 |

This demonstrates exceptionally consistent listing-level calendar coverage.

No listing has:

* missing calendar dates within its own 365-row window,
* fewer than 365 rows,
* more than 365 rows.

---

#### Overall Date Coverage

The complete dataset spans:

* Earliest date: **2026-06-15**
* Latest date: **2027-06-27**
* Unique dates across all records: **378**
* Missing dates: **0**

Although every listing contains exactly 365 dates, the full dataset contains 378 unique dates because listings are divided across different calendar windows.

This is not a duplicate or completeness problem.

It is a temporal comparability consideration.

---

#### Calendar Window Consistency

Four calendar windows were observed:

| First Calendar Date | Last Calendar Date | Unique Dates per Listing | Listing Count |
| ------------------- | ------------------ | -----------------------: | ------------: |
| 2026-06-15          | 2027-06-14         |                      365 |           499 |
| 2026-06-16          | 2027-06-15         |                      365 |         6,450 |
| 2026-06-24          | 2027-06-23         |                      365 |         3,420 |
| 2026-06-28          | 2027-06-27         |                      365 |            96 |

The listing counts sum to:

**10,465**

Therefore, every listing belongs to exactly one observed 365-day calendar window.

This is a strong structural result.

However, date-specific or seasonal comparisons should account for the fact that listings do not all share the same exact date range.

---

#### Availability Data Quality

The `available` field contains exactly two values:

* `False`
* `True`

Observed distribution:

| Availability | Row Count | Percentage |
| ------------ | --------: | ---------: |
| False        | 2,820,970 |     73.85% |
| True         |   998,755 |     26.15% |

There are:

* 0 missing availability values,
* 2 expected Boolean categories.

This is an excellent domain-validity result.

However, the field should be interpreted carefully.

A value of `False` means:

**The calendar date is marked unavailable.**

It does not necessarily prove:

* a confirmed booking,
* occupancy,
* guest stay.

Therefore, availability data is structurally strong but has interpretive limitations.

---

#### Listing-Level Availability Range

Observed listing-level availability statistics are:

* Minimum available days: **0**
* Maximum available days: **365**
* Average available days: approximately **95.44**

For unavailable days:

* Minimum unavailable days: **0**
* Maximum unavailable days: **365**
* Average unavailable days: approximately **269.56**

These ranges are logically valid because each listing has exactly 365 total calendar records.

The following identity holds:

```text
available_days + unavailable_days = 365
```

for each listing.

This supports reliable creation of:

* available-day count,
* unavailable-day count,
* availability rate.

---

#### Minimum-Night Quality

Observed `minimum_nights` statistics are:

* Minimum: **1**
* Maximum: **800**
* Median: **3**
* Average: approximately **4.10**
* Unique values: **56**

Positive quality observations:

* No missing values.
* No zero or negative values were observed in the reported range.
* All values are integer-based.

Main concern:

* The maximum of **800 nights** is unusually large.

This value should be investigated before any decision to:

* remove,
* cap,
* replace,
* transform it.

It may represent a valid extreme configuration or a data-quality issue.

---

#### Maximum-Night Quality

Observed `maximum_nights` statistics are:

* Minimum: **1**
* Maximum: **2,147,483,647**
* Median: **30**
* Average: approximately **410,752.61**
* Unique values: **169**

The most important concern is the extreme value:

`2,147,483,647`

This corresponds to the maximum signed 32-bit integer and may potentially represent:

* an unlimited maximum stay,
* a technical default,
* a sentinel,
* a source-system artifact.

Its exact meaning is not confirmed.

The extreme values also distort the arithmetic mean.

Therefore:

**Median maximum nights = 30**

is a more representative measure of typical stay-rule central tendency than the average of approximately 410,752.61.

---

#### Logical Stay-Rule Quality

The dataset was explicitly checked for rows where:

`minimum_nights > maximum_nights`

Observed results:

* Invalid rows: **0**
* Invalid percentage: **0.0000%**

Therefore, all 3,819,725 rows satisfy:

```text
minimum_nights <= maximum_nights
```

This is an excellent logical-integrity result.

---

#### Price-Field Availability

The actual schema contains no:

* `price`
* `adjusted_price`

fields.

Therefore, the calendar dataset cannot support:

* daily price analysis,
* weekday vs weekend pricing,
* seasonal price changes,
* monthly dynamic pricing analysis.

This is a source-schema limitation rather than a data-quality error.

It must be documented clearly to prevent unsupported analyses.

---

#### Extreme-Value Assessment

The two most important extreme values are:

* `minimum_nights = 800`
* `maximum_nights = 2,147,483,647`

These should be classified as:

**Validation priorities**

rather than automatically as errors.

Recommended treatment process:

1. Detect the extreme value.
2. Identify affected listings and dates.
3. Check whether values remain constant or vary over time.
4. Compare with corresponding listing-level stay-rule fields where available.
5. Document the decision.
6. Preserve original raw values.

This avoids arbitrary removal.

---

#### Temporal Comparability Risk

The dataset's four different calendar windows create a moderate analytical risk.

For listing-level annual availability metrics, comparability is strong because every listing has exactly 365 records.

For exact date-level or month-level analysis, coverage differs.

For example, not all listings are represented on:

* 2026-06-15,
* 2026-06-16,
* 2026-06-24,
* 2026-06-28.

Therefore, temporal aggregations should use the actual number of observed rows as the denominator.

---

#### Memory and Processing Quality

The dataset contains more than 3.8 million rows.

It was successfully profiled using DuckDB without loading the entire file into a Pandas DataFrame.

This is an appropriate engineering choice for an 8 GB RAM environment.

The recommended strategy for downstream use is:

```text
Raw Calendar
3,819,725 rows
        │
        │ aggregate with DuckDB
        ▼
Listing-Level Calendar Summary
10,465 rows
        │
        │ join
        ▼
Enriched Listing Master
```

This preserves memory efficiency and avoids unnecessary many-to-many or one-to-many expansion during enrichment.

---

#### Main Data Quality Strengths

The strongest characteristics of `calendar.csv.gz` are:

1. **0 missing values across all five columns.**
2. **0 complete duplicate rows.**
3. **0 duplicate `(listing_id, date)` combinations.**
4. **Complete candidate composite key.**
5. **Exactly 365 rows per listing.**
6. **Exactly 365 unique dates per listing.**
7. **100.00% ID match with summary listings.**
8. **100.00% detailed-listing-to-calendar coverage.**
9. **Exact set confirmation of the 96 summary-only/calendar-extra IDs.**
10. **Valid Boolean availability values.**
11. **0 logically invalid minimum/maximum stay combinations.**
12. **Strong integer and date typing in DuckDB.**

---

#### Main Data Quality Concerns

The main concerns are:

1. Four different 365-day calendar windows reduce exact date-level comparability.
2. The calendar contains 96 listings absent from detailed listings.
3. The cause of the 96-listing omission from detailed listings remains unknown.
4. `minimum_nights` reaches an extreme value of 800.
5. `maximum_nights` reaches 2,147,483,647.
6. The mean `maximum_nights` value is severely distorted by extreme values.
7. No calendar-level price fields exist.
8. Unavailable dates cannot be interpreted as confirmed bookings.
9. Availability cannot be equated directly with occupancy.
10. The 3.8-million-row size requires memory-aware processing.

---

#### Data Quality Risk Classification

| Quality Area                           | Assessment                                       |
| -------------------------------------- | ------------------------------------------------ |
| Schema consistency                     | Excellent                                        |
| Missing-value quality                  | Excellent                                        |
| Duplicate quality                      | Excellent                                        |
| Composite-key quality                  | Excellent                                        |
| Listing-ID completeness                | Excellent                                        |
| Summary-listing referential integrity  | Excellent                                        |
| Detailed-listing referential integrity | Very Good, with 96 known additional calendar IDs |
| Calendar coverage per listing          | Excellent                                        |
| Availability domain validity           | Excellent                                        |
| Stay-rule logical consistency          | Excellent                                        |
| Temporal comparability                 | Moderate concern                                 |
| Extreme-value quality                  | Requires validation                              |
| Calendar pricing capability            | Not available in source                          |

---

#### Recommended Cleaning and Validation Actions

Recommended downstream actions are:

* preserve the raw calendar file unchanged,
* retain `(listing_id, date)` as the logical composite key,
* aggregate calendar records by `listing_id` before enrichment,
* create `available_days`,
* create `unavailable_days`,
* create `availability_rate`,
* investigate extreme `minimum_nights` values,
* investigate `maximum_nights = 2,147,483,647`,
* avoid interpreting unavailable dates as confirmed bookings,
* avoid calling calendar unavailability true occupancy,
* account for multiple calendar windows in time-based analysis,
* document the absence of calendar price fields,
* revalidate referential integrity whenever future data snapshots are processed.

---

#### Overall Data Quality Assessment

The `calendar.csv.gz` dataset has excellent structural quality.

It provides:

* complete identifiers,
* complete dates,
* complete Boolean availability data,
* complete stay-rule fields,
* zero duplicates,
* a valid composite key,
* exactly 365 records per listing,
* perfect alignment with the summary listing population.

The main concerns are not basic completeness or duplication.

Instead, they involve:

* interpretation of unavailable dates,
* different 365-day calendar windows,
* extreme stay-rule values,
* absence of price fields,
* the known 96-listing difference from detailed listings.

**Overall assessment: Excellent-quality calendar data that is highly suitable for DuckDB aggregation, availability feature engineering, stay-rule analysis, and listing-level enrichment, provided that temporal differences, extreme values, and availability interpretation are handled carefully and documented transparently.**

## 7. Detailed Reviews Dataset Familiarization

The `reviews.csv.gz` dataset contains detailed review-level records for Amsterdam Airbnb listings.

This dataset is expected to complement the summary `reviews.csv` file by providing additional review-level attributes such as review identifiers, reviewer information, and review comments.

Because the compressed detailed reviews file may contain hundreds of thousands of rows and large text fields, it will be profiled directly with DuckDB instead of loading the full dataset into Pandas.

The dataset will be inspected for:

- File structure
- Row count
- Column count
- Column names
- Data types
- Missing values
- Unique-value counts
- Sample values
- Duplicate rows
- Candidate primary keys
- Candidate composite keys
- Listing relationships
- Date coverage
- Review identifier quality
- Reviewer information
- Comment completeness
- Relationship with `reviews.csv`
- Relationship with `listings.csv`
- Relationship with `listings.csv.gz`
- Business-domain meaning
- Dataset limitations
- Data quality issues

The expected relationship is:

`reviews.csv.gz.listing_id` → `listings.csv.id`

However, all keys and cross-dataset relationships will be explicitly validated from the actual data before conclusions are made.

Release the calendar DuckDB connection

In [150]:
calendar_connection.close()

print("Calendar DuckDB connection closed successfully.")

Calendar DuckDB connection closed successfully.


Create a DuckDB connection for detailed reviews

In [151]:
# Create a separate in-memory DuckDB connection
# for the detailed reviews dataset.

reviews_connection = duckdb.connect(database=":memory:")

print("DuckDB reviews connection created successfully.")

DuckDB reviews connection created successfully.


Prepare the detailed reviews file path

In [152]:
detailed_reviews_file_path = str(
    DATA_FILES["detailed_reviews"]
).replace("\\", "/")

print("Detailed reviews file path:")
print(detailed_reviews_file_path)

Detailed reviews file path:
c:/Users/saths/Desktop/MyProjects/Expernetic Data Challenge/airbnb-data-challenge/data/raw/amsterdam/reviews.csv.gz


Create a DuckDB view over reviews.csv.gz

In [153]:
reviews_connection.execute(f"""
    CREATE OR REPLACE VIEW detailed_reviews_raw AS
    SELECT *
    FROM read_csv_auto(
        '{detailed_reviews_file_path}',
        compression='gzip',
        header=true,
        sample_size=-1,
        all_varchar=false
    )
""")

print("DuckDB detailed reviews view created successfully.")

DuckDB detailed reviews view created successfully.


Inspect the actual schema

In [154]:
detailed_reviews_schema_df = reviews_connection.execute("""
    DESCRIBE detailed_reviews_raw
""").fetchdf()

print("Detailed Reviews Schema")
print("-" * 60)

display(detailed_reviews_schema_df)

Detailed Reviews Schema
------------------------------------------------------------


,column_name,column_type,null,key,default,extra
0,listing_id,BIGINT,YES,None,None,None
1,id,BIGINT,YES,None,None,None
2,date,DATE,YES,None,None,None
3,reviewer_id,BIGINT,YES,None,None,None
4,reviewer_name,VARCHAR,YES,None,None,None
5,comments,VARCHAR,YES,None,None,None


Show all column names

In [155]:
detailed_reviews_columns = (
    detailed_reviews_schema_df["column_name"]
    .tolist()
)

print("Detailed Reviews Column Names")
print("-" * 60)

for index, column in enumerate(
    detailed_reviews_columns,
    start=1
):
    print(f"{index}. {column}")

Detailed Reviews Column Names
------------------------------------------------------------
1. listing_id
2. id
3. date
4. reviewer_id
5. reviewer_name
6. comments


Calculate dataset size

In [156]:
detailed_reviews_row_count = (
    reviews_connection.execute("""
        SELECT COUNT(*)
        FROM detailed_reviews_raw
    """).fetchone()[0]
)

detailed_reviews_column_count = len(
    detailed_reviews_columns
)

print("Detailed Reviews Dataset Size")
print("-" * 50)

print(
    f"Rows: "
    f"{detailed_reviews_row_count:,}"
)

print(
    f"Columns: "
    f"{detailed_reviews_column_count}"
)

Detailed Reviews Dataset Size
--------------------------------------------------
Rows: 545,162
Columns: 6


Preview the first five rows

In [157]:
detailed_reviews_sample_df = (
    reviews_connection.execute("""
        SELECT *
        FROM detailed_reviews_raw
        LIMIT 5
    """).fetchdf()
)

display(detailed_reviews_sample_df)

,listing_id,id,date,reviewer_id,reviewer_name,comments
0,28871,82539,2010-08-22,163752,Dave,Very nice place and people. Great location! Made our vacation very fun.
1,28871,21518912,2014-10-19,20597737,Zuzana,"We were at the double room and I don't have anything wrong to say. The room was clean and cozy, ..."
2,28871,26293687,2015-02-09,20543694,Melissa,"Edwin is friendly and humorous guide, his house is<br/>warm and sweet, I really enjoyed it when ..."
3,28871,32450201,2015-05-18,26175845,Franz,"Nos ha gustado mucho el sitio de Edwin, esta increiblemente bien ubicado y es encantador, Edwin ..."
4,28871,32686822,2015-05-20,29217313,Maria,I didn't actually see Edwin because he was out of town on business. He left someone to meet me a...


Build missing-value summary

In [158]:
detailed_reviews_missing_expressions = []

for column in detailed_reviews_columns:
    expression = (
        f'SUM('
        f'CASE WHEN "{column}" IS NULL '
        f'THEN 1 ELSE 0 END'
        f') AS "{column}"'
    )

    detailed_reviews_missing_expressions.append(
        expression
    )

detailed_reviews_missing_query = f"""
    SELECT
        {
            ", ".join(
                detailed_reviews_missing_expressions
            )
        }
    FROM detailed_reviews_raw
"""

detailed_reviews_missing_counts = (
    reviews_connection
    .execute(detailed_reviews_missing_query)
    .fetchdf()
    .iloc[0]
)

detailed_reviews_missing_summary_df = pd.DataFrame({
    "column_name": detailed_reviews_columns,
    "missing_count": [
        int(detailed_reviews_missing_counts[column])
        for column in detailed_reviews_columns
    ]
})

detailed_reviews_missing_summary_df[
    "missing_percentage"
] = (
    detailed_reviews_missing_summary_df[
        "missing_count"
    ]
    / detailed_reviews_row_count
    * 100
).round(2)

detailed_reviews_missing_summary_df = (
    detailed_reviews_missing_summary_df
    .sort_values(
        by="missing_percentage",
        ascending=False
    )
    .reset_index(drop=True)
)

display(detailed_reviews_missing_summary_df)

,column_name,missing_count,missing_percentage
0,listing_id,0,0.0
1,id,0,0.0
2,date,0,0.0
3,reviewer_id,0,0.0
4,reviewer_name,1,0.0
5,comments,0,0.0


Calculate exact unique-value counts

In [159]:
detailed_reviews_unique_results = []

for column in detailed_reviews_columns:
    unique_count = reviews_connection.execute(f"""
        SELECT COUNT(DISTINCT "{column}")
        FROM detailed_reviews_raw
    """).fetchone()[0]

    detailed_reviews_unique_results.append({
        "column_name": column,
        "unique_count": int(unique_count)
    })

detailed_reviews_unique_summary_df = pd.DataFrame(
    detailed_reviews_unique_results
)

display(detailed_reviews_unique_summary_df)

,column_name,unique_count
0,listing_id,9432
1,id,545162
2,date,5327
3,reviewer_id,523777
4,reviewer_name,72308
5,comments,528743


Create compact column profile

In [160]:
detailed_reviews_profile_df = (
    detailed_reviews_schema_df[
        [
            "column_name",
            "column_type"
        ]
    ]
    .merge(
        detailed_reviews_missing_summary_df,
        on="column_name",
        how="left"
    )
    .merge(
        detailed_reviews_unique_summary_df,
        on="column_name",
        how="left"
    )
)

display(detailed_reviews_profile_df)

,column_name,column_type,missing_count,missing_percentage,unique_count
0,listing_id,BIGINT,0,0.0,9432
1,id,BIGINT,0,0.0,545162
2,date,DATE,0,0.0,5327
3,reviewer_id,BIGINT,0,0.0,523777
4,reviewer_name,VARCHAR,1,0.0,72308
5,comments,VARCHAR,0,0.0,528743


Check expected important columns

In [161]:
expected_detailed_reviews_columns = [
    "listing_id",
    "id",
    "date",
    "reviewer_id",
    "reviewer_name",
    "comments"
]

print("Expected Detailed Reviews Column Check")
print("-" * 60)

for column in expected_detailed_reviews_columns:
    status = (
        "FOUND"
        if column in detailed_reviews_columns
        else "NOT FOUND"
    )

    print(f"{column}: {status}")

Expected Detailed Reviews Column Check
------------------------------------------------------------
listing_id: FOUND
id: FOUND
date: FOUND
reviewer_id: FOUND
reviewer_name: FOUND
comments: FOUND


Display safe sample values for each column

In [162]:
detailed_reviews_sample_values = []

for column in detailed_reviews_columns:
    sample_values = reviews_connection.execute(f"""
        SELECT DISTINCT CAST("{column}" AS VARCHAR)
        FROM detailed_reviews_raw
        WHERE "{column}" IS NOT NULL
        LIMIT 3
    """).fetchall()

    sample_values = [
        (
            value[0][:100] + "..."
            if value[0] is not None
            and len(value[0]) > 100
            else value[0]
        )
        for value in sample_values
    ]

    detailed_reviews_sample_values.append({
        "column_name": column,
        "sample_values": sample_values
    })

detailed_reviews_sample_values_df = pd.DataFrame(
    detailed_reviews_sample_values
)

display(detailed_reviews_sample_values_df)

,column_name,sample_values
0,listing_id,"[965841585626848134, 958488169480389455, 972431973085759996]"
1,id,"[21518912, 78066145, 80350934]"
2,date,"[2013-03-18, 2014-04-23, 2014-06-17]"
3,reviewer_id,"[20523419, 59958770, 110548978]"
4,reviewer_name,"[Gonnie, Gonzalo, Noortje]"
5,comments,"[It's nothing fancy, but it met my needs perfectly. Great shower water pressure and temperature..."


Check basic minimum and maximum values dynamically

In [163]:
basic_profile_columns = [
    column
    for column in [
        "listing_id",
        "id",
        "date",
        "reviewer_id"
    ]
    if column in detailed_reviews_columns
]

detailed_reviews_range_results = []

for column in basic_profile_columns:
    result = reviews_connection.execute(f"""
        SELECT
            MIN("{column}") AS minimum_value,
            MAX("{column}") AS maximum_value
        FROM detailed_reviews_raw
    """).fetchdf()

    detailed_reviews_range_results.append({
        "column_name": column,
        "minimum_value": (
            result.loc[0, "minimum_value"]
        ),
        "maximum_value": (
            result.loc[0, "maximum_value"]
        )
    })

detailed_reviews_range_summary_df = pd.DataFrame(
    detailed_reviews_range_results
)

display(detailed_reviews_range_summary_df)

,column_name,minimum_value,maximum_value
0,listing_id,28871,1704341300366137848
1,id,79730,1717984606216649127
2,date,2010-08-16 00:00:00,2026-06-28 00:00:00
3,reviewer_id,1,1711012061390045280


Check comment-length statistics if comments exists

In [164]:
if "comments" in detailed_reviews_columns:

    detailed_reviews_comment_length_df = (
        reviews_connection.execute("""
            SELECT
                MIN(LENGTH(comments))
                    AS min_comment_length,
                MAX(LENGTH(comments))
                    AS max_comment_length,
                MEDIAN(LENGTH(comments))
                    AS median_comment_length,
                AVG(LENGTH(comments))
                    AS avg_comment_length
            FROM detailed_reviews_raw
            WHERE comments IS NOT NULL
        """).fetchdf()
    )

    print("Review Comment Length Summary")
    print("-" * 50)

    display(
        detailed_reviews_comment_length_df
    )

else:
    print(
        "No comments column exists in the "
        "detailed reviews dataset."
    )

Review Comment Length Summary
--------------------------------------------------


,min_comment_length,max_comment_length,median_comment_length,avg_comment_length
0,1,6164,189.0,247.759804


Check empty-string comments separately

In [165]:
if "comments" in detailed_reviews_columns:

    detailed_reviews_empty_comments_count = (
        reviews_connection.execute("""
            SELECT COUNT(*)
            FROM detailed_reviews_raw
            WHERE comments IS NOT NULL
              AND TRIM(comments) = ''
        """).fetchone()[0]
    )

    print("Empty Comment Validation")
    print("-" * 50)

    print(
        f"Empty but non-null comments: "
        f"{detailed_reviews_empty_comments_count:,}"
    )

else:
    detailed_reviews_empty_comments_count = None

Empty Comment Validation
--------------------------------------------------
Empty but non-null comments: 0


Full duplicate-row analysis

In [166]:
detailed_reviews_duplicate_summary_df = (
    reviews_connection.execute("""
        SELECT
            COUNT(*) AS duplicate_groups,
            COALESCE(
                SUM(row_count - 1),
                0
            ) AS duplicate_extra_rows
        FROM (
            SELECT
                listing_id,
                id,
                date,
                reviewer_id,
                reviewer_name,
                comments,
                COUNT(*) AS row_count
            FROM detailed_reviews_raw
            GROUP BY
                listing_id,
                id,
                date,
                reviewer_id,
                reviewer_name,
                comments
            HAVING COUNT(*) > 1
        )
    """).fetchdf()
)

detailed_reviews_duplicate_group_count = int(
    detailed_reviews_duplicate_summary_df.loc[
        0,
        "duplicate_groups"
    ]
)

detailed_reviews_duplicate_extra_rows = int(
    detailed_reviews_duplicate_summary_df.loc[
        0,
        "duplicate_extra_rows"
    ]
)

detailed_reviews_duplicate_percentage = (
    detailed_reviews_duplicate_extra_rows
    / detailed_reviews_row_count
    * 100
)

print("Detailed Reviews Full Duplicate-Row Analysis")
print("-" * 60)

print(
    f"Total rows: "
    f"{detailed_reviews_row_count:,}"
)

print(
    f"Duplicate groups: "
    f"{detailed_reviews_duplicate_group_count:,}"
)

print(
    f"Extra duplicate rows: "
    f"{detailed_reviews_duplicate_extra_rows:,}"
)

print(
    f"Duplicate percentage: "
    f"{detailed_reviews_duplicate_percentage:.4f}%"
)

Detailed Reviews Full Duplicate-Row Analysis
------------------------------------------------------------
Total rows: 545,162
Duplicate groups: 0
Extra duplicate rows: 0
Duplicate percentage: 0.0000%


Review ID primary-key validation

In [167]:
detailed_reviews_id_validation_df = (
    reviews_connection.execute("""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT id)
                AS unique_review_ids,
            SUM(
                CASE
                    WHEN id IS NULL THEN 1
                    ELSE 0
                END
            ) AS missing_review_ids
        FROM detailed_reviews_raw
    """).fetchdf()
)

print("Review ID Primary-Key Validation")
print("-" * 50)

display(detailed_reviews_id_validation_df)

Review ID Primary-Key Validation
--------------------------------------------------


,total_rows,unique_review_ids,missing_review_ids
0,545162,545162,0.0


In [168]:
detailed_reviews_duplicate_review_ids = (
    reviews_connection.execute("""
        SELECT COUNT(*)
        FROM (
            SELECT
                id
            FROM detailed_reviews_raw
            GROUP BY id
            HAVING COUNT(*) > 1
        )
    """).fetchone()[0]
)

print(
    f"Duplicate review IDs: "
    f"{detailed_reviews_duplicate_review_ids:,}"
)

Duplicate review IDs: 0


Validate (listing_id, date) combinations

In [169]:
detailed_reviews_listing_date_summary_df = (
    reviews_connection.execute("""
        SELECT
            COUNT(*) AS duplicate_pair_groups,
            COALESCE(
                SUM(row_count - 1),
                0
            ) AS extra_rows_from_repeated_pairs
        FROM (
            SELECT
                listing_id,
                date,
                COUNT(*) AS row_count
            FROM detailed_reviews_raw
            GROUP BY
                listing_id,
                date
            HAVING COUNT(*) > 1
        )
    """).fetchdf()
)

detailed_reviews_duplicate_pair_groups = int(
    detailed_reviews_listing_date_summary_df.loc[
        0,
        "duplicate_pair_groups"
    ]
)

detailed_reviews_extra_pair_rows = int(
    detailed_reviews_listing_date_summary_df.loc[
        0,
        "extra_rows_from_repeated_pairs"
    ]
)

print("Repeated (listing_id, date) Analysis")
print("-" * 55)

print(
    f"Repeated pair groups: "
    f"{detailed_reviews_duplicate_pair_groups:,}"
)

print(
    f"Extra rows from repeated pairs: "
    f"{detailed_reviews_extra_pair_rows:,}"
)

Repeated (listing_id, date) Analysis
-------------------------------------------------------
Repeated pair groups: 12,942
Extra rows from repeated pairs: 22,541


In [170]:
if detailed_reviews_duplicate_pair_groups > 0:

    detailed_reviews_repeated_pair_examples_df = (
        reviews_connection.execute("""
            SELECT
                listing_id,
                date,
                COUNT(*) AS review_count
            FROM detailed_reviews_raw
            GROUP BY
                listing_id,
                date
            HAVING COUNT(*) > 1
            ORDER BY review_count DESC
            LIMIT 20
        """).fetchdf()
    )

    display(
        detailed_reviews_repeated_pair_examples_df
    )

,listing_id,date,review_count
0,50383849,2023-07-02,13
1,50383849,2022-04-18,13
2,50383849,2022-08-05,13
3,45045046,2022-09-22,12
4,50383849,2023-01-06,12
5,50383849,2023-07-15,11
6,50383849,2022-03-16,11
7,50383849,2021-09-05,11
8,50383849,2023-08-12,11
9,50383849,2023-01-15,11


Prove repeated listing-date rows have distinct review IDs

In [171]:
repeated_pair_review_id_validation_df = (
    reviews_connection.execute("""
        WITH repeated_pairs AS (
            SELECT
                listing_id,
                date
            FROM detailed_reviews_raw
            GROUP BY
                listing_id,
                date
            HAVING COUNT(*) > 1
        )

        SELECT
            COUNT(*) AS repeated_pair_rows,
            COUNT(DISTINCT d.id)
                AS distinct_review_ids
        FROM detailed_reviews_raw AS d
        INNER JOIN repeated_pairs AS r
            ON d.listing_id = r.listing_id
           AND d.date = r.date
    """).fetchdf()
)

print(
    "Review-ID Validation Within Repeated "
    "(listing_id, date) Groups"
)
print("-" * 70)

display(repeated_pair_review_id_validation_df)

Review-ID Validation Within Repeated (listing_id, date) Groups
----------------------------------------------------------------------


,repeated_pair_rows,distinct_review_ids
0,35483,35483


Inspect detailed examples of repeated same-day reviews

In [172]:
detailed_reviews_same_day_examples_df = (
    reviews_connection.execute("""
        WITH repeated_pairs AS (
            SELECT
                listing_id,
                date
            FROM detailed_reviews_raw
            GROUP BY
                listing_id,
                date
            HAVING COUNT(*) > 1
            ORDER BY COUNT(*) DESC
            LIMIT 5
        )

        SELECT
            d.listing_id,
            d.date,
            d.id AS review_id,
            d.reviewer_id,
            d.reviewer_name,
            LEFT(d.comments, 120)
                AS comment_preview
        FROM detailed_reviews_raw AS d
        INNER JOIN repeated_pairs AS r
            ON d.listing_id = r.listing_id
           AND d.date = r.date
        ORDER BY
            d.listing_id,
            d.date,
            d.id
    """).fetchdf()
)

display(detailed_reviews_same_day_examples_df)

,listing_id,date,review_id,reviewer_id,reviewer_name,comment_preview
0,45045046,2022-09-22,721493305279091893,82092512,Jean,"Comfortable rooms, good location. I enjoyed my stay at The Tire Station"
1,45045046,2022-09-22,721508638503114066,353432889,Anton,"Great cute hotel, coffee shop downstairs super convenient and tasty for a quick breakfast"
2,45045046,2022-09-22,721512633534020412,59133184,Julia,Everything was perfect. We liked it a lot and would visit it again!
3,45045046,2022-09-22,721516168454179242,69012276,Christian,Personnel sympathique. Chambre impeccable. Excellent petit déjeuner. A recommander sans hésiter....
4,45045046,2022-09-22,721524353681298859,471808706,Mirjam,Leider war mein Zimmer nicht so sauber und gepflegt wie ich es erwartet hatte.<br/>Hiermit meine...
5,45045046,2022-09-22,721530482968071640,478718476,Rana,Good place to stay
6,45045046,2022-09-22,721543872102974959,470619363,Estefhany Laurent,"Súper recomendado tiene todo, el aeropuerto cerca, museos cerca, el barrio rojo está un poco lej..."
7,45045046,2022-09-22,721546347092216217,263121299,Neomi,we had a great time. nice coffee house in the lobby (expensive) and nice staff. the room was ple...
8,45045046,2022-09-22,721547361590999888,89264973,Nicol,"Welcming staff, comfy room, well located, would recommend."
9,45045046,2022-09-22,721555020458923572,326746663,Agi,"Great place to stay! Very clean rooms and great, polite stuff"


Listing ID validation

In [173]:
detailed_reviews_listing_id_validation_df = (
    reviews_connection.execute("""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT listing_id)
                AS unique_listing_ids,
            SUM(
                CASE
                    WHEN listing_id IS NULL THEN 1
                    ELSE 0
                END
            ) AS missing_listing_ids
        FROM detailed_reviews_raw
    """).fetchdf()
)

print("Detailed Reviews Listing ID Validation")
print("-" * 55)

display(detailed_reviews_listing_id_validation_df)

Detailed Reviews Listing ID Validation
-------------------------------------------------------


,total_rows,unique_listing_ids,missing_listing_ids
0,545162,9432,0.0


Extract compact detailed-review listing ID set

In [174]:
detailed_reviews_listing_ids = set(
    reviews_connection.execute("""
        SELECT DISTINCT listing_id
        FROM detailed_reviews_raw
        WHERE listing_id IS NOT NULL
    """).fetchnumpy()["listing_id"].tolist()
)

print("Detailed-review listing ID set created.")

print(
    f"Unique listing IDs: "
    f"{len(detailed_reviews_listing_ids):,}"
)

Detailed-review listing ID set created.
Unique listing IDs: 9,432


Relationship with summary listing

In [175]:
detailed_reviews_not_in_summary = (
    detailed_reviews_listing_ids
    - summary_listing_ids
)

summary_without_detailed_reviews = (
    summary_listing_ids
    - detailed_reviews_listing_ids
)

detailed_reviews_summary_common_ids = (
    detailed_reviews_listing_ids
    & summary_listing_ids
)

detailed_reviews_to_summary_coverage = (
    len(detailed_reviews_summary_common_ids)
    / len(detailed_reviews_listing_ids)
    * 100
    if detailed_reviews_listing_ids
    else 0
)

summary_with_detailed_reviews_coverage = (
    len(detailed_reviews_summary_common_ids)
    / len(summary_listing_ids)
    * 100
    if summary_listing_ids
    else 0
)

print(
    "Detailed Reviews-to-Summary Listings Relationship"
)
print("-" * 65)

print(
    f"Unique detailed-review listing IDs: "
    f"{len(detailed_reviews_listing_ids):,}"
)

print(
    f"Unique summary listing IDs: "
    f"{len(summary_listing_ids):,}"
)

print(
    f"Common listing IDs: "
    f"{len(detailed_reviews_summary_common_ids):,}"
)

print(
    f"Review listing IDs absent from summary listings: "
    f"{len(detailed_reviews_not_in_summary):,}"
)

print(
    f"Summary listings without detailed reviews: "
    f"{len(summary_without_detailed_reviews):,}"
)

print(
    f"Detailed-reviews-to-summary coverage: "
    f"{detailed_reviews_to_summary_coverage:.2f}%"
)

print(
    f"Summary listings with detailed reviews: "
    f"{summary_with_detailed_reviews_coverage:.2f}%"
)

Detailed Reviews-to-Summary Listings Relationship
-----------------------------------------------------------------
Unique detailed-review listing IDs: 9,432
Unique summary listing IDs: 10,465
Common listing IDs: 9,432
Review listing IDs absent from summary listings: 0
Summary listings without detailed reviews: 1,033
Detailed-reviews-to-summary coverage: 100.00%
Summary listings with detailed reviews: 90.13%


Relationship with detailed listings

In [176]:
detailed_reviews_not_in_detailed_listings = (
    detailed_reviews_listing_ids
    - detailed_listing_ids
)

detailed_listings_without_detailed_reviews = (
    detailed_listing_ids
    - detailed_reviews_listing_ids
)

detailed_reviews_detailed_common_ids = (
    detailed_reviews_listing_ids
    & detailed_listing_ids
)

detailed_reviews_to_detailed_coverage = (
    len(detailed_reviews_detailed_common_ids)
    / len(detailed_reviews_listing_ids)
    * 100
    if detailed_reviews_listing_ids
    else 0
)

detailed_listings_with_reviews_coverage = (
    len(detailed_reviews_detailed_common_ids)
    / len(detailed_listing_ids)
    * 100
    if detailed_listing_ids
    else 0
)

print(
    "Detailed Reviews-to-Detailed Listings Relationship"
)
print("-" * 70)

print(
    f"Unique detailed-review listing IDs: "
    f"{len(detailed_reviews_listing_ids):,}"
)

print(
    f"Unique detailed listing IDs: "
    f"{len(detailed_listing_ids):,}"
)

print(
    f"Common listing IDs: "
    f"{len(detailed_reviews_detailed_common_ids):,}"
)

print(
    "Review listing IDs absent from detailed listings: "
    f"{len(detailed_reviews_not_in_detailed_listings):,}"
)

print(
    "Detailed listings without detailed reviews: "
    f"{len(detailed_listings_without_detailed_reviews):,}"
)

print(
    f"Detailed-reviews-to-detailed coverage: "
    f"{detailed_reviews_to_detailed_coverage:.2f}%"
)

print(
    f"Detailed listings with reviews: "
    f"{detailed_listings_with_reviews_coverage:.2f}%"
)

Detailed Reviews-to-Detailed Listings Relationship
----------------------------------------------------------------------
Unique detailed-review listing IDs: 9,432
Unique detailed listing IDs: 10,369
Common listing IDs: 9,346
Review listing IDs absent from detailed listings: 86
Detailed listings without detailed reviews: 1,023
Detailed-reviews-to-detailed coverage: 99.09%
Detailed listings with reviews: 90.13%


Check whether the 96 summary-only listings have reviews

In [177]:
summary_only_with_detailed_reviews = (
    summary_only_listing_ids
    & detailed_reviews_listing_ids
)

summary_only_without_detailed_reviews = (
    summary_only_listing_ids
    - detailed_reviews_listing_ids
)

print("96 Summary-Only Listings Review Validation")
print("-" * 60)

print(
    f"Total summary-only listings: "
    f"{len(summary_only_listing_ids):,}"
)

print(
    f"Summary-only listings with detailed reviews: "
    f"{len(summary_only_with_detailed_reviews):,}"
)

print(
    f"Summary-only listings without detailed reviews: "
    f"{len(summary_only_without_detailed_reviews):,}"
)

96 Summary-Only Listings Review Validation
------------------------------------------------------------
Total summary-only listings: 96
Summary-only listings with detailed reviews: 86
Summary-only listings without detailed reviews: 10


Create DuckDB view for summary reviews.csv

In [178]:
summary_reviews_file_path = str(
    DATA_FILES["summary_reviews"]
).replace("\\", "/")

reviews_connection.execute(f"""
    CREATE OR REPLACE VIEW summary_reviews_raw AS
    SELECT *
    FROM read_csv_auto(
        '{summary_reviews_file_path}',
        header=true,
        sample_size=-1,
        all_varchar=false
    )
""")

print(
    "DuckDB summary reviews view created successfully."
)

DuckDB summary reviews view created successfully.


Compare summary and detailed review structure

In [179]:
summary_reviews_duckdb_summary_df = (
    reviews_connection.execute("""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT listing_id)
                AS unique_listing_ids,
            MIN(date) AS earliest_date,
            MAX(date) AS latest_date
        FROM summary_reviews_raw
    """).fetchdf()
)

detailed_reviews_duckdb_summary_df = (
    reviews_connection.execute("""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT listing_id)
                AS unique_listing_ids,
            MIN(date) AS earliest_date,
            MAX(date) AS latest_date
        FROM detailed_reviews_raw
    """).fetchdf()
)

print("Summary Reviews")
display(summary_reviews_duckdb_summary_df)

print("Detailed Reviews")
display(detailed_reviews_duckdb_summary_df)

Summary Reviews


,total_rows,unique_listing_ids,earliest_date,latest_date
0,545162,9432,2010-08-16,2026-06-28


Detailed Reviews


,total_rows,unique_listing_ids,earliest_date,latest_date
0,545162,9432,2010-08-16,2026-06-28


Exact multiset comparison between summary and detailed reviews

In [180]:
summary_vs_detailed_review_pair_mismatches_df = (
    reviews_connection.execute("""
        WITH summary_pairs AS (
            SELECT
                listing_id,
                date,
                COUNT(*) AS row_count
            FROM summary_reviews_raw
            GROUP BY
                listing_id,
                date
        ),

        detailed_pairs AS (
            SELECT
                listing_id,
                date,
                COUNT(*) AS row_count
            FROM detailed_reviews_raw
            GROUP BY
                listing_id,
                date
        )

        SELECT
            COALESCE(
                s.listing_id,
                d.listing_id
            ) AS listing_id,

            COALESCE(
                s.date,
                d.date
            ) AS date,

            COALESCE(
                s.row_count,
                0
            ) AS summary_row_count,

            COALESCE(
                d.row_count,
                0
            ) AS detailed_row_count

        FROM summary_pairs AS s

        FULL OUTER JOIN detailed_pairs AS d
            ON s.listing_id = d.listing_id
           AND s.date = d.date

        WHERE
            COALESCE(s.row_count, 0)
            !=
            COALESCE(d.row_count, 0)
    """).fetchdf()
)

print(
    "Summary vs Detailed Reviews Pair-Level Comparison"
)
print("-" * 65)

print(
    f"Mismatched (listing_id, date) groups: "
    f"{len(summary_vs_detailed_review_pair_mismatches_df):,}"
)

Summary vs Detailed Reviews Pair-Level Comparison
-----------------------------------------------------------------
Mismatched (listing_id, date) groups: 0


In [181]:
if (
    len(
        summary_vs_detailed_review_pair_mismatches_df
    ) == 0
):

    print(
        "The summary reviews file is an exact "
        "listing_id/date projection of the detailed "
        "reviews file, including repeated pairs."
    )

else:
    display(
        summary_vs_detailed_review_pair_mismatches_df.head(20)
    )

The summary reviews file is an exact listing_id/date projection of the detailed reviews file, including repeated pairs.


Review date validation

In [182]:
detailed_reviews_date_summary_df = (
    reviews_connection.execute("""
        SELECT
            MIN(date) AS earliest_date,
            MAX(date) AS latest_date,
            COUNT(DISTINCT date)
                AS unique_dates,
            COUNT(*) FILTER (
                WHERE date IS NULL
            ) AS missing_dates
        FROM detailed_reviews_raw
    """).fetchdf()
)

print("Detailed Reviews Date Validation")
print("-" * 50)

display(detailed_reviews_date_summary_df)

Detailed Reviews Date Validation
--------------------------------------------------


,earliest_date,latest_date,unique_dates,missing_dates
0,2010-08-16,2026-06-28,5327,0


Reviews per listing summary

In [183]:
detailed_reviews_per_listing_summary_df = (
    reviews_connection.execute("""
        WITH review_counts AS (
            SELECT
                listing_id,
                COUNT(*) AS review_count
            FROM detailed_reviews_raw
            GROUP BY listing_id
        )

        SELECT
            COUNT(*) AS listings_with_reviews,
            MIN(review_count)
                AS minimum_reviews,
            MAX(review_count)
                AS maximum_reviews,
            MEDIAN(review_count)
                AS median_reviews,
            AVG(review_count)
                AS average_reviews
        FROM review_counts
    """).fetchdf()
)

print("Detailed Reviews per Listing Summary")
print("-" * 55)

display(detailed_reviews_per_listing_summary_df)

Detailed Reviews per Listing Summary
-------------------------------------------------------


,listings_with_reviews,minimum_reviews,maximum_reviews,median_reviews,average_reviews
0,9432,1,5603,13.0,57.799194


Reviews per listing summary

In [184]:
detailed_reviews_per_listing_summary_df = (
    reviews_connection.execute("""
        WITH review_counts AS (
            SELECT
                listing_id,
                COUNT(*) AS review_count
            FROM detailed_reviews_raw
            GROUP BY listing_id
        )

        SELECT
            COUNT(*) AS listings_with_reviews,
            MIN(review_count)
                AS minimum_reviews,
            MAX(review_count)
                AS maximum_reviews,
            MEDIAN(review_count)
                AS median_reviews,
            AVG(review_count)
                AS average_reviews
        FROM review_counts
    """).fetchdf()
)

print("Detailed Reviews per Listing Summary")
print("-" * 55)

display(detailed_reviews_per_listing_summary_df)

Detailed Reviews per Listing Summary
-------------------------------------------------------


,listings_with_reviews,minimum_reviews,maximum_reviews,median_reviews,average_reviews
0,9432,1,5603,13.0,57.799194


Top listings by detailed review count

In [185]:
detailed_reviews_top_listings_df = (
    reviews_connection.execute("""
        SELECT
            listing_id,
            COUNT(*) AS review_count
        FROM detailed_reviews_raw
        GROUP BY listing_id
        ORDER BY review_count DESC
        LIMIT 10
    """).fetchdf()
)

display(detailed_reviews_top_listings_df)

,listing_id,review_count
0,50383849,5603
1,32485135,4082
2,45045046,3257
3,50452179,1993
4,35927687,1773
5,30762441,1706
6,905677609043113133,1611
7,802052,1551
8,746060021045144380,1398
9,46591985,1304


Reviewer ID quality

In [186]:
detailed_reviews_reviewer_summary_df = (
    reviews_connection.execute("""
        WITH reviewer_counts AS (
            SELECT
                reviewer_id,
                COUNT(*) AS review_count
            FROM detailed_reviews_raw
            GROUP BY reviewer_id
        )

        SELECT
            COUNT(*) AS unique_reviewers,
            MIN(review_count)
                AS minimum_reviews_per_reviewer,
            MAX(review_count)
                AS maximum_reviews_per_reviewer,
            MEDIAN(review_count)
                AS median_reviews_per_reviewer,
            AVG(review_count)
                AS average_reviews_per_reviewer
        FROM reviewer_counts
    """).fetchdf()
)

print("Reviewer ID Summary")
print("-" * 45)

display(detailed_reviews_reviewer_summary_df)

Reviewer ID Summary
---------------------------------------------


,unique_reviewers,minimum_reviews_per_reviewer,maximum_reviews_per_reviewer,median_reviews_per_reviewer,average_reviews_per_reviewer
0,523777,1,13,1.0,1.040828


Reviewers with multiple reviews

In [187]:
multiple_reviewers_summary_df = (
    reviews_connection.execute("""
        SELECT
            COUNT(*) AS reviewers_with_multiple_reviews
        FROM (
            SELECT
                reviewer_id
            FROM detailed_reviews_raw
            GROUP BY reviewer_id
            HAVING COUNT(*) > 1
        )
    """).fetchdf()
)

display(multiple_reviewers_summary_df)

,reviewers_with_multiple_reviews
0,18043


Check repeated review comments

In [188]:
repeated_comments_summary_df = (
    reviews_connection.execute("""
        SELECT
            COUNT(*) AS repeated_comment_groups,
            COALESCE(
                SUM(comment_count - 1),
                0
            ) AS extra_rows_from_repeated_comments
        FROM (
            SELECT
                comments,
                COUNT(*) AS comment_count
            FROM detailed_reviews_raw
            GROUP BY comments
            HAVING COUNT(*) > 1
        )
    """).fetchdf()
)

print("Repeated Review Comment Analysis")
print("-" * 50)

display(repeated_comments_summary_df)

Repeated Review Comment Analysis
--------------------------------------------------


,repeated_comment_groups,extra_rows_from_repeated_comments
0,2928,16419.0


Inspect repeated-comment examples safely

In [189]:
repeated_comment_examples_df = (
    reviews_connection.execute("""
        SELECT
            LEFT(comments, 120)
                AS comment_preview,
            COUNT(*) AS occurrence_count,
            COUNT(DISTINCT listing_id)
                AS distinct_listings,
            COUNT(DISTINCT reviewer_id)
                AS distinct_reviewers
        FROM detailed_reviews_raw
        GROUP BY comments
        HAVING COUNT(*) > 1
        ORDER BY occurrence_count DESC
        LIMIT 20
    """).fetchdf()
)

display(repeated_comment_examples_df)

,comment_preview,occurrence_count,distinct_listings,distinct_reviewers
0,.,1033,629,1024
1,Good,297,201,297
2,Top,246,197,245
3,Great,240,175,238
4,Great place,234,184,234
5,Great location,217,178,216
6,Perfect,210,178,209
7,Great stay,203,173,202
8,Great stay!,202,173,199
9,Great place!,198,167,198


Validate the 1,033 no-review pattern

In [190]:
print("1,033 No-Review Pattern Validation")
print("-" * 55)

print(
    f"Summary listings without detailed reviews: "
    f"{len(summary_without_detailed_reviews):,}"
)

1,033 No-Review Pattern Validation
-------------------------------------------------------
Summary listings without detailed reviews: 1,033


Final detailed reviews dataset summary

In [191]:
detailed_reviews_dataset_summary = {
    "dataset_name": "detailed_reviews",
    "file_name": DATA_FILES[
        "detailed_reviews"
    ].name,

    "row_count": (
        detailed_reviews_row_count
    ),

    "column_count": (
        detailed_reviews_column_count
    ),

    "unique_listing_ids": len(
        detailed_reviews_listing_ids
    ),

    "unique_review_ids": int(
        detailed_reviews_id_validation_df.loc[
            0,
            "unique_review_ids"
        ]
    ),

    "duplicate_extra_rows": (
        detailed_reviews_duplicate_extra_rows
    ),

    "candidate_primary_key": (
        "id"
        if (
            detailed_reviews_duplicate_review_ids == 0
            and
            detailed_reviews_id_validation_df.loc[
                0,
                "missing_review_ids"
            ] == 0
        )
        else None
    ),

    "repeated_listing_date_extra_rows": (
        detailed_reviews_extra_pair_rows
    ),

    "detailed_reviews_to_summary_listing_coverage": round(
        detailed_reviews_to_summary_coverage,
        2
    ),

    "detailed_reviews_to_detailed_listing_coverage": round(
        detailed_reviews_to_detailed_coverage,
        2
    ),

    "summary_listings_without_detailed_reviews": (
        len(summary_without_detailed_reviews)
    ),

    "missing_values_total": int(
        detailed_reviews_missing_summary_df[
            "missing_count"
        ].sum()
    ),

    "empty_comments": (
        detailed_reviews_empty_comments_count
    ),
}

detailed_reviews_dataset_summary

{'dataset_name': 'detailed_reviews',
 'file_name': 'reviews.csv.gz',
 'row_count': 545162,
 'column_count': 6,
 'unique_listing_ids': 9432,
 'unique_review_ids': 545162,
 'duplicate_extra_rows': 0,
 'candidate_primary_key': 'id',
 'repeated_listing_date_extra_rows': 22541,
 'detailed_reviews_to_summary_listing_coverage': 100.0,
 'detailed_reviews_to_detailed_listing_coverage': 99.09,
 'summary_listings_without_detailed_reviews': 1033,
 'missing_values_total': 1,
 'empty_comments': 0}

Save compact profiling outputs

In [192]:
detailed_reviews_profile_path = (
    OUTPUT_DIR
    / "reviews_csv_gz_column_profile.csv"
)

detailed_reviews_profile_df.to_csv(
    detailed_reviews_profile_path,
    index=False
)

detailed_reviews_pair_profile_path = (
    OUTPUT_DIR
    / "reviews_csv_gz_repeated_listing_date_examples.csv"
)

if detailed_reviews_duplicate_pair_groups > 0:
    detailed_reviews_repeated_pair_examples_df.to_csv(
        detailed_reviews_pair_profile_path,
        index=False
    )

detailed_reviews_top_listings_path = (
    OUTPUT_DIR
    / "reviews_csv_gz_top_listings_by_review_count.csv"
)

detailed_reviews_top_listings_df.to_csv(
    detailed_reviews_top_listings_path,
    index=False
)

print(
    "Detailed reviews profiling outputs "
    "saved successfully."
)

print(
    f"\nColumn profile:\n"
    f"{detailed_reviews_profile_path}"
)

if detailed_reviews_duplicate_pair_groups > 0:
    print(
        f"\nRepeated listing-date examples:\n"
        f"{detailed_reviews_pair_profile_path}"
    )

print(
    f"\nTop listings by review count:\n"
    f"{detailed_reviews_top_listings_path}"
)

Detailed reviews profiling outputs saved successfully.

Column profile:
c:\Users\saths\Desktop\MyProjects\Expernetic Data Challenge\airbnb-data-challenge\outputs\data_quality\reviews_csv_gz_column_profile.csv

Repeated listing-date examples:
c:\Users\saths\Desktop\MyProjects\Expernetic Data Challenge\airbnb-data-challenge\outputs\data_quality\reviews_csv_gz_repeated_listing_date_examples.csv

Top listings by review count:
c:\Users\saths\Desktop\MyProjects\Expernetic Data Challenge\airbnb-data-challenge\outputs\data_quality\reviews_csv_gz_top_listings_by_review_count.csv


### Findings and Interpretation

The `reviews.csv.gz` dataset contains detailed review-level records for Amsterdam Airbnb listings.

The grain of the dataset is:

**One row represents one individual review.**

The dataset contains:

- **545,162 rows**
- **6 columns**
- **9,432 unique listing IDs**
- **545,162 unique review IDs**
- **523,777 unique reviewer IDs**
- **5,327 unique review dates**
- **0 complete duplicate rows**
- **1 missing value across the entire dataset**
- **0 empty but non-null comments**

The six fields are:

- `listing_id`
- `id`
- `date`
- `reviewer_id`
- `reviewer_name`
- `comments`

Overall, the detailed reviews dataset is structurally strong and provides the review-level identifiers and reviewer information that were absent from the summary `reviews.csv` file.

---

#### Dataset Structure

Observed DuckDB data types are:

| Column | Data Type |
|---|---|
| `listing_id` | BIGINT |
| `id` | BIGINT |
| `date` | DATE |
| `reviewer_id` | BIGINT |
| `reviewer_name` | VARCHAR |
| `comments` | VARCHAR |

These data types are appropriate for the observed data.

Important positive observations include:

- `listing_id` is stored as `BIGINT`, preserving very large listing identifiers safely.
- `id` is stored as `BIGINT` and uniquely identifies reviews.
- `date` is correctly recognized as a `DATE`.
- `reviewer_id` is stored as `BIGINT`.
- review names and comments are stored as text.

---

#### Dataset Size

Observed results:

- Total rows: **545,162**
- Total columns: **6**
- Unique listing IDs: **9,432**
- Unique review IDs: **545,162**
- Unique reviewer IDs: **523,777**

This means the dataset contains more than half a million individual review records across 9,432 Airbnb listings.

---

#### Missing-Value Assessment

The dataset contains only one missing value.

| Column | Missing Count |
|---|---:|
| `listing_id` | 0 |
| `id` | 0 |
| `date` | 0 |
| `reviewer_id` | 0 |
| `reviewer_name` | 1 |
| `comments` | 0 |

Therefore:

- every review has a listing identifier,
- every review has a unique review identifier,
- every review has a date,
- every review has a reviewer identifier,
- every review has a non-null comment,
- only one review is missing the reviewer name.

The single missing reviewer name represents approximately:

**0.00018% of all rows**

and is therefore negligible from a dataset-completeness perspective.

---

#### Review ID as Candidate Primary Key

The `id` field was explicitly validated.

Observed results:

- Total rows: **545,162**
- Unique review IDs: **545,162**
- Missing review IDs: **0**
- Duplicate review IDs: **0**

Therefore:

**Candidate primary key: `id`**

The `id` field uniquely and completely identifies every individual review.

This is a stronger identifier than the `(listing_id, date)` combination because multiple genuine reviews can occur for the same listing on the same date.

---

#### Complete Duplicate-Row Analysis

No complete duplicate rows were found.

Observed results:

- Total rows: **545,162**
- Duplicate groups: **0**
- Extra duplicate rows: **0**
- Duplicate percentage: **0.0000%**

Therefore, the detailed review dataset contains no exact duplicate records across all six fields.

This is a strong data-quality result.

---

#### Repeated `(listing_id, date)` Combinations

The combination:

`(listing_id, date)`

is not unique.

Observed results:

- Repeated `(listing_id, date)` groups: **12,942**
- Extra rows from repeated pairs: **22,541**

This exactly matches the earlier observation in `reviews.csv`, where 22,541 rows appeared as repeated listing-date combinations.

However, the detailed dataset reveals that these are not necessarily duplicate reviews.

---

#### Explanation of the 22,541 Repeated Summary Review Rows

The detailed review dataset provides a unique review identifier:

`id`

For all rows belonging to repeated `(listing_id, date)` groups:

- Total rows in repeated groups: **35,483**
- Distinct review IDs: **35,483**

Therefore, every review in these repeated listing-date groups has its own unique review ID.

This strongly confirms that the repeated `(listing_id, date)` combinations are legitimate separate review records rather than duplicate review IDs.

The relationship is:

```text
Same listing
    +
Same review date
    │
    ▼
Multiple distinct review IDs
````

For example, listing:

`50383849`

received:

* 13 reviews on 2022-04-18,
* 13 reviews on 2022-08-05,
* 13 reviews on 2023-07-02.

These records have distinct:

* review IDs,
* reviewer IDs,
* reviewer names,
* comments.

Therefore, the earlier 22,541 repeated rows in `reviews.csv` should not be removed as duplicates.

They represent valid multiple-review events occurring for the same listing on the same date.

---

#### Highest Same-Day Review Counts

Examples of high same-day review counts include:

| Listing ID | Date       | Reviews on Same Date |
| ---------- | ---------- | -------------------: |
| `50383849` | 2022-04-18 |                   13 |
| `50383849` | 2022-08-05 |                   13 |
| `50383849` | 2023-07-02 |                   13 |
| `50383849` | 2023-01-06 |                   12 |
| `45045046` | 2022-09-22 |                   12 |

These examples reinforce that multiple guests can leave reviews for the same listing on the same date.

Therefore:

**`(listing_id, date)` is not a candidate key for the detailed review dataset.**

The correct review-level identifier is:

`id`

---

#### Relationship with Summary `reviews.csv`

The summary review file contains only:

* `listing_id`
* `date`

The detailed file contains:

* `listing_id`
* `id`
* `date`
* `reviewer_id`
* `reviewer_name`
* `comments`

Both files contain:

* **545,162 rows**
* **9,432 unique listing IDs**
* **5,327 unique review dates**

An exact pair-level multiset comparison was performed.

Observed result:

* Mismatched `(listing_id, date)` groups: **0**

Therefore:

**The summary `reviews.csv` file is an exact projection of `reviews.csv.gz` onto the `listing_id` and `date` columns, including repeated listing-date combinations.**

This can be represented as:

```text
reviews.csv.gz
    listing_id
    id
    date
    reviewer_id
    reviewer_name
    comments
          │
          │ select listing_id, date
          ▼
reviews.csv
    listing_id
    date
```

This is a very strong cross-dataset consistency result.

---

#### Relationship with Summary Listings

Detailed review listing IDs were compared against:

`listings.csv.id`

Observed results:

* Unique detailed-review listing IDs: **9,432**
* Unique summary listing IDs: **10,465**
* Common listing IDs: **9,432**
* Review listing IDs absent from summary listings: **0**
* Summary listings without detailed reviews: **1,033**

Coverage:

* Detailed-reviews-to-summary coverage: **100.00%**
* Summary listings with at least one detailed review: **90.13%**

Therefore, every listing represented in the detailed review dataset exists in the summary listings dataset.

The relationship is:

```text
listings.csv
    id
     │
     │ one-to-many
     ▼
reviews.csv.gz
    listing_id
```

One listing may have:

* zero reviews,
* one review,
* many reviews.

---

#### The 1,033 No-Review Pattern

The summary listings dataset contains:

* **10,465 listings**

The detailed reviews dataset contains reviews for:

* **9,432 listings**

The difference is:

**1,033 listings**

Therefore:

**1,033 summary listings have no records in the detailed reviews dataset.**

Earlier profiling also found:

* 1,033 missing `last_review` values in `listings.csv`,
* 1,033 missing `reviews_per_month` values in `listings.csv`,
* 1,033 listings with no records in `reviews.csv`.

This is a very strong structural missingness pattern.

However, the exact listing-ID sets for missing `last_review` and missing `reviews_per_month` have not yet been explicitly compared against the 1,033 no-review listing IDs.

Therefore, the current conclusion should be:

**The counts align perfectly and strongly suggest structural missingness associated with listings that have no review history, but exact row-level ID validation is still required before declaring a complete set-level match.**

---

#### Relationship with Detailed Listings

Detailed review listing IDs were also compared against:

`listings.csv.gz.id`

Observed results:

* Unique detailed-review listing IDs: **9,432**
* Unique detailed listing IDs: **10,369**
* Common listing IDs: **9,346**
* Review listing IDs absent from detailed listings: **86**
* Detailed listings without detailed reviews: **1,023**

Coverage:

* Detailed-reviews-to-detailed-listings coverage: **99.09%**
* Detailed listings with at least one review: **90.13%**

Therefore, 86 listing IDs represented in the detailed review dataset are absent from `listings.csv.gz`.

These 86 IDs belong to the previously identified group of 96 summary-only listings.

---

#### Review Status of the 96 Summary-Only Listings

The 96 listings that:

* exist in `listings.csv`,
* exist in `calendar.csv.gz`,
* do not exist in `listings.csv.gz`

were compared with the detailed review population.

Observed results:

* Total summary-only listings: **96**
* Summary-only listings with detailed reviews: **86**
* Summary-only listings without detailed reviews: **10**

Therefore:

**86 of the 96 summary-only listings have review history, while 10 have no detailed review records.**

This explains why the detailed reviews dataset contains:

* 86 listing IDs absent from the detailed listings file.

The relationship can be represented as:

```text
96 Summary-Only Listings
        │
        ├── 86 have detailed reviews
        │
        └── 10 have no detailed reviews
```

This is an important cross-dataset finding.

---

#### Review Date Coverage

Observed review date range:

* Earliest review date: **2010-08-16**
* Latest review date: **2026-06-28**
* Unique review dates: **5,327**
* Missing dates: **0**

This exactly aligns with the summary review dataset.

The review history therefore spans almost 16 years.

However, older reviews describe historical guest experiences, while current listing attributes may reflect a later listing snapshot.

Therefore, historical reviews should not automatically be interpreted as if all current listing characteristics were unchanged throughout the full review period.

---

#### Reviews per Listing

Observed review-count statistics are:

* Listings with at least one review: **9,432**
* Minimum reviews per reviewed listing: **1**
* Maximum reviews per listing: **5,603**
* Median reviews per reviewed listing: **13**
* Average reviews per reviewed listing: approximately **57.80**

The large difference between:

* median: **13**
* average: **57.80**
* maximum: **5,603**

shows that the review-count distribution is strongly right-skewed.

A relatively small number of listings have extremely large review volumes.

Therefore, the median is more representative of a typical reviewed listing than the mean.

---

#### Listings with the Highest Review Counts

The top reviewed listings are:

| Rank |           Listing ID | Review Count |
| ---: | -------------------: | -----------: |
|    1 |           `50383849` |        5,603 |
|    2 |           `32485135` |        4,082 |
|    3 |           `45045046` |        3,257 |
|    4 |           `50452179` |        1,993 |
|    5 |           `35927687` |        1,773 |
|    6 |           `30762441` |        1,706 |
|    7 | `905677609043113133` |        1,611 |
|    8 |             `802052` |        1,551 |
|    9 | `746060021045144380` |        1,398 |
|   10 |           `46591985` |        1,304 |

These values should not automatically be treated as errors.

They may represent:

* highly active listings,
* hotel-like properties,
* professionally managed accommodation,
* listings with long operating histories.

The extreme counts should be investigated and interpreted contextually.

---

#### Reviewer Identifier Analysis

The dataset contains:

* **523,777 unique reviewer IDs**
* across **545,162 reviews**

Reviewer-level statistics are:

* Minimum reviews per reviewer: **1**
* Maximum reviews per reviewer: **13**
* Median reviews per reviewer: **1**
* Average reviews per reviewer: approximately **1.04**

Additionally:

* **18,043 reviewers** appear more than once.

Therefore, most reviewers appear only once in this Amsterdam dataset, but some reviewers have reviewed multiple listings or generated multiple review records.

The `reviewer_id` field should not be treated as unique at the review-row level.

---

#### Reviewer Name Completeness

The `reviewer_name` field contains:

* **1 missing value**
* **72,308 unique non-null names**

Reviewer names are not suitable as unique identifiers because:

* many different reviewers may share the same name,
* names may vary in capitalization or spelling,
* one name can correspond to many reviewer IDs.

Therefore:

`reviewer_id`

is the preferred reviewer-level identifier.

---

#### Comment Completeness

The `comments` field is highly complete.

Observed results:

* Missing comments: **0**
* Empty but non-null comments: **0**
* Minimum comment length: **1 character**
* Maximum comment length: **6,164 characters**
* Median comment length: **189 characters**
* Average comment length: approximately **247.76 characters**

This makes the dataset potentially suitable for optional text analysis.

However, text processing is not necessary for the core assignment unless time permits.

---

#### Repeated Review Comments

The dataset contains:

* **528,743 unique comment values**
* across **545,162 total reviews**

Repeated-comment analysis found:

* Repeated comment groups: **2,928**
* Extra rows from repeated comments: **16,419**

Examples of frequently repeated comments include:

| Comment          | Occurrences |
| ---------------- | ----------: |
| `.`              |       1,033 |
| `Good`           |         297 |
| `Top`            |         246 |
| `Great`          |         240 |
| `Great place`    |         234 |
| `Great location` |         217 |
| `Perfect`        |         210 |
| `Great stay`     |         203 |
| `Great stay!`    |         202 |

Repeated text should not automatically be interpreted as duplicate review records.

Possible reasons include:

* generic short reviews,
* common phrases,
* repeated templates,
* automated text,
* identical guest wording,
* placeholder-like content.

Because review IDs remain unique and complete duplicate rows are absent, repeated comment text alone is not sufficient evidence of duplicate reviews.

---

#### Notable `"."` Comment Pattern

The single-character comment:

`.`

appears:

* **1,033 times**
* across **629 distinct listings**
* from **1,024 distinct reviewers**

This is notable because the occurrence count of 1,033 matches the number of summary listings without reviews.

However, these are different concepts and no relationship has been established between them.

Therefore, this matching count should be treated as coincidental unless explicit ID-level evidence demonstrates otherwise.

The `"."` comments are valid non-empty text values but may have very limited analytical content.

---

#### Comment Text Is Multilingual

The sample review comments contain multiple languages, including examples in:

* English,
* French,
* German,
* Spanish,
* Dutch,
* Portuguese,
* Italian,
* other languages.

Therefore, any future sentiment analysis or NLP work would need to account for multilingual text.

A simple English-only text model may produce biased or incomplete results.

Given the short project timeline, extensive multilingual NLP should remain optional.

---

#### Summary Reviews and Detailed Reviews Consistency

The two review files are highly consistent.

Observed validation shows:

* Same row count: **545,162**
* Same unique listing count: **9,432**
* Same date range
* Same number of unique dates: **5,327**
* Exact `(listing_id, date)` multiplicity match
* Zero pair-level mismatches

Therefore:

**`reviews.csv` is a compact summary projection of `reviews.csv.gz`, while `reviews.csv.gz` adds review ID, reviewer ID, reviewer name, and comment text.**

This means the detailed reviews file should be preferred when review-level uniqueness or reviewer/comment analysis is required.

---

#### Main Data Quality Strengths

The strongest characteristics of `reviews.csv.gz` are:

1. **545,162 unique review IDs across 545,162 rows.**
2. **0 missing review IDs.**
3. **0 duplicate review IDs.**
4. **0 complete duplicate rows.**
5. **0 missing listing IDs.**
6. **0 missing dates.**
7. **0 missing reviewer IDs.**
8. **0 missing comments.**
9. **Only 1 missing reviewer name.**
10. **0 empty but non-null comments.**
11. **100.00% review-listing ID coverage against summary listings.**
12. **Exact pair-level projection match with `reviews.csv`.**
13. **The 22,541 repeated summary review rows are explained by distinct review IDs.**
14. **All available dates are valid DuckDB `DATE` values.**

---

#### Main Data Quality Concerns

The most important concerns are:

1. Historical reviews span nearly 16 years and may not align perfectly with current listing attributes.
2. 86 reviewed listing IDs are absent from `listings.csv.gz`.
3. Review counts are highly right-skewed.
4. Some listings have extremely high review counts.
5. Repeated comments occur frequently, although they are not automatically duplicates.
6. Some comments contain minimal information such as `"."`, `"-"`, or very short generic phrases.
7. Reviewer names are not unique identifiers.
8. Multilingual comments make NLP more complex.
9. The exact ID-level relationship between the 1,033 no-review listings and missing `last_review` / `reviews_per_month` values has not yet been explicitly validated.

---

#### Overall Interpretation

The `reviews.csv.gz` dataset is the authoritative review-level source in the Amsterdam Airbnb dataset collection.

Its preferred candidate primary key is:

`id`

The central relationship is:

```text
Listing
   │
   │ one
   ▼
Detailed Reviews
   │
   │ many
   ▼
Individual Review
```

The dataset also resolves an important question from the earlier `reviews.csv` familiarization:

**The 22,541 repeated `(listing_id, date)` rows are not duplicate review IDs. They represent legitimate distinct review records sharing the same listing and review date.**

The strongest cross-dataset findings are:

* `reviews.csv` is an exact projection of `reviews.csv.gz` onto `listing_id` and `date`.
* All 9,432 reviewed listing IDs exist in the summary listings dataset.
* 1,033 summary listings have no review records.
* 86 of the 96 summary-only listings have review history.
* 10 of the 96 summary-only listings have no detailed reviews.

**Overall assessment: Excellent-quality review-level data that is highly suitable for review aggregation, listing enrichment, relationship validation, reviewer-level analysis, and optional text analysis, with the main cautions being temporal mismatch with current listing attributes, highly skewed review counts, repeated generic comment text, and the presence of 86 reviewed listing IDs absent from the detailed listings file.**

### Candidate Keys and Relationships

The `reviews.csv.gz` dataset contains detailed review-level records for Amsterdam Airbnb listings.

The grain of the dataset is:

**One row represents one individual review.**

The dataset contains:

- **545,162 rows**
- **545,162 unique review IDs**
- **9,432 unique listing IDs**
- **523,777 unique reviewer IDs**
- **0 missing review IDs**
- **0 duplicate review IDs**
- **0 complete duplicate rows**

These results provide strong evidence for a clear review-level primary key and several important cross-dataset relationships.

---

#### Candidate Primary Key

The `id` field was explicitly validated as the strongest candidate primary key.

Observed results:

- Total rows: **545,162**
- Unique `id` values: **545,162**
- Missing `id` values: **0**
- Duplicate `id` values: **0**

Therefore:

**Candidate primary key: `id`**

The `id` field uniquely identifies every review record.

The relationship is:

```text
Review ID
   │
   ▼
One unique review record
````

Because the source is a CSV file rather than a relational database table with formally enforced constraints, `id` should be described as a candidate primary key based on observed uniqueness and completeness.

---

#### Why `listing_id` Is Not a Primary Key

The `listing_id` field contains:

* **9,432 unique values**
* across **545,162 review rows**

Therefore, one listing may have many reviews.

The relationship is:

```text
Listing
   │
   │ one
   ▼
Reviews
   │
   │ many
```

This can be represented as:

```text
Listing 1 ───────< Many Reviews
```

Therefore:

`listing_id`

acts as a foreign-key-like relationship field rather than a review-level primary key.

---

#### Why `reviewer_id` Is Not a Primary Key

The `reviewer_id` field contains:

* **523,777 unique reviewer IDs**
* across **545,162 reviews**

Additionally:

* **18,043 reviewers** appear more than once.

Therefore, a reviewer may be associated with multiple review records.

The relationship is:

```text
Reviewer
    │
    │ one
    ▼
Reviews
    │
    │ many
```

The observed reviewer-level statistics are:

* Minimum reviews per reviewer: **1**
* Maximum reviews per reviewer: **13**
* Median reviews per reviewer: **1**
* Average reviews per reviewer: approximately **1.04**

Therefore:

`reviewer_id`

is a reviewer-level identifier, not a review-row primary key.

---

#### Why `reviewer_name` Is Not a Key

The `reviewer_name` field contains:

* **72,308 unique non-null values**
* **1 missing value**

Reviewer names cannot reliably identify individual reviewers because:

* multiple people may share the same name,
* one reviewer name may appear with different reviewer IDs,
* spelling and capitalization may vary,
* names may change over time.

Therefore:

`reviewer_name`

should be treated as a descriptive field only.

The preferred reviewer identifier is:

`reviewer_id`

---

#### Why `(listing_id, date)` Is Not a Candidate Key

The combination:

`(listing_id, date)`

was explicitly validated and found to be non-unique.

Observed results:

* Repeated `(listing_id, date)` groups: **12,942**
* Extra rows from repeated pairs: **22,541**

Therefore:

**`(listing_id, date)` is not a candidate key.**

Multiple distinct reviews can occur for the same listing on the same date.

For example:

| Listing ID | Date       | Reviews on Same Date |
| ---------: | ---------- | -------------------: |
| `50383849` | 2022-04-18 |                   13 |
| `50383849` | 2022-08-05 |                   13 |
| `50383849` | 2023-07-02 |                   13 |
| `50383849` | 2023-01-06 |                   12 |
| `45045046` | 2022-09-22 |                   12 |

These repeated listing-date groups contain genuinely different review records.

---

#### Review-ID Validation Within Repeated Listing-Date Groups

The repeated `(listing_id, date)` groups were further validated using the unique review ID.

Observed results:

* Total rows belonging to repeated listing-date groups: **35,483**
* Distinct review IDs in those groups: **35,483**

Therefore, every review record within the repeated listing-date groups has its own unique review ID.

This proves that the repeated pairs are not duplicate review identifiers.

The structure is:

```text
Same listing
    +
Same review date
    │
    ▼
Multiple unique review IDs
```

Therefore, the correct review-level key remains:

`id`

---

#### Relationship with Summary Reviews

The summary `reviews.csv` dataset contains only:

* `listing_id`
* `date`

The detailed `reviews.csv.gz` dataset contains:

* `listing_id`
* `id`
* `date`
* `reviewer_id`
* `reviewer_name`
* `comments`

An exact multiset comparison was performed between both datasets using:

`(listing_id, date)`

and preserving the number of occurrences of each pair.

Observed result:

* Mismatched `(listing_id, date)` groups: **0**

Therefore:

**The summary `reviews.csv` file is an exact projection of the detailed `reviews.csv.gz` file onto the `listing_id` and `date` columns, including repeated listing-date combinations.**

The relationship is:

```text
reviews.csv.gz
    listing_id
    id
    date
    reviewer_id
    reviewer_name
    comments
          │
          │ project selected columns
          ▼
reviews.csv
    listing_id
    date
```

This is a complete and validated relationship.

---

#### Relationship with Summary Listings

The detailed review listing IDs were explicitly compared against:

`listings.csv.id`

The relationship is:

```text
listings.csv
    id
     │
     │ one-to-many
     ▼
reviews.csv.gz
    listing_id
```

Validation results:

* Unique detailed-review listing IDs: **9,432**
* Unique summary listing IDs: **10,465**
* Common listing IDs: **9,432**
* Review listing IDs absent from summary listings: **0**
* Summary listings without detailed reviews: **1,033**

Coverage:

* Detailed-reviews-to-summary-listings coverage: **100.00%**
* Summary listings with at least one detailed review: **90.13%**

Therefore:

* every reviewed listing exists in the summary listings dataset,
* 1,033 summary listings have no detailed review records.

The cardinality is:

```text
Summary Listing 1 ───────< Zero or Many Reviews
```

---

#### Relationship with Detailed Listings

The detailed review listing IDs were also compared against:

`listings.csv.gz.id`

The relationship is:

```text
listings.csv.gz
    id
     │
     │ one-to-many
     ▼
reviews.csv.gz
    listing_id
```

Validation results:

* Unique detailed-review listing IDs: **9,432**
* Unique detailed listing IDs: **10,369**
* Common listing IDs: **9,346**
* Review listing IDs absent from detailed listings: **86**
* Detailed listings without detailed reviews: **1,023**

Coverage:

* Detailed-reviews-to-detailed-listings coverage: **99.09%**
* Detailed listings with at least one review: **90.13%**

Therefore:

* most detailed-review listing IDs are represented in `listings.csv.gz`,
* 86 reviewed listing IDs are absent from the detailed listings file.

These 86 records are not orphaned from the overall listing population because they exist in `listings.csv`.

---

#### Relationship with the 96 Summary-Only Listings

Earlier validation identified:

* **96 listing IDs** present in `listings.csv`,
* also present in `calendar.csv.gz`,
* but absent from `listings.csv.gz`.

These 96 listing IDs were compared with the detailed review population.

Observed results:

* Total summary-only listings: **96**
* Summary-only listings with detailed reviews: **86**
* Summary-only listings without detailed reviews: **10**

Therefore:

**The 86 detailed-review listing IDs absent from `listings.csv.gz` are explained by reviewed listings belonging to the 96-record summary-only population.**

The relationship is:

```text
96 Summary-Only Listings
        │
        ├── 86 have detailed reviews
        │
        └── 10 have no detailed reviews
```

This is an important cross-dataset consistency result.

---

#### Relationship with Calendar Data

There is no direct review-level relationship between:

`reviews.csv.gz`

and:

`calendar.csv.gz`

because both datasets are child datasets of the listing entity.

The indirect relationship is through:

`listing_id`

The structure is:

```text
                    Listing
                       │
             ┌─────────┴─────────┐
             ▼                   ▼
          Reviews             Calendar
          listing_id          listing_id
```

The grains differ:

```text
Reviews:
One row per individual review
```

```text
Calendar:
One row per listing-date combination
```

Therefore, raw review rows and raw calendar rows should not be directly joined without aggregation because this could create a large many-to-many expansion.

The safer workflow is:

```text
Detailed Reviews
545,162 rows
      │
      │ aggregate by listing_id
      ▼
Review Summary per Listing
      │
      │ join on listing_id
      ▼
Enriched Listing Master
```

and separately:

```text
Calendar
3,819,725 rows
      │
      │ aggregate by listing_id
      ▼
Calendar Summary per Listing
      │
      │ join on listing_id
      ▼
Enriched Listing Master
```

---

#### Relationship with Reviewer Entity

The `reviewer_id` field can be interpreted as a logical reviewer identifier.

The relationship is:

```text
Reviewer
    │
    │ reviewer_id
    │ one
    ▼
Review Records
    │ many
```

However, the project does not currently contain a separate reviewer reference table.

Therefore, `reviewer_id` represents an embedded entity identifier rather than a formally linked dimension table.

Potential reviewer-level analysis could include:

* reviews per reviewer,
* repeat reviewer counts,
* listings reviewed per reviewer.

However, this is optional and not necessary for the core assignment.

---

#### Relationship Between Review ID and Reviewer ID

The two identifiers serve different purposes:

```text
id
=
Unique review identifier
```

```text
reviewer_id
=
Identifier of the person/account associated with the review
```

One reviewer may create more than one review.

Therefore:

```text
Reviewer 1 ───────< Many Reviews
```

The `id` field remains the row-level candidate primary key.

---

#### Relationship with Review Comments

Each review record contains one `comments` value.

The structure is:

```text
Review ID
   │
   ▼
Review Record
   │
   ├── Listing ID
   ├── Review Date
   ├── Reviewer ID
   ├── Reviewer Name
   └── Comments
```

The `comments` field is not unique.

Observed results:

* Total review rows: **545,162**
* Unique comment values: **528,743**
* Repeated comment groups: **2,928**
* Extra rows from repeated comments: **16,419**

Repeated comments do not affect review identity because the `id` field remains unique.

---

#### Repeated Comment Text Is Not a Key Violation

Examples of repeated comment values include:

* `.`
* `Good`
* `Top`
* `Great`
* `Great place`
* `Great location`
* `Perfect`

These values may occur across many:

* listings,
* reviewers,
* review dates.

Therefore:

`comments`

cannot be used as a key.

Repeated text alone is not evidence of duplicate review records.

The review ID remains the authoritative row identifier.

---

#### Relationship with the 1,033 No-Review Listings

The summary listing population contains:

* **10,465 listings**

The detailed reviews file contains review records for:

* **9,432 listings**

Difference:

```text
10,465 - 9,432 = 1,033
```

Therefore:

**1,033 summary listings have no records in `reviews.csv.gz`.**

Earlier profiling also identified exactly 1,033:

* missing `last_review` values,
* missing `reviews_per_month` values,
* listings without summary review rows.

This is a strong structural pattern.

However, exact ID-level validation between these missing listing attributes and the no-review listing set is still required before declaring complete row-level equivalence.

---

#### Relationship Cardinality Summary

##### Summary Listings to Detailed Reviews

```text
listings.csv.id
       1
       │
       ▼
      many
reviews.csv.gz.listing_id
```

Status: **Validated**

Review-to-listing coverage: **100.00%**

Listings with reviews: **90.13%**

Listings without reviews: **1,033**

---

##### Detailed Listings to Detailed Reviews

```text
listings.csv.gz.id
        1
        │
        ▼
       many
reviews.csv.gz.listing_id
```

Status: **Validated with known coverage difference**

Detailed-review-to-detailed-listings coverage: **99.09%**

Reviewed listing IDs absent from detailed listings: **86**

---

##### Summary Reviews to Detailed Reviews

```text
reviews.csv
    listing_id
    date
        │
        ▼
Exact projection of
reviews.csv.gz
```

Status: **Fully validated**

Mismatched `(listing_id, date)` groups: **0**

---

##### Reviewer to Review

```text
reviewer_id
     1
     │
     ▼
    many
review.id
```

Status: **Observed logical relationship**

Maximum reviews from one reviewer: **13**

---

#### Relationship Validation Status

| Relationship                                       | Status                           | Result                                                |
| -------------------------------------------------- | -------------------------------- | ----------------------------------------------------- |
| `id` as review primary key                         | Validated                        | 545,162 unique values, 0 missing, 0 duplicates        |
| `(listing_id, date)` as key                        | Rejected                         | 12,942 repeated groups                                |
| Repeated pair review IDs                           | Validated                        | 35,483 rows, 35,483 distinct review IDs               |
| `reviews.csv.gz.listing_id` → `listings.csv.id`    | Validated                        | 100.00% review-to-summary coverage                    |
| `reviews.csv.gz.listing_id` → `listings.csv.gz.id` | Validated with known difference  | 99.09% review-to-detailed coverage                    |
| Summary-only listings with reviews                 | Validated                        | 86 of 96                                              |
| Summary-only listings without reviews              | Validated                        | 10 of 96                                              |
| `reviews.csv` → `reviews.csv.gz` projection        | Fully validated                  | 0 pair-level mismatches                               |
| `reviewer_id` → review records                     | Observed                         | One reviewer may have multiple reviews                |
| 1,033 no-review listings                           | Validated by review relationship | Exact linkage to missing listing fields still pending |

---

#### Overall Key and Relationship Interpretation

The `reviews.csv.gz` dataset has a clear and reliable review-level structure.

Its preferred candidate primary key is:

`id`

The main foreign-key-like field is:

`listing_id`

which connects each review to a listing.

The central relationship structure is:

```text
                         Listing
                            │
                            │ listing_id
                            ▼
                         Review
                      review.id
                    /     |      \
                   ▼      ▼       ▼
              Reviewer   Date   Comments
              reviewer_id
```

The most important validated findings are:

* every review has a unique `id`,
* there are no duplicate review IDs,
* `(listing_id, date)` is not unique,
* repeated listing-date rows represent distinct review IDs,
* `reviews.csv` is an exact projection of `reviews.csv.gz`,
* every reviewed listing exists in `listings.csv`,
* 1,033 summary listings have no reviews,
* 86 of the 96 summary-only listings have detailed review history,
* 10 of the 96 summary-only listings have no detailed reviews.

Overall, the dataset demonstrates strong key integrity and clear one-to-many relationships, making it highly suitable for review aggregation, listing enrichment, reviewer analysis, and cross-dataset validation.

### Business-Domain Meaning

The `reviews.csv.gz` dataset represents detailed guest-review activity for Amsterdam Airbnb listings.

The grain of the dataset is:

**One row represents one individual review submitted for one listing.**

Each review record contains:

- `listing_id`
- `id`
- `date`
- `reviewer_id`
- `reviewer_name`
- `comments`

The dataset contains:

- **545,162 individual reviews**
- **9,432 reviewed listings**
- **545,162 unique review IDs**
- **523,777 unique reviewer IDs**
- review history from **2010-08-16 to 2026-06-28**

Its main business role is to provide detailed evidence about guest engagement, listing activity, reviewer participation, and textual guest feedback.

---

#### Main Business Entity Represented

The primary business entity is the:

**Review**

A review represents one piece of guest feedback associated with a particular Airbnb listing.

The logical structure is:

```text
Listing
   │
   │ receives
   ▼
Review
   │
   ├── Review ID
   ├── Review Date
   ├── Reviewer ID
   ├── Reviewer Name
   └── Review Comment
````

The review is uniquely identified by:

`id`

while:

`listing_id`

connects the review back to the corresponding listing.

---

#### Listing-to-Review Relationship

The central business relationship is:

```text
Listing 1 ───────< Zero or Many Reviews
```

A listing may have:

* no reviews,
* one review,
* many reviews.

Observed results:

* Total summary listings: **10,465**
* Listings with at least one detailed review: **9,432**
* Listings without detailed reviews: **1,033**

Therefore:

* **90.13%** of summary listings have at least one detailed review.
* **9.87%** have no detailed review records.

This relationship can support later listing-level features such as:

* total review count,
* first review date,
* latest review date,
* review activity period,
* recent review count,
* review frequency.

---

#### Review ID

The `id` field uniquely identifies one individual review.

Observed results:

* Total rows: **545,162**
* Unique review IDs: **545,162**
* Missing review IDs: **0**
* Duplicate review IDs: **0**

Therefore:

`id`

is the preferred review-level identifier.

Its role is different from:

* `listing_id`, which identifies the reviewed listing,
* `reviewer_id`, which identifies the reviewer.

The relationship is:

```text
Review ID
   │
   ▼
One Individual Review
```

---

#### Listing Identifier

The `listing_id` field connects each review to an Airbnb listing.

The relationship is:

```text
Listing
   │
   │ one
   ▼
Reviews
   │
   │ many
```

Observed results:

* Unique reviewed listings: **9,432**
* Missing listing IDs: **0**

Every reviewed listing ID exists in the summary listings dataset.

This makes `listing_id` the main foreign-key-like field for review aggregation and listing enrichment.

---

#### Reviewer Entity

The `reviewer_id` field represents the person or account associated with a review.

Observed results:

* Unique reviewer IDs: **523,777**
* Minimum reviews per reviewer: **1**
* Maximum reviews per reviewer: **13**
* Median reviews per reviewer: **1**
* Average reviews per reviewer: approximately **1.04**
* Reviewers appearing more than once: **18,043**

Therefore, most reviewers appear once in this Amsterdam dataset, while a smaller number have multiple review records.

The logical relationship is:

```text
Reviewer 1 ───────< One or More Reviews
```

However, the project does not contain a separate reviewer dimension or reviewer reference table.

Therefore, reviewer analysis should remain limited to the identifiers and attributes available in the review dataset.

---

#### Reviewer Name

The `reviewer_name` field contains the displayed name associated with the reviewer.

Observed results:

* Unique non-null reviewer names: **72,308**
* Missing reviewer names: **1**

Reviewer names are useful for descriptive inspection but should not be treated as unique identifiers.

Multiple reviewers may share names such as:

* Maria
* John
* Anna
* David

Therefore:

`reviewer_id`

is the preferred reviewer-level identifier.

---

#### Review Date

The `date` field represents the date associated with a review record.

Observed coverage:

* Earliest date: **2010-08-16**
* Latest date: **2026-06-28**
* Unique review dates: **5,327**
* Missing dates: **0**

This supports temporal analysis such as:

* reviews by year,
* reviews by month,
* recent review activity,
* listing review lifespan,
* time since last review.

For example:

```text
review_lifespan
=
latest_review_date - first_review_date
```

and:

```text
days_since_last_review
=
reference_date - latest_review_date
```

These can become useful derived listing-level features.

---

#### Historical Review Activity

The review history spans nearly 16 years.

This means the dataset provides a long-term view of guest activity.

Possible business questions include:

* Which listings have the longest review histories?
* Which listings receive frequent recent reviews?
* Which listings show declining or increasing review activity?
* Which neighbourhoods contain more actively reviewed listings?
* Which room types receive more reviews?

However, current listing attributes should not automatically be assumed to have remained unchanged throughout the full historical review period.

---

#### Reviews per Listing

Observed review-count statistics are:

* Listings with reviews: **9,432**
* Minimum reviews per reviewed listing: **1**
* Maximum reviews per listing: **5,603**
* Median reviews per reviewed listing: **13**
* Average reviews per reviewed listing: approximately **57.80**

The distribution is highly right-skewed.

This means:

* many listings have relatively few reviews,
* a smaller number have extremely high review volumes.

Therefore, the median is more representative of a typical reviewed listing than the mean.

---

#### Review Count as an Activity Proxy

Review count can be useful as a proxy for listing activity.

For example:

```text
Higher review count
        │
        ▼
Potentially greater historical guest activity
```

However, review count must not be interpreted as exact booking count.

Not every guest necessarily leaves a review.

Therefore:

**Review count is an activity proxy, not a direct booking count.**

This distinction is important for trustworthy business interpretation.

---

#### Highly Reviewed Listings

The listings with the highest detailed review counts include:

|           Listing ID | Review Count |
| -------------------: | -----------: |
|           `50383849` |        5,603 |
|           `32485135` |        4,082 |
|           `45045046` |        3,257 |
|           `50452179` |        1,993 |
|           `35927687` |        1,773 |
|           `30762441` |        1,706 |
| `905677609043113133` |        1,611 |
|             `802052` |        1,551 |
| `746060021045144380` |        1,398 |
|           `46591985` |        1,304 |

These listings may represent:

* highly active accommodations,
* long-running listings,
* hotel-like properties,
* professionally managed listings.

The high counts should not automatically be considered errors.

---

#### Same-Day Multiple Reviews

The dataset contains multiple cases where a single listing receives several reviews on the same date.

Observed results:

* Repeated `(listing_id, date)` groups: **12,942**
* Extra rows from repeated groups: **22,541**
* Total rows within repeated groups: **35,483**
* Distinct review IDs within those rows: **35,483**

Therefore, these records represent different reviews rather than duplicate review IDs.

For example, listing:

`50383849`

received:

* 13 reviews on 2022-04-18,
* 13 reviews on 2022-08-05,
* 13 reviews on 2023-07-02.

This can occur when multiple guests submit separate feedback associated with the same listing and date.

---

#### Relationship with Summary Reviews

The `reviews.csv` file contains only:

* `listing_id`
* `date`

The detailed review file contains the same review population but adds:

* unique review ID,
* reviewer ID,
* reviewer name,
* review comment.

The exact relationship is:

```text
Detailed Review
    listing_id
    id
    date
    reviewer_id
    reviewer_name
    comments
          │
          │ select only listing_id and date
          ▼
Summary Review
    listing_id
    date
```

The files were explicitly compared and produced:

* **0 mismatched `(listing_id, date)` groups**

Therefore, the summary reviews file is an exact reduced projection of the detailed review dataset.

---

#### Review Comments

The `comments` field stores the textual review left by the reviewer.

Observed results:

* Missing comments: **0**
* Empty but non-null comments: **0**
* Unique comments: **528,743**
* Minimum comment length: **1 character**
* Maximum comment length: **6,164 characters**
* Median comment length: **189 characters**
* Average comment length: approximately **247.76 characters**

This field contains potentially valuable qualitative information about guest experiences.

---

#### Potential Business Uses of Review Comments

Review comments could theoretically support analysis of:

* guest satisfaction,
* cleanliness,
* location,
* communication,
* check-in experience,
* recurring complaints,
* positive themes,
* service quality.

For example:

```text
Review Comments
       │
       ▼
Keyword or Theme Extraction
       │
       ▼
Common Positive and Negative Topics
```

However, extensive NLP is optional and should not delay the core engineering, EDA, statistical analysis, or documentation tasks.

---

#### Repeated Comment Text

Repeated comment text was identified.

Observed results:

* Repeated comment groups: **2,928**
* Extra rows from repeated comments: **16,419**

Examples include:

* `.`
* `Good`
* `Top`
* `Great`
* `Great place`
* `Great location`
* `Perfect`
* `Great stay`
* `Nice place`

Repeated text does not necessarily indicate duplicate review records.

For example, many different guests can independently write:

`Great`

Therefore, repeated text should not be used as a duplicate-removal rule.

---

#### Minimal-Information Comments

Some comments contain very limited text, such as:

* `.`
* `-`
* `👍`
* `Good`
* `Top`

The comment:

`.`

appears **1,033 times**.

These records are still valid review records because:

* review IDs are unique,
* reviewer IDs exist,
* listing IDs exist,
* dates exist.

However, minimal-text reviews may provide limited value for:

* sentiment analysis,
* theme extraction,
* text summarization.

If text analysis is later performed, a minimum-text-length or content-quality filter may be considered.

Such filtering should apply only to the analytical text subset, not to the original raw dataset.

---

#### Multilingual Guest Feedback

The sample comments demonstrate that the dataset contains multiple languages.

Examples include:

* English,
* French,
* German,
* Spanish,
* Dutch,
* Portuguese,
* Italian,
* other languages.

This reflects Amsterdam's international guest population.

However, multilingual comments make NLP more challenging.

An English-only sentiment model may produce unreliable results for non-English reviews.

Therefore, multilingual NLP should only be attempted if sufficient time and appropriate tooling are available.

---

#### Relationship with the 1,033 Listings Without Reviews

The summary listings dataset contains:

* **10,465 listings**

The detailed reviews dataset represents:

* **9,432 listings**

Therefore:

**1,033 summary listings have no review records.**

Earlier profiling also found exactly 1,033 missing values in:

* `last_review`
* `reviews_per_month`

in the summary listings dataset.

This strongly suggests that listings without review records also account for these missing review-related fields.

However, exact row-level set comparison is still required before stating this as fully proven.

---

#### Relationship with the 96 Summary-Only Listings

Earlier analysis identified 96 listings that:

* exist in `listings.csv`,
* exist in `calendar.csv.gz`,
* do not exist in `listings.csv.gz`.

Detailed review validation showed:

* **86 of these 96 listings have review records**
* **10 have no review records**

Therefore:

```text id="15xly6"
96 Summary-Only Listings
        │
        ├── 86 reviewed
        └── 10 without reviews
```

This is important when selecting the base table for the enriched listing master.

Using only `listings.csv.gz` as the base would exclude 86 listings that still have valid review history.

---

#### Listing-Level Review Aggregation

The raw dataset contains:

**545,162 review rows**

For efficient enrichment, it should first be aggregated by:

`listing_id`

Potential derived features include:

* `review_count`
* `first_review_date`
* `last_review_date`
* `review_lifespan_days`
* `recent_review_count`
* `unique_reviewer_count`
* `average_comment_length`

The recommended workflow is:

```text id="q5l6ts"
Detailed Reviews
545,162 rows
      │
      │ aggregate by listing_id
      ▼
Listing-Level Review Summary
9,432 reviewed listings
      │
      │ join on listing ID
      ▼
Enriched Listing Master
```

This preserves the one-row-per-listing analytical grain.

---

#### Relationship with Calendar Data

Reviews and calendar data describe different parts of listing activity.

```text
Listing
   │
   ├────────► Reviews
   │            │
   │            └── historical guest feedback
   │
   └────────► Calendar
                │
                └── future availability and stay rules
```

Reviews are primarily historical.

Calendar data is primarily forward-looking around the scrape period.

These datasets should be aggregated separately before joining to the listing master.

Directly joining raw review rows with raw calendar rows could create an enormous many-to-many expansion.

---

#### Historical Versus Current Attributes

The review dataset begins in:

**2010**

while the listing datasets represent a much later current snapshot.

Therefore, current attributes such as:

* room type,
* property type,
* price,
* host status,
* availability,
* amenities,

may not represent the state of the listing at the time of an old review.

This creates an important temporal limitation.

For example:

```text
Review from 2012
       │
       ▼
Should not automatically be assumed to describe
the exact 2026 listing configuration.
```

---

#### Business Questions Supported by This Dataset

The detailed review dataset can support questions such as:

* Which listings have the most review activity?
* Which neighbourhoods have higher review counts?
* Which room types receive more reviews?
* Which listings have recent review activity?
* Which listings have long periods without recent reviews?
* Which reviewers appear more than once?
* What themes appear in guest comments?
* Which listings have particularly long review histories?

These questions can support market understanding and listing-level segmentation.

---

#### Review Metrics for Enriched Listing Master

Useful listing-level derived metrics may include:

```text
total_reviews
```

```text
first_review_date
```

```text
last_review_date
```

```text
review_lifespan_days
```

```text
days_since_last_review
```

```text
unique_reviewer_count
```

```text
recent_review_count
```

Only features with clear business meaning should be included.

---

#### Analytical Importance

The detailed review dataset is valuable for:

* listing activity measurement,
* temporal review analysis,
* review-history enrichment,
* reviewer analysis,
* relationship validation,
* optional text analysis,
* validating summary review records.

Its strongest advantage over `reviews.csv` is the presence of:

* unique review IDs,
* reviewer IDs,
* reviewer names,
* comments.

These additional fields make it possible to distinguish genuinely separate reviews that share the same listing and date.

---

#### Overall Business Interpretation

The `reviews.csv.gz` dataset represents individual guest-review activity associated with Amsterdam Airbnb listings.

Its central business structure is:

```text id="bmd7oh"
Listing
   │
   │ receives
   ▼
Review
   │
   ├── Review ID
   ├── Date
   ├── Reviewer
   └── Comment
```

The most important business interpretations are:

* review count is an activity proxy, not an exact booking count,
* one listing may receive many reviews,
* one reviewer may appear in multiple review records,
* multiple valid reviews can occur for the same listing on the same date,
* comment text provides qualitative feedback but is multilingual,
* historical reviews may not perfectly align with current listing attributes,
* raw review rows should be aggregated before listing-level enrichment.

Overall, the dataset provides a detailed and reliable source for understanding guest-review activity, listing engagement, historical review patterns, and optional qualitative feedback analysis.

### Dataset Limitations

The `reviews.csv.gz` dataset is structurally strong and highly complete, but several limitations must be considered before using it for review aggregation, listing enrichment, EDA, statistical analysis, reviewer analysis, or text analysis.

The dataset contains:

- **545,162 review rows**
- **6 columns**
- **9,432 reviewed listings**
- **545,162 unique review IDs**
- **523,777 unique reviewer IDs**
- **5,327 unique review dates**
- **0 complete duplicate rows**
- **1 missing value across the entire dataset**

Although the dataset has excellent structural quality, its business interpretation requires caution because review activity is not equivalent to booking activity, historical reviews span many years, comments are multilingual, and some reviewed listings are absent from the detailed listings file.

---

#### 1. Review Count Is Not Booking Count

The dataset contains one row per review, not one row per reservation or completed stay.

Therefore:

```text
Number of reviews
≠
Number of bookings
````

Not every guest necessarily leaves a review.

A listing with:

* 100 reviews

may have had more than 100 bookings.

Therefore, review count should be interpreted as:

**A proxy for guest activity or engagement**

rather than an exact booking count.

This distinction is important when comparing:

* listing popularity,
* neighbourhood demand,
* room-type performance,
* host activity.

---

#### 2. Reviewed Listings Represent Only Part of the Listing Population

The summary listings dataset contains:

* **10,465 listings**

The detailed reviews dataset contains reviews for:

* **9,432 listings**

Therefore:

* **1,033 listings have no detailed review records**
* reviewed-listing coverage is **90.13%**

This means any analysis using only reviewed listings excludes approximately:

**9.87% of the summary listing population**

This can create selection bias because listings without reviews may differ systematically from reviewed listings.

For example, they may be:

* newly created,
* inactive,
* less frequently booked,
* recently added to the platform,
* otherwise different from established listings.

No specific cause should be assumed without evidence.

---

#### 3. Historical Reviews and Current Listing Attributes May Not Align Temporally

The review dataset spans:

* Earliest review date: **2010-08-16**
* Latest review date: **2026-06-28**

This covers almost 16 years.

However, the listing datasets represent a much later snapshot around 2026.

Therefore, a historical review may not correspond to the listing's current:

* price,
* room type,
* property type,
* number of beds,
* amenities,
* host status,
* superhost status,
* availability.

For example:

```text
Review from 2012
       │
       ▼
Current 2026 listing attributes
```

should not automatically be assumed to describe the exact same listing configuration.

This creates an important temporal interpretation limitation.

---

#### 4. Current Host Attributes May Not Reflect Historical Review Periods

The detailed listings dataset contains current or recent host-related fields such as:

* `host_id`
* `host_is_superhost`
* `host_identity_verified`
* `calculated_host_listings_count`

However, these current host attributes may not have been the same when older reviews were submitted.

For example:

* the listing may have changed hosts,
* a host may have gained or lost superhost status,
* a host portfolio may have changed over time.

Therefore, historical review activity should not automatically be attributed to current host conditions without caution.

---

#### 5. The Detailed Review Population Does Not Fully Match `listings.csv.gz`

The detailed review dataset contains:

* **9,432 unique listing IDs**

The detailed listings dataset contains:

* **10,369 unique listing IDs**

Relationship validation found:

* **9,346 common listing IDs**
* **86 reviewed listing IDs absent from `listings.csv.gz`**
* **1,023 detailed listings without review records**

Coverage:

* Detailed reviews to detailed listings: **99.09%**
* Detailed listings with reviews: **90.13%**

Therefore, using only `listings.csv.gz` as the base for review enrichment would exclude valid review records associated with 86 listing IDs.

---

#### 6. The 86 Reviewed Listings Missing from Detailed Listings Require Careful Handling

The 86 reviewed listing IDs absent from `listings.csv.gz` belong to the previously identified group of 96 summary-only listings.

Observed results:

* Total summary-only listings: **96**
* Summary-only listings with reviews: **86**
* Summary-only listings without reviews: **10**

Therefore, these 86 review listing IDs are not orphaned from the overall listing population.

They exist in:

* `listings.csv`
* `calendar.csv.gz`
* `reviews.csv`
* `reviews.csv.gz`

but are absent from:

* `listings.csv.gz`

The reason for their absence from the detailed listings file remains unconfirmed.

---

#### 7. Review Counts Are Highly Right-Skewed

Observed review counts per reviewed listing are:

* Minimum: **1**
* Maximum: **5,603**
* Median: **13**
* Average: approximately **57.80**

The large difference between:

* median = 13
* average = 57.80
* maximum = 5,603

shows that a relatively small number of listings have very large review volumes.

Therefore:

* the mean may be strongly influenced by high-volume listings,
* median and quantiles may provide more representative summaries.

This should be considered during EDA and statistical comparisons.

---

#### 8. Extremely High Review Counts Require Contextual Validation

The most reviewed listing has:

**5,603 reviews**

Other listings also have thousands of reviews.

Examples include:

* 4,082 reviews
* 3,257 reviews
* 1,993 reviews
* 1,773 reviews

These values should not automatically be removed as errors.

They may represent:

* hotel-like accommodations,
* high-volume professionally managed properties,
* long-running listings,
* genuinely popular listings.

The correct approach is to investigate and document extreme values before deciding whether any special treatment is required.

---

#### 9. `(listing_id, date)` Is Not Unique

The summary reviews dataset contains only:

* `listing_id`
* `date`

Earlier profiling found:

* **12,942 repeated `(listing_id, date)` groups**
* **22,541 extra rows from repeated pairs**

The detailed reviews dataset confirms that these are legitimate separate reviews because:

* repeated-pair rows: **35,483**
* distinct review IDs within those rows: **35,483**

Therefore, `(listing_id, date)` must not be treated as a unique review key.

Removing repeated listing-date rows would incorrectly delete valid reviews.

---

#### 10. Repeated Comment Text Does Not Necessarily Mean Duplicate Reviews

The dataset contains:

* **528,743 unique comments**
* across **545,162 reviews**

Repeated-comment analysis found:

* **2,928 repeated comment groups**
* **16,419 extra rows from repeated comments**

Examples include:

* `.`
* `Good`
* `Top`
* `Great`
* `Great place`
* `Perfect`

Repeated text can occur naturally because multiple people may independently write the same short phrase.

Therefore:

```text
Repeated comment text
≠
Duplicate review
```

Duplicate detection should rely on:

* review ID,
* full-row equality,
* other strong evidence.

---

#### 11. Some Comments Contain Very Little Information

Observed comment lengths are:

* Minimum: **1 character**
* Maximum: **6,164 characters**
* Median: **189 characters**
* Average: approximately **247.76 characters**

Some comments contain minimal content, such as:

* `.`
* `-`
* `👍`
* `Good`
* `Top`

These records are structurally valid but may contribute little value to:

* sentiment analysis,
* topic modeling,
* theme extraction.

If NLP is performed later, a text-quality filter may be appropriate for the analytical subset.

The raw records should still remain unchanged.

---

#### 12. The `"."` Comment Appears 1,033 Times

The comment:

`.`

appears:

* **1,033 times**
* across **629 distinct listings**
* from **1,024 distinct reviewers**

This is unusual but should not automatically be treated as a duplicate pattern.

The number 1,033 also happens to match:

* the number of summary listings without reviews.

However, no relationship between these two facts has been established.

Therefore, the matching count should be treated as coincidence unless explicit ID-level evidence proves otherwise.

---

#### 13. Multilingual Comments Complicate Text Analysis

The comments include multiple languages, such as:

* English,
* French,
* German,
* Spanish,
* Dutch,
* Portuguese,
* Italian,
* other languages.

Therefore, English-only NLP methods may produce incomplete or biased results.

Potential challenges include:

* incorrect sentiment scores,
* poor keyword extraction,
* language-specific slang,
* mixed-language comments.

Given the short project timeline, multilingual NLP should remain optional.

---

#### 14. HTML-Like Tags Appear in Review Text

Some comments contain text such as:

```text
<br/>
```

This suggests that review comments may contain HTML-like formatting artifacts.

Before NLP or text mining, comments may require cleaning such as:

* HTML tag removal,
* whitespace normalization,
* line-break normalization.

However, the original raw comment should remain preserved.

---

#### 15. Reviewer Names Are Not Unique Identifiers

The dataset contains:

* **72,308 unique reviewer names**
* **523,777 unique reviewer IDs**

Many different reviewers can share the same displayed name.

Therefore:

`reviewer_name`

should not be used for:

* deduplication,
* reviewer-level joins,
* reviewer identity validation.

The correct reviewer-level identifier is:

`reviewer_id`

---

#### 16. One Reviewer May Appear More Than Once

Observed reviewer statistics include:

* Unique reviewers: **523,777**
* Reviewers with multiple reviews: **18,043**
* Maximum reviews per reviewer: **13**

Therefore, `reviewer_id` is not unique at the review-row level.

This is expected because a reviewer may submit multiple reviews.

The review-level primary key remains:

`id`

---

#### 17. Reviewer Identity Is Limited to the Available Fields

The dataset contains:

* `reviewer_id`
* `reviewer_name`

It does not contain richer reviewer attributes such as:

* reviewer location,
* account creation date,
* verified status,
* demographic details.

Therefore, reviewer-level analysis is limited.

The project should avoid making unsupported claims about reviewer characteristics.

---

#### 18. Review Comments May Contain Subjective or Noisy Text

Review text may include:

* personal opinions,
* spelling mistakes,
* emojis,
* sarcasm,
* slang,
* mixed languages,
* very short comments,
* formatting artifacts.

Therefore, comments are not clean structured data.

Any text analysis should include appropriate preprocessing and cautious interpretation.

---

#### 19. Review Sentiment Is Not Directly Available

The dataset does not contain an explicit sentiment label.

Therefore, identifying:

* positive reviews,
* neutral reviews,
* negative reviews

would require a derived NLP model or rule-based method.

Such derived sentiment should not be presented as source-provided truth.

It should be clearly labeled as:

**Model-derived sentiment**

if implemented.

---

#### 20. Review Text May Mention Factors Outside Listing Control

Comments may discuss:

* weather,
* transport,
* neighbourhood noise,
* airline delays,
* personal expectations,
* external attractions.

Therefore, sentiment in a comment may not always directly reflect listing quality.

This limits simple interpretation of text sentiment as a pure host or property performance measure.

---

#### 21. Review Date Does Not Necessarily Equal Stay Date

The dataset provides:

`date`

as the review date.

It does not provide:

* check-in date,
* checkout date,
* actual stay date.

Therefore, the project should not assume that:

```text
review date = guest stay date
```

unless supported by source documentation.

This limits exact booking-timeline reconstruction.

---

#### 22. No Explicit Review Rating Is Present in the Detailed Review File

The detailed reviews file contains:

* review text,
* reviewer information,
* review date,

but no explicit numeric rating column.

Numeric review scores exist in the detailed listings dataset at the listing level.

Therefore, review-level sentiment and listing-level review scores should not be confused.

The file does not allow direct analysis of:

```text
one individual review text
vs
one individual numeric rating
```

because individual review ratings are not provided.

---

#### 23. The Dataset Is Not a Complete Booking Transaction Log

The detailed review file does not contain:

* reservation IDs,
* booking values,
* check-in dates,
* checkout dates,
* payment information,
* booked-night counts.

Therefore, it cannot independently support:

* revenue reconstruction,
* exact occupancy calculation,
* exact booking frequency.

Its role is review and guest-feedback analysis.

---

#### 24. Old Reviews May Belong to Listings That Changed Significantly

Over a long review history, a listing may have changed:

* host,
* room configuration,
* property type,
* amenities,
* price,
* policies.

Therefore, aggregating all historical reviews together assumes continuity that may not always exist.

This is acceptable for broad activity measures but should be acknowledged.

---

#### 25. Average Review Count Can Be Misleading

The average number of reviews per reviewed listing is approximately:

**57.80**

However, the median is only:

**13**

Because of the strongly skewed distribution, relying only on the average may overstate typical review activity.

Recommended summaries include:

* median,
* quartiles,
* percentiles,
* logarithmic scales for visualization.

---

#### 26. Review Activity May Be Influenced by Listing Age

Older listings have had more time to accumulate reviews.

Therefore, comparing total review counts without considering listing age may unfairly favor older listings.

A possible derived metric is:

```text
review_rate
=
total_reviews / active_time
```

However, this requires a reliable definition of listing start or review-active period.

Because `host_since` is entirely missing in the detailed listings file, review dates may need to be used carefully as a proxy.

---

#### 27. First Review Date Is Not Necessarily Listing Creation Date

The earliest review associated with a listing does not prove when the listing was created.

A listing may have existed before receiving its first review.

Therefore:

```text
first_review_date
≠
confirmed listing creation date
```

It may be used as an observed activity-start proxy only when labeled clearly.

---

#### 28. Detailed and Summary Review Files Are Redundant for Some Tasks

The exact comparison confirmed that `reviews.csv` is a projection of `reviews.csv.gz` onto:

* `listing_id`
* `date`

Therefore, using both files simultaneously for review counts could double count records.

The correct approach is to choose the appropriate source:

* use `reviews.csv` when only listing and date are needed,
* use `reviews.csv.gz` when review ID, reviewer, or comment information is required.

Do not concatenate the two datasets.

---

#### 29. Raw Review-to-Calendar Joins Can Create Massive Row Expansion

The detailed review dataset contains:

* **545,162 rows**

The calendar dataset contains:

* **3,819,725 rows**

Both contain multiple rows per listing.

Joining them directly on:

`listing_id`

could create a very large many-to-many expansion.

The safer workflow is:

```text
Detailed Reviews
      │
      │ aggregate by listing_id
      ▼
Review Summary
      │
      ▼
Listing Master
```

and separately:

```text
Calendar
      │
      │ aggregate by listing_id
      ▼
Calendar Summary
      │
      ▼
Listing Master
```

This preserves one row per listing and protects memory.

---

#### 30. Text Analysis Could Be Expensive

Although the detailed reviews file contains only 545,162 rows, the `comments` column includes large variable-length text.

Full NLP processing may require:

* additional memory,
* tokenization,
* language detection,
* sentiment models,
* extra dependencies,
* longer execution time.

Given the limited project timeline, core engineering and analytical requirements should remain the priority.

---

#### 31. Personal Names Should Be Handled Carefully

The dataset contains reviewer names.

Although they are part of the source data, they are not necessary for most analytical tasks.

For:

* public reports,
* visualizations,
* exported analytical datasets,

it is generally better to avoid displaying unnecessary personal names unless required.

Reviewer IDs are sufficient for most aggregation purposes.

---

#### 32. CSV Relationships Are Not Formally Enforced

Although the following relationships were validated:

* review ID uniqueness,
* listing relationships,
* summary-to-detailed review consistency,

the files are CSV-based.

Therefore:

* primary keys are not enforced by a database,
* foreign keys are not enforced,
* future snapshots may differ.

All key and relationship checks should be repeated in a reusable pipeline.

---

#### 33. The Exact 1,033 Structural Missingness Relationship Is Not Yet Fully Proven at ID Level

The following counts all equal 1,033:

* summary listings without detailed reviews,
* summary listings without summary review rows,
* missing `last_review` values,
* missing `reviews_per_month` values.

This strongly suggests structural missingness.

However, exact set-level validation between:

* no-review listing IDs,
* missing `last_review` listing IDs,
* missing `reviews_per_month` listing IDs

has not yet been explicitly performed.

Therefore, the correct current statement is:

**The counts align perfectly and strongly suggest a shared structural missingness pattern, but exact ID-level equivalence remains to be validated.**

---

#### Overall Limitation Summary

The most important limitations of `reviews.csv.gz` are:

1. Review count is not equivalent to booking count.
2. 1,033 summary listings have no review records.
3. Historical reviews span nearly 16 years and may not match current listing attributes.
4. 86 reviewed listings are absent from `listings.csv.gz`.
5. Review counts are highly right-skewed.
6. Very high review counts require contextual interpretation.
7. `(listing_id, date)` is not unique.
8. Repeated comment text does not necessarily indicate duplicate reviews.
9. Some comments contain minimal information.
10. Comments are multilingual and may contain HTML-like artifacts.
11. Reviewer names are not unique identifiers.
12. Reviewer-level attributes are limited.
13. Review date is not confirmed stay date.
14. Individual numeric review ratings are not included.
15. The dataset is not a booking or transaction log.
16. Raw review-to-calendar joins could create massive many-to-many expansion.
17. NLP can add substantial time and technical complexity.
18. The exact ID-level 1,033 structural missingness relationship remains to be validated.

Despite these limitations, the dataset remains highly valuable for:

* review aggregation,
* listing enrichment,
* review-history analysis,
* reviewer activity analysis,
* relationship validation,
* optional text analysis.

**Overall assessment: Excellent-quality review-level data with strong identifiers, minimal missingness, no duplicates, and excellent consistency with the summary review file. Its main limitations are interpretive rather than structural, especially the distinction between reviews and bookings, temporal mismatch with current listing attributes, multilingual free text, skewed review counts, and incomplete overlap with the detailed listings population.**

### Data Quality Assessment

The `reviews.csv.gz` dataset demonstrates excellent structural quality, strong key integrity, minimal missingness, and high cross-dataset consistency.

The dataset contains:

- **545,162 rows**
- **6 columns**
- **9,432 unique listing IDs**
- **545,162 unique review IDs**
- **523,777 unique reviewer IDs**
- **5,327 unique review dates**
- **0 complete duplicate rows**
- **0 duplicate review IDs**
- **1 missing value across the entire dataset**
- **0 empty but non-null comments**

Overall, the dataset is highly reliable for review-level analysis, listing enrichment, relationship validation, reviewer aggregation, and optional text analysis.

---

#### Structural Quality

The six columns are:

- `listing_id`
- `id`
- `date`
- `reviewer_id`
- `reviewer_name`
- `comments`

Observed DuckDB data types are:

| Column | Data Type |
|---|---|
| `listing_id` | BIGINT |
| `id` | BIGINT |
| `date` | DATE |
| `reviewer_id` | BIGINT |
| `reviewer_name` | VARCHAR |
| `comments` | VARCHAR |

The schema is compact and logically consistent with the dataset grain:

**One row represents one individual review.**

Important positive observations include:

- large identifiers are safely represented using `BIGINT`,
- dates are correctly represented as `DATE`,
- reviewer names and comments are stored as text,
- no important type-conversion issue was identified during familiarization.

---

#### Candidate Primary-Key Quality

The `id` field was explicitly validated as the preferred candidate primary key.

Observed results:

- Total rows: **545,162**
- Unique review IDs: **545,162**
- Missing review IDs: **0**
- Duplicate review IDs: **0**

Therefore:

**Candidate primary key: `id`**

The field is:

- complete,
- unique,
- appropriate for individual review identity.

This is an excellent key-quality result.

Because the data is stored in CSV format rather than a relational database, the primary key is logically identified rather than formally enforced.

---

#### Complete Duplicate-Row Quality

The full six-column records were tested for exact duplicates.

Observed results:

- Total rows: **545,162**
- Duplicate groups: **0**
- Extra duplicate rows: **0**
- Duplicate percentage: **0.0000%**

Therefore, no complete duplicate review records were found.

This is a strong positive data-quality characteristic.

---

#### Missing-Value Quality

Only one missing value exists across all 545,162 rows.

| Column | Missing Count | Missing Percentage |
|---|---:|---:|
| `listing_id` | 0 | 0.00% |
| `id` | 0 | 0.00% |
| `date` | 0 | 0.00% |
| `reviewer_id` | 0 | 0.00% |
| `reviewer_name` | 1 | approximately 0.00018% |
| `comments` | 0 | 0.00% |

This means:

- every review has a listing identifier,
- every review has a unique review identifier,
- every review has a date,
- every review has a reviewer identifier,
- every review has a comment,
- only one review lacks a reviewer name.

The single missing reviewer name is negligible for most analytical purposes.

---

#### Listing Identifier Quality

The `listing_id` field contains:

- **9,432 unique listing IDs**
- **0 missing values**

It is suitable as the main foreign-key-like field linking reviews to listing datasets.

Observed relationship with summary listings:

- Review listing IDs: **9,432**
- Summary listing IDs: **10,465**
- Common IDs: **9,432**
- Review IDs absent from summary listings: **0**

Coverage:

- Detailed reviews to summary listings: **100.00%**

Therefore, every reviewed listing exists in the summary listing dataset.

This is an excellent referential-integrity result.

---

#### Relationship Quality with Summary Listings

The relationship is:

```text
listings.csv.id
       1
       │
       ▼
      many
reviews.csv.gz.listing_id
````

Observed results:

* Summary listings: **10,465**
* Listings with detailed reviews: **9,432**
* Summary listings without detailed reviews: **1,033**
* Reviewed-listing coverage: **90.13%**

This means:

* 90.13% of summary listings have at least one detailed review,
* 9.87% have no detailed review records.

The absence of reviews for 1,033 listings is not automatically a data-quality error.

It may represent structural missingness associated with listings that have no review history.

---

#### Relationship Quality with Detailed Listings

The detailed reviews dataset was also compared with:

`listings.csv.gz.id`

Observed results:

* Unique detailed-review listing IDs: **9,432**
* Unique detailed listing IDs: **10,369**
* Common listing IDs: **9,346**
* Review listing IDs absent from detailed listings: **86**
* Detailed listings without detailed reviews: **1,023**

Coverage:

* Detailed reviews to detailed listings: **99.09%**
* Detailed listings with reviews: **90.13%**

The 86 review listing IDs absent from the detailed listings file were explained by the known summary-only listing population.

---

#### Quality of the 96 Summary-Only Listing Relationship

Earlier analysis identified 96 listings that:

* appear in `listings.csv`,
* appear in `calendar.csv.gz`,
* do not appear in `listings.csv.gz`.

Review validation showed:

* **86 of the 96 have detailed reviews**
* **10 of the 96 have no detailed reviews**

Therefore, the 86 review listing IDs absent from `listings.csv.gz` are not unexplained orphan records.

They belong to the known 96-record summary-only population.

This is a strong cross-dataset consistency result.

---

#### Summary and Detailed Reviews Consistency

The summary `reviews.csv` file and detailed `reviews.csv.gz` file were explicitly compared.

Both contain:

* **545,162 rows**
* **9,432 unique listing IDs**
* review dates from **2010-08-16 to 2026-06-28**
* **5,327 unique review dates**

An exact multiset comparison was performed on:

`(listing_id, date)`

while preserving the number of occurrences of every pair.

Observed result:

* Mismatched `(listing_id, date)` groups: **0**

Therefore:

**`reviews.csv` is an exact projection of `reviews.csv.gz` onto the `listing_id` and `date` columns, including repeated listing-date pairs.**

This is one of the strongest cross-dataset quality findings in the project.

---

#### Repeated `(listing_id, date)` Quality Assessment

The detailed review dataset contains:

* **12,942 repeated `(listing_id, date)` groups**
* **22,541 extra rows from repeated pairs**

At first glance, this might appear to indicate duplicate data.

However, review-level validation showed:

* Total rows in repeated groups: **35,483**
* Distinct review IDs within those groups: **35,483**

Therefore, every row within the repeated listing-date groups has its own unique review ID.

This confirms that the repeated pairs represent legitimate distinct review records.

The correct interpretation is:

```text
Same listing
+
Same review date
≠
Duplicate review
```

Therefore, these 22,541 extra repeated-pair rows should not be removed during cleaning.

---

#### Review Date Quality

The `date` field has:

* Earliest date: **2010-08-16**
* Latest date: **2026-06-28**
* Unique dates: **5,327**
* Missing dates: **0**

DuckDB successfully interpreted the field as:

`DATE`

This indicates strong date-format consistency.

No missing date records were found.

The main limitation is temporal rather than structural: historical reviews may not perfectly align with current listing characteristics.

---

#### Reviews-per-Listing Quality

Observed review-count statistics are:

* Listings with reviews: **9,432**
* Minimum reviews per reviewed listing: **1**
* Maximum reviews per listing: **5,603**
* Median reviews: **13**
* Average reviews: approximately **57.80**

This distribution is highly right-skewed.

The large difference between:

* median = 13,
* average ≈ 57.80,
* maximum = 5,603

shows that a relatively small number of listings have extremely large review counts.

Therefore, for descriptive analysis:

* the median should be preferred for typical review activity,
* high-count listings should be investigated rather than automatically removed.

---

#### High-Volume Listing Validation Priority

The highest review counts include:

|           Listing ID | Review Count |
| -------------------: | -----------: |
|           `50383849` |        5,603 |
|           `32485135` |        4,082 |
|           `45045046` |        3,257 |
|           `50452179` |        1,993 |
|           `35927687` |        1,773 |
|           `30762441` |        1,706 |
| `905677609043113133` |        1,611 |
|             `802052` |        1,551 |
| `746060021045144380` |        1,398 |
|           `46591985` |        1,304 |

These values should be classified as:

**Validation priorities**

rather than automatic errors.

They may represent:

* highly active properties,
* hotel-like listings,
* professionally managed accommodation,
* long-running listings.

---

#### Reviewer Identifier Quality

The `reviewer_id` field contains:

* **523,777 unique reviewer IDs**
* **0 missing values**

Observed reviewer-level statistics:

* Minimum reviews per reviewer: **1**
* Maximum reviews per reviewer: **13**
* Median reviews per reviewer: **1**
* Average reviews per reviewer: approximately **1.04**
* Reviewers with multiple reviews: **18,043**

This indicates that most reviewers appear once, while a smaller number appear more than once.

The field is suitable for reviewer-level aggregation.

However, it is not unique at the review-row level.

---

#### Reviewer Name Quality

The `reviewer_name` field contains:

* **72,308 unique non-null values**
* **1 missing value**

The field is nearly complete but should not be used as a unique identifier because:

* many reviewers may share the same name,
* names may vary in spelling or capitalization.

The preferred reviewer-level identifier is:

`reviewer_id`

---

#### Comment Completeness Quality

The `comments` field has:

* Missing values: **0**
* Empty but non-null values: **0**

Observed comment lengths are:

* Minimum length: **1 character**
* Maximum length: **6,164 characters**
* Median length: **189 characters**
* Average length: approximately **247.76 characters**

This is a very strong completeness result.

Every review contains some text.

However, some comments contain only minimal content such as:

* `.`
* `-`
* `👍`

These are structurally valid but may have limited analytical value for NLP.

---

#### Repeated Comment Quality

Observed results:

* Total reviews: **545,162**
* Unique comments: **528,743**
* Repeated comment groups: **2,928**
* Extra rows from repeated comments: **16,419**

Examples include:

* `.`
* `Good`
* `Top`
* `Great`
* `Great place`
* `Great location`
* `Perfect`

Repeated text alone is not evidence of duplicate review records.

Because:

* review IDs are unique,
* complete duplicate rows are absent,
* different reviewers may independently use the same phrase.

Therefore, repeated comments should not be removed automatically.

---

#### Minimal-Content Comment Assessment

The single-character comment:

`.`

appears:

* **1,033 times**
* across **629 distinct listings**
* from **1,024 distinct reviewers**

This value is structurally valid but contains almost no meaningful text information.

For optional NLP analysis, low-information comments may be:

* excluded from the analytical text subset,
* flagged using a minimum-content rule.

However, the raw records should remain unchanged.

---

#### Multilingual Text Quality Consideration

The sample comments include multiple languages.

This creates no structural data-quality problem.

However, it introduces an analytical challenge for:

* sentiment analysis,
* topic extraction,
* keyword analysis,
* text classification.

An English-only NLP model could produce incomplete or biased results.

Therefore, multilingual text processing should remain optional unless suitable tools and sufficient time are available.

---

#### HTML-Like Formatting Artifacts

Some comments contain HTML-like strings such as:

```text
<br/>
```

These should be considered text-cleaning artifacts.

If NLP is performed later, preprocessing may include:

* HTML-tag removal,
* whitespace normalization,
* line-break normalization.

The original raw comments should remain preserved.

---

#### The 1,033 No-Review Pattern

The review relationship found:

* **1,033 summary listings without detailed reviews**

Earlier summary-listing profiling also found exactly:

* **1,033 missing `last_review` values**
* **1,033 missing `reviews_per_month` values**

The summary `reviews.csv` file also lacks review records for exactly:

* **1,033 listings**

This is a strong structural pattern.

However, exact ID-level equivalence between:

* no-review listing IDs,
* missing `last_review` IDs,
* missing `reviews_per_month` IDs

has not yet been explicitly tested.

Therefore, the correct interpretation is:

**The counts align perfectly and strongly suggest structural missingness, but exact row-level equivalence remains to be validated before final imputation decisions.**

---

#### Temporal Consistency Risk

The review history spans nearly 16 years.

Current listing attributes may differ from those that existed when older reviews were created.

Possible changes include:

* price,
* property type,
* room configuration,
* amenities,
* host,
* superhost status.

Therefore, the dataset is structurally valid but has a temporal comparability limitation when joined to current listing attributes.

---

#### Main Data Quality Strengths

The strongest characteristics of `reviews.csv.gz` are:

1. **545,162 unique review IDs across 545,162 rows.**
2. **0 missing review IDs.**
3. **0 duplicate review IDs.**
4. **0 complete duplicate rows.**
5. **0 missing listing IDs.**
6. **0 missing dates.**
7. **0 missing reviewer IDs.**
8. **0 missing comments.**
9. **Only one missing reviewer name.**
10. **0 empty but non-null comments.**
11. **100.00% review-to-summary-listing coverage.**
12. **Exact projection match with `reviews.csv`.**
13. **Repeated listing-date combinations are explained by distinct review IDs.**
14. **Strong date typing and completeness.**
15. **86 review IDs absent from detailed listings are explained by the known summary-only population.**

---

#### Main Data Quality Concerns

The main concerns are:

1. 1,033 summary listings have no review records.
2. Historical reviews may not match current listing attributes.
3. 86 reviewed listings are absent from `listings.csv.gz`.
4. Review counts are strongly right-skewed.
5. Some listings have extremely high review counts.
6. Repeated comment text exists but should not be treated automatically as duplicate data.
7. Some comments contain minimal information.
8. Comments are multilingual.
9. HTML-like formatting artifacts occur in review text.
10. Reviewer names are not unique identifiers.
11. Review count is not equivalent to booking count.
12. Exact ID-level validation of the 1,033 structural missingness pattern is still pending.

---

#### Data Quality Risk Classification

| Quality Area                                | Assessment                                      |
| ------------------------------------------- | ----------------------------------------------- |
| Schema consistency                          | Excellent                                       |
| Review ID integrity                         | Excellent                                       |
| Missing-value quality                       | Excellent                                       |
| Complete duplicate quality                  | Excellent                                       |
| Listing relationship with summary listings  | Excellent                                       |
| Listing relationship with detailed listings | Very Good, with 86 known reviewed IDs absent    |
| Summary-vs-detailed review consistency      | Excellent                                       |
| Date quality                                | Excellent                                       |
| Comment completeness                        | Excellent                                       |
| Reviewer ID completeness                    | Excellent                                       |
| Review-count distribution                   | Strongly skewed; requires robust summaries      |
| Repeated comments                           | Requires interpretation, not automatic cleaning |
| Temporal comparability                      | Moderate concern                                |
| NLP readiness                               | Good, but multilingual and noisy                |

---

#### Recommended Cleaning and Validation Actions

Recommended downstream actions are:

* preserve the raw `reviews.csv.gz` file unchanged,
* retain `id` as the preferred review-level key,
* retain `listing_id` as the listing relationship field,
* do not deduplicate on `(listing_id, date)`,
* do not remove repeated comments automatically,
* aggregate reviews by `listing_id` before joining to the listing master,
* calculate listing-level review counts,
* calculate first and latest review dates,
* calculate review lifespan or recency where useful,
* validate the exact 1,033-listing missingness pattern before imputation,
* use robust statistics for review-count analysis,
* clean HTML-like text only if NLP is performed,
* consider filtering low-information comments only for text analysis,
* avoid unnecessary use of reviewer names in public analytical outputs.

---

#### Overall Data Quality Assessment

The `reviews.csv.gz` dataset has excellent structural and relational quality.

It provides:

* a complete and unique review identifier,
* almost no missing data,
* no duplicate rows,
* excellent consistency with the summary review file,
* complete review-to-summary-listing referential integrity,
* strong reviewer and comment completeness.

Its main issues are analytical rather than structural:

* review activity is strongly skewed,
* historical reviews may not align with current listing attributes,
* repeated generic comments require interpretation,
* some text contains little analytical content,
* multilingual text increases NLP complexity,
* 86 valid reviewed listings are absent from the detailed listings file.

**Overall assessment: Excellent-quality review-level data that is highly suitable for review aggregation, listing enrichment, relationship validation, and optional text analysis, with the main cautions being temporal mismatch, skewed review activity, multilingual text, repeated generic comments, and incomplete overlap with the detailed listings population.**

## 8. Overall Dataset Familiarization Summary

Dataset familiarization has now been completed for all seven Amsterdam Airbnb source files:

- `neighbourhoods.csv`
- `listings.csv`
- `reviews.csv`
- `listings.csv.gz`
- `neighbourhoods.geojson`
- `calendar.csv.gz`
- `reviews.csv.gz`

This section consolidates the most important structural characteristics, candidate keys, cross-dataset relationships, data-quality issues, and downstream processing decisions identified during familiarization.

The purpose of this summary is not to repeat every profiling result already documented in the previous sections. Instead, it provides a compact dataset-level view that will guide the next stages of the project:

**Automated Profiling → Data Validation → Cleaning → Enrichment → Analytical Modeling**

In [195]:
# Create a compact overall summary of all seven source datasets.
# The values below are based on the validated profiling results
# produced during Dataset Familiarization.

overall_dataset_summary = pd.DataFrame([
    {
        "file_name": "neighbourhoods.csv",
        "rows_or_features": neighbourhoods_dataset_summary["row_count"],
        "columns_or_properties": neighbourhoods_dataset_summary["column_count"],
        "grain": "One row per neighbourhood",
        "candidate_key": "neighbourhood",
        "main_relationships": (
            "Joins to listings.csv.neighbourhood, "
            "listings.csv.gz.neighbourhood_cleansed, "
            "and neighbourhoods.geojson.neighbourhood"
        ),
        "major_quality_issues": (
            "neighbourhood_group is 100% missing"
        ),
        "processing_strategy": (
            "Use as the neighbourhood reference dimension"
        ),
    },

    {
        "file_name": "listings.csv",
        "rows_or_features": summary_listings_dataset_summary["row_count"],
        "columns_or_properties": summary_listings_dataset_summary["column_count"],
        "grain": "One row per listing",
        "candidate_key": "id",
        "main_relationships": (
            "Links to reviews and calendar through listing_id; "
            "links to neighbourhood reference through neighbourhood"
        ),
        "major_quality_issues": (
            "38.17% missing price; neighbourhood_group 100% missing; "
            "1,033 listings have no review history; "
            "96 listings have missing host_id"
        ),
        "processing_strategy": (
            "Use as the canonical 10,465-listing population"
        ),
    },

    {
        "file_name": "reviews.csv",
        "rows_or_features": summary_reviews_dataset_summary["row_count"],
        "columns_or_properties": summary_reviews_dataset_summary["column_count"],
        "grain": "One review event per listing and date",
        "candidate_key": "None",
        "main_relationships": (
            "listing_id → listings.csv.id; "
            "exact projection of reviews.csv.gz onto listing_id and date"
        ),
        "major_quality_issues": (
            "No unique review ID; 22,541 repeated listing-date rows; "
            "(listing_id, date) is not unique"
        ),
        "processing_strategy": (
            "Retain for validation; prefer detailed reviews for enrichment"
        ),
    },

    {
        "file_name": "listings.csv.gz",
        "rows_or_features": detailed_listings_dataset_summary["row_count"],
        "columns_or_properties": detailed_listings_dataset_summary["column_count"],
        "grain": "One row per detailed listing",
        "candidate_key": "id",
        "main_relationships": (
            "Detailed listing subset of listings.csv; "
            "joins to neighbourhoods through neighbourhood_cleansed"
        ),
        "major_quality_issues": (
            "57 of 90 columns contain missing values; "
            "13 columns are completely empty; "
            "38.50% of price values are missing; "
            "96 summary listings are absent"
        ),
        "processing_strategy": (
            "Use as the rich listing-enrichment source; "
            "do not discard the 96 summary-only listings"
        ),
    },

    {
        "file_name": "neighbourhoods.geojson",
        "rows_or_features": geojson_dataset_summary["feature_count"],
        "columns_or_properties": geojson_dataset_summary["property_column_count"],
        "grain": "One geographic feature per neighbourhood",
        "candidate_key": "neighbourhood",
        "main_relationships": (
            "100% neighbourhood-name coverage with neighbourhoods.csv "
            "and detailed listings"
        ),
        "major_quality_issues": (
            "neighbourhood_group is 100% missing; "
            "contains boundaries but no business metrics"
        ),
        "processing_strategy": (
            "Use for optional geographic enrichment and mapping"
        ),
    },

    {
        "file_name": "calendar.csv.gz",
        "rows_or_features": calendar_dataset_summary["row_count"],
        "columns_or_properties": calendar_dataset_summary["column_count"],
        "grain": "One row per listing per calendar date",
        "candidate_key": "(listing_id, date)",
        "main_relationships": (
            "listing_id → listings.csv.id; "
            "100% coverage of all 10,465 summary listings"
        ),
        "major_quality_issues": (
            "No price or adjusted_price fields; "
            "extreme stay-rule values require validation; "
            "96 listing IDs are absent from detailed listings"
        ),
        "processing_strategy": (
            "Aggregate with DuckDB before joining to listing-level data"
        ),
    },

    {
        "file_name": "reviews.csv.gz",
        "rows_or_features": detailed_reviews_dataset_summary["row_count"],
        "columns_or_properties": detailed_reviews_dataset_summary["column_count"],
        "grain": "One row per individual review",
        "candidate_key": "id",
        "main_relationships": (
            "listing_id → listings.csv.id; "
            "detailed source corresponding exactly to summary reviews"
        ),
        "major_quality_issues": (
            "One missing reviewer name; "
            "86 reviewed listing IDs are absent from detailed listings; "
            "historical reviews may not align with current listing attributes"
        ),
        "processing_strategy": (
            "Use as the preferred review-level source; "
            "aggregate to listing level before enrichment"
        ),
    },
])

pd.set_option("display.max_colwidth", None)

display(overall_dataset_summary)

,file_name,rows_or_features,columns_or_properties,grain,candidate_key,main_relationships,major_quality_issues,processing_strategy
0,neighbourhoods.csv,22,2,One row per neighbourhood,neighbourhood,"Joins to listings.csv.neighbourhood, listings.csv.gz.neighbourhood_cleansed, and neighbourhoods.geojson.neighbourhood",neighbourhood_group is 100% missing,Use as the neighbourhood reference dimension
1,listings.csv,10465,19,One row per listing,id,Links to reviews and calendar through listing_id; links to neighbourhood reference through neighbourhood,"38.17% missing price; neighbourhood_group 100% missing; 1,033 listings have no review history; 96 listings have missing host_id","Use as the canonical 10,465-listing population"
2,reviews.csv,545162,2,One review event per listing and date,None,listing_id → listings.csv.id; exact projection of reviews.csv.gz onto listing_id and date,"No unique review ID; 22,541 repeated listing-date rows; (listing_id, date) is not unique",Retain for validation; prefer detailed reviews for enrichment
3,listings.csv.gz,10369,90,One row per detailed listing,id,Detailed listing subset of listings.csv; joins to neighbourhoods through neighbourhood_cleansed,57 of 90 columns contain missing values; 13 columns are completely empty; 38.50% of price values are missing; 96 summary listings are absent,Use as the rich listing-enrichment source; do not discard the 96 summary-only listings
4,neighbourhoods.geojson,22,2,One geographic feature per neighbourhood,neighbourhood,100% neighbourhood-name coverage with neighbourhoods.csv and detailed listings,neighbourhood_group is 100% missing; contains boundaries but no business metrics,Use for optional geographic enrichment and mapping
5,calendar.csv.gz,3819725,5,One row per listing per calendar date,"(listing_id, date)","listing_id → listings.csv.id; 100% coverage of all 10,465 summary listings",No price or adjusted_price fields; extreme stay-rule values require validation; 96 listing IDs are absent from detailed listings,Aggregate with DuckDB before joining to listing-level data
6,reviews.csv.gz,545162,6,One row per individual review,id,listing_id → listings.csv.id; detailed source corresponding exactly to summary reviews,One missing reviewer name; 86 reviewed listing IDs are absent from detailed listings; historical reviews may not align with current listing attributes,Use as the preferred review-level source; aggregate to listing level before enrichment


### Overall Cross-Dataset Relationships

The seven source files represent several connected business entities and different levels of data granularity.

The main relationship structure is:

```text
                         Host
                          │
                          │ host_id
                          ▼
                    ┌─────────────┐
                    │   Listing   │
                    └─────────────┘
                     │     │     │
          listing_id │     │     │ neighbourhood
                     │     │     ▼
                     │     │  ┌──────────────────┐
                     │     │  │  Neighbourhood   │
                     │     │  └──────────────────┘
                     │     │           │
                     │     │           │ neighbourhood
                     │     │           ▼
                     │     │  ┌──────────────────┐
                     │     │  │ GeoJSON Boundary │
                     │     │  └──────────────────┘
                     │     │
                     │     └───────────────────────┐
                     ▼                             ▼
              ┌───────────┐                ┌────────────┐
              │  Reviews  │                │  Calendar  │
              └───────────┘                └────────────┘
               many rows                    many rows
               per listing                  per listing

The main validated relationships are:

listings.csv.id is the main listing identifier for the full 10,465-listing population.
listings.csv.gz.id contains 10,369 of those listings and provides substantially richer attributes.
The 96 listings present in listings.csv but absent from listings.csv.gz should not be silently discarded.
calendar.csv.gz.listing_id has 100% coverage of all 10,465 summary listings.
Every listing contains exactly 365 calendar rows.
reviews.csv.gz.listing_id links detailed individual reviews to listings.
reviews.csv is an exact projection of reviews.csv.gz onto listing_id and date, including repeated occurrences.
reviews.csv.gz.id is the preferred candidate primary key for individual reviews.
reviews.csv does not contain enough information to uniquely identify individual review records.
neighbourhoods.csv.neighbourhood provides the main neighbourhood reference key.
All 22 neighbourhoods match successfully across the neighbourhood CSV, detailed listings, and GeoJSON boundary data.

### Overall Findings and Processing Decisions

Dataset familiarization identified a structurally connected collection of listing, review, calendar, and geographic datasets with generally strong key quality and cross-dataset consistency.

The most important findings are:

1. **The canonical listing population contains 10,465 listings.**

   The summary `listings.csv` dataset contains all 10,465 listing IDs represented in the calendar data and should therefore be preserved as the canonical listing population.

2. **The detailed listing dataset is richer but incomplete relative to the summary listing population.**

   `listings.csv.gz` contains 10,369 listings, meaning that 96 summary listings are absent from the detailed source. These listings should not be silently removed during enrichment.

3. **Calendar data is structurally excellent but must be processed efficiently.**

   The calendar contains 3,819,725 rows, representing exactly 365 records for every one of the 10,465 listings. It contains no missing values or duplicate `(listing_id, date)` combinations.

   Because of its size, calendar data should be aggregated using DuckDB before being joined to the listing-level analytical dataset.

4. **Detailed reviews should be preferred over summary reviews for review-level enrichment.**

   The detailed and summary review datasets contain the same 545,162 review events when compared as multisets of `(listing_id, date)` values.

   However, only `reviews.csv.gz` contains a unique review identifier, reviewer information, and comments. Therefore, the detailed reviews file should be treated as the preferred review-level analytical source.

5. **Raw review and calendar rows should never be directly joined to the listing table before aggregation.**

   Both datasets have one-to-many relationships with listings. Direct raw joins could produce massive intermediate datasets and create row multiplication.

   Therefore:

   - calendar data should first be aggregated to one row per listing,
   - review data should first be aggregated to one row per listing,
   - only compact aggregated results should be joined to the listing master dataset.

6. **Neighbourhood relationships demonstrate complete observed consistency.**

   All 22 Amsterdam neighbourhoods match successfully across:

   - `neighbourhoods.csv`
   - `listings.csv`
   - `listings.csv.gz`
   - `neighbourhoods.geojson`

   This provides a strong foundation for neighbourhood-level enrichment and optional geographic analysis.

7. **Missing values require context-dependent treatment.**

   Missing values should not be handled with a universal strategy such as filling all null values with zero.

   Important examples include:

   - missing price does not mean zero price,
   - missing review scores do not mean zero-quality reviews,
   - missing review fields may indicate the absence of review history,
   - fully empty fields may be excluded from processed outputs while remaining untouched in raw source data.

8. **Availability should not be interpreted as verified occupancy.**

   Calendar availability does not prove whether an unavailable date was booked. Therefore, any derived measure based on unavailable dates must be explicitly described as an availability-based proxy rather than true occupancy.

9. **Review count should not be interpreted as booking count.**

   Not every guest leaves a review. Therefore, review frequency may be used as a proxy for guest activity but not as an exact measure of reservations.

10. **The project will follow an 8 GB RAM-aware processing strategy.**

    Large datasets will be processed independently using DuckDB or aggregation-based techniques. The project will avoid retaining all seven datasets in memory simultaneously.

---